In [ ]:
def main(datasource, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    if isinstance(datasource, dict):
        bar1m = datasource.get(
            "bar1m",
            datasource.get("bigalpha_2026_stock_bar1m", "bigalpha_2026_stock_bar1m"),
        )
    else:
        bar1m = datasource or "bigalpha_2026_stock_bar1m"

    sql = f"""
    WITH raw AS (
      SELECT
        date::DATE::DATETIME AS trading_day,
        instrument::string AS instrument,
        date AS minute_time,
        EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) AS hhmm,
        open,
        high,
        low,
        close,
        volume,
        amount,
        deal_number,
        ask_price1,
        ask_price2,
        ask_price3,
        ask_price4,
        ask_price5,
        bid_price1,
        bid_price2,
        bid_price3,
        bid_price4,
        bid_price5,
        ask_volume1,
        ask_volume2,
        ask_volume3,
        ask_volume4,
        ask_volume5,
        bid_volume1,
        bid_volume2,
        bid_volume3,
        bid_volume4,
        bid_volume5,
        ask_num_orders1,
        ask_num_orders2,
        ask_num_orders3,
        ask_num_orders4,
        ask_num_orders5,
        bid_num_orders1,
        bid_num_orders2,
        bid_num_orders3,
        bid_num_orders4,
        bid_num_orders5
      FROM {bar1m}
    ),
    windowed AS (
      SELECT
        trading_day,
        instrument,
        minute_time,
        hhmm,
        open,
        high,
        low,
        close,
        volume,
        amount,
        deal_number,
        ask_price1,
        ask_price2,
        ask_price3,
        ask_price4,
        ask_price5,
        bid_price1,
        bid_price2,
        bid_price3,
        bid_price4,
        bid_price5,
        ask_volume1,
        ask_volume2,
        ask_volume3,
        ask_volume4,
        ask_volume5,
        bid_volume1,
        bid_volume2,
        bid_volume3,
        bid_volume4,
        bid_volume5,
        ask_num_orders1,
        ask_num_orders2,
        ask_num_orders3,
        ask_num_orders4,
        ask_num_orders5,
        bid_num_orders1,
        bid_num_orders2,
        bid_num_orders3,
        bid_num_orders4,
        bid_num_orders5,
        LAG(close, 1) OVER (PARTITION BY trading_day, instrument ORDER BY minute_time) AS close_lag_1
      FROM raw
    ),
    primitive AS (
      SELECT
        trading_day,
        instrument,
        hhmm,
        (((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0)) - (COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0))) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)), 0)) * LN(1 + GREATEST(COALESCE(amount, 0), 0)) AS p0,
        (((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0)) - (COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0))) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)), 0)) * LN(1 + GREATEST(COALESCE(deal_number, 0), 0)) AS p1,
        LEAST(GREATEST((((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5) - ((COALESCE(ask_price1,0)*COALESCE(bid_volume1,0) + COALESCE(bid_price1,0)*COALESCE(ask_volume1,0)) / NULLIF(COALESCE(ask_volume1,0)+COALESCE(bid_volume1,0),0))) / NULLIF(ABS((COALESCE(ask_price1, 0) - COALESCE(bid_price1, 0))), 0), -2), 2) AS p2,
        LEAST(GREATEST(((((COALESCE(ask_price1,0)*COALESCE(ask_volume1,0) + COALESCE(bid_price1,0)*COALESCE(bid_volume1,0) + COALESCE(ask_price2,0)*COALESCE(ask_volume2,0) + COALESCE(bid_price2,0)*COALESCE(bid_volume2,0) + COALESCE(ask_price3,0)*COALESCE(ask_volume3,0) + COALESCE(bid_price3,0)*COALESCE(bid_volume3,0) + COALESCE(ask_price4,0)*COALESCE(ask_volume4,0) + COALESCE(bid_price4,0)*COALESCE(bid_volume4,0) + COALESCE(ask_price5,0)*COALESCE(ask_volume5,0) + COALESCE(bid_price5,0)*COALESCE(bid_volume5,0)) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)), 0)) - ((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5)) / NULLIF(ABS(((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5)), 0)), -0.2), 0.2) AS p3,
        (((COALESCE(ask_volume1, 0)) - (COALESCE(bid_volume1, 0))) / NULLIF((COALESCE(ask_volume1, 0)) + (COALESCE(bid_volume1, 0)), 0)) - (((COALESCE(ask_num_orders1, 0)) - (COALESCE(bid_num_orders1, 0))) / NULLIF((COALESCE(ask_num_orders1, 0)) + (COALESCE(bid_num_orders1, 0)), 0)) AS p4,
        (((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0)) - (COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0))) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)), 0)) * (LEAST(GREATEST(close / NULLIF(close_lag_1, 0) - 1, -0.2), 0.2)) AS p5,
        (COALESCE(ask_num_orders1,0)+COALESCE(bid_num_orders1,0)) / NULLIF((COALESCE(ask_num_orders1, 0) + COALESCE(ask_num_orders2, 0) + COALESCE(ask_num_orders3, 0) + COALESCE(ask_num_orders4, 0) + COALESCE(ask_num_orders5, 0) + COALESCE(bid_num_orders1, 0) + COALESCE(bid_num_orders2, 0) + COALESCE(bid_num_orders3, 0) + COALESCE(bid_num_orders4, 0) + COALESCE(bid_num_orders5, 0)),0) AS p6,
        POWER(COALESCE(ask_volume1,0) / NULLIF(COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0),0), 2) + POWER(COALESCE(ask_volume2,0) / NULLIF(COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0),0), 2) + POWER(COALESCE(ask_volume3,0) / NULLIF(COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0),0), 2) + POWER(COALESCE(ask_volume4,0) / NULLIF(COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0),0), 2) + POWER(COALESCE(ask_volume5,0) / NULLIF(COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0),0), 2) AS p7,
        LEAST(GREATEST((COALESCE(bid_price1, 0) - COALESCE(bid_price5, 0)) / NULLIF(ABS(((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5)), 0), -0.2), 0.2) AS p8,
        POWER(COALESCE(ask_volume1,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(ask_volume2,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(ask_volume3,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(ask_volume4,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(ask_volume5,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(bid_volume1,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(bid_volume2,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(bid_volume3,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(bid_volume4,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) + POWER(COALESCE(bid_volume5,0) / NULLIF((COALESCE(ask_volume1, 0) + COALESCE(ask_volume2, 0) + COALESCE(ask_volume3, 0) + COALESCE(ask_volume4, 0) + COALESCE(ask_volume5, 0) + COALESCE(bid_volume1, 0) + COALESCE(bid_volume2, 0) + COALESCE(bid_volume3, 0) + COALESCE(bid_volume4, 0) + COALESCE(bid_volume5, 0)),0), 2) AS p9,
        LEAST(GREATEST(((COALESCE(bid_price3, 0) - COALESCE(bid_price5, 0)) - (COALESCE(bid_price1, 0) - COALESCE(bid_price3, 0))) / NULLIF(ABS(((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5)), 0), -0.2), 0.2) AS p10,
        (COALESCE(ask_num_orders1,0)-COALESCE(bid_num_orders1,0)) / NULLIF(COALESCE(ask_num_orders1,0)+COALESCE(bid_num_orders1,0),0) AS p11,
        ((COALESCE(ask_num_orders1, 0)) - (COALESCE(bid_num_orders1, 0))) / NULLIF((COALESCE(ask_num_orders1, 0)) + (COALESCE(bid_num_orders1, 0)), 0) AS p12,
        LEAST(GREATEST((((COALESCE(ask_price5, 0)-COALESCE(ask_price1, 0))-(COALESCE(bid_price1, 0)-COALESCE(bid_price5, 0))) / NULLIF(ABS(((COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0)) * 0.5)), 0)), -0.2), 0.2) AS p13
      FROM windowed
    ),
    daily AS (
      SELECT
        trading_day AS date,
        instrument,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p0 END)) AS DOUBLE), 8) AS f0,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1001 AND 1030 THEN p0 END)) AS DOUBLE), 8) AS f1,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p0 END)) AS DOUBLE), 8) AS f2,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p0 END)) AS DOUBLE), 8) AS f3,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1401 AND 1430 THEN p0 END)) AS DOUBLE), 8) AS f4,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1431 AND 1456 THEN p0 END)) AS DOUBLE), 8) AS f5,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p1 END)) AS DOUBLE), 8) AS f6,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1001 AND 1030 THEN p1 END)) AS DOUBLE), 8) AS f7,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p1 END)) AS DOUBLE), 8) AS f8,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p1 END)) AS DOUBLE), 8) AS f9,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1401 AND 1430 THEN p1 END)) AS DOUBLE), 8) AS f10,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1431 AND 1456 THEN p1 END)) AS DOUBLE), 8) AS f11,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p2 END)) AS DOUBLE), 8) AS f12,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1001 AND 1030 THEN p2 END)) AS DOUBLE), 8) AS f13,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p2 END)) AS DOUBLE), 8) AS f14,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p2 END)) AS DOUBLE), 8) AS f15,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1401 AND 1430 THEN p2 END)) AS DOUBLE), 8) AS f16,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1431 AND 1456 THEN p2 END)) AS DOUBLE), 8) AS f17,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p3 END)) AS DOUBLE), 8) AS f18,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1001 AND 1030 THEN p3 END)) AS DOUBLE), 8) AS f19,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p3 END)) AS DOUBLE), 8) AS f20,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p3 END)) AS DOUBLE), 8) AS f21,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1401 AND 1430 THEN p3 END)) AS DOUBLE), 8) AS f22,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1431 AND 1456 THEN p3 END)) AS DOUBLE), 8) AS f23,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p4 END)) AS DOUBLE), 8) AS f24,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1001 AND 1030 THEN p4 END)) AS DOUBLE), 8) AS f25,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p4 END)) AS DOUBLE), 8) AS f26,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p4 END)) AS DOUBLE), 8) AS f27,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1401 AND 1430 THEN p4 END)) AS DOUBLE), 8) AS f28,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1431 AND 1456 THEN p4 END)) AS DOUBLE), 8) AS f29,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p5 END)) AS DOUBLE), 8) AS f30,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1001 AND 1030 THEN p5 END)) AS DOUBLE), 8) AS f31,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p5 END)) AS DOUBLE), 8) AS f32,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p5 END)) AS DOUBLE), 8) AS f33,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1401 AND 1430 THEN p5 END)) AS DOUBLE), 8) AS f34,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1431 AND 1456 THEN p5 END)) AS DOUBLE), 8) AS f35,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p6 END)) AS DOUBLE), 8) AS f36,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1431 AND 1456 THEN p6 END)) AS DOUBLE), 8) AS f37,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p7 END)) AS DOUBLE), 8) AS f38,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p7 END)) AS DOUBLE), 8) AS f39,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1001 AND 1030 THEN p8 END)) AS DOUBLE), 8) AS f40,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1431 AND 1456 THEN p8 END)) AS DOUBLE), 8) AS f41,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p9 END)) AS DOUBLE), 8) AS f42,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p9 END)) AS DOUBLE), 8) AS f43,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 935 AND 1000 THEN p10 END)) AS DOUBLE), 8) AS f44,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p10 END)) AS DOUBLE), 8) AS f45,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p11 END)) AS DOUBLE), 8) AS f46,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p11 END)) AS DOUBLE), 8) AS f47,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p12 END)) AS DOUBLE), 8) AS f48,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p12 END)) AS DOUBLE), 8) AS f49,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1031 AND 1130 THEN p13 END)) AS DOUBLE), 8) AS f50,
        ROUND(CAST((AVG(CASE WHEN hhmm BETWEEN 1301 AND 1400 THEN p13 END)) AS DOUBLE), 8) AS f51
      FROM primitive
      GROUP BY trading_day, instrument
    )
    SELECT date, instrument, f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, f11, f12, f13, f14, f15, f16, f17, f18, f19, f20, f21, f22, f23, f24, f25, f26, f27, f28, f29, f30, f31, f32, f33, f34, f35, f36, f37, f38, f39, f40, f41, f42, f43, f44, f45, f46, f47, f48, f49, f50, f51
    FROM daily
    """
    data = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    if data.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    raw_columns = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 'f31', 'f32', 'f33', 'f34', 'f35', 'f36', 'f37', 'f38', 'f39', 'f40', 'f41', 'f42', 'f43', 'f44', 'f45', 'f46', 'f47', 'f48', 'f49', 'f50', 'f51']
    data = data[["date", "instrument", *raw_columns]].copy()
    data["date"] = pd.to_datetime(data["date"]).dt.normalize()
    data["instrument"] = data["instrument"].astype(str)
    for column in raw_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data = data.replace([np.inf, -np.inf], np.nan)

    def cs_rank(series):
        values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
        values = values.fillna(values.median())
        return values.rank(method="average", pct=True).fillna(0.5) * 2.0 - 1.0

    def rank_factor(series):
        return pd.Series(series, index=data.index).groupby(
            data["date"], group_keys=False
        ).transform(cs_rank)

    ranks = data.groupby("date", group_keys=False)[raw_columns].transform(cs_rank)
    model_x = ranks[raw_columns].to_numpy(dtype=np.float32, copy=True)
    trajectory_path = model_x[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]].reshape(len(model_x), 6, 6)
    trajectory_delta = np.diff(trajectory_path, axis=2)
    trajectory_acceleration = np.diff(trajectory_delta, axis=2)
    trajectory_x = np.concatenate([trajectory_path.reshape(len(model_x), -1), trajectory_delta.reshape(len(model_x), -1), trajectory_acceleration.reshape(len(model_x), -1), trajectory_path.std(axis=2), trajectory_path.max(axis=2) - trajectory_path.min(axis=2), trajectory_path[:, :, -1] - trajectory_path[:, :, 0]], axis=1).astype(np.float32)
    trajectory_mean = trajectory_x.mean(axis=1, keepdims=True)
    trajectory_var = ((trajectory_x - trajectory_mean) ** 2).mean(axis=1, keepdims=True)
    trajectory_z = (trajectory_x - trajectory_mean) / np.sqrt(trajectory_var + 1e-5)
    trajectory_z = trajectory_z * np.asarray([1.0435916185379028, 1.0252060890197754, 1.01901376247406, 1.0278170108795166, 1.0344901084899902,
     1.04720139503479, 1.0223039388656616, 0.983145534992218, 1.0083588361740112, 1.0147465467453003,
     1.0062955617904663, 0.992838978767395, 0.9780881404876709, 1.0130220651626587, 1.0165895223617554,
     1.0196049213409424, 1.0195813179016113, 0.9844934344291687, 1.0508449077606201, 1.0540835857391357,
     1.0448859930038452, 1.0277612209320068, 1.059321641921997, 1.0542826652526855, 0.9584662914276123,
     0.968350887298584, 0.9530364274978638, 0.9838131666183472, 1.0045230388641357, 0.9741045236587524,
     1.0105680227279663, 1.0075886249542236, 1.0056028366088867, 1.0125744342803955, 0.9597110152244568,
     0.9699638485908508, 1.008545994758606, 1.0187450647354126, 0.9778996706008911, 0.9839233756065369,
     0.9635571837425232, 0.9987901449203491, 0.9840212464332581, 1.0010097026824951, 0.997912585735321,
     0.9885827302932739, 0.9689964652061462, 0.9857212901115417, 0.9739572405815125, 0.9897066354751587,
     0.9924063682556152, 0.9618442058563232, 0.9869581460952759, 0.9979312419891357, 0.9984500408172607,
     0.9902166724205017, 0.9780611395835876, 0.9794018268585205, 0.9700267314910889, 0.9702320098876953,
     0.991535484790802, 0.9951997995376587, 0.9908397197723389, 0.9861522912979126, 0.9786731600761414,
     0.9935278296470642, 0.9720842242240906, 0.971200168132782, 0.990865170955658, 0.9802972078323364,
     0.9966786503791809, 0.9943286180496216, 0.9996606707572937, 0.9684103727340698, 0.9677457809448242,
     0.9471152424812317, 0.9444975852966309, 0.9735001921653748, 0.9803889989852905, 0.9839008450508118,
     0.953537106513977, 0.9860909581184387, 0.9775059223175049, 0.9658229351043701, 0.9588581919670105,
     0.987654983997345, 0.9766333699226379, 0.9803755283355713, 0.9880310297012329, 0.9824546575546265,
     1.0447425842285156, 1.0232858657836914, 1.0418843030929565, 1.0095447301864624, 1.0524468421936035,
     1.04079270362854, 1.0644690990447998, 1.0406321287155151, 1.0863715410232544, 1.0214457511901855,
     1.098791241645813, 1.0570881366729736, 0.9945033192634583, 1.0302022695541382, 0.9441423416137695,
     0.9869786500930786, 0.9768158793449402, 0.9749419689178467], dtype=np.float32) + np.asarray([0.06758091598749161, 0.058497652411460876, 0.06323225051164627, 0.05250667408108711,
     0.06215159595012665, 0.044161804020404816, 0.06676114350557327, 0.05099865049123764,
     0.0566207692027092, 0.05095464363694191, 0.03790488466620445, 0.05544327571988106,
     -0.05632008984684944, -0.05589093640446663, -0.06594517081975937, 0.04128719121217728,
     -0.015556059777736664, 0.0018505697371438146, -0.07666695863008499, -0.07427488267421722,
     -0.07551363110542297, -0.08553845435380936, -0.0676514133810997, -0.049820009618997574,
     0.019135253503918648, 0.021486250683665276, 0.042022719979286194, 0.07160034030675888,
     0.019375255331397057, -0.0034105447120964527, 0.049383778125047684, 0.024039864540100098,
     0.05680388957262039, 0.04256431385874748, 0.07533576339483261, -0.002753393491730094,
     -0.04567540064454079, -0.039750128984451294, -0.03439665213227272, 0.01813509128987789,
     0.050988275557756424, -0.0011495561338961124, -0.07736828178167343, -0.07035335153341293,
     -0.09105333685874939, 0.04282572492957115, 0.030140981078147888, -0.037296466529369354,
     -0.05933086574077606, 0.011903621256351471, -0.034704044461250305, 0.0008017098298296332,
     0.02446422539651394, -0.05712727829813957, -0.05995246395468712, -0.057620611041784286,
     0.01744799315929413, -0.024951277300715446, 0.014366324059665203, 0.01536405086517334,
     0.057901736348867416, -0.04954720288515091, 0.0008539062109775841, -0.058378901332616806,
     0.022550048306584358, -0.05054204910993576, 0.012566297315061092, -0.010821591131389141,
     -0.0010271217906847596, 0.05959433317184448, -0.05216657370328903, -0.014746966771781445,
     -0.005463971756398678, 0.034165527671575546, -0.006247612647712231, 0.020116539672017097,
     -0.06596548110246658, -0.020300280302762985, -0.027946701273322105, -0.07206158339977264,
     0.03961736708879471, -0.06608384102582932, -0.011075062677264214, 0.02634444460272789,
     -0.03936157003045082, 0.014820437878370285, 0.03385406360030174, -0.056525975465774536,
     -0.04989864304661751, -0.029273778200149536, 0.0577940009534359, 0.0272993016988039,
     0.043360888957977295, 0.03431526944041252, 0.05503375083208084, 0.027642928063869476,
     0.0756719708442688, 0.05373469740152359, 0.06784205138683319, 0.03935021907091141,
     0.09232071787118912, 0.0688570961356163, -0.07296692579984665, -0.047864094376564026,
     0.06736619770526886, -0.012753617018461227, 0.005371909122914076, -0.023983215913176537], dtype=np.float32)
    trajectory_h1 = trajectory_z @ np.asarray([[-0.0335259884595871, 0.028691191226243973, -0.08863846957683563, -0.00126658717636019,
      -0.05074431747198105, 0.0017980425618588924, -0.06791437417268753, 0.06921141594648361,
      0.053454119712114334, 0.039802148938179016, -0.05489398539066315, 0.05935373529791832,
      0.015360106714069843, -0.06710033863782883, -0.046692848205566406, 0.07445590943098068,
      -0.10035953670740128, -0.022990310564637184, 0.011894287541508675, -0.05419193208217621,
      -0.016087161377072334, -0.06463350355625153, -0.09019137918949127, 0.002967905020341277,
      0.05706988647580147, -0.030141683295369148, 0.03654509782791138, -0.0002256079897051677,
      -0.058111004531383514, 0.020983919501304626, 0.054239340126514435, 0.056079328060150146,
      0.06868930160999298, 0.056485388427972794, 0.06548748165369034, 0.023110512644052505,
      0.0016414511483162642, 0.046248164027929306, 0.08232826739549637, 0.08383768796920776,
      -0.05193262919783592, -0.026875145733356476, 0.04480961337685585, -0.020665457472205162,
      0.057234928011894226, -0.03751247003674507, 0.054555248469114304, 0.06045825406908989,
      0.015433032065629959, -0.07050880789756775, 0.007283959072083235, -0.009061003103852272,
      0.06813109666109085, 0.0015857798280194402, -0.04799560829997063, -0.019071390852332115,
      -0.07034420222043991, -0.08286582678556442, 0.005669119767844677, -0.08202105015516281,
      0.09886372089385986, -0.060198813676834106, 0.0400858148932457, 0.0371689572930336,
      -0.07508645206689835, 0.09692244231700897, 0.06573638319969177, 0.05706143006682396,
      0.07530467957258224, 0.006590657867491245, -0.08761250227689743, 0.06800362467765808,
      -0.039948470890522, 0.07738850265741348, -0.024037456139922142, 0.0011346472892910242,
      0.06476432830095291, -0.05771419405937195, -0.0784125104546547, -0.02625446952879429,
      -0.05588477477431297, 0.07326631993055344, -0.025783920660614967, 0.07096554338932037,
      0.07197368144989014, -0.051129259169101715, 0.017850236967206, -0.018022416159510612,
      0.056567028164863586, 0.07054484635591507, -0.012811366468667984, -0.06782308965921402,
      0.0587843656539917, -0.10472933948040009, 0.04019381105899811, 0.030054179951548576,
      -0.08588523417711258, -0.09517332911491394, 0.05941055342555046, 0.03968040645122528,
      -0.10223595798015594, 0.05961860343813896, 0.0692964643239975, -0.03572878986597061,
      -0.007264391053467989, -0.01639580726623535, -0.02369890734553337, -0.01164992619305849],
     [0.053368836641311646, 0.02013152278959751, -0.05751558765769005, 0.12492943555116653,
      0.11238780617713928, 0.13824757933616638, 0.05831383168697357, 0.08803670108318329,
      0.0440593957901001, 0.005539920646697283, 0.014173267409205437, 0.07550055533647537,
      -0.008063613437116146, -0.04944229871034622, -0.09645707160234451, -0.07133214920759201,
      0.005399656016379595, 0.05114072933793068, 0.020882440730929375, -0.02851603738963604,
      0.09780624508857727, 0.10093404352664948, -0.032313354313373566, 0.1426815241575241,
      0.02964850887656212, -0.02415396273136139, -0.06323681026697159, -0.014024050906300545,
      -0.020260611549019814, -0.10673058778047562, -0.04251134768128395, 0.0489874929189682,
      -0.06025931239128113, 0.03415335714817047, -0.04859447851777077, 0.03363290801644325,
      0.08668143302202225, -0.07532159984111786, -0.03676248714327812, 0.0386272557079792,
      -0.08274057507514954, -0.05986721068620682, 0.07314411550760269, 0.006871126592159271,
      0.047867923974990845, -0.10386009514331818, 0.0238886047154665, 0.019266312941908836,
      -0.05860406532883644, 0.012624763883650303, 0.0842597484588623, -0.08266671746969223,
      -0.043429721146821976, -0.061906684190034866, 0.03891129419207573, -0.05342881381511688,
      -0.05171946436166763, -0.07995830476284027, 0.004418188706040382, -0.040274303406476974,
      0.031975701451301575, -0.04412377253174782, 0.03257576376199722, 0.07615267485380173,
      0.08093135803937912, 0.044520337134599686, -0.07219310104846954, -0.03394084423780441,
      0.07187020778656006, 0.06579005718231201, -0.0503910630941391, 0.08623643964529037,
      -0.022884400561451912, -0.0499814972281456, -0.07405347377061844, 0.02061464823782444,
      -0.06992936134338379, -0.03633897751569748, 0.07134336233139038, -0.08637057989835739,
      0.02649180218577385, -0.03077193908393383, -0.021827708929777145, -0.04533957690000534,
      0.015210389159619808, 0.07475819438695908, 0.038595959544181824, 0.03907546401023865,
      0.05069070681929588, -0.028940126299858093, -0.09036599844694138, -0.0014633388491347432,
      0.020482482388615608, -0.031129304319620132, 0.03466818481683731, -0.00215260311961174,
      -0.09192205965518951, -0.046784933656454086, 0.03176584839820862, -0.005001723766326904,
      -0.06238601729273796, -0.11453904211521149, 0.006024856586009264, -0.05626698583364487,
      -0.0011884559644386172, -0.013579444028437138, 0.05799409747123718, 0.013018099591135979],
     [-0.08691395074129105, -0.023643191903829575, -0.1284216195344925, -0.1015787124633789,
      -0.12100311368703842, -0.12784387171268463, -0.08717517554759979, 0.043917182832956314,
      0.05698416009545326, -0.0668562576174736, -0.03845328465104103, 0.06164592504501343,
      0.031001009047031403, 0.05915055051445961, -0.07132969796657562, -0.04578876122832298,
      -0.03625406324863434, -0.0006411278154700994, 0.17393584549427032, 0.15733662247657776,
      0.10217158496379852, 0.08156127482652664, 0.13505339622497559, 0.052218351513147354,
      -0.03262781724333763, -0.04423341900110245, 0.08850617706775665, 0.07691782712936401,
      0.018906766548752785, -0.02516598254442215, 0.07747625559568405, -0.015243114903569221,
      -0.018549541011452675, -0.05103682726621628, -0.058336447924375534, 0.04137615114450455,
      0.07691198587417603, 0.10613761842250824, -0.007579282391816378, 0.0579022541642189,
      0.018863467499613762, -0.03628820180892944, 0.08080216497182846, -0.0392146110534668,
      0.01862153597176075, -0.012106691487133503, 0.0005644228076562285, 0.0849948301911354,
      0.09538883715867996, 0.01300740148872137, 0.020011991262435913, 0.036641158163547516,
      -0.029082629829645157, -0.016157491132616997, -0.03555206209421158, 0.08242929726839066,
      0.05782123655080795, 0.0004173368215560913, 0.014892921783030033, 0.006485295481979847,
      -0.01742394082248211, 0.06727142632007599, 0.1153322234749794, 0.0507306307554245,
      -0.010487789288163185, 0.0796932578086853, -0.047002654522657394, 0.03796713799238205,
      -0.04210096597671509, 0.016026372089982033, -0.00465004937723279, 0.0018342641415074468,
      0.026745637878775597, -0.0826232060790062, 0.08000840991735458, 0.0989200547337532,
      0.10724643617868423, -0.006385658401995897, 0.09771546721458435, 0.09077434241771698,
      -0.0016358505235984921, 0.0684475377202034, 0.0756266862154007, 0.0105785196647048,
      -0.03303977847099304, 0.06140383705496788, -0.06742317974567413, -0.03352365642786026,
      -0.027672139927744865, -0.05952272564172745, 0.02694779448211193, -0.09463702142238617,
      -0.165178582072258, -0.06801677495241165, 0.019876599311828613, -0.04027624800801277,
      -0.09818847477436066, -0.08391193300485611, -0.08715112507343292, -0.0878361165523529,
      -0.1453591138124466, -0.09977081418037415, 0.08323530852794647, 0.13183875381946564,
      0.005124771501868963, 0.03563641011714935, 0.00035365598159842193, -0.055357519537210464],
     [0.034198418259620667, 0.07764708250761032, 0.054650161415338516, 0.07586277276277542,
      0.03708083927631378, -0.06469536572694778, 0.05468650162220001, -0.09028681367635727,
      0.06186532601714134, 0.05948895588517189, 0.007736108731478453, -0.035689499229192734,
      -0.023232603445649147, 0.07381448149681091, -0.07095728069543839, -0.04829911142587662,
      -0.07327645272016525, -0.09710577875375748, -0.08218497782945633, 0.03393067792057991,
      0.07766887545585632, -0.07340443134307861, 0.10146992653608322, -0.08259069919586182,
      0.0250532403588295, 0.006198754534125328, -0.0044360994361341, 0.014978070743381977,
      0.021772190928459167, 0.007399615366011858, 0.0585472397506237, -0.004315910395234823,
      0.032542821019887924, -0.036955028772354126, 0.053691308945417404, -0.073993980884552,
      0.0012423087609931827, 0.0971129983663559, 0.06779923290014267, 0.00796604435890913,
      0.07740470767021179, 0.0374261774122715, -0.03652995452284813, -0.004078412428498268,
      0.09001576900482178, -0.03895217180252075, 0.06503849476575851, -0.045216839760541916,
      -0.0445527657866478, -0.019937194883823395, 0.10274849832057953, -0.005885331425815821,
      -0.04951244592666626, 0.06874015182256699, 0.0722288191318512, -0.026694513857364655,
      0.07536766678094864, -0.06308519840240479, -0.0407828614115715, -0.09448966383934021,
      -0.030893336981534958, 0.07244382798671722, -0.06822682917118073, 0.0886283740401268,
      -0.017006924375891685, 0.053401511162519455, 0.09252700954675674, 0.039570219814777374,
      -0.08407118171453476, 0.03920913115143776, -0.022932691499590874, -0.026426522061228752,
      0.03942342847585678, 0.08707905560731888, -0.022083215415477753, -0.08727998286485672,
      -0.07545999437570572, 0.10114879906177521, -0.004904789384454489, 0.06774819642305374,
      0.009800935164093971, 0.044890739023685455, -0.014967218041419983, 0.0941678136587143,
      -0.05770362913608551, 0.10287013649940491, -0.005453699268400669, 0.03513132035732269,
      -0.013903905637562275, 0.08413811773061752, -0.06980650871992111, -0.028906194493174553,
      0.0636555626988411, 0.05698132514953613, 0.011105458252131939, 0.04353448748588562,
      0.057822950184345245, 0.06026318296790123, 0.0337996669113636, -0.0266746673732996,
      -0.07705795764923096, 0.10474573075771332, -0.08345308154821396, -0.05539889261126518,
      0.022447070106863976, -0.06616806983947754, -0.006647750269621611, 0.07067793607711792],
     [0.052905887365341187, -0.011435143649578094, -0.021353241056203842, 0.07836923003196716,
      0.061424411833286285, -0.08000302314758301, -0.07585275918245316, -0.11967397481203079,
      0.05380115285515785, -0.042299896478652954, -0.07914496958255768, 0.05075051635503769,
      0.04912104085087776, -0.004765606950968504, 0.05414928123354912, 0.049814339727163315,
      -0.07409732788801193, -0.036082521080970764, 0.002313806675374508, -0.05299842730164528,
      -0.040583763271570206, -0.07024361938238144, -0.013682425022125244, -0.032044894993305206,
      0.033571626991033554, 0.05327736586332321, -0.012224131263792515, -0.048967547714710236,
      -0.04213785007596016, 0.04478442296385765, 0.036994051188230515, -0.015246623195707798,
      -0.021649139001965523, -0.04749245569109917, 0.034521620720624924, -0.054009921848773956,
      0.012803693301975727, -0.041754789650440216, 0.05299144238233566, -0.043210580945014954,
      0.04119676724076271, -0.003660433227196336, 0.061294812709093094, 0.04170737415552139,
      0.04956142231822014, -0.07435107231140137, -0.06256762146949768, -0.08608464896678925,
      0.03419112786650658, 0.030482828617095947, 0.12275132536888123, 0.011274893768131733,
      0.09367499500513077, 0.04984012618660927, 0.03249164670705795, 0.061120033264160156,
      -0.1082526370882988, 0.1027260571718216, 0.031052036210894585, -0.05970580875873566,
      0.09552770853042603, -0.11599823087453842, 0.06792296469211578, 0.025900542736053467,
      0.05081837624311447, 0.01661667972803116, 0.05880744382739067, -0.07362207025289536,
      -0.0949845239520073, -0.006347096990793943, -0.07460620254278183, -0.10717994719743729,
      -0.052538130432367325, 0.0755474790930748, -0.0023424047976732254, 0.08705011010169983,
      -0.016820482909679413, 0.0593332014977932, -0.06569274514913559, -0.10300233215093613,
      -0.08291200548410416, -0.0821881964802742, 0.10032499581575394, -0.006332038901746273,
      0.011644313111901283, -0.04881773889064789, 0.06938454508781433, 0.047233402729034424,
      0.06143061816692352, -0.08020210266113281, -0.032856959849596024, 0.09526780247688293,
      0.027936911210417747, -0.04701968654990196, 0.027036499232053757, -0.045661620795726776,
      -0.03362652286887169, -0.04973864555358887, -0.008685840293765068, 0.07702237367630005,
      -0.04177209734916687, 0.018700256943702698, -0.014512328431010246, -0.031913936138153076,
      0.04640055075287819, 0.009372023865580559, 0.07853786647319794, 0.005251489579677582],
     [-0.07543276995420456, 0.09459474682807922, 0.11311813443899155, 0.041047677397727966,
      0.015805017203092575, -0.05282605439424515, 0.07680812478065491, 0.056062888354063034,
      0.073404461145401, 0.003291935892775655, 0.12681785225868225, -0.02312823198735714,
      -0.07925786823034286, 0.07629497349262238, -0.00966835767030716, -0.09742026776075363,
      -0.014848629012703896, -0.04146776348352432, 0.050866980105638504, -0.04186534881591797,
      0.04358061030507088, -0.10561610758304596, -0.02851826697587967, 0.04854203388094902,
      0.037275005131959915, 0.03606810048222542, 0.007197067607194185, -0.0372207947075367,
      0.01557037141174078, -0.018078835681080818, 0.0507224015891552, -0.0767543688416481,
      0.08215872943401337, -0.05645504221320152, 0.04262932389974594, -0.008204212412238121,
      0.1048189178109169, -0.007216873113065958, 0.03889914229512215, -0.000425845937570557,
      -0.0028988404665142298, 0.026936683803796768, 0.056493211537599564, 0.02545681782066822,
      -0.0785687044262886, 0.09396710991859436, -0.042447056621313095, 0.03914849832653999,
      0.009594096802175045, 0.07844708114862442, -0.037763528525829315, 0.06709452718496323,
      -0.014420628547668457, 0.08764149993658066, 0.09087972342967987, 0.021244686096906662,
      0.027727486565709114, 0.035057809203863144, 0.08328276127576828, 0.010808060877025127,
      -0.011598415672779083, 0.09919176250696182, 0.09685119986534119, 0.02750604785978794,
      0.033937279134988785, -0.03273816779255867, 0.002181309973821044, -0.04130581393837929,
      0.061390042304992676, -0.014894112013280392, 0.08757079392671585, 0.05626455694437027,
      -0.03107050620019436, 0.05818118155002594, 0.00828024372458458, 0.06553051620721817,
      0.03363269940018654, 0.003391158999875188, -0.08306334167718887, 0.05258772894740105,
      -0.04895278438925743, -0.025855688378214836, -0.0249740369617939, 0.023716451600193977,
      -0.02713824436068535, -0.08822835981845856, -0.04635544866323471, 0.05131044238805771,
      0.03943030536174774, 0.02338620088994503, -0.047940444201231, 0.07394178211688995,
      -0.027968399226665497, -0.07051065564155579, -0.00867125391960144, -0.003659948008134961,
      -0.11830802261829376, -0.03566288203001022, -0.08703045547008514, -0.03251286596059799,
      -0.0381658598780632, 0.019388634711503983, -0.018586158752441406, 0.011304285377264023,
      0.01232575811445713, 0.04629157483577728, 0.07068879902362823, 0.02788686379790306],
     [-0.013306362554430962, 0.05565505474805832, -0.048268791288137436, -0.05963457375764847,
      0.07199922204017639, 0.1298571527004242, -0.029091451317071915, 0.10455747693777084,
      -0.048317793756723404, -0.005124826915562153, 0.013108598999679089, 0.12984995543956757,
      0.060387443751096725, -0.08474285900592804, -0.058981843292713165, -0.08326796442270279,
      -0.11372702568769455, 0.05485326796770096, -0.00043649590224958956, -0.020757636055350304,
      -0.11245540529489517, 0.04029776155948639, -0.11002525687217712, -0.08994192630052567,
      -0.10618971288204193, -0.07392130047082901, -0.049538616091012955, 0.03423486649990082,
      0.035766951739788055, -0.05390065908432007, -0.06391732394695282, 0.08944873511791229,
      -0.08003262430429459, 0.06943219155073166, -0.04250246286392212, -0.10136683285236359,
      -0.041252557188272476, 0.002558488631621003, -0.04004288092255592, -0.05473880469799042,
      -0.07625848799943924, -0.029690761119127274, 0.020873485133051872, 0.035440605133771896,
      0.09296897798776627, -0.10206855833530426, 0.030750559642910957, 0.04378441348671913,
      0.06529530882835388, 0.002619263017550111, -0.03704267367720604, -0.06669671088457108,
      0.04970788210630417, 0.031148696318268776, -0.046285565942525864, -0.01698698103427887,
      -0.06654611974954605, 0.0818093866109848, -0.1081586480140686, -0.10444913804531097,
      0.04691950976848602, -0.06483840197324753, 0.017509540542960167, 0.05266791954636574,
      -0.05889568850398064, -0.017173392698168755, 0.049630798399448395, -0.040000636130571365,
      0.08041469007730484, 0.016751911491155624, -0.043175701051950455, 0.08621490746736526,
      -0.0932464450597763, 0.00999415386468172, -0.085342176258564, -0.02678311988711357,
      -0.025912314653396606, -0.06956758350133896, -0.021134288981556892, 0.02917608991265297,
      -0.07974736392498016, -0.07995449006557465, -0.023342926055192947, 0.05316370725631714,
      0.05243205279111862, 0.0227857306599617, 0.059854499995708466, -0.08931616693735123,
      0.0829075276851654, -0.05083629488945007, -0.008876921609044075, -0.003119897795841098,
      0.04574056714773178, -0.026260970160365105, -0.07704263925552368, 0.01242003496736288,
      -0.08039020001888275, -0.028606025502085686, 0.04806964844465256, -0.011311783455312252,
      -0.08414484560489655, -0.04084789380431175, -0.026997903361916542, -0.025492403656244278,
      0.0659668818116188, 0.003688650671392679, -0.04850549250841141, -0.01719534397125244],
     [0.08255238085985184, 0.06118563190102577, -0.09434419870376587, -0.04189058020710945,
      0.023785697296261787, 0.007840761914849281, -0.059207551181316376, 0.07599854469299316,
      0.015172860585153103, -0.012237351387739182, -0.014639707282185555, 0.02752045914530754,
      0.03835980221629143, -0.0834319069981575, -0.10821504890918732, -0.0800078883767128,
      0.042912136763334274, -0.05853604897856712, 0.04979385435581207, -0.06667446345090866,
      0.023583099246025085, -0.07474027574062347, 0.04021332785487175, -0.07130170613527298,
      0.04058576002717018, 0.07222981005907059, -0.05241250991821289, -0.0168850626796484,
      0.033578190952539444, -0.009922455064952374, 0.09260749071836472, 0.023816227912902832,
      0.04516691341996193, 0.047830093652009964, -0.0499332956969738, 0.035844605416059494,
      -0.0058175246231257915, -0.03428509086370468, -0.039892785251140594, 0.02753177471458912,
      -0.046513233333826065, -0.08002879470586777, 0.09447469562292099, -0.007415427826344967,
      0.008832636289298534, -0.044942282140254974, 0.06687063723802567, -0.01209845021367073,
      0.08883564919233322, 0.05731597915291786, -0.029225412756204605, 0.06954306364059448,
      0.01715240627527237, 0.09913931787014008, 0.1003505066037178, -0.015205063857138157,
      -0.057674530893564224, -0.008078012615442276, -0.08890542387962341, 0.007096719928085804,
      -0.03495907410979271, 0.0006862576119601727, -0.03229473903775215, 0.03081415221095085,
      0.04930715262889862, -0.042885180562734604, 0.05733809992671013, 0.00369504583068192,
      0.10548248142004013, 0.03888793662190437, 0.06848904490470886, 0.03425376117229462,
      0.03237496316432953, -0.015902576968073845, 0.04000972583889961, 0.05247272551059723,
      -0.06764844059944153, -0.026540657505393028, -0.03424324095249176, 0.10374859720468521,
      -0.08506670594215393, -0.10533925145864487, -0.03585569187998772, -0.04690024256706238,
      0.015541047789156437, 0.04652240127325058, 0.01478621643036604, -0.024684393778443336,
      0.017184721305966377, -0.09828510135412216, 0.029605934396386147, 0.05880440026521683,
      0.044821444898843765, 0.003927071113139391, 0.01122796256095171, -0.03787196800112724,
      -0.039431020617485046, -0.029199181124567986, -0.04337691515684128, -0.03661762923002243,
      -0.061747487634420395, -0.040125876665115356, 0.10411777347326279, -0.019264372065663338,
      -0.06801850348711014, -0.04454287141561508, -0.07292243093252182, -0.03748679533600807],
     [0.033890724182128906, -0.07376625388860703, -0.09251823276281357, -0.00018443394219502807,
      -0.10555169731378555, -0.011943064630031586, -0.110806405544281, -0.09023112803697586,
      0.06638839095830917, 0.05148325115442276, -0.09248267859220505, 0.05894983559846878,
      0.09054914861917496, 0.06491689383983612, 0.0058190301060676575, 0.06540238112211227,
      0.02778320387005806, -0.06002466380596161, -0.024469951167702675, -0.004447296727448702,
      0.07142110913991928, 0.02486550435423851, 0.11215712130069733, 0.1222403272986412,
      -0.01578003726899624, -0.02024657651782036, -0.05352616310119629, 0.0975872054696083,
      -0.0233296025544405, -0.025536267086863518, 0.0013137833448126912, 0.015272360295057297,
      0.04728180542588234, 0.06028785929083824, -0.11089051514863968, -0.10395815968513489,
      -0.02952820248901844, -0.014815120957791805, 0.025618933141231537, -0.03892571106553078,
      -0.03561335802078247, 0.06010119989514351, 0.03731606528162956, 0.010605738498270512,
      0.0727972462773323, -0.009513736702501774, -0.013717740774154663, 0.026549486443400383,
      -0.0004318856808822602, -0.08868993073701859, 0.0417822040617466, 0.023320408537983894,
      -0.05814666673541069, 0.05001167953014374, 0.09261457622051239, 0.05313697084784508,
      0.007814135402441025, 0.08410435169935226, -0.00455261068418622, -0.059433210641145706,
      -0.027013571932911873, -0.0730096772313118, -0.04879099130630493, 0.10139481723308563,
      0.063570536673069, 0.08045075088739395, 0.024821987375617027, -0.056547101587057114,
      0.03669550642371178, 0.02618747390806675, 0.05698046088218689, 0.033551134169101715,
      -0.013841322623193264, -3.381227361387573e-05, -0.035932086408138275, -0.0181532371789217,
      0.027117960155010223, -0.07363991439342499, -0.007101507391780615, 0.09713353961706161,
      -0.017614712938666344, 0.0868595540523529, 0.037938158959150314, 0.03585651144385338,
      0.05824778601527214, 0.003409213852137327, -0.06351999938488007, -0.050979357212781906,
      0.10436014831066132, 0.010176988318562508, -0.012545997276902199, -0.06450532376766205,
      -0.11509257555007935, -0.04295088350772858, -0.085482656955719, -0.0017631691880524158,
      -0.01616203971207142, -0.02699950523674488, -0.01833740435540676, -0.1437394767999649,
      -0.12350378185510635, -0.00041830886038951576, -0.04499062895774841, 0.10382887721061707,
      0.10961642116308212, -0.009154709987342358, -0.03186377137899399, 0.08934980630874634],
     [-0.12447840720415115, -5.816730117658153e-05, -0.08050951361656189, -0.14778538048267365,
      -0.1480184942483902, 0.0035654238890856504, 0.0008714473806321621, 0.0158110149204731,
      -0.019695255905389786, 0.03150780871510506, 0.03432970121502876, 0.027170313522219658,
      0.10399451851844788, -0.007769563235342503, -0.032330453395843506, -0.011144183576107025,
      0.06140795722603798, 0.09344244003295898, 0.06447894871234894, -0.037624385207891464,
      0.10912206768989563, 0.09124545007944107, 0.06092844903469086, 0.058696284890174866,
      0.0853177085518837, 0.056243933737277985, -0.037308480590581894, 0.08965180814266205,
      -0.035772278904914856, 0.03203606605529785, -0.03893787041306496, 0.05815728008747101,
      -0.043936748057603836, -0.05786395072937012, 0.060883160680532455, 0.041969045996665955,
      -0.05133863538503647, 0.08440583944320679, -0.04403431713581085, 0.09549231827259064,
      -0.053142812103033066, 0.026164066046476364, -0.01590864546597004, -0.00793758500367403,
      0.05751917511224747, 0.003114235820248723, -0.08993614464998245, 0.0004615147481672466,
      -0.05100309103727341, -0.030082765966653824, -0.03922543674707413, -0.04864143207669258,
      -0.03152076527476311, -0.02399657852947712, 0.04962509498000145, 0.007164639420807362,
      0.004624930210411549, -0.029044149443507195, -0.05489092692732811, -0.02217097580432892,
      0.07195670902729034, -0.013312991708517075, -0.09155900776386261, 0.015198839828372002,
      -0.0654386356472969, -0.033833034336566925, 0.0018894164822995663, 0.026172269135713577,
      0.006027581170201302, 0.0795799121260643, -0.07468178868293762, 0.01915820874273777,
      -0.0012126791989430785, 0.06206092610955238, 0.09908368438482285, -0.07809050381183624,
      -0.045842062681913376, 0.059445783495903015, -0.050627924501895905, -0.03625977411866188,
      -0.0262623131275177, 0.06654021888971329, -0.019328534603118896, 0.07651499658823013,
      -0.041906747967004776, 0.0034226232673972845, 0.06643015146255493, 0.06921250373125076,
      0.027488350868225098, 0.034773580729961395, -0.046204131096601486, 0.07347463071346283,
      0.06279988586902618, -0.10813106596469879, -0.08429082483053207, 0.0671769380569458,
      0.0045425244607031345, -0.07141852378845215, 0.004410352557897568, -0.016028182581067085,
      -0.020248256623744965, -0.05343972519040108, -0.057060591876506805, -0.03858916088938713,
      -0.10082516074180603, -0.07889873534440994, 0.06393469870090485, -0.025637447834014893],
     [-0.15427136421203613, -0.05148767679929733, -0.11869321763515472, -0.00632404163479805,
      -0.003367021679878235, -0.01847929134964943, -0.12794961035251617, -0.04466502368450165,
      -0.09504834562540054, 0.03920558840036392, -0.06258447468280792, -0.0551520511507988,
      0.030671989545226097, 0.0920613706111908, 0.009804686531424522, 0.007570865098387003,
      0.059238843619823456, 0.004038089886307716, 0.14176979660987854, 0.022050578147172928,
      0.1461547315120697, 0.12147130072116852, 0.060829050838947296, 0.08250661939382553,
      0.04686305671930313, 0.006081766448915005, 0.082253597676754, 0.017875464633107185,
      0.002233345527201891, -0.05011482536792755, 0.03244888037443161, -0.03613527491688728,
      -0.09139367938041687, -0.03411438316106796, 0.0558830127120018, -0.07916209846735,
      0.08891311287879944, -0.04012472182512283, -0.05904044955968857, -0.029745949432253838,
      -0.08961408585309982, 0.08185862749814987, 0.1091076210141182, 0.05986251309514046,
      0.028703097254037857, -0.07396965473890305, -0.059996072202920914, 0.01686854287981987,
      0.09077799320220947, -0.015733866021037102, -0.015340538695454597, -0.05330701172351837,
      0.07614422589540482, -0.006334908772259951, 0.12295173108577728, 0.039600569754838943,
      0.04269340634346008, 0.009415941312909126, -0.026423566043376923, -0.011555065400898457,
      0.03508555144071579, 0.011808543466031551, 0.046231791377067566, 0.08723533898591995,
      0.02355366386473179, 0.07140903174877167, 0.06274984776973724, 0.06174657121300697,
      -0.02234140783548355, 0.028098978102207184, 0.06767018139362335, -0.09811526536941528,
      -0.06058932840824127, -0.05420658364892006, -0.03370632603764534, 0.01212073303759098,
      -0.01605142466723919, 0.011604057624936104, 0.10088929533958435, 0.08430454134941101,
      0.036466412246227264, 0.04113198444247246, 0.0024628834798932076, 0.04376644268631935,
      -0.017835283651947975, 0.005957764573395252, -0.06045442819595337, -0.033096794039011,
      0.052320048213005066, 0.07264667004346848, -0.11265255510807037, -0.06189738214015961,
      -0.10988117754459381, 0.024382416158914566, -0.0376395508646965, -0.04590526968240738,
      -0.19174064695835114, -0.13020294904708862, -0.10069664567708969, -0.09764382988214493,
      -0.04133555293083191, -0.10291367769241333, 0.08114701509475708, 0.04542544111609459,
      0.04358761012554169, -0.08526363968849182, -0.03891937807202339, 0.05421432480216026],
     [0.09366130083799362, -0.06864181160926819, -0.030826110392808914, 0.05648982524871826,
      0.024665692821145058, -0.05852502956986427, 0.016361098736524582, 0.08289030939340591,
      0.00019608372531365603, -0.08251786231994629, 0.04136379063129425, 0.07568249851465225,
      0.009577925316989422, -0.04670441523194313, 0.03421924263238907, -0.04603897035121918,
      -0.031611520797014236, -0.048978693783283234, 0.021898075938224792, -0.04033435881137848,
      -0.059063371270895004, -0.003713710932061076, 0.055803216993808746, -0.01926051639020443,
      0.05478210002183914, 0.0648723915219307, -0.029948152601718903, -0.03375203534960747,
      -0.021509740501642227, 0.028546636924147606, -0.09318810701370239, -0.09816187620162964,
      -0.11036177724599838, -0.10758223384618759, -0.06214520335197449, 0.02437753602862358,
      0.07656533271074295, 0.06079857051372528, -0.046137843281030655, 0.11096584051847458,
      0.03625607118010521, -0.09237658232450485, 0.07151994854211807, -0.033614546060562134,
      -0.04205906391143799, 0.009100708179175854, -0.09101981669664383, -0.020236559212207794,
      0.05466478317975998, -0.08383046835660934, 0.0435970313847065, -0.06200592592358589,
      -0.07288246601819992, 0.08584164828062057, -0.049905553460121155, -0.002264962298795581,
      0.10015669465065002, 0.10131169855594635, 0.08327911049127579, 0.06016159430146217,
      0.019760070368647575, 0.006856021471321583, -0.06784664839506149, -0.04684489965438843,
      0.028694817796349525, 0.058768633753061295, 0.008985357359051704, -0.015533655881881714,
      -0.05660323053598404, -0.020115245133638382, 0.08871394395828247, 0.044049713760614395,
      0.06369148194789886, 0.0005134579259902239, -0.07367826998233795, -0.06261882185935974,
      0.05808396264910698, 0.03906401991844177, 0.06656927615404129, -0.09435286372900009,
      0.06553727388381958, -0.01730269007384777, -0.02937113121151924, 0.050596799701452255,
      0.04842739924788475, -0.018967751413583755, -0.0681367814540863, -0.0024846461601555347,
      -0.05791288986802101, -0.08542800694704056, 0.0829060897231102, 0.05576476827263832,
      0.05143699422478676, -0.02212102897465229, 0.0608302503824234, 0.05278962478041649,
      0.09834601730108261, 0.08392126113176346, -0.08892884850502014, 0.07995124906301498,
      0.00876662228256464, -0.04446595534682274, -0.010990161448717117, -0.07840093225240707,
      -0.010533815249800682, -0.01239350251853466, -0.017594730481505394, -0.08525906503200531],
     [-0.1028020977973938, 0.03050682693719864, 0.01847434975206852, 0.01356507744640112,
      0.05378642678260803, -0.09074759483337402, -0.01908423937857151, 0.016278300434350967,
      -0.06657132506370544, -0.08202136307954788, -0.02047865092754364, 0.0302217286080122,
      0.0722636803984642, 0.07795143127441406, 0.03340190649032593, 0.0062385727651417255,
      -0.10352534800767899, -0.10999525338411331, 0.11079437285661697, 0.08638481795787811,
      0.08816532045602798, -0.06248846277594566, 0.0031198577489703894, 0.10926181077957153,
      0.03298596665263176, 0.07562100887298584, -0.09473616629838943, 0.021891599521040916,
      0.01586422510445118, -0.07894468307495117, -0.09098262339830399, -0.005960762966424227,
      -0.03850097209215164, -0.06860018521547318, 0.0015918432036414742, -0.12106586992740631,
      0.026681816205382347, 0.05817737802863121, -0.04698628559708595, -0.03384005278348923,
      0.018151545897126198, 0.017910288646817207, -0.05823608860373497, 0.051566604524850845,
      0.011086652055382729, 0.019884690642356873, -0.002977948170155287, -0.04886627569794655,
      0.08383377641439438, 0.07638415694236755, 0.0014015963533893228, 0.07777754217386246,
      -0.020997213199734688, 0.021846385672688484, -0.02161647193133831, -0.010254070162773132,
      -0.054586026817560196, -0.004917886573821306, -0.03466752544045448, -0.03509281948208809,
      0.01566557213664055, 0.0026049953885376453, 0.07128327339887619, 0.031047414988279343,
      -0.07285775244235992, 0.06164153292775154, 0.01927565596997738, -0.03395267575979233,
      -0.029945306479930878, 0.030908852815628052, 0.044011715799570084, 0.05054644122719765,
      0.02969134971499443, 0.028034556657075882, 0.03444802761077881, 0.02800239622592926,
      0.015801871195435524, -0.06339925527572632, -0.01935836859047413, 0.02327693998813629,
      0.022239841520786285, -0.07914916425943375, 0.08404669165611267, -0.031214991584420204,
      0.09705792367458344, -0.08929862082004547, -0.08913800120353699, 0.0709151178598404,
      0.004743695724755526, -0.04602864384651184, -0.09865488111972809, -0.04069356247782707,
      -0.053779736161231995, 0.08831752091646194, -0.012181686237454414, -0.06571230292320251,
      0.039331912994384766, -0.08503612130880356, -0.012244881130754948, -0.06484843045473099,
      0.025760933756828308, -0.07324647158384323, -0.07088080048561096, -0.022617459297180176,
      0.04411374032497406, 0.05073804780840874, 0.0009149276302196085, 0.04000577703118324],
     [-0.05176512151956558, -0.051240626722574234, -0.021841250360012054, -0.018827367573976517,
      -0.03919902816414833, -0.06185760721564293, -0.10479643940925598, 0.08897557854652405,
      0.0804978758096695, -0.0695749968290329, 0.0025518762413412333, 0.08003963530063629,
      0.05629529058933258, 0.02127416804432869, -0.038257066160440445, 0.019523654133081436,
      0.03422350808978081, -0.01720256730914116, 0.026052772998809814, -0.0612858310341835,
      0.07513777911663055, 0.01781393587589264, 0.052729424089193344, 0.08738285303115845,
      -0.05967666953802109, -0.062021926045417786, -0.05618796870112419, 0.03427485376596451,
      0.11357312649488449, -0.057721611112356186, 0.03047110140323639, 0.028758075088262558,
      0.00586180854588747, 0.09384769201278687, -0.027077585458755493, -0.07189774513244629,
      -0.05364592745900154, -0.09531862288713455, -0.08651721477508545, -0.08104228228330612,
      0.05062944069504738, -0.005793254356831312, -0.08653765171766281, 0.08972492814064026,
      0.046198464930057526, -0.03152541071176529, -0.05859721451997757, -0.034570273011922836,
      -0.05875587463378906, -0.02289508655667305, 0.05121234804391861, 0.015806976705789566,
      -0.0018644509837031364, 0.09378772974014282, -0.02112162485718727, 0.006703568156808615,
      -0.016481729224324226, 0.04753286764025688, 0.0452871099114418, -0.03840288892388344,
      0.008466997183859348, -0.07261455059051514, -0.08009329438209534, -0.007591557689011097,
      0.06080792099237442, 0.02233380265533924, -0.027669280767440796, -0.05784707888960838,
      0.06913013756275177, 0.01230581197887659, 0.007894270122051239, -0.0536617636680603,
      0.0363006629049778, -0.04901672899723053, -0.08966724574565887, -0.0814964547753334,
      -0.08212451636791229, 0.09154017269611359, -0.05130019038915634, -0.062344253063201904,
      0.06695111840963364, -0.0698755756020546, 0.017472226172685623, -0.07660127431154251,
      -0.07378174364566803, -0.04662100598216057, 0.022187134250998497, 0.001556379604153335,
      0.04380063712596893, 0.08670089393854141, -0.1275503933429718, -0.033212170004844666,
      -0.02620231918990612, -0.016935154795646667, -0.09385755658149719, -0.11343115568161011,
      0.04268763214349747, 0.009684578515589237, -0.09828308969736099, 0.014987949281930923,
      -0.09918559342622757, -0.03957930579781532, 0.08525123447179794, 0.04276333004236221,
      -0.06158118695020676, -0.015652824193239212, -0.07662338763475418, 0.03438185155391693],
     [0.11021007597446442, -0.06685981154441833, -0.028024841099977493, 0.004954727832227945,
      0.0907583087682724, 0.043275345116853714, 0.08555278927087784, -0.006387154571712017,
      0.0009899394353851676, -0.07545841485261917, 0.09635210782289505, 0.06524865329265594,
      0.023356355726718903, -0.0091662323102355, 0.032805394381284714, -0.05330540984869003,
      0.03341813012957573, -0.11313842236995697, 0.06435028463602066, -0.01896241307258606,
      -0.059580713510513306, 0.1056351438164711, -0.017095960676670074, 0.0689874067902565,
      0.0017753938445821404, -0.06517749279737473, 0.04817755147814751, 0.04107506573200226,
      -0.10634255409240723, 0.04698899760842323, -0.02493635192513466, -0.0106250811368227,
      -0.039661016315221786, 0.040105778723955154, -0.07501111924648285, 0.022327251732349396,
      0.05947807803750038, 0.06871698796749115, 0.03509412705898285, 0.027723608538508415,
      -0.08611062169075012, -0.08147070556879044, 0.05988766625523567, 0.0057501960545778275,
      0.05977994203567505, 0.07477264106273651, -0.022236835211515427, -0.029577411711215973,
      -0.025158921256661415, -0.06900584697723389, 0.05939612165093422, 0.10571771115064621,
      -0.041289083659648895, 0.005152831319719553, 0.004848157521337271, 0.0711415708065033,
      -0.08647602796554565, 0.09315099567174911, 0.056892771273851395, 0.09890936315059662,
      -0.006804116535931826, 0.0914260745048523, 0.04278748482465744, 0.03730772063136101,
      -0.0674152597784996, -0.0648985356092453, 0.01816481165587902, 0.0009861676953732967,
      0.003024347824975848, -0.07232791185379028, 0.08384912461042404, -0.009624280966818333,
      0.08132893592119217, -0.06136665865778923, 0.02476074919104576, 0.013352843932807446,
      -0.026730652898550034, 0.04516298696398735, 0.0612507238984108, -0.07235737890005112,
      0.035513997077941895, 0.01911698281764984, 0.05774542689323425, 0.09140037000179291,
      0.07175473868846893, -0.07381663471460342, 0.03201764449477196, 0.06406527012586594,
      -0.059229250997304916, -0.022084223106503487, -0.007484113797545433, -0.0669875368475914,
      -0.08127943426370621, -0.0999051108956337, -0.07228998094797134, 0.01256685983389616,
      -0.09925764054059982, -0.0938219502568245, -0.025489911437034607, -0.10464087873697281,
      -0.07934416085481644, 0.050726518034935, -0.052310433238744736, -0.026191849261522293,
      0.018453175202012062, -0.06597951054573059, -0.11140655726194382, -0.041976943612098694],
     [-0.03767960146069527, 0.021727008745074272, -0.06781776994466782, 0.04590468853712082,
      -0.09076730906963348, 0.07771395891904831, -0.05082827806472778, -0.016009317710995674,
      0.006176152266561985, 0.02402331307530403, -0.060361359268426895, 0.04555999115109444,
      -0.011427733115851879, -0.07785765081644058, 0.013467700220644474, 0.07671470940113068,
      -0.0496782548725605, 0.03556371107697487, 0.03368908539414406, -0.12240629643201828,
      -0.09195327758789062, 0.02019546367228031, 0.02502315863966942, -0.10497774183750153,
      -0.0014683264307677746, -0.04997521638870239, -0.04365310072898865, -0.05319059267640114,
      -0.06965181231498718, -0.06776827573776245, 0.07009459286928177, 0.04526838660240173,
      0.0771896243095398, -0.05562140420079231, -0.11250606179237366, -0.018011458218097687,
      -0.050245948135852814, 0.037237271666526794, 0.045386504381895065, 0.08555129170417786,
      -0.07717099040746689, -0.013367410749197006, -0.014194044284522533, 0.10415700078010559,
      0.09951560944318771, 0.003977644257247448, -0.06770472973585129, -0.05054417625069618,
      0.07948825508356094, -0.08739915490150452, -0.044238463044166565, -0.01133109349757433,
      -0.05511583015322685, 0.0083200354129076, -0.06940381973981857, 0.021477552130818367,
      -0.07341741025447845, 0.04235729202628136, 0.02486061118543148, -0.058095548301935196,
      -0.0046834880486130714, 0.023281991481781006, -0.04638740420341492, -0.04473651573061943,
      -0.07840213924646378, 0.03248550742864609, -0.02309379167854786, 0.059923551976680756,
      -0.040089331567287445, -0.02094658836722374, -0.043028898537158966, 0.07265741378068924,
      -0.0701838880777359, 0.08298413455486298, 0.07081368565559387, -0.026645788922905922,
      0.06906574219465256, 0.050695255398750305, -0.04377139359712601, 0.027825262397527695,
      -0.0343489833176136, -0.054321374744176865, 0.01793665811419487, 0.026338472962379456,
      0.016602888703346252, -0.039873428642749786, -0.04582279920578003, 0.03617829829454422,
      -0.01841709017753601, 0.07973158359527588, -0.04885605350136757, 0.0344027578830719,
      0.040627747774124146, 0.03687805309891701, -0.06874361634254456, 0.13093237578868866,
      -0.05490003153681755, 0.06561895459890366, 0.10684124380350113, 0.06895992159843445,
      -0.011634284630417824, 0.09436999261379242, 0.04123253375291824, 0.02760830521583557,
      -0.029856199398636818, 0.018910478800535202, 0.07341645658016205, 0.10920678079128265],
     [-0.14280395209789276, -0.10217973589897156, 0.036624811589717865, -0.028048750013113022,
      -0.04654441028833389, -0.01780824363231659, -0.11448639631271362, -0.1007494255900383,
      0.009649135172367096, -0.06600832939147949, 0.05825609341263771, -0.003202403662726283,
      -0.005199736915528774, 0.1095610111951828, -0.05652647465467453, 0.02299182116985321,
      -0.01859177276492119, -0.007026569917798042, 0.03484390303492546, 0.051295507699251175,
      0.1137135922908783, 0.07981714606285095, 0.04536610469222069, 0.01802225597202778,
      0.0014024947304278612, -0.03680165484547615, 0.02420762926340103, 0.10129767656326294,
      0.05522949993610382, 0.02509959787130356, 0.07047626376152039, 0.05764399468898773,
      -0.06749001145362854, 0.012572974897921085, -0.015456101857125759, -0.0033378624357283115,
      0.036588020622730255, 0.06943348050117493, 0.048586614429950714, 0.014921990223228931,
      -0.12034077197313309, 0.011350526474416256, 0.0838499516248703, 0.0813683420419693,
      0.07095933705568314, 0.04882054775953293, -0.07260724902153015, 0.09903806447982788,
      -0.008545131422579288, 0.008428413420915604, 0.0816970020532608, 0.07132688164710999,
      -0.011872568167746067, -0.0054975515231490135, 0.07817215472459793, 0.06532737612724304,
      -0.02047448419034481, 0.03978428617119789, 0.013309052214026451, 0.09743484854698181,
      0.08565611392259598, -0.030720366165041924, 0.0739479511976242, 0.08686988055706024,
      0.1081472858786583, 0.02878512814640999, -0.00949242152273655, -0.07944227755069733,
      0.025305602699518204, -0.06850504875183105, -0.06853008270263672, 0.043961334973573685,
      -0.05046042054891586, 0.04036588966846466, 0.0006812164792791009, 0.0012933332473039627,
      0.008234580047428608, 0.07727373391389847, 0.034560803323984146, 0.036217667162418365,
      -0.035190802067518234, -0.019668027758598328, -0.05933225527405739, 0.011178555898368359,
      0.011643677949905396, -0.04881695657968521, 0.0017389061395078897, -0.05885463207960129,
      -0.0425923652946949, 0.06815949082374573, -0.04077223315834999, -0.04142192006111145,
      -0.008693923242390156, -0.12097418308258057, -0.10829204320907593, -0.031126998364925385,
      -0.11720433086156845, 0.028469344601035118, -0.14088845252990723, -0.11530385911464691,
      -0.10692866146564484, -0.0780545026063919, 0.017291678115725517, 0.008604289032518864,
      -0.007930606603622437, 0.052907127887010574, -0.03230797499418259, 0.0785798579454422],
     [-0.03911475092172623, -0.04680081456899643, -0.051235381513834, -0.06306033581495285,
      -0.015312965027987957, -0.07497292757034302, -0.08823748677968979, 0.07715660333633423,
      -0.07315876334905624, -0.023349419236183167, 0.07284820824861526, 0.02251594327390194,
      0.051760006695985794, 0.057139281183481216, -0.09577576071023941, -0.03842690959572792,
      -0.11534421145915985, 0.05197901278734207, 0.11756227910518646, 0.042176488786935806,
      -0.08011822402477264, 0.08630060404539108, 0.010070748627185822, 0.0654626116156578,
      -0.05768256261944771, -0.03362525627017021, 0.06517105549573898, -0.056542299687862396,
      -0.02559582144021988, -0.007695007137954235, -0.012451005168259144, 0.05584965646266937,
      -0.07400862127542496, -0.04906751215457916, -0.08749201893806458, -0.02620270475745201,
      -0.005343340802937746, 0.04555252566933632, 0.011302913539111614, -0.06376001983880997,
      -0.08717430382966995, -0.08274225890636444, -0.05182451009750366, -0.07181932032108307,
      0.11262407898902893, 0.06381583958864212, -0.08155232667922974, 0.02396542765200138,
      -0.0112466374412179, -0.06763199716806412, -0.0990203395485878, 0.04462077096104622,
      -0.08372058719396591, -0.01705148071050644, 0.04732722043991089, -0.04369605705142021,
      -0.06253260374069214, 0.013208580203354359, 0.027258530259132385, -0.012899935245513916,
      0.029307087883353233, -0.06323859840631485, -0.03267306089401245, -0.030220696702599525,
      -0.07479487359523773, -0.0673583447933197, 0.012433856725692749, 0.03138101473450661,
      -0.043874409049749374, -0.10226771235466003, -0.04800504073500633, -0.058137111365795135,
      0.043008286505937576, -0.031712766736745834, 0.037085700780153275, 0.00698381382972002,
      0.02207244746387005, 0.016961919143795967, -0.003851755289360881, -0.06875847280025482,
      -0.011858822777867317, 0.05787157267332077, -0.07613671571016312, 0.0005195922567509115,
      0.03548751398921013, -0.07710467278957367, -0.053279679268598557, -0.017102409154176712,
      0.0063453358598053455, -0.048704758286476135, 0.018612612038850784, 0.0841284990310669,
      0.03697914630174637, 0.03335285931825638, -0.09994735568761826, -0.004419262055307627,
      -0.04230248183012009, 0.04352656006813049, -0.06813421100378036, -0.06675346940755844,
      0.0369488000869751, -0.04466759040951729, 0.10913320630788803, 0.06946597248315811,
      -0.0024703287053853273, 0.0030167216900736094, -0.07104791700839996, -0.0878828838467598],
     [-0.03934907540678978, 0.07587753981351852, -0.07538291066884995, -0.00045489874901250005,
      0.06640434265136719, -0.02271207422018051, -0.03861358389258385, -0.02112262137234211,
      0.023935992270708084, -0.006722211372107267, -0.005682679824531078, -0.06449570506811142,
      -0.08402267843484879, -0.048062022775411606, -0.005926128476858139, 0.029099248349666595,
      -0.07117538154125214, -0.08274977654218674, -0.014817276038229465, 0.07927344739437103,
      -0.0815286636352539, -0.05228932574391365, -0.05142262205481529, -0.04383578151464462,
      0.0030411744955927134, 0.013831076212227345, 0.06960780173540115, -0.09230393916368484,
      -0.07435335218906403, -0.04299112781882286, 0.012569439597427845, -0.10073433816432953,
      -0.043115053325891495, -0.053850941359996796, 0.009099316783249378, -0.08263015002012253,
      -0.06762675195932388, 0.0050414022989571095, -0.04318011924624443, -0.07031775265932083,
      -0.05520583316683769, -0.08492118120193481, -0.008727083913981915, 0.03546876460313797,
      0.02489740028977394, 0.07993347942829132, 0.04155471548438072, -0.022902892902493477,
      -0.07608606666326523, -0.08455625921487808, -0.06489454954862595, -0.07729573547840118,
      0.030667925253510475, 0.07813341170549393, 0.001244605635292828, 0.07644489407539368,
      -0.06477706879377365, 0.04393693059682846, -0.001559560769237578, 0.03653745725750923,
      0.004827870521694422, 0.06563062220811844, 0.037535738199949265, 0.08477959036827087,
      0.05683608725667, -0.08032605797052383, 0.008701753802597523, -0.019195497035980225,
      0.07011517882347107, -0.05928901582956314, 0.016083307564258575, -0.0199893768876791,
      -0.06900576502084732, -0.00036919739795848727, 0.07601531594991684, 0.07052113860845566,
      0.09828755259513855, -0.06576526165008545, -0.0664801225066185, -0.035113390535116196,
      -0.052257031202316284, -0.019858714193105698, 0.006597420200705528, 0.07519962638616562,
      -0.07593777775764465, 0.012106694281101227, 0.07191149890422821, 0.012692973017692566,
      -0.05930707976222038, -0.07936985045671463, 0.06652683019638062, 0.06444363296031952,
      -0.08164867758750916, 0.008283720351755619, -0.07186836749315262, 0.11815391480922699,
      0.004421509802341461, -0.04550239071249962, 0.07197301834821701, 0.08252068608999252,
      0.05041970685124397, -0.001933717867359519, 0.05435202643275261, 0.02261332795023918,
      -0.049495283514261246, 0.006705095525830984, 0.059496697038412094, -0.02848675288259983],
     [0.08827140182256699, 0.02710600383579731, -0.06502072513103485, 0.09100329130887985,
      -0.029006626456975937, -0.023695509880781174, -0.026446228846907616, -0.10184346139431,
      0.06898292899131775, -0.09159287065267563, -0.0035855229943990707, 0.08571372181177139,
      -0.015072164125740528, 0.029272254556417465, -0.027145659551024437, -0.049114029854536057,
      -0.0402216836810112, -0.007584586273878813, -0.0709916427731514, 0.01788770779967308,
      -0.07825985550880432, -0.007387696765363216, 0.009667744860053062, 0.0513954684138298,
      0.004867879673838615, 0.08334491401910782, 0.02165914885699749, 0.01751740276813507,
      -0.005636171437799931, 0.0008410501759499311, 0.05458708107471466, 0.005767651833593845,
      -0.00902265403419733, 0.018915409222245216, -0.008195478469133377, 0.1079881340265274,
      0.015027305111289024, -0.010226421989500523, 0.005321578122675419, 0.03310253471136093,
      0.02509678713977337, -6.605526141356677e-05, -0.08074206113815308, 0.033123839646577835,
      0.027009252458810806, -0.06783164292573929, 0.04361758381128311, 0.019893763586878777,
      0.07266559451818466, -0.08211839944124222, 0.016377201303839684, -0.04503055661916733,
      0.01736222580075264, 0.06374096870422363, 0.05089385807514191, 0.032683346420526505,
      0.05875873938202858, -0.07976032793521881, -0.06845666468143463, 0.056906793266534805,
      0.03879012539982796, -0.08142915368080139, 0.003149981377646327, -0.10201022028923035,
      0.07451414316892624, 0.07438787817955017, -0.04645206406712532, -0.011859574355185032,
      0.06290831416845322, 0.0715564638376236, -0.0004302743764128536, -0.06139159947633743,
      0.031532298773527145, 0.03449253737926483, -0.06926616281270981, -0.06342755258083344,
      0.014330828562378883, 0.0009963088668882847, -0.027178047224879265, 0.04857444018125534,
      0.021217457950115204, 0.014251815155148506, -0.08118342608213425, 0.006945027969777584,
      0.05234464257955551, -0.03023619018495083, 0.04935681074857712, -0.08454277366399765,
      0.07351534813642502, 0.04403770714998245, 0.05109740421175957, -0.013244660571217537,
      -0.07666349411010742, -0.007668656297028065, 0.02212636172771454, -0.023813985288143158,
      -0.03195956349372864, 0.06955581158399582, 0.03531178832054138, -0.05094030871987343,
      -0.05022801086306572, 0.0839763656258583, 0.07907815277576447, 0.02232883870601654,
      0.037642452865839005, -0.056545209139585495, -0.02382590062916279, -0.03324730321764946],
     [0.04304623603820801, 0.0755237564444542, -0.03415728360414505, -0.08874095231294632,
      0.00826052576303482, 0.07071045786142349, 0.007425122428685427, 0.05681975185871124,
      -0.09041111916303635, 0.05655858293175697, -0.04998604580760002, -0.07594205439090729,
      -0.03380745276808739, -0.0036342847160995007, 0.1023099422454834, 0.006812996696680784,
      0.0262941662222147, -0.07031795382499695, 0.030153576284646988, -0.06954082101583481,
      0.005662058480083942, 0.02884136699140072, -0.05784783512353897, -0.06758391857147217,
      0.006162768695503473, -0.06607338786125183, -0.0599459670484066, -0.0015775331994518638,
      0.09031234681606293, -0.05481064319610596, 0.05998848378658295, 0.03471257537603378,
      -0.02915848046541214, -0.009090601466596127, 0.07630935311317444, 0.08343470841646194,
      -0.013963147066533566, 0.07836858183145523, 0.061709925532341, -0.05949435383081436,
      0.021862512454390526, 0.017447153106331825, 0.08489231765270233, -0.03357682749629021,
      0.009785962291061878, 0.025125497952103615, -0.06102918088436127, 0.06256289035081863,
      0.04219905659556389, 0.09091619402170181, -0.05937010422348976, -0.04199986159801483,
      0.09717205166816711, -0.014210214838385582, 0.019640570506453514, 0.09116921573877335,
      -0.008339098654687405, -0.011871661059558392, -0.0835781991481781, -0.006234871223568916,
      0.05416729673743248, 0.04487402364611626, 0.08385512977838516, -0.03920692950487137,
      0.012362007983028889, -0.05773090571165085, 0.05167256295681, 0.052993278950452805,
      0.10211760550737381, 0.024865826591849327, -0.06089656054973602, 0.08165540546178818,
      -0.007511228788644075, 0.06423301249742508, -0.04524633288383484, 0.0822347030043602,
      0.08838766068220139, -0.021450549364089966, -0.04075891897082329, -0.08016427606344223,
      -0.007872995920479298, -0.09978536516427994, 0.002056489698588848, -0.001432335819117725,
      -0.051876191049814224, -0.019168494269251823, -0.028249848634004593, -0.0013052434660494328,
      0.04923485592007637, 0.10651058703660965, -0.06927814334630966, -0.0071374597027897835,
      -0.059460077434778214, -0.052453018724918365, 0.0822720006108284, -0.053366176784038544,
      -0.0002599194413051009, -0.09964224696159363, -0.12263128161430359, 0.08177508413791656,
      0.009943901561200619, -0.0036408707965165377, 0.09002973139286041, -0.0703076422214508,
      0.011439291760325432, -0.04435036703944206, -0.10235324501991272, -0.03657563030719757],
     [-0.013993972912430763, -0.040210843086242676, 0.04945944622159004, -0.026876887306571007,
      -0.12204284220933914, -0.04373466223478317, 0.022743068635463715, -0.027855562046170235,
      0.024645142257213593, -0.07720572501420975, -0.0391940213739872, -0.12253642827272415,
      0.051221348345279694, 0.08728081732988358, 0.08720774203538895, 0.0848151296377182,
      0.05738990753889084, -0.022148845717310905, -0.07364459335803986, 0.03081856481730938,
      0.013875298202037811, 0.039839938282966614, -0.05913979932665825, -0.002244001952931285,
      -0.03579612821340561, 0.08607751131057739, -0.024273889139294624, -0.0375254787504673,
      0.10397851467132568, -0.05459459125995636, -0.0848461389541626, -0.012036093510687351,
      -0.03277716785669327, 0.06732773035764694, 0.05415821075439453, -0.0006476120906881988,
      -0.02847062423825264, 0.01984328217804432, 0.012680796906352043, -0.06367224454879761,
      -0.01748073473572731, -0.03915625065565109, 0.0004768935905303806, -0.027312086895108223,
      0.07019145786762238, -0.059650126844644547, -0.08501952141523361, 0.08777826279401779,
      -0.03729863837361336, -0.029122449457645416, 0.10589525103569031, 0.041021861135959625,
      0.07862250506877899, 0.019801311194896698, -0.03961377963423729, 0.02548368275165558,
      0.05009860545396805, 0.037424128502607346, -0.009047841653227806, -0.01832803152501583,
      -0.06277947127819061, -0.010415393859148026, -0.025087222456932068, 0.0980740338563919,
      0.09743300825357437, 0.07526577264070511, 0.009192114695906639, 0.023712031543254852,
      -0.04487768933176994, -0.020762132480740547, 0.028674887493252754, -0.06642377376556396,
      -0.08994460850954056, 0.04669554904103279, -0.07767866551876068, -0.04255147650837898,
      -0.023695245385169983, -0.07966943830251694, 0.04886437579989433, 0.012574369087815285,
      -0.05598287284374237, -0.015291309915482998, -0.033797282725572586, 0.05137228965759277,
      0.0591534860432148, -0.06450425088405609, 0.024805370718240738, -0.06600069999694824,
      -0.07372067868709564, 0.03489919751882553, -0.012717952951788902, 0.09359200298786163,
      -0.040379833430051804, 0.08794142305850983, 0.05729823186993599, -0.061997197568416595,
      0.009400937706232071, -0.006584826856851578, 0.1296164095401764, 0.020285632461309433,
      0.1032695323228836, 0.13382241129875183, -0.09218692779541016, -0.09017586708068848,
      -0.0562300868332386, 0.09422846138477325, 0.03186957910656929, -0.03289416432380676],
     [0.09986070543527603, -0.04207570478320122, 0.08723579347133636, 0.031145425513386726,
      0.09750230610370636, 0.017763102427124977, 0.010097309947013855, 0.015852224081754684,
      0.04160153120756149, -0.0028835975099354982, 0.02159610204398632, -0.033362697809934616,
      0.08488010615110397, 0.0131875891238451, -0.08762113749980927, 0.029336079955101013,
      -0.03289562463760376, -0.016911692917346954, 0.09514877200126648, 0.044436778873205185,
      -0.0616704560816288, 0.09420128911733627, 0.03981446102261543, 0.038574304431676865,
      -0.01796330325305462, -0.026328906416893005, 0.02953105978667736, -0.027974961325526237,
      -0.04386124387383461, -0.07235245406627655, -0.06509633362293243, -0.11823145300149918,
      -0.0320650115609169, 0.005105131771415472, -0.05280480906367302, -0.021207701414823532,
      -0.05496871471405029, -0.07437407225370407, -0.06551147997379303, 0.03379083052277565,
      0.04254402965307236, -0.07599959522485733, -0.0190386064350605, -0.02640618197619915,
      0.0939561054110527, 0.019716190174221992, -0.04455488175153732, 0.09385556727647781,
      -0.02105073072016239, -0.05921778827905655, -0.029907066375017166, 0.02756616845726967,
      0.06840814650058746, -0.09113296866416931, -0.05959611386060715, -0.07127071917057037,
      0.06619475781917572, -0.03832090646028519, -0.018258098512887955, -0.02696424536406994,
      -0.01408325880765915, -0.08423855155706406, -0.024966692551970482, 0.08092831820249557,
      0.019779657945036888, 0.09540698677301407, -0.04098782315850258, -0.03231286630034447,
      0.02978503331542015, -0.06824365258216858, 0.05009988695383072, 0.03876149281859398,
      0.03775188699364662, 0.004343284759670496, -0.08099523931741714, -0.02438807301223278,
      0.05122748017311096, -0.03362682834267616, 0.0738144963979721, 0.04092104732990265,
      0.025403745472431183, -0.04998304322361946, -0.039749667048454285, -0.06536951661109924,
      0.07444826513528824, 0.026637524366378784, -0.06692866235971451, 0.06449540704488754,
      0.10855806618928909, -0.02912793681025505, -0.07099423557519913, -0.0770336166024208,
      -0.1383114606142044, 0.00047163237468339503, 0.01873674802482128, -0.07856867462396622,
      -0.10183096677064896, -0.001753288903273642, 0.02444046176970005, 0.03499255329370499,
      -0.13300016522407532, -0.012620432302355766, 0.05516716092824936, -0.06728887557983398,
      -0.08662790805101395, 0.09123299270868301, 0.09033824503421783, 0.1089990958571434],
     [-0.003035134170204401, -0.04316505417227745, -0.13440798223018646, -0.12189429998397827,
      -0.047448065131902695, -0.07219645380973816, -0.11071336269378662, -0.07539857178926468,
      -0.06033528596162796, 0.039804067462682724, -0.031075475737452507, 0.009691945277154446,
      -0.026240497827529907, -0.040305107831954956, 0.012574787251651287, 0.005081762094050646,
      0.08509910106658936, 0.04420081153512001, 0.03626907616853714, 0.10090140253305435,
      0.11558640748262405, 0.07652993500232697, 0.09346775710582733, 0.13861173391342163,
      -0.03212277963757515, 0.07461483776569366, -0.06524452567100525, -0.03910888358950615,
      0.07877364754676819, 0.062230728566646576, 0.08245126157999039, -0.055306028574705124,
      -0.03361271694302559, -0.009127367287874222, -0.0814775675535202, 0.07798206806182861,
      -2.950284215330612e-05, 0.07339087128639221, -0.0382690504193306, -0.09225913882255554,
      -0.033229850232601166, -0.002006026217713952, 0.0066278716549277306, -0.06455453485250473,
      0.0040935068391263485, -0.00017074536299332976, -0.0020097405649721622, -0.054204463958740234,
      0.09203453361988068, 0.04520175978541374, 0.02114766836166382, -0.004762783646583557,
      -0.08852412551641464, 0.06218960881233215, -0.06568309664726257, -0.06410790234804153,
      -0.06933031231164932, 0.010847208090126514, -0.06900469213724136, -0.06921248883008957,
      -0.004173023626208305, 0.06988539546728134, 0.01515266951173544, 0.07869222015142441,
      -0.05740736797451973, 0.02977532520890236, -0.07337231934070587, 0.0891537219285965,
      -0.01797115057706833, -0.1011916920542717, 0.09864961355924606, 0.032400570809841156,
      -0.07612311094999313, 0.05726761743426323, -0.07322608679533005, -0.08265224099159241,
      0.05679904669523239, 0.04776041954755783, 0.033678147941827774, 0.012629128061234951,
      -0.03444804251194, 0.04179298132658005, -0.07505221664905548, -0.028955357149243355,
      0.046533141285181046, 0.0795513242483139, -0.06032705679535866, 0.038942284882068634,
      -0.028861792758107185, 0.05140367150306702, -0.0655016079545021, -0.028299586847424507,
      -0.08254134654998779, 0.04829156771302223, -0.011288325302302837, 0.03678039833903313,
      -0.0926019623875618, -0.005474538076668978, -0.03284111246466637, 0.030529988929629326,
      -0.0659797340631485, -0.14714962244033813, -0.0845254510641098, -0.061948321759700775,
      0.07961945980787277, 0.03906647861003876, -0.080706886947155, -0.044119514524936676],
     [-0.0960095077753067, 0.03043115697801113, 0.055994514375925064, -0.10276126116514206,
      -0.08359488844871521, -0.05904826149344444, 0.04618952050805092, 0.08493480831384659,
      0.08468541502952576, -0.04150767624378204, -0.013103476725518703, -0.0682702586054802,
      -0.05023611709475517, -0.0040674759075045586, 0.09100539982318878, 0.038496121764183044,
      -0.007237627170979977, -0.026773052290081978, 0.05172588676214218, 0.003299556439742446,
      0.06854955852031708, -0.07911935448646545, 0.07038377225399017, -0.052359119057655334,
      -0.07797925174236298, 0.0025995199102908373, -0.08072946965694427, 0.06112111359834671,
      0.04947924241423607, 0.0352947823703289, -0.07773832231760025, 0.005772984586656094,
      -0.07727735489606857, 0.09767292439937592, -0.07893542945384979, 0.013319415040314198,
      0.04223959892988205, -0.055003076791763306, -0.027977628633379936, -0.05330070108175278,
      0.025242548435926437, 0.017974993214011192, -0.06589391827583313, -0.02487216144800186,
      -0.009647189639508724, 0.08134844899177551, 0.006734215654432774, 0.05247640982270241,
      0.01307433657348156, 0.06963641941547394, 0.042442332953214645, -0.03516870737075806,
      -0.01081345696002245, -0.07352031022310257, -0.009111342951655388, -0.041182614862918854,
      0.04516366124153137, -0.01453376468271017, 0.04270271584391594, -0.033562399446964264,
      -0.05331965163350105, -0.04265856370329857, -0.0918111503124237, 0.038148075342178345,
      -0.05434264615178108, 0.03540188446640968, 0.08688358962535858, -9.71479166764766e-05,
      0.05703474581241608, -0.02816145494580269, -0.02315131574869156, -0.030769111588597298,
      -0.05821593850851059, 0.06534642726182938, -0.08820564299821854, 0.03316107019782066,
      0.012259703129529953, 0.05916459113359451, -0.027422070503234863, 0.055083125829696655,
      0.035516157746315, -0.03531729802489281, 0.011289888061583042, -0.011440083384513855,
      -0.07312662154436111, 0.01942295767366886, -0.010655843652784824, -0.0417572557926178,
      0.0007311294903047383, 0.06946377456188202, 0.06008252874016762, -0.02275869995355606,
      -0.06357185542583466, -0.08792579919099808, -0.04421218857169151, -0.005952916108071804,
      -0.1142413318157196, 0.061705801635980606, 0.01635703817009926, 0.08083294332027435,
      -0.006836255546659231, 0.029349857941269875, -0.021136529743671417, 0.038271714001894,
      0.008474561385810375, 0.021489432081580162, -0.06333699077367783, 0.04703618958592415],
     [-0.09732970595359802, -0.09031115472316742, 0.0014263525372371078, -0.05996466800570488,
      -0.12820672988891602, -0.0017698351293802261, -0.019338829442858696, 0.018239900469779968,
      -0.11344780027866364, 0.005809409078210592, -0.06586595624685287, -0.12493759393692017,
      -0.04941033944487572, 0.02061629295349121, 0.016490280628204346, 0.04223073273897171,
      -0.029132215306162834, -0.044222380965948105, 0.015744464471936226, 0.07505946606397629,
      0.01681392453610897, 0.10915423929691315, 0.15053151547908783, 0.004492343403398991,
      -0.023214153945446014, -0.010085875168442726, 0.07413223385810852, -0.043073974549770355,
      0.08765790611505508, 0.025842055678367615, 0.0029601568821817636, 0.00808748509734869,
      -0.06180505454540253, -0.016747964546084404, 0.04825413227081299, 0.0075613693334162235,
      0.052771151065826416, 0.09469985961914062, -0.07023613899946213, 0.1157679334282875,
      0.06194264069199562, 0.02266315184533596, 0.09162575751543045, 0.08743832260370255,
      -0.06274916231632233, -0.006339710671454668, -0.08078448474407196, 0.08422902226448059,
      0.037422724068164825, -0.034933820366859436, -0.015562032349407673, 0.03913256153464317,
      -0.08443452417850494, -0.05861310288310051, -0.05381046235561371, 0.059535667300224304,
      -0.08616415411233902, 0.06959276646375656, 0.05047987401485443, 0.10140883922576904,
      -0.02003076672554016, -0.0019183721160516143, -0.038724225014448166, -0.08685368299484253,
      0.07123629003763199, 0.05656389519572258, 0.020318463444709778, -0.0019468381069600582,
      -0.02545980177819729, 0.04073537141084671, -0.02130867913365364, -0.07108153402805328,
      -0.01824340783059597, -0.013685422949492931, 0.041739750653505325, -0.09156177937984467,
      -0.013117589056491852, -0.0662938803434372, 0.09442096948623657, -0.054083410650491714,
      0.045656703412532806, -0.007443731185048819, -0.029148567467927933, 0.07270509004592896,
      0.005845994688570499, -0.08723679184913635, 0.06197765842080116, 0.1038307175040245,
      -0.037669312208890915, 0.022361034527420998, -0.09424938261508942, 0.05653635784983635,
      -0.07905618101358414, -0.0644427016377449, -0.04443281516432762, -0.07925640791654587,
      -0.09114391356706619, -0.1373916119337082, -0.015419159084558487, -0.07404430955648422,
      0.014569198712706566, -0.14608153700828552, -0.03526986017823219, 0.11231856793165207,
      -0.0005483019049279392, -0.06372637301683426, 0.019822586327791214, -0.06270153820514679],
     [0.02178761176764965, -0.005426286719739437, -0.08080031722784042, -0.001614305074326694,
      -0.0423889197409153, -6.092805779189803e-05, 0.017044277861714363, 0.06603585183620453,
      -0.08616277575492859, -0.056364115327596664, 0.021154718473553658, -0.09171359241008759,
      0.032685209065675735, -0.04018234834074974, -0.014458554796874523, 0.056662704795598984,
      -0.08071641623973846, -0.098185695707798, -0.00530982157215476, 0.08251160383224487,
      -0.09184861183166504, -0.052506912499666214, -0.0031628236174583435, -0.07016148418188095,
      -0.07879745215177536, 0.055636726319789886, 0.036709364503622055, 0.028374869376420975,
      -0.10842355340719223, -0.049360908567905426, 0.07118813693523407, 0.0803639143705368,
      -0.09249473363161087, 0.06129228323698044, -0.05255977064371109, 0.06623818725347519,
      -0.0035691524390131235, 0.015487409196794033, 0.03373633325099945, 0.03793453797698021,
      -0.05306077003479004, -0.012786795385181904, -0.04129977524280548, -0.04607703536748886,
      -0.03014594316482544, 0.0669255256652832, 0.03741714730858803, -0.06693688035011292,
      -0.08188970386981964, 0.03584221750497818, 0.08102969825267792, 0.03515952453017235,
      -0.026263494044542313, 0.10543613880872726, -0.07819332927465439, 0.03331334888935089,
      0.11009272187948227, -0.08691780269145966, 0.0914003923535347, 0.0844268649816513,
      0.004284348338842392, 0.01822415366768837, 0.024293387308716774, -0.051940493285655975,
      0.06144728884100914, -0.04741432145237923, -0.10073912888765335, 0.051890891045331955,
      -0.022288141772150993, -0.02245298959314823, -0.05023839324712753, 0.10319510102272034,
      -0.06194290146231651, -0.03189144283533096, 0.03057916648685932, 0.076354019343853,
      -0.0923532024025917, 0.08065439015626907, -0.03997750207781792, -0.042275335639715195,
      0.02221325784921646, -0.07973605394363403, -0.010261640883982182, -0.057285092771053314,
      0.01882038824260235, 0.03996201604604721, 0.07895350456237793, -0.0232014749199152,
      0.03615077957510948, -0.08678681403398514, 0.002957755234092474, 0.029804915189743042,
      0.006926369853317738, -0.03240646421909332, 0.09818775206804276, -0.02100609987974167,
      0.054112326353788376, 0.036114759743213654, -0.03556573763489723, -0.029651157557964325,
      -0.08216799795627594, 0.02242700755596161, 0.08934520184993744, -0.06828168034553528,
      0.07291193306446075, -0.034421246498823166, 0.060937754809856415, -0.0567103773355484],
     [-0.029084181413054466, -0.05193471908569336, 0.043843645602464676, -0.07230763137340546,
      0.00030239528859965503, -0.11311857402324677, -0.04535256326198578, -0.03519746661186218,
      0.08339890837669373, -0.08201941102743149, -0.05501025542616844, -0.052459850907325745,
      -0.06571818888187408, 0.0552561916410923, -0.065425343811512, 0.007933162152767181,
      -0.0688357874751091, -0.0020565339364111423, 0.06405023485422134, 0.10060853511095047,
      0.013969341292977333, 0.014181514270603657, 0.11199643462896347, 0.0298735611140728,
      0.058845192193984985, 0.03104911558330059, 0.07662197947502136, -0.10221771150827408,
      -0.02384536899626255, 0.04336961731314659, -0.0778728798031807, 0.06820359081029892,
      0.002931079128757119, 0.05654323846101761, 0.04379981756210327, -0.08469797670841217,
      0.047266170382499695, -0.022074321284890175, 0.06734246015548706, 0.0992979183793068,
      -0.07217206060886383, -0.015599919483065605, -0.04534091800451279, 0.01300282683223486,
      -0.02346138097345829, -0.014232521876692772, -0.07747015357017517, 0.04640708118677139,
      -0.018393494188785553, 0.05582557991147041, -0.064309261739254, 0.03108857199549675,
      -0.02195497788488865, 0.016812501475214958, 0.05784207582473755, 0.06138090416789055,
      -0.020282652229070663, 0.06978221982717514, 0.004641961306333542, 0.06428848206996918,
      -0.0048338440246880054, -0.06478127092123032, 0.0728394165635109, 0.003695492632687092,
      -0.07090670615434647, -0.02546032704412937, 0.0880206748843193, 0.036246031522750854,
      0.04511220008134842, -0.024675052613019943, 0.0373770110309124, 0.08961974829435349,
      0.01391922403126955, -0.016442712396383286, 0.05498579889535904, -0.03670830279588699,
      -0.018114058300852776, 0.05251173675060272, 0.101175457239151, -0.06165322661399841,
      -0.025804007425904274, 0.10374286770820618, -0.01426684856414795, 0.10343196243047714,
      -0.060289740562438965, -0.012419101782143116, -0.06779750436544418, 0.04835706949234009,
      0.03398861363530159, -0.012057587504386902, -0.11166740208864212, 0.05767578259110451,
      -0.08217692375183105, 0.08340725302696228, 0.07387486845254898, -0.01278107799589634,
      0.04668355733156204, 0.001201303326524794, -0.05444236844778061, -0.10802119225263596,
      -0.10578345507383347, -0.12046420574188232, 0.12624825537204742, 0.125696063041687,
      -0.04361835494637489, -0.0850001722574234, -7.780292071402073e-05, -0.0766816958785057],
     [-0.03684648871421814, -0.0991796925663948, -0.06704667210578918, 0.04541710019111633,
      -0.10717273503541946, 0.005229703616350889, -0.1105131134390831, -0.06126009672880173,
      -0.083965003490448, -0.03214848414063454, -0.00798753835260868, -0.01095852255821228,
      -0.06487682461738586, 0.006995152682065964, 0.0852762907743454, 0.04346969351172447,
      0.050683584064245224, 0.02160617895424366, 0.13138191401958466, 0.09349054843187332,
      -0.012017841450870037, 0.11848333477973938, 0.09224076569080353, 0.00878024473786354,
      -0.04797706380486488, -0.041872087866067886, 0.032624948769807816, -0.05667866766452789,
      -0.09941170364618301, 0.019378338009119034, -0.03149189054965973, 0.04376688227057457,
      -0.061253905296325684, 0.054274678230285645, 0.05995681509375572, -0.0050756181590259075,
      -0.05552930384874344, 0.02230026200413704, 0.04246409609913826, 0.02580871433019638,
      0.10188555717468262, 0.033990953117609024, 0.10628583282232285, 0.10507722944021225,
      0.08172601461410522, -0.04660825431346893, 0.0010837602894753218, 0.022197963669896126,
      0.018718354403972626, 0.012339623644948006, 0.06815675646066666, 0.05838383734226227,
      -0.007122873794287443, -0.026823610067367554, 0.06716199219226837, -0.040805064141750336,
      -0.03321604058146477, -0.022812504321336746, 0.047299936413764954, 0.0892459973692894,
      -0.08061015605926514, 0.020574292168021202, 0.010565640404820442, -0.002933197421953082,
      0.10300273448228836, 0.05463346093893051, 0.03967803716659546, -0.042322490364313126,
      0.03038564883172512, -0.07763955742120743, -0.06440261006355286, 0.04866495728492737,
      0.07154703885316849, 0.03771454840898514, -0.014905870892107487, -0.03633018210530281,
      0.0763818696141243, 0.041587863117456436, 0.10156814008951187, 0.02024291269481182,
      0.03747587651014328, 0.00040730112232267857, 0.0410069078207016, -0.04472823068499565,
      0.05552281439304352, -0.02377474121749401, 0.032758813351392746, 0.048971667885780334,
      -0.03503381088376045, -0.0007240394479595125, 0.04230731725692749, -0.03326525166630745,
      0.01985391229391098, -0.056078072637319565, -0.0046220747753977776, 0.02479587309062481,
      -0.13201208412647247, -0.09807353466749191, 0.011148009449243546, -0.05286511033773422,
      -0.13499382138252258, -0.09503805637359619, -0.06035667285323143, -0.01948140189051628,
      -0.10221873968839645, -0.06190677732229233, -0.042905598878860474, -0.0121785132214427],
     [0.010194179601967335, 0.0797479972243309, 0.0863003060221672, -0.0038898864295333624,
      -0.06095663458108902, 0.048926059156656265, -0.08842688053846359, 0.07741668820381165,
      0.05777944251894951, 0.08913476765155792, -0.057375602424144745, -0.0627995952963829,
      0.07314927130937576, 0.020238611847162247, 0.006082913372665644, -0.08374669402837753,
      -0.006789480801671743, -0.0565849207341671, -0.03291955217719078, 0.10865550488233566,
      -0.04599103704094887, 0.048133302479982376, 0.08084512501955032, 0.0840899869799614,
      0.03932114690542221, 0.02350696548819542, -0.10301681607961655, -0.012340138666331768,
      -0.11595384776592255, 0.03394303843379021, 0.0024362634867429733, -0.04436898231506348,
      0.005659455433487892, -0.027572279796004295, -0.09514147788286209, 0.06408052891492844,
      -0.07427728176116943, -0.014479171484708786, 0.07695692032575607, -0.009336816146969795,
      0.008784467354416847, -0.09205954521894455, 0.02783115766942501, 0.019650723785161972,
      -0.04779694974422455, 0.09650371968746185, 0.05024950951337814, -0.011541971936821938,
      0.07976821810007095, -0.04746078699827194, 0.03826500475406647, 0.05750042572617531,
      0.07249943166971207, -0.001201495761051774, 0.09063959121704102, 0.10814779996871948,
      0.012992366217076778, -0.014544034376740456, -0.05027364194393158, -0.045288823544979095,
      -0.09211195260286331, 0.04178144782781601, 0.0050875041633844376, 0.06511763483285904,
      -0.05079633742570877, -0.07051432132720947, -0.04759113863110542, -0.06197637692093849,
      -0.06648847460746765, 0.056188955903053284, 0.0032588897738605738, 0.09607891738414764,
      0.08243998140096664, -0.05913400650024414, 0.09803812950849533, -0.043551452457904816,
      -0.017639340832829475, -0.01091304887086153, -0.07690540701150894, 0.09864237159490585,
      0.056398674845695496, -0.053948480635881424, 0.07906977087259293, -0.0742993876338005,
      0.09824756532907486, -0.041813112795352936, -0.021793190389871597, 0.04439358040690422,
      -0.07530830800533295, 0.03992091119289398, 0.009127115830779076, -0.0732153132557869,
      -0.018696008250117302, 0.04581036791205406, -0.1099340096116066, 0.0733053907752037,
      0.032468944787979126, 0.04012136533856392, 0.01179112121462822, -0.10694611817598343,
      -0.0013136385241523385, -0.06088602542877197, 0.04379268363118172, 0.019610801711678505,
      0.07804320752620697, -0.02419774793088436, 0.0708574429154396, 0.055708542466163635],
     [-0.031214531511068344, 0.007268627639859915, 0.028699304908514023, 0.016419697552919388,
      -0.1178155243396759, 0.007827448658645153, 0.07828781008720398, -0.022966019809246063,
      -0.023756779730319977, -0.10630127042531967, 0.03505994379520416, -0.04775412753224373,
      0.07220445573329926, -0.03611552342772484, 0.06884003430604935, -0.0059180897660553455,
      0.08002118021249771, 0.002971955807879567, -0.055058639496564865, -0.03798292204737663,
      0.07549961656332016, 0.03815028816461563, -0.04343352094292641, 0.04526159539818764,
      -0.030712004750967026, -0.041879426687955856, -0.014517547562718391, 0.00735853984951973,
      0.041043903678655624, 0.00871206447482109, -0.044340986758470535, 0.05219756439328194,
      -0.020692694932222366, 0.04655316844582558, -0.0647154450416565, -0.07112018018960953,
      0.001979395281523466, 0.07229752838611603, -0.05445237085223198, -0.0868058130145073,
      0.03600300848484039, 0.007885794155299664, -0.048063330352306366, 0.05214936286211014,
      -0.0559232160449028, 0.0691499337553978, -0.04937536641955376, 0.06970158964395523,
      -0.01860698126256466, 0.01623246818780899, 0.061015576124191284, 0.01615055464208126,
      0.06010560318827629, 0.062365397810935974, -0.041998445987701416, 0.04882315918803215,
      -0.0543716736137867, 0.10612010955810547, -0.04318644851446152, -0.030198147520422935,
      0.04153960570693016, 0.02573074959218502, -0.04310965538024902, 0.13987569510936737,
      0.054109398275613785, -0.017605729401111603, 0.08419332653284073, 0.09785698354244232,
      -0.038678817451000214, -0.06424377113580704, -0.015486671589314938, 0.09583543241024017,
      -0.05436631292104721, -0.08887649327516556, -0.03623267263174057, -0.04780614376068115,
      -0.04792680963873863, 0.04951651766896248, -0.09024252742528915, -0.07467548549175262,
      -0.07487630844116211, -0.006238081026822329, -0.02109641395509243, 0.06550019234418869,
      -0.04982849210500717, 0.06306758522987366, -0.04210389405488968, 0.02398749068379402,
      -0.09765774756669998, -0.037057943642139435, -0.03563963994383812, -0.05049721524119377,
      0.11439836770296097, -0.057780712842941284, 0.08321240544319153, 0.04853787273168564,
      -0.027241049334406853, -0.06236082687973976, 0.0797029510140419, 0.09385192394256592,
      0.03886156156659126, -0.03213212639093399, -0.05286266654729843, 0.044181615114212036,
      -0.07512462139129639, 0.002222832292318344, 0.07563653588294983, -0.011212443001568317],
     [-0.058409977704286575, -0.021139666438102722, 0.05904535949230194, 0.04995783045887947,
      0.013220109045505524, -0.028731143102049828, -0.05791677534580231, 0.0591244176030159,
      0.01072204764932394, 0.06472005695104599, -0.042782049626111984, 0.016519902274012566,
      0.0008725774823687971, -0.1043296754360199, -0.11260706186294556, -0.10400360077619553,
      -0.02444778010249138, -0.05935254693031311, 0.057448212057352066, 0.041664354503154755,
      0.03879127651453018, -0.12613710761070251, -0.10019700974225998, -0.06655897200107574,
      0.0359141007065773, 0.07218244671821594, 0.07152622193098068, 0.07036428898572922,
      -0.06553931534290314, 0.036716949194669724, 0.04597685858607292, -0.0700807198882103,
      -0.026754654943943024, -0.08877327293157578, 0.03753592446446419, 0.0032024693209677935,
      -0.06315378844738007, 0.05960351601243019, -0.0021913759410381317, -0.07811032235622406,
      -0.01881721243262291, -0.08293282240629196, 0.08947756886482239, 0.02586330659687519,
      0.060644857585430145, 0.04797865077853203, -0.02487318590283394, -0.03491973131895065,
      0.06210467964410782, -0.05687481537461281, -0.031066935509443283, -0.0570843443274498,
      0.0947016254067421, 0.08244817703962326, -0.047152336686849594, -0.061336640268564224,
      0.05582529678940773, 0.028171522542834282, 0.008674930781126022, -0.07576826959848404,
      0.047815922647714615, -0.04436720907688141, -0.09245423972606659, 0.04866233468055725,
      -0.09094615280628204, 0.029746251180768013, -0.05221674591302872, 0.03312245011329651,
      0.07558134198188782, -0.07785854488611221, 0.08760455995798111, 0.022625761106610298,
      0.0871381014585495, -0.018713150173425674, -0.006401102524250746, 0.03896549716591835,
      -0.05727233737707138, -0.06874866038560867, 0.05641782283782959, -0.026055580005049706,
      0.034887928515672684, 0.07874879240989685, 0.07899248600006104, 0.07929348945617676,
      -0.017636604607105255, -0.0008934941724874079, -0.01681671477854252, 0.11200057715177536,
      -0.12244346737861633, 0.07867132872343063, -0.005523063242435455, 0.08410602062940598,
      0.10583197325468063, 0.04445653781294823, -0.04501540958881378, 0.07123728096485138,
      -0.027711959555745125, 0.04546898975968361, 0.12610329687595367, -0.019490882754325867,
      -0.06436444073915482, 0.0757463201880455, -0.09854964911937714, 0.017655858770012856,
      -0.08654022216796875, -0.04353690519928932, -0.09118949621915817, 0.05775465816259384],
     [0.048005539923906326, 0.005818245466798544, -0.002111058682203293, 0.06352626532316208,
      -0.06637474149465561, -0.008096512407064438, 0.044434644281864166, 0.05806705728173256,
      0.07128677517175674, -0.05873454734683037, 0.015502545982599258, 0.0009087243815883994,
      0.0031881690956652164, 0.04050110653042793, -0.03696855157613754, -0.045041196048259735,
      0.0422314777970314, -0.01422264613211155, 0.11507440358400345, 0.11510318517684937,
      0.0661608949303627, 0.03051099367439747, -0.04472746327519417, 0.050296634435653687,
      0.0741235613822937, 0.05170236527919769, 0.0943864956498146, 0.0981435477733612,
      -0.02352937124669552, 0.05609016865491867, 0.06876347213983536, -0.02770882099866867,
      -0.02876005694270134, 0.07102297246456146, 0.0005004875711165369, -0.019177241250872612,
      -0.017633065581321716, -0.09176956117153168, 0.07136492431163788, -0.018318070098757744,
      0.005460487212985754, 0.08304239064455032, -0.0791100263595581, 0.054019831120967865,
      -0.03201975300908089, -0.05250925198197365, 0.06110147386789322, -0.05461100861430168,
      -0.08838534355163574, -0.013860387727618217, 0.06072079390287399, -0.025740398094058037,
      -0.04371963441371918, -0.08042504638433456, -0.025156965479254723, 0.00874368567019701,
      -0.05896200239658356, -0.03083980828523636, 0.08994293957948685, 0.01186367403715849,
      0.03359602391719818, 0.04110439866781235, -0.08913303166627884, -0.02981787919998169,
      -0.0533483624458313, 0.042945705354213715, 0.05606178939342499, -0.015135918743908405,
      -0.0221710167825222, 0.07684139907360077, 0.04484692960977554, 0.019675180315971375,
      0.09405224770307541, 0.041084546595811844, 0.07447399944067001, 0.025725893676280975,
      0.08611056208610535, 0.002210225909948349, -0.032577209174633026, -0.0413951501250267,
      -0.025669001042842865, 0.03350291773676872, 0.009282097220420837, -0.04094735532999039,
      -0.017643047496676445, -0.056884560734033585, 0.06439948081970215, 0.045516740530729294,
      0.0665040984749794, 0.014186305925250053, -0.09403184801340103, 0.03145263344049454,
      -0.03064691089093685, 0.01341405138373375, -0.07825803011655807, -0.04184604436159134,
      -0.03970055654644966, 0.014415639452636242, -0.09264223277568817, -0.04575560986995697,
      -0.022343555465340614, -0.06283653527498245, -0.06491683423519135, 0.07977402210235596,
      -0.0015017194673418999, 0.021060341969132423, -0.06957060098648071, 0.06769294291734695],
     [-0.03128504753112793, -0.0671427845954895, -0.0015338666271418333, -0.013355273753404617,
      -0.0655796006321907, -0.022492650896310806, 0.07247111946344376, 0.00635403161868453,
      -0.05464433506131172, 0.043329715728759766, -0.012600467540323734, -0.08584297448396683,
      0.04614149034023285, 0.003926984500139952, 0.07572511583566666, -0.007804025895893574,
      0.0644088163971901, -0.03416886553168297, -0.03551185131072998, 0.05425557121634483,
      0.1065359115600586, 0.11055559664964676, -0.048118479549884796, -0.0446837842464447,
      -0.055972859263420105, -0.07948336005210876, 0.011789137497544289, -0.023417674005031586,
      0.021246496587991714, -0.04561211168766022, -0.03837886080145836, 0.056362755596637726,
      -0.06404346972703934, 0.02400924265384674, -0.0126985227689147, 0.057611554861068726,
      -0.05333013832569122, -0.04972614347934723, -0.01334309857338667, -0.05916875973343849,
      0.07702547311782837, 0.11522417515516281, 0.03463307395577431, 0.03945303335785866,
      0.04941468685865402, -0.061628032475709915, -0.055906668305397034, -0.060656748712062836,
      0.04419950395822525, -0.05534927174448967, -0.04736148193478584, -0.07982158660888672,
      0.024097051471471786, -0.0366988442838192, 0.039797231554985046, -0.093015655875206,
      0.062207289040088654, 0.05656439810991287, -0.06856978684663773, 0.014679684303700924,
      0.06330753117799759, -0.0052494751289486885, -0.04228765144944191, 0.061344362795352936,
      0.05027548596262932, -0.06143815815448761, -0.008463852107524872, -0.05400623008608818,
      -0.03098522312939167, 0.03258508816361427, 0.03126446530222893, 0.023342402651906013,
      -0.0791388675570488, -0.06189240142703056, 0.025792153552174568, 0.08224568516016006,
      0.088690847158432, 0.0487155057489872, 0.07497967034578323, -0.05752837657928467,
      -0.06314797699451447, 0.07055987417697906, -0.09767191857099533, -0.021031199023127556,
      0.07872599363327026, 0.08270802348852158, 0.054371852427721024, -0.01660255528986454,
      -0.014166322536766529, -0.026675516739487648, -0.039548054337501526, -0.06268586218357086,
      -0.050195518881082535, -0.100877545773983, -0.13412457704544067, -0.1196068525314331,
      -0.05993996933102608, 0.05409115552902222, -0.07133615016937256, 0.029076196253299713,
      -0.11699226498603821, -0.15087999403476715, 0.016947010532021523, 0.09600021690130234,
      -0.07127192616462708, 0.07363787293434143, 0.08848206698894501, -0.07009290903806686],
     [-0.06380367279052734, -0.13816902041435242, -0.007258793339133263, -0.06836900860071182,
      -0.08869786560535431, -0.009118204936385155, -0.0993972048163414, -0.01903672143816948,
      -0.015132185071706772, -0.05960677191615105, -0.01862795278429985, 0.03637517988681793,
      0.07192827016115189, -0.021947180852293968, 0.05620236322283745, -0.013061613775789738,
      -0.007386294659227133, 0.0823223739862442, 0.03076450526714325, 0.0010572989704087377,
      0.04358309879899025, 0.15482939779758453, 0.13587826490402222, 0.1250496357679367,
      -0.009185491129755974, -0.02849007397890091, 0.004969348665326834, -0.028604216873645782,
      0.020895710214972496, -0.09162494540214539, -0.03130732476711273, -0.011081306263804436,
      0.034759800881147385, -0.02896459773182869, 0.008805761113762856, -0.01746291108429432,
      0.02310468815267086, -0.02960314229130745, -0.061071328818798065, -0.06817813217639923,
      0.03827233240008354, 0.012891042046248913, 0.00023667224741075188, -0.0856010690331459,
      0.028980888426303864, -0.06111428514122963, -0.038427386432886124, -0.05013767629861832,
      0.04482358694076538, 0.04839088022708893, -0.01899377815425396, 0.020168108865618706,
      -0.011328229680657387, 0.07711441814899445, 0.028086019679903984, -0.07641424983739853,
      0.021797947585582733, -0.006323807407170534, 0.10354071110486984, 0.029682347550988197,
      0.013620913028717041, -0.01996546983718872, 0.0683518722653389, 0.06268429011106491,
      0.07586175948381424, -0.04421915486454964, -0.04086054116487503, -0.004295469261705875,
      -0.07374204695224762, -0.009765489026904106, -0.013493851758539677, 0.03160780668258667,
      0.012907538563013077, 0.009138680063188076, -0.04140769690275192, -0.030587874352931976,
      0.01773916371166706, 0.06603541970252991, 0.05334891006350517, 0.09682814031839371,
      0.033469587564468384, -0.059805549681186676, 0.09843087941408157, -0.05809218809008598,
      0.07720067352056503, -0.04652997478842735, -0.06626143306493759, -0.0010590751189738512,
      0.039236538112163544, 0.03898585960268974, -0.08083844184875488, -0.0980299860239029,
      -0.13009457290172577, -0.08077538758516312, -0.08732448518276215, -0.12668296694755554,
      0.010749793611466885, -0.15296243131160736, -0.10679705440998077, -0.005111606325954199,
      0.001319167553447187, -0.058815084397792816, -0.08701586723327637, 0.10480904579162598,
      0.01011672429740429, 0.042689695954322815, 0.06563451141119003, 0.055928848683834076],
     [-0.006119123660027981, -0.08128651976585388, 0.006802405696362257, -0.03379852697253227,
      0.008421286940574646, 0.04522835835814476, -0.06374312192201614, -0.07298107445240021,
      0.04777207598090172, 0.06480158865451813, 0.002694162540137768, -0.04115622118115425,
      0.04667633771896362, -0.05143749713897705, 0.11274823546409607, 0.06729664653539658,
      0.1193438321352005, -0.07436015456914902, -0.06215403228998184, -0.04615860432386398,
      -0.03457704931497574, -0.08064847439527512, -0.022823220118880272, -0.01337716355919838,
      0.09077481925487518, -0.015625325962901115, -0.07291305810213089, -0.05731210857629776,
      -0.0712670385837555, 0.055361077189445496, 0.0024329996667802334, -0.08477923274040222,
      0.00015351576439570636, 0.08243656158447266, 0.04620371013879776, 0.09445638209581375,
      -0.059604328125715256, 0.05068163573741913, -0.094085693359375, -0.07103588432073593,
      -0.03198588266968727, -0.09573957324028015, 0.08011947572231293, 0.06315908581018448,
      -0.002575264312326908, 0.013806444592773914, 0.050361618399620056, -0.005298379808664322,
      -0.05453357845544815, -0.0640689805150032, -0.06237240880727768, 0.018373141065239906,
      0.0016435729339718819, -0.05165835842490196, -0.04955381155014038, 0.08166558295488358,
      0.024236053228378296, -0.06368685513734818, 0.051504138857126236, -0.06720080226659775,
      -0.029682205989956856, -0.020405100658535957, -0.0038173107896000147, 0.002524609211832285,
      -0.029834797605872154, 0.03360314667224884, 0.04501790180802345, -0.017605986446142197,
      0.03964938968420029, -0.04877062514424324, -0.028418833389878273, -0.048480208963155746,
      -0.005583261605352163, -0.00870346836745739, 0.007831568829715252, -0.0841781497001648,
      0.07111766189336777, -0.0102301724255085, -0.04031679779291153, 0.007848418317735195,
      0.029921939596533775, 0.04588612541556358, -0.03441116213798523, -0.056333716958761215,
      -0.07932262122631073, 0.06628914177417755, -0.09659449756145477, 0.04555216431617737,
      -0.01839655265212059, -0.019648076966404915, -0.09692896157503128, -0.05633541941642761,
      -0.06932234019041061, 0.0029239223804324865, 0.09117257595062256, -0.05904243513941765,
      0.008041052147746086, -0.02217232994735241, -0.03214061260223389, -0.06760004907846451,
      0.0774579718708992, -0.06333237886428833, 0.03870674967765808, -0.09686820954084396,
      0.07400690019130707, -0.012457525357604027, -0.08393840491771698, 0.1011219248175621],
     [-0.09579014778137207, -0.0943465456366539, -0.11082382500171661, -0.12651611864566803,
      0.036236487329006195, -0.027994846925139427, 0.026839278638362885, 0.05542825162410736,
      -0.10492017865180969, -0.08950303494930267, -0.07837608456611633, -0.1090771034359932,
      0.09233307093381882, -0.00794905424118042, 0.051555026322603226, 0.0904085710644722,
      0.059555213898420334, -0.03360773250460625, 0.11295723915100098, 0.04621676355600357,
      0.1504095196723938, -0.007516095414757729, 0.0416487380862236, 0.027432415634393692,
      0.03100917488336563, -0.09682115912437439, -0.0850674957036972, 0.012720873579382896,
      0.06347087770700455, 0.0734047144651413, -0.004298393614590168, -0.09926112741231918,
      -0.07293490320444107, 0.054331645369529724, 0.03988407552242279, -0.04242226853966713,
      0.024001440033316612, 0.09905312955379486, 0.05155631899833679, -0.02597069926559925,
      0.010193669237196445, 0.04504205659031868, 0.08215072005987167, 0.08078952878713608,
      0.04752664640545845, 0.0542156957089901, -0.011959556490182877, 0.02123405411839485,
      -0.0875229462981224, 0.0608309805393219, 0.06550199538469315, -0.023745350539684296,
      0.04739302769303322, -0.0041949027217924595, -0.07988350093364716, 0.07494892925024033,
      -0.017002729699015617, -0.004928335081785917, 0.028007173910737038, 0.0024324983824044466,
      -0.07977277040481567, -0.03387204930186272, -0.07995487749576569, -0.023256151005625725,
      -0.013927042484283447, -0.0367002859711647, 0.08282997459173203, -0.008403263986110687,
      -0.05636484548449516, -0.07100031524896622, 0.06705716252326965, 0.08857603371143341,
      0.0942218005657196, 0.0012657264014706016, -0.02990877442061901, 0.02634568139910698,
      -0.029477732256054878, -0.086161769926548, -0.03858602046966553, -0.0755368322134018,
      -0.06837201118469238, 0.000572612916585058, 0.010004846379160881, 0.04272386431694031,
      -0.07638534158468246, 0.023159973323345184, 0.0840301513671875, 0.08540484309196472,
      0.08911234140396118, -0.054209690541028976, -0.013649898581206799, -0.09013576805591583,
      -0.020197732374072075, -0.1316075474023819, -0.08481866866350174, -0.0283485259860754,
      0.015058293007314205, -0.12426242977380753, -0.1449393928050995, -0.0357787162065506,
      -0.07295489311218262, -0.031022556126117706, 0.103545181453228, 0.015051420778036118,
      -0.017608340829610825, 0.09179675579071045, 0.03131634742021561, -0.061837244778871536],
     [-0.10839410871267319, -0.005716058425605297, -0.13667045533657074, 0.00563607644289732,
      -0.0934557244181633, -0.03871475160121918, -0.02940419316291809, -0.10264017432928085,
      -0.11258844286203384, -0.12278176099061966, 0.011509840376675129, 0.02090914361178875,
      0.055854637175798416, 0.11442280560731888, -0.011379240080714226, -0.03945353254675865,
      -0.04412505775690079, -0.04557473585009575, 0.13486652076244354, 0.05049517750740051,
      0.16161875426769257, 0.06810637563467026, 0.001791241578757763, 0.04290539026260376,
      0.04659242182970047, 0.05053167790174484, 0.11401442438364029, 0.03015400469303131,
      -0.05755307152867317, 0.11213474720716476, 0.01530762854963541, -0.09323486685752869,
      -0.037340857088565826, 0.02292536571621895, -0.07938903570175171, -0.03189823031425476,
      0.04626882076263428, 0.1335771232843399, -0.022678343579173088, 0.033001482486724854,
      -0.009674222208559513, 0.05336304008960724, -0.015171029604971409, 0.06638982146978378,
      0.0997324287891388, -0.10145621746778488, -0.02984049916267395, 0.11424355208873749,
      -0.026338940486311913, -0.04659043252468109, -0.027046093717217445, 0.0688699260354042,
      0.11531716585159302, 0.045634523034095764, -0.0046747722662985325, 0.0033401125110685825,
      0.07960467785596848, 0.0980905070900917, 0.09700478613376617, 0.06871282309293747,
      -0.0027384301647543907, 0.023072076961398125, 0.08527044206857681, -0.014693138189613819,
      0.07866465300321579, -0.002253200626000762, -0.06633565574884415, -0.004167919047176838,
      0.006624115165323019, 0.034387946128845215, -0.02247738279402256, 0.041601214557886124,
      0.02929995022714138, -0.06757362931966782, 0.04464148357510567, 0.0844256654381752,
      0.003520333906635642, 0.08032336086034775, -0.001602581120096147, -0.032185573130846024,
      -0.03963571414351463, 0.08470788598060608, 0.00790529977530241, -0.02795613557100296,
      0.0061170682311058044, -0.04814457893371582, -0.02035992220044136, 0.05965558812022209,
      -0.0148420175537467, -0.0186997689306736, 0.022516021504998207, -0.1058603897690773,
      -0.10122322291135788, -0.0021366707514971495, -0.037068143486976624, 0.005801461637020111,
      -0.08740062266588211, -0.03565201163291931, -0.1057954803109169, -0.06370232254266739,
      -0.009212007746100426, -0.12290016561746597, 0.005373646505177021, 0.128545343875885,
      -0.031560178846120834, 0.08287010341882706, 0.06425043195486069, 0.08543207496404648],
     [-0.0007451922865584493, 0.03328823670744896, -0.0983492061495781, -0.08353693038225174,
      0.06992980092763901, 0.041114479303359985, 0.04382742568850517, 0.03823278099298477,
      -0.0901048481464386, 0.046619515866041183, -0.047592367976903915, -0.041410163044929504,
      -0.04014774411916733, -0.008060690946877003, 0.07435016334056854, 0.0007821016479283571,
      0.046450331807136536, 0.0792231336236, -0.08926013112068176, -0.08464367687702179,
      0.03585810959339142, 0.012179305776953697, 0.08185098320245743, -0.005707925651222467,
      -0.06151203066110611, 0.08822783082723618, 0.0019361189333721995, -0.05247081443667412,
      0.1020766869187355, -0.034984756261110306, -0.07492928951978683, 0.0693100169301033,
      0.08478810638189316, -0.08535458147525787, 0.1008860394358635, 0.008702265098690987,
      0.01545116025954485, 0.0359097383916378, 0.08080106973648071, 0.056556545197963715,
      0.015156546607613564, -0.020143525674939156, -0.02317434921860695, 0.06328801810741425,
      -0.022993121296167374, 0.01470092125236988, -0.005909864325076342, 0.03603121638298035,
      -0.05523499473929405, 0.0493810661137104, 0.03846132755279541, 0.08079694956541061,
      -0.0725359320640564, 0.012710698880255222, 0.05906379222869873, 0.03285261616110802,
      0.03287750110030174, 0.014179622754454613, -0.06170462816953659, 0.0016298829577863216,
      0.0251237154006958, 0.04187368229031563, 0.03218571096658707, -0.09081713855266571,
      0.03952476382255554, -0.0431005135178566, 0.07109370827674866, 0.050883643329143524,
      -0.08324053138494492, -0.04743024706840515, -0.059918615967035294, 0.10475204885005951,
      -0.07226121425628662, 0.04166446626186371, 0.09661109000444412, 0.01418650709092617,
      0.004951436538249254, 0.06994523108005524, 0.08911049365997314, -0.04118851572275162,
      -0.05151494964957237, 0.05319615453481674, -0.04261506721377373, -0.0017387637635692954,
      0.04601883143186569, 0.026730192825198174, 0.021036135032773018, 0.053057316690683365,
      0.09428747743368149, 0.0672631487250328, -0.09395329654216766, 0.08316092193126678,
      0.09745089709758759, -0.09565915167331696, -0.021940292790532112, -0.018913304433226585,
      -0.06646183133125305, -0.003115616273134947, 0.020333023741841316, -0.09753341972827911,
      -0.05723083019256592, 0.0494021400809288, -0.05763324722647667, -0.10398752242326736,
      -0.06580987572669983, 0.027900805696845055, 0.03294840455055237, 0.040108680725097656],
     [-0.03713695704936981, -0.09480725973844528, -0.1211121529340744, 0.030892355367541313,
      -0.005888503976166248, -0.039008524268865585, -0.05801286920905113, -0.09896041452884674,
      -0.05751617252826691, 0.0638977587223053, -0.10757118463516235, -0.03693975508213043,
      0.05414383485913277, 0.04463580623269081, -0.04383910074830055, 0.0890946239233017,
      -0.041118793189525604, 0.04732202738523483, 0.0061977319419384, 0.14273226261138916,
      0.06014344096183777, 0.061963725835084915, 0.00433989567682147, 0.09336626529693604,
      0.02573791705071926, 0.10451368242502213, -0.04511120542883873, -0.0030498492997139692,
      0.03055380843579769, 0.02799425832927227, -0.018027251586318016, -0.0025359303690493107,
      -0.05789255350828171, -0.12002792954444885, 0.003461152780801058, 0.03385566920042038,
      0.07464636117219925, 0.06110440939664841, 0.07386134564876556, 0.01092072669416666,
      0.009410678409039974, 0.08191966265439987, 0.05292299762368202, 0.0006351290503516793,
      0.05647427961230278, 0.00033099966822192073, 0.06961791217327118, 0.07982870936393738,
      -0.07548719644546509, -0.06345554441213608, 0.10546820610761642, 0.07826577126979828,
      0.0288372915238142, -0.026145149022340775, 0.052099891006946564, 0.014726029708981514,
      -0.05472967401146889, 0.06373035162687302, 0.06295706331729889, -0.02721346542239189,
      -0.01475945021957159, 0.021108685061335564, 0.0325152762234211, 0.05012262985110283,
      -0.06097976118326187, 0.02887078933417797, -0.06445013731718063, -0.07955513149499893,
      -0.030140656977891922, -0.07057224214076996, 0.05985593423247337, 0.06640929728746414,
      -0.04214378073811531, -0.08141206204891205, -0.056106992065906525, 0.036004163324832916,
      -0.04217394441366196, 0.03073132410645485, 0.06999900937080383, 0.07668566703796387,
      0.036053504794836044, 0.06291855126619339, -0.07662192732095718, 0.003085323376581073,
      0.08853746950626373, -0.048521675169467926, -0.06313822418451309, -0.08683998882770538,
      0.0017994327936321497, 0.03438199311494827, -0.1480431705713272, -0.037392862141132355,
      -0.059365686029195786, 0.03306354209780693, -0.06666567176580429, 0.0044512394815683365,
      -0.09481252729892731, -0.03316299617290497, -0.041943278163671494, -0.06682014465332031,
      -0.053067781031131744, 0.014583278447389603, 0.005455655045807362, 0.012649350799620152,
      -0.06842587143182755, -0.05004454776644707, 0.06243709847331047, -0.06038661673665047],
     [0.027396729215979576, 0.0243200846016407, 0.07258699834346771, 0.09629634022712708,
      0.03859987482428551, 0.0973879024386406, 0.01695035770535469, 0.06617997586727142,
      0.07801977545022964, 0.010283912532031536, -0.057163745164871216, 0.08196920901536942,
      0.027396421879529953, -0.05164067819714546, 0.061364736407995224, -0.042891521006822586,
      -0.0419088751077652, -0.07306147366762161, 0.0182945616543293, 0.042103976011276245,
      0.07865922898054123, 0.05725780501961708, 0.10468414425849915, 0.011933826841413975,
      0.05084551125764847, 0.05981893837451935, -0.07848457247018814, -0.05832146853208542,
      -0.058037955313920975, 0.05379059538245201, -0.009662569500505924, -0.035172462463378906,
      0.0014497792581096292, -0.10844558477401733, -0.0365876704454422, 0.02256309799849987,
      -0.004725190810859203, -0.08159537613391876, 0.04661281779408455, 0.03915712237358093,
      -0.05363212525844574, -0.0868934839963913, 0.02716401405632496, 0.0043184817768633366,
      0.08249824494123459, 0.02563190460205078, 0.09572134912014008, 0.02749061957001686,
      -0.0027527306228876114, 0.0577193908393383, -0.01712310127913952, 0.011922636069357395,
      -0.06745088845491409, 0.059625595808029175, 0.02892698161303997, -0.05343855172395706,
      -0.02834175154566765, -0.07124456018209457, -0.05796860158443451, 0.021337028592824936,
      0.10953254252672195, -0.10623474419116974, 0.0024059887509793043, -0.1054302230477333,
      -0.031341344118118286, 0.032367877662181854, 0.06226448342204094, 0.028352024033665657,
      -0.05841871723532677, -0.027664700523018837, 0.06266177445650101, -0.016974974423646927,
      0.01349480077624321, -0.06079607084393501, 0.06335549801588058, -0.0026754175778478384,
      -0.018775006756186485, 0.03373485431075096, -0.07355580478906631, -0.06352423131465912,
      -0.007208654657006264, -0.007369880564510822, -0.028882039710879326, -0.0837382972240448,
      -0.10499486327171326, -0.013349844142794609, -0.07026030868291855, -0.03400168567895889,
      -0.003831342561170459, 0.09271960705518723, 0.014916318468749523, 0.03913756459951401,
      -0.125191792845726, 0.020063284784555435, -0.10188474506139755, 0.006456897594034672,
      -0.11947526037693024, -0.02512887306511402, -0.15031157433986664, -0.08630784600973129,
      -0.0526704303920269, -0.05761285871267319, -0.014133662916719913, -0.06939315050840378,
      0.0019258527318015695, -0.026882316917181015, 0.08453680574893951, 0.07550906389951706],
     [-0.03130386024713516, 0.05175630748271942, 0.012139406986534595, 0.032187320291996,
      -0.08035281300544739, -0.037182148545980453, -0.03711280599236488, 0.061421144753694534,
      0.04042421653866768, 0.0584867037832737, -0.06778071075677872, -0.022616609930992126,
      0.07033766806125641, 0.09543690085411072, 0.012066338211297989, 0.06154951453208923,
      0.006992380600422621, 0.052376385778188705, -0.023084361106157303, 0.04781809821724892,
      -0.03579273447394371, -3.60608974006027e-05, -0.04243950545787811, -0.09052686393260956,
      -0.0238143689930439, 0.019581444561481476, 0.06369952112436295, -0.060527119785547256,
      -0.033760253340005875, 0.011910486966371536, -0.01976894587278366, 0.06014668941497803,
      -0.07842663675546646, -0.0654979720711708, -0.00479365698993206, 0.08198030292987823,
      0.07259033620357513, 0.07598832249641418, -0.070513516664505, 0.007803173735737801,
      -0.10300533473491669, 0.030683498829603195, -0.05876047536730766, 0.0841595008969307,
      -0.07253491133451462, 0.024319251999258995, -0.0463067926466465, -0.011704267002642155,
      -0.037506114691495895, -0.031094763427972794, 0.015759004279971123, 0.048740848898887634,
      0.007907653227448463, 0.08084584027528763, 0.013322600163519382, 0.03405514732003212,
      -0.027958540245890617, -0.049338266253471375, -0.049487438052892685, 0.04521547630429268,
      -0.043425302952528, -0.07654087245464325, -0.01557448785752058, 0.024237636476755142,
      0.013379225507378578, 0.021938655525445938, -0.0018366083968430758, 0.056717321276664734,
      0.07805825024843216, 0.0777154341340065, 0.03198961913585663, 0.08974776417016983,
      0.008249393664300442, 0.05277284234762192, -0.04491213336586952, 0.09203991293907166,
      0.02489042654633522, 0.004487906116992235, 0.06490588188171387, 0.014046130701899529,
      -0.03675999492406845, 0.06171062961220741, -0.04376513138413429, 0.02665630541741848,
      0.031013531610369682, -0.048041731119155884, 0.05236951634287834, 0.09533960372209549,
      -0.015119647607207298, 0.09258191287517548, -0.07706037908792496, 0.01229801680892706,
      0.04372062906622887, -0.04894734174013138, -0.06482566148042679, -0.05839526280760765,
      -0.1055256575345993, -0.001946888049133122, -0.10292986780405045, 0.04938257113099098,
      -0.07674051076173782, -0.012080773711204529, -0.04404548183083534, 0.07090010493993759,
      0.03481285274028778, 0.05767057463526726, 0.06774086505174637, -0.00407826155424118],
     [-0.08694925159215927, -0.12285327166318893, -0.06544586271047592, 0.045332733541727066,
      -0.0324944406747818, -0.05767960101366043, -0.05033634975552559, 0.01630985550582409,
      0.06034744158387184, -0.07677821069955826, 0.06998094171285629, -0.08733637630939484,
      0.0015227788826450706, 0.06967486441135406, -0.04448515549302101, -0.029762543737888336,
      -0.07315965741872787, 0.038809772580862045, -0.03396940231323242, 0.08013596385717392,
      0.07365889847278595, 0.09999743849039078, 0.04105683043599129, -0.0099297184497118,
      -0.0757395550608635, -0.05138826370239258, -0.010475054383277893, 0.05196559429168701,
      0.02520167827606201, -0.06928107887506485, 0.08110740035772324, 0.01986946538090706,
      -0.052909284830093384, -0.011634032242000103, -0.013923936523497105, 0.02074495516717434,
      -0.011576223187148571, 0.0672374740242958, -0.05410585179924965, -0.08345059305429459,
      -0.024493273347616196, 0.08972996473312378, -0.027529258280992508, 0.05652384087443352,
      0.03788040578365326, -0.06482630968093872, 0.014990507625043392, -0.06971725821495056,
      -0.005044697783887386, -0.02103525772690773, -0.01080809161067009, -0.022567329928278923,
      -0.011398072354495525, 0.036499593406915665, -0.01714986562728882, 0.05317341536283493,
      0.06351223587989807, -0.01412453968077898, 0.08466145396232605, -0.0063999369740486145,
      0.0056814709678292274, -0.013024779036641121, -0.068463034927845, -0.024967307224869728,
      -0.0752626284956932, 0.03144051134586334, -0.0026795396115630865, 0.0409977026283741,
      -0.0753851979970932, 0.040274668484926224, -0.07435572147369385, 0.04696110635995865,
      -0.032011695206165314, -0.0006902266759425402, 0.09535276144742966, 0.012148489244282246,
      -0.07255440950393677, -0.01070617139339447, 0.0314633771777153, -0.0781342163681984,
      -0.030125882476568222, 0.024111350998282433, -0.009360972791910172, 0.06754125654697418,
      -0.06402561813592911, -0.004724219441413879, 0.05646719038486481, 0.07459846884012222,
      -0.05950097367167473, -0.04438238590955734, 0.010307244025170803, 0.05341283604502678,
      -0.12628506124019623, 0.033064380288124084, -0.011166156269609928, 0.024642687290906906,
      -0.07700847834348679, -0.11468744277954102, -0.04704597219824791, -0.08854218572378159,
      -0.06980009377002716, -0.10248623788356781, 0.03864102438092232, -0.04970548301935196,
      -0.045002009719610214, 0.07903780788183212, 0.0246548093855381, -0.036922696977853775],
     [-0.04314279183745384, -0.039631668478250504, -0.13709789514541626, -0.011102467775344849,
      -0.11358699947595596, -0.06510154902935028, -0.019363196566700935, -0.01618097722530365,
      -0.10887592285871506, -0.03987378254532814, -0.03178190439939499, -0.09641683101654053,
      0.05355396494269371, -0.032509271055459976, 0.09611029922962189, -0.03232318162918091,
      -0.10873524099588394, -0.029453905299305916, 0.1251797080039978, 0.09988334774971008,
      0.002329959301277995, 0.0865168422460556, -0.011432109400629997, -0.03519728034734726,
      -0.07495039701461792, -0.025461917743086815, 0.040859173983335495, -0.06896624714136124,
      -0.0747184306383133, 0.03126033768057823, 0.020993327721953392, -0.04645397514104843,
      0.04955698549747467, -0.05484502390027046, -0.03866869956254959, -0.05392463877797127,
      0.025404561311006546, 0.12186528742313385, -0.0024020352866500616, 0.017355697229504585,
      0.06268122047185898, 0.060663413256406784, -0.05631613731384277, -0.02893134392797947,
      -0.017822669818997383, -0.01825033873319626, 0.09080474823713303, 0.08117996156215668,
      -0.01318302284926176, 0.06988570094108582, -0.04959909990429878, 0.004514352884143591,
      0.019086629152297974, 0.05383538454771042, -0.06804623454809189, 0.015079892240464687,
      0.0605512335896492, -0.07209763675928116, -0.04931163415312767, 0.0004173942725174129,
      -0.10632599145174026, -0.03364235907793045, -0.040133655071258545, 0.031840454787015915,
      0.05654532462358475, -0.05514024198055267, -0.011763953603804111, -0.0727686807513237,
      0.010760637000203133, -0.08640607446432114, 0.07043202966451645, 0.08701958507299423,
      -0.03974473103880882, 0.03762645274400711, 0.08748108148574829, 0.06562557816505432,
      0.06552188843488693, -0.021130511537194252, -0.026994623243808746, 0.06397397071123123,
      0.11195410788059235, 0.009620980359613895, -0.024820908904075623, -0.06462118774652481,
      0.010307512246072292, 0.007687246892601252, 0.040271785110235214, 0.0800226554274559,
      0.03750447928905487, 0.010268262587487698, 0.05241928994655609, 0.03216830641031265,
      -0.1051926389336586, -0.09186404943466187, -0.09327820688486099, -0.04370512440800667,
      -0.0011398326605558395, -0.011284619569778442, -0.0014851521700620651, -0.07160618901252747,
      0.003581801662221551, 0.0258994922041893, 0.05215064436197281, 0.0465521514415741,
      0.026522377505898476, -0.03021506778895855, 0.021796472370624542, 0.05867873504757881],
     [0.07296309620141983, -0.018908848986029625, -0.033399589359760284, 0.07774766534566879,
      -0.021569538861513138, 0.030933422967791557, 0.03101813793182373, -0.10207495093345642,
      -0.09193253517150879, -0.0934438407421112, -0.07521780580282211, -0.043249454349279404,
      -0.07246822863817215, -0.06884589791297913, 0.07299020886421204, -0.10159777849912643,
      -0.05290215462446213, 0.04773922637104988, -0.06044594198465347, 0.03349445387721062,
      -0.01650192402303219, -0.03196076303720474, 0.1150389164686203, -0.030030660331249237,
      0.00278498325496912, 0.037450313568115234, 0.04845033586025238, 0.028885861858725548,
      0.025662165135145187, -0.004454098641872406, 0.00839113350957632, 0.07258494198322296,
      0.09118324518203735, 0.0560908317565918, -0.08270479738712311, 0.06422116607427597,
      0.004123707767575979, 0.01867627166211605, 0.011804500594735146, -0.011388287879526615,
      -0.07368291169404984, -0.044341620057821274, 0.0484003908932209, -0.03522772341966629,
      0.09397516399621964, 0.023318367078900337, -0.02254057303071022, -0.029004696756601334,
      0.0547349713742733, 0.040969911962747574, -0.07403009384870529, -0.012459331192076206,
      0.055628012865781784, 0.007751191966235638, -0.012560057453811169, -0.025916941463947296,
      0.01620286889374256, 0.042843129485845566, 0.03513966128230095, 0.09287964552640915,
      -0.052090175449848175, -0.035012200474739075, -0.05285540968179703, -0.03633597865700722,
      0.06526853144168854, 0.08064384013414383, -0.0022515738382935524, 0.06455031037330627,
      0.10302115976810455, 0.0747879147529602, -0.024208297953009605, 0.011638419702649117,
      -0.05594504624605179, -0.033131204545497894, 0.012254701927304268, -0.048165593296289444,
      -0.06974419206380844, 0.01687491498887539, 0.08720088750123978, 0.00975851621478796,
      -0.08089916408061981, 0.0879579707980156, 0.05517853423953056, -0.04361974447965622,
      0.09238947182893753, 0.043608106672763824, 0.08313647657632828, 0.0033664165530353785,
      0.0153351416811347, 0.010891441255807877, -0.05298372358083725, 0.009541674517095089,
      0.01973939873278141, 0.0689837709069252, 0.023911498486995697, -0.03646570444107056,
      0.02496899850666523, -0.08265286684036255, 0.052498314529657364, -0.037767961621284485,
      -0.04358363524079323, 0.039617542177438736, 0.06339964270591736, -0.05743441730737686,
      0.05311284586787224, -0.045585621148347855, 0.02754155732691288, -0.03690740466117859],
     [-0.08519081026315689, -0.14457036554813385, -0.0010030101984739304, -0.03465063124895096,
      -0.0001264085731236264, -0.08863086998462677, -0.05974356085062027, -0.05041291564702988,
      -0.09798230230808258, -0.06526677310466766, -0.02011510543525219, -0.09998258203268051,
      -0.04463278874754906, -0.007416508626192808, 0.11196278035640717, 0.09380488842725754,
      0.01159208919852972, 0.10769356787204742, 0.12080249190330505, 0.09263350814580917,
      0.056377846747636795, 0.08401231467723846, 0.08963464200496674, 0.14089050889015198,
      -0.08532245457172394, 0.07397879660129547, -0.08064311742782593, 0.043767645955085754,
      0.022998711094260216, 0.049367040395736694, -0.05843240022659302, 0.04517048969864845,
      -0.01756647787988186, -0.017305336892604828, 0.06595099717378616, 0.07194218784570694,
      0.01493113860487938, -0.03546690568327904, 0.07033431529998779, -0.016180813312530518,
      0.04594923183321953, -0.05449192225933075, -0.07612445950508118, 0.040837302803993225,
      0.06687232851982117, -0.07926809787750244, 0.03591843321919441, 0.07807917147874832,
      -0.04484415054321289, -0.02388712577521801, -0.02838335558772087, -0.0672515332698822,
      0.02350633218884468, -0.020667098462581635, 0.04743523895740509, -0.05866025760769844,
      0.0949551910161972, 0.059789299964904785, -0.06350277364253998, 0.04513629153370857,
      -0.06262017041444778, -0.06809171289205551, -0.07677653431892395, -0.0541335828602314,
      -0.00302209728397429, 0.04151986166834831, -0.03885692358016968, 0.05426786094903946,
      0.06823572516441345, -0.06887564063072205, -0.058730337768793106, -0.08874548226594925,
      0.026636354625225067, -0.027790414169430733, -0.033375199884176254, 0.03146853670477867,
      0.001008721417747438, 0.06903839111328125, -0.03967420756816864, -0.04654430225491524,
      -0.061072446405887604, -0.0009406312019564211, -0.07892891764640808, -0.01357663981616497,
      0.016189470887184143, -0.09099908918142319, 0.046113643795251846, 0.0015046485932543874,
      -0.03926631063222885, -0.0022950535640120506, 0.0016012298874557018, -0.0192873477935791,
      0.028700554743409157, 0.036328885704278946, -0.0664113461971283, -0.12793461978435516,
      0.006389020476490259, 0.0033779528457671404, 0.0020216209813952446, 0.025818923488259315,
      -0.02304891124367714, -0.033842649310827255, 0.0685962364077568, -0.02868492342531681,
      -0.04576018825173378, 0.0005306519451551139, 0.02922365441918373, -0.06864580512046814],
     [0.08228703588247299, 0.10301259160041809, 0.0010487432591617107, 0.10433066636323929,
      0.11172494292259216, 0.07389714568853378, -0.0007177114603109658, 0.05473216623067856,
      -0.08418139815330505, 0.024724040180444717, 0.05259288474917412, 0.046419017016887665,
      0.022001497447490692, -0.019366752356290817, -0.05986717715859413, -0.03628193214535713,
      -0.0071886698715388775, -0.019641349092125893, 0.07290109246969223, 0.04125640541315079,
      -0.046911075711250305, 0.10683464258909225, 0.10193835943937302, 0.022173143923282623,
      -0.0005360858631320298, 0.05630796030163765, -0.004211667459458113, -0.06416766345500946,
      0.03278245031833649, -0.07005441188812256, 0.004393856972455978, 0.023489585146307945,
      -0.05581636354327202, -0.11870884150266647, -0.014628617092967033, -0.024678070098161697,
      -0.05822690576314926, -0.02274511754512787, 0.09618613123893738, -0.026372866705060005,
      0.06721174716949463, 0.03943684697151184, 0.018389098346233368, 0.07231248915195465,
      0.035789959132671356, 0.0747356191277504, 0.07371409237384796, 0.009214074350893497,
      0.01591063290834427, 0.02278762124478817, 0.052408721297979355, -0.06413369625806808,
      0.0547395758330822, 0.013645395636558533, 0.08026579767465591, -0.031727440655231476,
      -0.050361741334199905, -0.08182213455438614, -0.002038927050307393, -0.010976159945130348,
      0.062467820942401886, 0.015337531454861164, 0.006271054036915302, -0.044757891446352005,
      -0.0685320720076561, 0.08949355036020279, 0.011948123574256897, 0.06061285734176636,
      0.034006040543317795, 0.07706023007631302, 0.07126761227846146, 0.03274806588888168,
      0.04352257028222084, 0.0428478866815567, 0.012232061475515366, 0.0210682675242424,
      -0.05464411899447441, 0.06807765364646912, 0.007342413067817688, -0.0703657791018486,
      0.040397826582193375, 0.07346274703741074, 0.03827623277902603, 0.016005365177989006,
      -0.059315282851457596, 0.04131694510579109, -0.0637369304895401, 0.029956473037600517,
      -0.03534134849905968, -0.024081198498606682, 0.003637881949543953, 0.1057078167796135,
      0.03361286595463753, 0.003915709443390369, -0.0025259375106543303, 0.09049037098884583,
      -0.08207972347736359, -0.020576346665620804, -0.12924347817897797, -0.075784832239151,
      0.004141378682106733, 0.01669090986251831, 0.016188664361834526, -0.08514364063739777,
      0.07564874738454819, -0.0779251977801323, -0.005350710358470678, 0.017227647826075554],
     [-0.04715248942375183, 0.06132642552256584, -0.04481753334403038, 0.07724378257989883,
      0.11352887004613876, -0.04019106552004814, 0.06944175809621811, -0.06273860484361649,
      -0.078968845307827, 0.07151813060045242, 0.07994185388088226, -0.07419322431087494,
      0.06653231382369995, 0.03632621094584465, -0.06687623262405396, -0.09720465540885925,
      -0.0838264599442482, 0.07611172646284103, 0.06309942901134491, -0.0738053023815155,
      -0.00647792499512434, 0.016851074993610382, -0.06086573749780655, 0.0016962994122877717,
      0.06724496185779572, 0.014244701713323593, 0.038008663803339005, 0.0022885960061103106,
      -0.08555637300014496, -0.043241459876298904, -0.10479432344436646, -0.05738716945052147,
      0.059206098318099976, -0.08680001646280289, -0.09131426364183426, -0.03804786875844002,
      0.05153467133641243, -0.00996424537152052, 0.05911923944950104, -0.027725255116820335,
      -0.010051759891211987, 0.059395864605903625, 0.06517461687326431, -0.044154200702905655,
      -0.00973296258598566, -0.006305072922259569, 0.06467734277248383, -0.0768449679017067,
      -0.022842789068818092, 0.02303350903093815, -0.013239869847893715, 0.08879552036523819,
      0.005775676108896732, 0.0030217147432267666, -0.08275969326496124, 0.008397084660828114,
      -0.031665824353694916, 0.10343760251998901, 0.023273149505257607, -0.003664169693365693,
      -0.07320895045995712, 0.07628800719976425, -0.050047867000103, -0.05590386316180229,
      -0.024781202897429466, 0.08364038914442062, -0.011906055733561516, 0.01810368150472641,
      0.011801605112850666, 0.017477428540587425, -0.00976108480244875, 0.022412927821278572,
      -0.010299966670572758, 0.030946411192417145, 0.03621446341276169, -0.004644505213946104,
      0.07290738075971603, 0.05384024605154991, -0.013988668099045753, 0.018374206498265266,
      0.00028017949080094695, -0.08412536233663559, 0.10358768701553345, -0.05076458305120468,
      -0.08028870075941086, -0.022148337215185165, 0.02948174625635147, -0.06211084872484207,
      0.04168979823589325, 0.0768246129155159, -0.05880207195878029, 0.08188360184431076,
      -0.08077537268400192, 0.006766111124306917, -0.06521134823560715, -0.04540077596902847,
      0.015417269431054592, -0.06816502660512924, -0.08284786343574524, 0.014074951410293579,
      -0.05011261627078056, -0.004043390974402428, -0.02509678713977337, -0.05776388943195343,
      -0.07600904256105423, -0.07772394269704819, 0.027025768533349037, 0.04934932291507721],
     [-0.013213729485869408, 0.0809158980846405, -0.009107464924454689, 0.03203579783439636,
      0.05720013007521629, 0.06118041276931763, 0.011284733191132545, 0.03339814022183418,
      -0.03494158014655113, 0.07672706991434097, 0.05876654013991356, 0.04203756898641586,
      0.06916100531816483, -0.06412772834300995, 0.017487574368715286, 0.0687997043132782,
      0.04308879375457764, -0.06782391667366028, 0.06520140916109085, -0.07638518512248993,
      0.006623346824198961, -0.0015488883946090937, 0.01019541546702385, -0.04850301891565323,
      0.02202930487692356, -0.08446761965751648, -0.07327860593795776, -0.09829815477132797,
      -0.040869537740945816, 0.06817605346441269, -0.01820993795990944, 0.0762917622923851,
      0.061997830867767334, -0.05993608385324478, -0.03274480253458023, 0.08306948840618134,
      0.07756844907999039, -0.019462723284959793, 0.03439086675643921, 0.015364659018814564,
      0.006855717860162258, -0.003225883934646845, -0.06210845336318016, 0.0036181206814944744,
      0.07416243106126785, -0.07948015630245209, 0.07190945744514465, -0.09942654520273209,
      -0.027235209941864014, -0.05499458685517311, -0.07535236328840256, -0.04031653702259064,
      -0.0659060925245285, 0.010412050411105156, -0.07908906787633896, 0.0517193041741848,
      -0.00748201459646225, 0.011730801314115524, -0.05527800694108009, 0.028902225196361542,
      0.03172494098544121, -0.009996481239795685, -0.013764865696430206, 0.026644667610526085,
      0.011176214553415775, 0.028598610311746597, -0.058333877474069595, -0.0579519122838974,
      -0.0213032029569149, 0.07524839043617249, -0.05647820606827736, -0.05925309285521507,
      -3.2620362617308274e-05, -0.01687098853290081, -0.037200409919023514, -0.0009637697949074209,
      -0.056231334805488586, -0.0455750934779644, 0.02774844318628311, -0.016214756295084953,
      -0.05750608071684837, -0.09636107087135315, -0.0320032574236393, 0.034566979855298996,
      0.011632130481302738, 0.02255934104323387, 0.042295731604099274, 0.043650541454553604,
      -0.04019494354724884, -0.029039232060313225, -0.007338892202824354, 0.030834127217531204,
      -0.05215911567211151, -0.008022491820156574, 0.016066841781139374, 0.002699998440220952,
      0.03371934965252876, -0.04319925606250763, 0.001767573063261807, 0.0729670375585556,
      -0.06564080715179443, 0.022781025618314743, 0.028208382427692413, 0.04850540682673454,
      0.015681525692343712, -0.04187306389212608, 0.05029786005616188, -0.04628723859786987],
     [0.07694002240896225, 0.038339149206876755, 0.11159869283437729, -0.03440079838037491,
      -0.056988801807165146, -0.09084116667509079, 0.05957647040486336, 0.03857949748635292,
      -0.00122636288870126, 0.05232040211558342, 0.07708388566970825, -0.05117770656943321,
      -0.09478168189525604, 0.06624182313680649, 0.027316460385918617, 0.014301692135632038,
      0.061015911400318146, 0.06568120419979095, -0.027023468166589737, 0.03598271682858467,
      0.03155258670449257, -0.026944441720843315, -0.0016818877775222063, -0.06293836981058121,
      0.09584537893533707, -0.019284430891275406, 0.012935523875057697, -0.050561461597681046,
      -0.02947183884680271, 0.010352900251746178, -0.06059408187866211, -0.003866511397063732,
      -0.016545062884688377, -0.09314241260290146, -0.023548688739538193, -0.01308155246078968,
      -0.030734173953533173, 0.02275322750210762, -0.05919056013226509, 0.07096464931964874,
      -0.03939039260149002, 0.06750110536813736, -0.053336482495069504, -0.09076575934886932,
      0.012694456614553928, -0.07492994517087936, 0.028381051495671272, 0.03179081156849861,
      -0.02860046923160553, 0.008222411386668682, 0.06771010160446167, 0.003236648626625538,
      0.020473472774028778, -0.04512494057416916, -0.05113951861858368, -0.05159139260649681,
      -0.05354888737201691, -0.021957550197839737, -0.022243058308959007, 0.08514995127916336,
      -0.050251420587301254, 0.0757901668548584, -0.05335990712046623, 0.05611087754368782,
      0.010564415715634823, -0.05978098139166832, -0.03957962989807129, 0.01420708280056715,
      0.07234013080596924, -0.010186923667788506, -0.028691193088889122, -0.10908617079257965,
      -0.0394706204533577, 0.09340181946754456, 0.05675356090068817, -0.10181889683008194,
      -0.07621356844902039, 0.0470353439450264, -0.030193183571100235, 0.07241257280111313,
      0.08204756677150726, -0.09837532043457031, 0.00792597234249115, 0.05173203721642494,
      0.09264547377824783, 0.07395937293767929, 0.08973845839500427, -0.035815633833408356,
      -0.044214364141225815, 0.0032385666854679585, 0.04015524685382843, -0.04096294566988945,
      -0.0322946161031723, -0.04457266256213188, -0.08282575011253357, 0.06464798003435135,
      -0.0011634244583547115, 0.042039357125759125, 0.04044685140252113, -0.1036745086312294,
      -0.05350961163640022, 0.04616767168045044, 0.033872850239276886, -0.05902161821722984,
      -0.03893323242664337, -0.0768001452088356, -0.01975296251475811, -0.02476748824119568],
     [0.09653976559638977, -0.05614769086241722, 0.017734436318278313, -0.012205439619719982,
      0.07585622370243073, 0.058620575815439224, -0.07925812155008316, 0.060696233063936234,
      0.08546490967273712, -0.00664876401424408, -0.020259369164705276, 0.07190436869859695,
      -0.08572982996702194, -0.04667907580733299, 0.06656147539615631, -0.014968996867537498,
      0.041980765759944916, 0.031117573380470276, 0.06749730557203293, -0.019594453275203705,
      -0.09763353317975998, -0.015448018908500671, 0.0778949037194252, 0.022199982777237892,
      -0.0761270821094513, 0.043335266411304474, -0.07196911424398422, 0.08067647367715836,
      -0.07884927093982697, -0.07466107606887817, -0.08145658671855927, 0.061242830008268356,
      -0.05350976809859276, -0.10642418265342712, 0.04138221591711044, 0.04562794044613838,
      -0.033956270664930344, -0.05660875514149666, -0.07409358024597168, 0.032051850110292435,
      0.032949116080999374, 0.034044764935970306, -0.04509999603033066, -0.0938369631767273,
      0.03369488567113876, -0.0510835275053978, -0.09243910014629364, 0.07439149171113968,
      0.04798212647438049, 0.0697508156299591, -0.08123727887868881, 0.08991636335849762,
      0.047038573771715164, 0.021650556474924088, 0.051296621561050415, 0.07728295773267746,
      -0.08632263541221619, -0.01403388287872076, 0.07363805919885635, -0.07613229006528854,
      -0.08154218643903732, -0.0687500536441803, 0.07621095329523087, 0.002875232370570302,
      -0.04059416428208351, 0.025742139667272568, -0.012109413743019104, -0.014878777787089348,
      -0.0351003035902977, -0.07372049987316132, -0.04294285923242569, -0.009475727565586567,
      0.08627250790596008, -0.002151655498892069, -0.06760333478450775, -0.049635592848062515,
      -0.03185907006263733, 0.03334221988916397, -0.0022687409073114395, 0.06673714518547058,
      0.0339808389544487, 0.04929804429411888, 0.0788959190249443, -0.05026518926024437,
      -0.044799450784921646, 0.00665292888879776, 0.09078869223594666, -0.025174422189593315,
      0.020772939547896385, -0.068803571164608, 0.06905040889978409, 0.07203712314367294,
      0.019403766840696335, -0.05551271140575409, 0.03774423524737358, 0.044681794941425323,
      0.11081725358963013, 0.056599341332912445, 0.03160339966416359, 0.029644347727298737,
      0.0029960719402879477, 0.031035926192998886, -0.052845682948827744, 0.0752006247639656,
      -0.058309637010097504, 0.024844251573085785, 0.012373644858598709, 0.061436977237463],
     [-0.06735704094171524, -0.0459604449570179, -0.117583729326725, -0.04971158877015114,
      0.05307496339082718, -0.1279771775007248, -0.0069427392445504665, -0.09935072064399719,
      0.011168823577463627, -0.08174868673086166, -0.02297932840883732, -0.0020007137209177017,
      0.016592781990766525, -0.027377964928746223, -0.04706349968910217, 0.01722988486289978,
      0.11788638681173325, -0.08127107471227646, 0.1077752485871315, 0.10141554474830627,
      0.07407635450363159, 0.05876024067401886, 0.10459834337234497, 0.15739159286022186,
      -0.08794175833463669, 0.017104027792811394, -0.0030699309427291155, 0.01994154043495655,
      0.03574646636843681, 0.03820158168673515, 0.046233244240283966, 0.02633913978934288,
      -0.036179713904857635, 0.04372130334377289, -0.02900700829923153, -0.053333114832639694,
      0.036445461213588715, 0.006167426239699125, 0.04639095067977905, 0.07011924684047699,
      -0.050723399966955185, -0.034680821001529694, 0.02822992391884327, 0.07163592427968979,
      0.013474524021148682, -0.09968697279691696, 0.0010856264270842075, 0.004253504332154989,
      -0.07309172302484512, -0.09325013309717178, 0.030742377042770386, 0.03505091741681099,
      -0.00936854537576437, 0.09675376862287521, -0.06037173420190811, 0.07265759259462357,
      -0.0004749129875563085, -0.03364114835858345, -0.05475063621997833, -0.02669350616633892,
      -0.07703370600938797, 0.020656706765294075, 0.03281136602163315, -0.033004093915224075,
      -0.07210014760494232, -0.053701210767030716, -0.02125602960586548, -0.07653263211250305,
      -0.06465861946344376, 0.01882306858897209, 0.0227751936763525, 0.029415016993880272,
      -0.0016516906907781959, 0.0822538286447525, 0.05280733481049538, 0.012630789540708065,
      0.08424834907054901, 0.0028844086918979883, 0.03904388099908829, 0.054808732122182846,
      -0.07115346938371658, 0.017338162288069725, -0.025886138901114464, 0.031067460775375366,
      0.000774287327658385, -0.008797736838459969, -0.05609608441591263, -0.04431847855448723,
      0.08808214217424393, 0.04534343257546425, -0.041888006031513214, -0.11896852403879166,
      0.016919562593102455, -0.04849905148148537, -0.015632908791303635, -0.09670691937208176,
      -0.12555181980133057, -0.11560294777154922, -0.08435440808534622, -0.056224651634693146,
      -0.04412829130887985, -0.03858621045947075, -0.01361409854143858, 0.1084243580698967,
      0.08121395111083984, -0.015040693804621696, 0.021050626412034035, 0.015278953127563],
     [0.08251956105232239, -0.09349428117275238, -0.05599772185087204, -0.008543460629880428,
      0.052441492676734924, 0.08312992006540298, -0.06372399628162384, 0.015349432826042175,
      0.06188526749610901, -0.08000272512435913, -0.047322794795036316, -0.06144319847226143,
      0.011424691416323185, 0.0437360517680645, -0.031102001667022705, -0.08293858170509338,
      -0.07973455637693405, 0.03494195267558098, -0.0609435997903347, -0.038829255849123,
      0.04231421649456024, -0.0841175988316536, -0.05980038642883301, -0.021870793774724007,
      0.04120662808418274, 0.008227309212088585, -0.09564018249511719, 0.015628671273589134,
      0.04089106246829033, 0.0026135409716516733, 0.06971866637468338, -0.005844258703291416,
      0.06881788372993469, -0.0011826578993350267, -0.01541118323802948, -0.056819938123226166,
      0.06616544723510742, 0.0803849846124649, 0.07406254857778549, -0.019945178180933,
      0.0544021800160408, 0.08309218287467957, -0.00036337843630462885, 0.08391502499580383,
      -0.07595466822385788, 0.10382837802171707, -0.02366846427321434, 0.05393276363611221,
      0.06167703866958618, 0.07521596550941467, 0.07105807214975357, 0.022478118538856506,
      -0.06168045476078987, 0.00960677582770586, -0.03497900068759918, 0.024108868092298508,
      0.046390436589717865, 0.030402475968003273, -0.03182992339134216, 0.052593715488910675,
      0.04022971913218498, 0.08270564675331116, 0.07674648612737656, -0.09118138253688812,
      -0.002483541378751397, -0.034674834460020065, 0.07298091053962708, -0.009749684482812881,
      0.04846181720495224, -0.08617917448282242, -0.0047803400084376335, -0.02581869624555111,
      0.04086742177605629, -0.05810686573386192, 0.03735214099287987, -0.06607213616371155,
      -0.016665268689393997, -0.06843867897987366, -0.006512127351015806, -0.08916351944208145,
      -0.0449678972363472, 0.03524647653102875, 0.08916305750608444, 0.02343648299574852,
      -0.0039803944528102875, -0.02842443436384201, -0.01881248876452446, 0.04123406112194061,
      0.015105081722140312, -0.06616005301475525, 0.0602680966258049, -0.020454902201890945,
      0.04345911741256714, -0.03895128518342972, 0.010568023659288883, 0.03131282329559326,
      -0.009127968922257423, 0.08788049966096878, 0.08070506900548935, -0.03783123195171356,
      0.07797974348068237, 0.007437488529831171, -0.014456182718276978, 0.08927026391029358,
      0.033818699419498444, 0.004300951492041349, 0.0860651507973671, 0.0636482685804367],
     [-0.11809572577476501, -0.003463776782155037, -0.022758375853300095, -0.069997638463974,
      -0.017785891890525818, -0.09424414485692978, -0.09312998503446579, -0.08771897107362747,
      -0.09065993130207062, -0.05256430432200432, -0.04819970205426216, -0.09372272342443466,
      0.0729355588555336, 0.04234239459037781, 0.05303914472460747, -0.009370253421366215,
      0.019557636231184006, -0.011859009973704815, 0.09489221125841141, 0.11043692380189896,
      0.02067839354276657, 0.1317373663187027, -0.011730470694601536, 0.08058170974254608,
      -0.07620196044445038, 0.03684346005320549, 0.06208019331097603, 0.06980923563241959,
      0.11506510525941849, 0.03388487920165062, -0.08411124348640442, 0.029815811663866043,
      -0.0992889255285263, -0.013634178787469864, -0.12539376318454742, -0.02456652745604515,
      0.14308582246303558, 0.09980551153421402, 0.015833090990781784, 0.007309046108275652,
      0.008497506380081177, 0.06325637549161911, 0.13826274871826172, 0.08180183917284012,
      0.02499336004257202, -0.13701824843883514, -0.004030282609164715, 0.024837711825966835,
      0.08765678107738495, -0.04418283700942993, 0.06917480379343033, 0.09367074817419052,
      0.10524164885282516, 0.032010503113269806, -0.028047407045960426, 0.07815713435411453,
      0.13646145164966583, 0.040275562554597855, 0.001535993069410324, -0.050711989402770996,
      0.04327232018113136, 0.012857413850724697, 0.020787997171282768, 0.11222801357507706,
      -0.01385566033422947, -0.010821967385709286, 0.06094491109251976, -0.03458914905786514,
      0.028684988617897034, -0.02250507101416588, -0.02295508421957493, -0.01861092820763588,
      0.012533452361822128, -0.05722665786743164, 0.017140721902251244, 0.02599659003317356,
      0.10100308060646057, 0.08775099366903305, -0.06840470433235168, 0.06333574652671814,
      -0.042427241802215576, -0.008910372853279114, 0.05169812589883804, 0.0959586650133133,
      -0.01883827894926071, -0.065885029733181, 0.022724522277712822, -0.012720877304673195,
      0.10838031768798828, -0.029271049425005913, -0.08724682033061981, -0.023518715053796768,
      -0.08150911331176758, -0.0745789036154747, -0.05154995247721672, -0.10982857644557953,
      -0.139608234167099, -0.09039810299873352, -0.1713879257440567, -0.09931347519159317,
      -0.17245222628116608, -0.12816259264945984, 0.12418469786643982, 0.06879688799381256,
      -0.0017380225472152233, 0.05509870499372482, 0.08982902765274048, -0.0019849492236971855],
     [-0.054321907460689545, -0.1127358004450798, 0.029741236940026283, -0.14906275272369385,
      -0.08995221555233002, 0.017862891778349876, 0.028657596558332443, -0.042848553508520126,
      -0.08200879395008087, 0.028505468741059303, 0.023068411275744438, 0.04226314648985863,
      -0.0697743371129036, 0.06840285658836365, 0.05248105898499489, -0.057979766279459,
      0.039923045784235, 0.02132987789809704, 0.06257200986146927, 0.12844255566596985,
      0.08710049837827682, 0.02943454310297966, 0.08280421048402786, 0.011297172866761684,
      0.014541243202984333, -0.028778115287423134, 0.08606242388486862, 0.0035788530949503183,
      -0.08339016139507294, 0.024770259857177734, 0.07159296423196793, 0.07169466465711594,
      -0.06032248213887215, -0.05356172099709511, -0.06793926656246185, 0.05553023889660835,
      0.059907667338848114, 0.06069312244653702, -0.0046881078742444515, -0.029938116669654846,
      -0.010477798990905285, 0.008089619688689709, 0.0650550052523613, 0.05137918144464493,
      0.027551893144845963, -0.0168522410094738, 0.05676266923546791, -0.04502708464860916,
      0.05216563120484352, 0.10691647231578827, 0.043142128735780716, -0.03723737224936485,
      -0.04343423619866371, 0.057998571544885635, 0.12347956001758575, -0.03609372675418854,
      -0.019874529913067818, 0.004950490314513445, 0.01706543378531933, 0.03529896214604378,
      -0.006105854641646147, -0.03561515361070633, 0.007946777157485485, -0.0497862733900547,
      -0.03303230553865433, -0.03457878530025482, 0.08749085664749146, -0.04591968283057213,
      -0.04712722823023796, -0.07362323999404907, -0.006164999678730965, -0.039437275379896164,
      -0.055833667516708374, -0.03853720426559448, 0.08004812896251678, 0.06584048271179199,
      -0.007276640739291906, 0.002742249984294176, 0.08742377161979675, 0.055481888353824615,
      0.006757785566151142, 0.026054702699184418, 0.0179207194596529, 0.05325568467378616,
      0.06451021134853363, 0.010182100348174572, -0.046754736453294754, -0.05531645938754082,
      0.055053383111953735, -0.029732443392276764, -0.11494585871696472, -0.08783335983753204,
      -0.0002837369102053344, 0.030847257003188133, -0.1128399446606636, -0.1608305126428604,
      -0.10192406922578812, -0.02250346541404724, -0.08022759854793549, -0.04310060665011406,
      -0.1334574818611145, -0.10382048040628433, -0.044758234173059464, 0.021238870918750763,
      0.07649822533130646, 0.018993595615029335, -0.0027356636710464954, 0.07374079525470734],
     [-0.054452091455459595, 0.011008297093212605, -0.0072853886522352695, -0.03580733388662338,
      0.009741083718836308, -0.01651793345808983, 0.00041454602614976466, 0.04638088494539261,
      0.05699468031525612, 0.07366537302732468, -0.009332621470093727, 0.04203499108552933,
      0.03608144447207451, -0.03281765431165695, -0.006963435560464859, -0.026367560029029846,
      0.06109125539660454, 0.02963186986744404, -0.10469373315572739, -0.11031216382980347,
      -0.08716665953397751, -0.04359923303127289, -0.03290444612503052, 0.030270427465438843,
      0.003334693843498826, -0.0029189609922468662, 0.05072091892361641, 0.020610881969332695,
      -0.023137997835874557, 0.039161570370197296, -0.02078104577958584, -0.08706849068403244,
      0.060815416276454926, -0.028030088171362877, 0.0954560860991478, 0.08604458719491959,
      -0.12298837304115295, 0.009560269303619862, -0.0620163157582283, 0.04302157461643219,
      -0.0048497989773750305, -0.026067741215229034, 0.04657221585512161, -0.07084982842206955,
      -0.07867887616157532, -0.04362144693732262, -0.01840297132730484, -0.0631510466337204,
      0.014430999755859375, 0.0016887147212401032, -0.06205436959862709, 0.034743648022413254,
      -0.0997748002409935, -0.06159894913434982, 0.014026639983057976, 0.020381584763526917,
      0.0012774417409673333, 0.052527233958244324, 0.020228229463100433, 0.04477672278881073,
      0.05651557445526123, -0.0344415009021759, -0.042713463306427, -0.09659628570079803,
      -0.011460617184638977, 0.0014964601723477244, 0.056072480976581573, -0.0013610312016680837,
      0.07093172520399094, 0.02793598175048828, 0.08942145854234695, -0.07990548759698868,
      0.022624511271715164, 0.047078244388103485, 0.04898745194077492, -0.04031835123896599,
      -0.005052861291915178, -0.04508864879608154, -0.01424444280564785, 0.06173879653215408,
      -0.051014434546232224, 0.02369319088757038, 0.016215071082115173, -0.09362559765577316,
      0.08354055881500244, 0.015142586082220078, -0.014607679098844528, -0.020189646631479263,
      0.05342656373977661, -0.10360094159841537, 0.004045306239277124, 0.03910243511199951,
      0.09859275817871094, 0.024628769606351852, 0.034926749765872955, 0.04689299315214157,
      0.03401818871498108, 0.12010252475738525, 0.08652950823307037, 0.058236025273799896,
      0.06240963563323021, 0.0788491815328598, -0.029581014066934586, 0.02458866685628891,
      0.06991693377494812, -0.042378347367048264, 0.04499501734972, 0.06484819203615189],
     [0.10590039193630219, 0.025579728186130524, 0.07907617092132568, 0.0026898987125605345,
      0.04982290789484978, 0.03127067908644676, -0.0686834305524826, -0.015885062515735626,
      0.07018880546092987, -0.03898445516824722, 0.03803301602602005, 0.0006194469751790166,
      -0.027541249990463257, -0.10271277278661728, -0.0431954599916935, -0.04240052402019501,
      0.02921803668141365, -0.05021457001566887, 0.07868871092796326, 0.01955682784318924,
      -0.06381199508905411, 0.032924834638834, -0.027146924287080765, 0.06237318739295006,
      -0.012213211506605148, -0.08166524767875671, 0.010584150440990925, -0.05702955275774002,
      0.05653063580393791, -0.008580855093896389, -0.015994900837540627, -0.018652863800525665,
      -0.08192452043294907, -0.06286241859197617, 0.08511504530906677, -0.0840243250131607,
      0.06059112772345543, -0.013892917893826962, 0.05402722954750061, 0.06785948574542999,
      -0.014983043074607849, -0.012941943481564522, -0.06320010870695114, -0.040295928716659546,
      -0.055268917232751846, 0.0701291486620903, 0.014949246309697628, -0.10355305671691895,
      -0.059147436171770096, 0.058064501732587814, -0.006711429916322231, -0.04181135818362236,
      -0.03001660667359829, 0.06981479376554489, -0.03865177556872368, 0.045075710862874985,
      -0.028402501717209816, -0.0928044468164444, 0.07675861567258835, 0.00343598541803658,
      -0.06747833639383316, -0.04184740036725998, -0.0034955868031829596, 0.0796477198600769,
      0.09203080832958221, -0.08171114325523376, -0.06113939732313156, -0.020726138725876808,
      0.005227791145443916, -0.01867583580315113, 0.038668785244226456, -0.047691863030195236,
      0.032887086272239685, 0.055584803223609924, -0.023402521386742592, 0.04288750886917114,
      0.018126320093870163, -0.08566343784332275, -0.053507786244153976, -0.004417772870510817,
      0.04582500830292702, -0.07576766610145569, -0.07292217761278152, -0.035260964184999466,
      -0.010921339504420757, -0.07287708669900894, -0.0967436209321022, 0.0005031856126151979,
      -0.05941581726074219, 0.07758399844169617, 0.11497371643781662, 0.08369499444961548,
      -0.08554353564977646, 0.08352752774953842, 0.008224694058299065, 0.07786959409713745,
      0.0946432575583458, 0.11431112140417099, 0.06718931347131729, -0.09124105423688889,
      0.06268204003572464, -0.04056158289313316, 0.08667939156293869, 0.03918905928730965,
      0.046316757798194885, -0.0005777113838121295, 0.058748096227645874, 0.10248276591300964],
     [0.03394455090165138, -0.1337815374135971, -0.0063931019976735115, -0.06937337666749954,
      -0.016519639641046524, -0.06688767671585083, -0.07108114659786224, 0.01453487854450941,
      0.010210391134023666, -0.13921217620372772, 0.019097890704870224, -0.053709354251623154,
      0.09319577366113663, 0.01869252882897854, 0.016539907082915306, 0.08409561216831207,
      0.06691256910562515, -0.006680861581116915, -0.03825398162007332, 0.06729808449745178,
      0.08278488367795944, 0.1109585165977478, -0.007161648012697697, 0.11046096682548523,
      0.01488581858575344, -0.0443909578025341, -0.0336122140288353, -0.06263022124767303,
      0.0011880308156833053, -0.02399301901459694, -0.015110159292817116, -0.06857415288686752,
      -0.055631373077631, 0.00929271336644888, -0.08230893313884735, -0.03394421562552452,
      0.013967331498861313, 0.046970169991254807, -0.018291091546416283, -0.0505482479929924,
      -0.10381209850311279, -0.00398677121847868, -0.03406374901533127, 0.06579757481813431,
      0.024787411093711853, -0.03779813274741173, -0.005647364072501659, -0.06982545554637909,
      0.0850570797920227, 0.024786673486232758, -0.05419307202100754, 0.06895903497934341,
      0.05835927650332451, -0.04868690297007561, -0.05639202520251274, 0.025869270786643028,
      0.06848672777414322, -0.07224380224943161, -0.05106846243143082, -0.0722079873085022,
      -0.006602196488529444, 0.03333817049860954, -0.08466823399066925, -0.04293772950768471,
      0.020313961431384087, 0.00851751584559679, 0.009553823620080948, -0.00715638417750597,
      -0.037960849702358246, 0.060009490698575974, 0.04410836473107338, -0.07421878725290298,
      -0.01804797165095806, 0.0573221780359745, -0.01802138052880764, -0.04222061112523079,
      -0.036965761333703995, 0.07913729548454285, 0.0753612220287323, -0.01729247160255909,
      -0.07072567194700241, -0.039418917149305344, 0.07081159949302673, -0.05611804500222206,
      -0.008264025673270226, -0.005407491698861122, 0.08395076543092728, 0.0012924820184707642,
      0.03264870122075081, 0.08170122653245926, -0.033419929444789886, -0.11617469787597656,
      -0.024692945182323456, -0.06498369574546814, -0.11075106263160706, 0.032274454832077026,
      -0.13966764509677887, -0.118410125374794, -0.112916961312294, -0.08687884360551834,
      -0.0021130084060132504, -0.0510907843708992, 0.02841317467391491, -0.007102165371179581,
      -0.0397573858499527, 0.011760042980313301, 0.10906821489334106, 0.07968254387378693],
     [-0.016731733456254005, 0.026541512459516525, -0.01818263903260231, -0.07275862246751785,
      -0.042206279933452606, -0.010437949560582638, -0.04755273088812828, 0.07069998979568481,
      -0.004677835386246443, -0.027310917153954506, -0.00518609257414937, -0.06615529954433441,
      -0.038507308810949326, 0.09753850102424622, -0.039118442684412, -0.04272996261715889,
      0.0805889293551445, 0.10488349944353104, 0.099551722407341, 0.060150500386953354,
      0.061376530677080154, 0.10960535705089569, -0.06309198588132858, -0.05764484405517578,
      0.060747627168893814, -0.004312619101256132, -0.015978973358869553, 0.01906747743487358,
      -0.04122437536716461, -0.061206161975860596, 0.0014055800857022405, -0.012980795465409756,
      0.00362396240234375, 0.017516521736979485, 0.00118909846059978, -0.08198749274015427,
      0.013804200105369091, -0.05630430579185486, 0.04971598461270332, 0.06595005840063095,
      0.04007507115602493, 0.08556916564702988, 0.047530412673950195, -0.010162094607949257,
      0.00040180995711125433, -0.06578877568244934, 0.036658626049757004, 0.06048816442489624,
      0.012405124492943287, 0.004619868937879801, -0.0293738953769207, 0.001985515933483839,
      -0.09825322031974792, -0.09245322644710541, 0.088189497590065, -0.08788483589887619,
      -0.010690162889659405, 0.028516488149762154, 0.045687235891819, 0.06967642903327942,
      -0.07625366002321243, -0.06523081660270691, 0.008257035166025162, 0.06742553412914276,
      -0.07381241768598557, 0.0454154871404171, -0.04214102029800415, -0.024727042764425278,
      -0.061563506722450256, -0.05562703683972359, -0.005695532076060772, -0.01698836125433445,
      0.03093760833144188, 0.0932941660284996, -0.004546659998595715, 0.048867352306842804,
      0.007464109919965267, 0.056278377771377563, 0.03908645734190941, -0.017824692651629448,
      0.042145758867263794, 0.06282758712768555, -0.006425410509109497, -0.003277770709246397,
      -0.07040940970182419, -0.0768304094672203, 0.07059133052825928, -0.025348125025629997,
      0.04005094990134239, 0.08299050480127335, -0.09772363305091858, 0.06468670815229416,
      -0.030504604801535606, 0.019436802715063095, 0.03589067608118057, -0.08507335186004639,
      -0.06776733696460724, -0.09867147356271744, -0.03341719135642052, -0.08461344987154007,
      0.040716446936130524, 0.029852356761693954, 0.07494869083166122, 0.05973856523633003,
      0.003229139605537057, 0.034484993666410446, 0.060638170689344406, -0.08627074211835861],
     [-0.08475072681903839, -0.10719993710517883, -0.11361149698495865, -0.06297486275434494,
      -0.08009762316942215, -0.008326196111738682, -0.1853705793619156, -0.02446332946419716,
      0.01899324171245098, -0.04913192614912987, 0.011682050302624702, -0.13251429796218872,
      -0.022853467613458633, 0.11407921463251114, 0.1458241194486618, -0.013176053762435913,
      0.08729592710733414, 0.02877092733979225, 0.16161632537841797, 0.18699580430984497,
      0.14587056636810303, 0.17331881821155548, 0.17625229060649872, 0.17851169407367706,
      0.029744533821940422, -0.008210676722228527, 0.013212107121944427, -0.06958102434873581,
      -0.06594927608966827, 0.027302853763103485, -0.04068078100681305, -0.08245354145765305,
      -0.0190262570977211, 0.011684902012348175, -0.05377968028187752, -0.052416443824768066,
      0.12025675922632217, 0.09951043874025345, 0.021162468940019608, 0.032936401665210724,
      -0.05423500016331673, 0.059235621243715286, -0.016188081353902817, 0.07420054823160172,
      0.08277113735675812, -0.07832036912441254, 0.04387243464589119, 0.02452581748366356,
      0.013056663796305656, 0.09084998071193695, -0.03982936963438988, -0.06100479140877724,
      -0.03599447384476662, 0.0878230631351471, 0.07173608988523483, 0.05903884768486023,
      0.03627631440758705, -0.023146549239754677, 0.12751483917236328, -0.0479964017868042,
      0.055601369589567184, 0.012565079145133495, 0.0232661385089159, 0.04324237257242203,
      0.10062569379806519, 0.05615095794200897, 0.034444913268089294, -0.026056736707687378,
      0.030757596716284752, -0.057678792625665665, 0.05105622485280037, -0.06828691810369492,
      -0.10912833362817764, -0.008045007474720478, -0.032774101942777634, -0.00814801175147295,
      -0.013460959307849407, 0.00669452827423811, 0.1065794825553894, 0.0611022487282753,
      0.06444644927978516, 0.12662407755851746, 0.03781687095761299, 0.03506103530526161,
      0.006299993954598904, 0.03289658948779106, 0.028196141123771667, 0.10815002024173737,
      0.10129308700561523, 0.05095088854432106, -0.15022464096546173, -0.040550317615270615,
      -0.1498100608587265, -0.059423625469207764, -0.06227171793580055, -0.017876654863357544,
      -0.04687124490737915, -0.06045409291982651, -0.15095007419586182, -0.16443714499473572,
      -0.14451132714748383, -0.12258550524711609, 0.06708718836307526, 0.034758374094963074,
      -0.07125942409038544, 0.07506386935710907, 0.016591720283031464, 0.04730994254350662],
     [0.06817303597927094, -0.061736010015010834, 0.07311630249023438, 0.07094458490610123,
      -0.07323700934648514, -0.059720125049352646, 0.03340233117341995, -0.0909024327993393,
      0.05603504180908203, -0.03180396556854248, -0.11811584234237671, -0.020609818398952484,
      0.01181670930236578, -0.004287199582904577, 0.09986968338489532, -0.010443437844514847,
      0.00396463880315423, 0.035067472606897354, 0.005730106960982084, -0.07988632470369339,
      -0.038674917072057724, 0.027890969067811966, -0.020069260150194168, 0.013413526117801666,
      -0.08505281805992126, 0.021398117765784264, -0.021948477253317833, 0.07516707479953766,
      -0.05007633939385414, -0.014746492728590965, 0.07468722015619278, 0.04579897224903107,
      0.0793161615729332, -0.038238149136304855, 0.07732289284467697, 0.01739594154059887,
      0.027143925428390503, -0.04653964191675186, -0.009981521405279636, -0.04152556136250496,
      0.06313607096672058, -0.05040553957223892, 0.052042774856090546, -0.02912907674908638,
      -0.05131789669394493, 0.0817764475941658, -0.10430414229631424, 0.00017586039029993117,
      -0.03585897758603096, -0.06440508365631104, -0.013516386970877647, 0.022418687120079994,
      -0.02518950030207634, -0.018639273941516876, -0.018802490085363388, -0.029983045533299446,
      -0.1272232085466385, -0.03784610703587532, 0.00628664530813694, 0.0657549574971199,
      0.08603750914335251, -0.015216786414384842, -0.009374179877340794, -0.08621753007173538,
      0.07953528314828873, -0.023967426270246506, 0.09170901775360107, 0.07594124227762222,
      0.07273824512958527, 0.08000998198986053, -0.020101839676499367, -5.159461943549104e-05,
      -0.08890524506568909, -0.02335881069302559, 0.038419563323259354, -0.03613421320915222,
      -0.09898044168949127, -0.062434855848550797, 0.041671399027109146, 0.06156526133418083,
      -0.042200881987810135, 0.07358290255069733, 0.10467446595430374, -0.056864067912101746,
      0.07421089708805084, 0.0010034770239144564, 0.1018611341714859, 0.02039356902241707,
      0.0998452827334404, 0.031315047293901443, 0.0813189297914505, -0.027886850759387016,
      0.12774617969989777, 0.021349864080548286, 0.07611344754695892, -0.011161446571350098,
      0.11468598246574402, -0.00135618110653013, 0.0848628506064415, 0.056815579533576965,
      -0.0023286438081413507, 0.13389967381954193, -0.08708363026380539, 0.018218718469142914,
      0.0659794956445694, 0.03782714158296585, -0.09063742309808731, -0.08383361995220184],
     [0.050861433148384094, -0.04006066545844078, -0.0377325713634491, -0.08654065430164337,
      -0.01822821795940399, -0.07688673585653305, -0.08205907046794891, 0.043000008910894394,
      0.026753082871437073, -0.003080674447119236, -0.02961314283311367, 0.048471465706825256,
      -0.029457632452249527, 0.01633373647928238, -0.013565554283559322, -0.01301829144358635,
      -0.08808472007513046, -0.022360974922776222, -0.011374431662261486, -0.0075438665226101875,
      -0.008487983606755733, -0.08559790253639221, -0.05085151642560959, -0.08230908215045929,
      -0.013709189370274544, -0.07372625917196274, 0.039832185953855515, 0.059436947107315063,
      0.018597889691591263, 0.030687270686030388, 0.06057104840874672, -0.08047637343406677,
      -0.08436193317174911, -0.08865629881620407, -0.0013001163024455309, 0.10759764164686203,
      0.005171643104404211, -0.04101787507534027, 0.06059093028306961, 0.05746586620807648,
      0.001320656854659319, 0.08825486153364182, 0.010292167775332928, 0.0003070377279073,
      0.08439737558364868, -0.04638126492500305, 0.01259648334234953, 0.0750812217593193,
      -0.08107295632362366, -0.09747055172920227, 0.05615006014704704, 0.04353027790784836,
      -0.02185150608420372, -0.052182234823703766, -0.01268806029111147, 0.025028996169567108,
      -0.022953510284423828, 0.07420216500759125, 0.023567387834191322, -0.07075957953929901,
      0.05018458515405655, 0.06662023812532425, -0.05688102915883064, 0.044808030128479004,
      0.016556501388549805, 0.054680220782756805, -0.02809407375752926, 0.08494960516691208,
      -0.029580967500805855, -0.092922642827034, -0.02960560843348503, -0.08952594548463821,
      0.06636050343513489, 0.03134891018271446, 0.08045390248298645, -0.07386501878499985,
      -0.003650052472949028, 0.0032568115275353193, -0.1004832461476326, 0.006310970522463322,
      0.0346754752099514, -0.07765047252178192, -0.032275136560201645, -0.05325499549508095,
      -0.010084221139550209, 0.05553346872329712, -0.07574321329593658, -0.023116637021303177,
      0.025848763063549995, -0.032734114676713943, 0.033125169575214386, -0.04163319617509842,
      -0.02412579022347927, -0.005372608546167612, -0.05418318510055542, 0.07460715621709824,
      0.03596113622188568, -0.011353440582752228, 0.007609589025378227, 0.016070932149887085,
      -0.013547691516578197, 0.0988818109035492, 0.04474351555109024, 0.09186835587024689,
      -0.09139290452003479, -0.07990903407335281, 0.020592501387000084, -0.059293244034051895],
     [-0.06966394186019897, 0.029757563024759293, 0.11337451636791229, 0.055585093796253204,
      0.05517290532588959, 0.06807322800159454, 0.06632883846759796, 0.04634927585721016,
      0.08850506693124771, -0.08106112480163574, 0.1035943403840065, 0.011319869197905064,
      -0.0874411016702652, -0.057482827454805374, -0.025520680472254753, 0.01607511192560196,
      -0.05138140171766281, -0.10133981704711914, 0.07213635742664337, 0.009743696078658104,
      0.02313421294093132, 0.019145542755723, -0.0013910026755183935, 0.006722064223140478,
      0.015165315009653568, -0.06302583962678909, 0.047942083328962326, 0.03814360126852989,
      -0.07746335864067078, -0.06973253935575485, -0.10770030319690704, -0.027297237887978554,
      -0.06461448222398758, -0.0396241769194603, -0.01495404914021492, 0.017498550936579704,
      0.002729743253439665, -0.09527469426393509, -0.09838595986366272, -0.07134298235177994,
      -0.07680904120206833, 0.04220658913254738, -0.013585188426077366, 0.07444427162408829,
      -0.02110297791659832, -0.072020024061203, -0.028772110119462013, 0.044854775071144104,
      0.05866175517439842, -0.07659793645143509, 0.04855334758758545, 0.0368010438978672,
      0.04365469142794609, 0.08326922357082367, 0.08900592476129532, -0.06641930341720581,
      0.05493147671222687, 0.05356663838028908, 0.07764073461294174, 0.044680286198854446,
      -0.05073642358183861, 0.015456764958798885, 0.007369651459157467, 0.07910659909248352,
      -0.08571674674749374, 0.04500749334692955, 0.02411196194589138, 0.01045607402920723,
      0.0886671170592308, -0.059445820748806, 0.058306001126766205, -0.023635851219296455,
      -0.05299505591392517, 0.03469592332839966, 0.08324041217565536, 0.08519892394542694,
      -0.03720865398645401, -0.05513176694512367, 0.014087601564824581, -0.004778311587870121,
      -0.04027998447418213, 0.02360484004020691, -0.08423326164484024, -0.012498650699853897,
      0.06862019002437592, -0.052800536155700684, 0.012888790108263493, 0.011315291747450829,
      -0.033022865653038025, 0.059839896857738495, 0.03351284936070442, -0.050093263387680054,
      0.05334436148405075, -0.014209180139005184, -0.04420781135559082, 0.054518233984708786,
      -0.017981112003326416, 0.057806290686130524, -0.019204648211598396, -0.0699688270688057,
      -0.1133844405412674, 0.06753798574209213, 0.05356218293309212, -0.06413581222295761,
      0.08808927983045578, 0.026524612680077553, -0.07545223087072372, -0.0506586991250515],
     [-0.027447717264294624, -0.061445895582437515, -0.028254801407456398, 0.03466152399778366,
      0.010109775699675083, 0.06767817586660385, 0.0910903662443161, -0.02946736291050911,
      0.06608632951974869, 0.01965833082795143, 0.09083285927772522, -0.0009657458867877722,
      -0.03633340448141098, 0.061478689312934875, -0.05466749519109726, -0.0778442919254303,
      -0.04314383119344711, 0.008578122593462467, -0.04384327679872513, 0.05101897940039635,
      0.08287645131349564, 0.00320471846498549, -0.09900180250406265, 0.053099166601896286,
      -0.010352648794651031, 0.033909592777490616, -0.008898617699742317, -0.02316437102854252,
      0.017300166189670563, -0.019602637737989426, 0.016756571829319, -0.04545571282505989,
      0.0764283612370491, -0.10137617588043213, 0.021513165906071663, -0.053992412984371185,
      0.04309175908565521, 0.0841275006532669, 0.08344672620296478, 0.06848415732383728,
      0.03929941728711128, -0.08315888047218323, 0.020333506166934967, -0.05682861804962158,
      -0.08091811835765839, -0.05705760046839714, 0.07118801772594452, -0.04889770969748497,
      -0.029222125187516212, 0.049536898732185364, 0.03479165956377983, -0.04401279613375664,
      0.08563343435525894, 0.01564234495162964, -0.06396589428186417, 0.07070688903331757,
      -0.0738762691617012, 0.08271554857492447, 0.002161024371162057, 0.03325433284044266,
      0.03070063702762127, 0.0452691987156868, 0.07238093763589859, -0.041836753487586975,
      -0.09309352189302444, -0.007574475836008787, -0.04642632603645325, -0.003263573395088315,
      -0.023685360327363014, 0.040649548172950745, 0.08768348395824432, -0.030307410284876823,
      -0.050466787070035934, 0.03231082111597061, 0.04325735569000244, -0.07134169340133667,
      0.09193872660398483, -0.06424113363027573, -0.03525453433394432, 0.075628861784935,
      0.061920370906591415, 0.04614970088005066, 0.038780733942985535, 0.009368067607283592,
      -0.062461260706186295, -0.08963044732809067, -0.06332802027463913, 0.027143843472003937,
      0.013672697357833385, 0.04977048933506012, 0.07128674536943436, -0.0503629669547081,
      -0.0028638439252972603, -0.09104503691196442, -0.10496155917644501, -0.09857433289289474,
      -0.02266065962612629, 0.07090553641319275, -0.11828160285949707, -0.08989778161048889,
      -0.03761347755789757, 0.04485166817903519, 0.04898018389940262, 0.05062433332204819,
      -0.016122546046972275, 0.0573447160422802, 0.01473206840455532, -0.037641581147909164],
     [-0.037180621176958084, 0.0380573570728302, 0.019217774271965027, -0.07668185979127884,
      -0.08068573474884033, 0.043796978890895844, -0.06270982325077057, 0.07936793565750122,
      -0.0485515370965004, -0.0990818440914154, 0.01934530958533287, -0.0885690450668335,
      0.02420714497566223, -0.032425761222839355, 0.10080995410680771, -0.03630872815847397,
      -0.04740440100431442, -0.04247881844639778, -0.10155065357685089, -0.08613710105419159,
      0.014715711586177349, 0.014513195492327213, 0.06689005345106125, 0.0358477421104908,
      -0.049393050372600555, -0.05374254658818245, 0.004216290544718504, 0.019792024046182632,
      0.11493628472089767, 0.05313263088464737, 0.0008208724320866168, -0.008617306128144264,
      0.08626916259527206, 0.044750720262527466, 0.06254029273986816, 0.11807192116975784,
      0.011481241323053837, -0.051184430718421936, 0.07056161016225815, -0.08294650167226791,
      -0.07534284144639969, -0.017082590609788895, -0.033938806504011154, -0.02123769000172615,
      -0.0037587746046483517, 0.03193429857492447, 0.018679950386285782, -0.06154104322195053,
      0.02300342358648777, 0.019082404673099518, 0.03387051820755005, -0.05340217426419258,
      0.05467776954174042, 0.018580099567770958, -0.031578537076711655, -0.03038315288722515,
      0.061628036201000214, -0.07013703137636185, -0.06140904501080513, 0.08414556831121445,
      0.11506826430559158, 0.04734368622303009, 0.05529292672872543, 0.007494606543332338,
      0.036992210894823074, 0.10542140901088715, -0.0016275339294224977, -0.03239172324538231,
      -0.08133182674646378, 0.0065736123360693455, 0.060517825186252594, 0.018626578152179718,
      0.052447881549596786, 0.045502159744501114, 0.0026249438524246216, -0.06008850038051605,
      0.062551349401474, -0.07417690008878708, 0.045835647732019424, 0.015513880178332329,
      0.014397481456398964, 0.017346646636724472, -0.0346597321331501, -0.01310777198523283,
      -0.0006900429143570364, 0.08205875754356384, 0.08304378390312195, -0.021078716963529587,
      0.01677868701517582, 0.054841410368680954, -0.03192950785160065, 0.02900839038193226,
      -0.056999679654836655, -0.014372343197464943, -0.029384225606918335, 0.012901983223855495,
      -0.10080897808074951, -0.0754074975848198, -0.0698227658867836, -0.020093489438295364,
      -0.11293455958366394, 0.021568678319454193, -0.044813212007284164, 0.09644748270511627,
      0.003960018046200275, 0.04918249696493149, -0.05332498252391815, 0.009259537793695927],
     [0.025623008608818054, 0.10519284009933472, 0.06749368458986282, -0.05658695101737976,
      -0.10490256547927856, -0.010315136052668095, 0.07158669084310532, 0.09178048372268677,
      -0.09381943941116333, -0.10487864166498184, -0.09144812822341919, 0.02204757370054722,
      0.05737115442752838, 0.06042786315083504, -0.01448122225701809, -0.02362462878227234,
      -0.058456771075725555, -0.023991964757442474, -0.01920594647526741, 0.06543807685375214,
      -0.029874520376324654, -0.06966723501682281, 0.009137708693742752, -0.059889521449804306,
      -0.036830466240644455, -0.06015954539179802, 0.03348980471491814, 0.062332041561603546,
      -0.07245605438947678, 0.017441147938370705, -0.04216829314827919, -0.045929089188575745,
      -0.05447346344590187, -0.03817494213581085, 0.03015040047466755, 0.11189578473567963,
      0.03470665216445923, -0.02634054608643055, 0.08455030620098114, -0.06452775746583939,
      0.08494845777750015, 0.0384591780602932, -0.008238251321017742, -0.036049261689186096,
      -0.051864150911569595, -0.05556356906890869, -0.038152698427438736, -0.033919669687747955,
      -0.023990565910935402, 0.010982922278344631, -0.050677940249443054, -0.04150107130408287,
      -0.03879484161734581, 0.0769912376999855, -0.020273415371775627, -0.0013325981562957168,
      0.024433927610516548, -0.006922558881342411, 0.04059344157576561, 0.08061483502388,
      -0.06576146930456161, 0.005774368066340685, -0.12028414756059647, 0.09158939868211746,
      0.007171823643147945, 0.09502672404050827, -0.021836210042238235, -0.019595693796873093,
      0.0010925759561359882, 0.025769991800189018, -0.015350558795034885, -0.024589624255895615,
      -0.04011497274041176, 0.054532118141651154, 0.044021520763635635, 0.06756206601858139,
      0.07187263667583466, 0.028622353449463844, 0.05430890619754791, 0.027144022285938263,
      0.04588792100548744, 0.00033171684481203556, -0.08960115164518356, 0.06833712011575699,
      -0.09850708395242691, -0.021087130531668663, -0.0614020898938179, 0.07641242444515228,
      -0.004118144046515226, 0.10420989990234375, 0.11189096421003342, 0.0943051427602768,
      0.1311734914779663, 0.04292609915137291, 0.12785956263542175, 0.05025005713105202,
      -0.006867794785648584, -0.04124102741479874, -0.010461178608238697, -0.0048988875932991505,
      -0.03505923971533775, -0.040849216282367706, 0.02641240507364273, 0.01709684357047081,
      0.0597258061170578, 0.016612807288765907, 0.07537911087274551, -0.0003634719760157168],
     [0.06733658164739609, 0.029382891952991486, 0.059875767678022385, 0.09467403590679169,
      -0.03322779759764671, -0.03282596915960312, 0.07415615767240524, 0.0781761109828949,
      -0.050157345831394196, 0.02756020426750183, -0.01924053393304348, 0.09811487048864365,
      -0.02509859949350357, 0.07158307731151581, 0.07401580363512039, -0.05703312158584595,
      -0.017977572977542877, -0.04905436187982559, -0.007162339054048061, 0.02031710371375084,
      0.044657669961452484, -0.0072829509153962135, 0.03844927251338959, -0.08659512549638748,
      0.052798859775066376, -0.03187895566225052, 0.020168287679553032, -0.019999219104647636,
      0.06762382388114929, 0.08016452193260193, -0.0564073845744133, -0.10392537713050842,
      0.05544675886631012, 0.0022203049156814814, 0.07945302873849869, 0.006610124837607145,
      0.05392967537045479, -0.08175718039274216, 0.08129492402076721, -0.008644881658256054,
      0.0830627977848053, 0.0015620392514392734, -0.04049941152334213, 0.008823061361908913,
      -0.030860327184200287, 0.0826219990849495, -0.06253957748413086, 0.01516587845981121,
      -0.06239820644259453, -0.024466637521982193, 0.09322129935026169, -0.0580889992415905,
      -0.036691244691610336, 0.017760414630174637, 0.05701768770813942, 0.07220768928527832,
      0.02711288072168827, -0.04759708419442177, 0.032674640417099, -0.008120362646877766,
      -0.008300523273646832, 0.003084764117375016, 0.07017789781093597, 0.007584829814732075,
      0.0235380120575428, -0.05490104854106903, 0.08073525875806808, 0.09999346733093262,
      0.03619571030139923, -0.044393375515937805, -0.08817819505929947, -0.003754963865503669,
      -0.012319613248109818, -0.07684831321239471, -0.02635936625301838, -0.03688688576221466,
      0.060923945158720016, -0.05517098307609558, 0.05444476008415222, 0.0780855044722557,
      -0.06311984360218048, -0.042287781834602356, -0.10372793674468994, -0.0640169084072113,
      -0.07888410985469818, -0.060031257569789886, 0.06776640564203262, 0.0701962411403656,
      0.011975348927080631, -0.07325218617916107, -0.07843345403671265, -0.01017904095351696,
      -0.09851058572530746, -0.057407982647418976, 0.06588141620159149, -0.06007172912359238,
      0.07348795235157013, -0.05234134569764137, -0.011817114427685738, 0.061926599591970444,
      -0.05452080816030502, -0.08433178067207336, 0.07122430205345154, -0.0010232334025204182,
      0.06647703796625137, -0.08722392469644547, -0.011824673041701317, -0.0593053363263607],
     [-0.042914606630802155, 0.053872428834438324, 0.04981175437569618, 0.02959674410521984,
      -0.0490831658244133, -0.07255683839321136, 0.07032010704278946, 0.020634062588214874,
      0.017338978126645088, -0.03190048411488533, -0.07693453133106232, 0.0787581130862236,
      0.05431085452437401, -0.07707186043262482, -0.08111061900854111, -0.009181631729006767,
      0.0145535534247756, 0.08052817732095718, -0.05741283670067787, -0.08260485529899597,
      0.0616234615445137, -0.0649079829454422, 0.10892478376626968, 0.04529718682169914,
      0.004390189424157143, -0.10583585500717163, -0.0974159687757492, 0.06644538044929504,
      0.07068225741386414, 0.014317495748400688, -0.006585007533431053, -0.057345855981111526,
      -0.021268947049975395, 0.013761462643742561, -0.0005243346095085144, -0.09653014689683914,
      -0.14007993042469025, -0.04289718344807625, 0.06759131699800491, 0.0674223005771637,
      0.07229669392108917, -0.14875786006450653, -0.029806390404701233, -0.058199863880872726,
      -0.06111839786171913, 0.0842166617512703, 0.011670585721731186, 0.021546881645917892,
      -0.06368371844291687, 0.0927775502204895, 0.03128330036997795, 0.02662632428109646,
      0.0004054174933116883, 0.027734344825148582, -0.09302259981632233, -0.06596498191356659,
      -0.04586965590715408, -0.08710635453462601, -0.04543028771877289, 0.08051585406064987,
      0.034400712698698044, -0.022706858813762665, -0.03210759162902832, -0.016271717846393585,
      -0.002442531753331423, -0.11106806993484497, 0.05797253176569939, 0.006125664804130793,
      0.003918668255209923, -0.03201178461313248, 0.061228472739458084, -0.07051882147789001,
      -0.07181929051876068, -0.05592835322022438, 0.08078493922948837, -0.03545713797211647,
      -0.01103964913636446, -0.042260557413101196, 0.1037786677479744, 0.014666257426142693,
      0.019644567742943764, 0.0021793595515191555, 0.008446482010185719, -0.023892005905508995,
      0.021225357428193092, -0.1240273043513298, 0.0702718049287796, -0.049897536635398865,
      -0.10430324077606201, 0.03782165050506592, 0.0599285364151001, -0.0470120869576931,
      0.039346303790807724, 0.04963142052292824, 0.13637687265872955, 0.040817223489284515,
      0.08404885232448578, -0.019765423610806465, 0.15659311413764954, 0.06258208304643631,
      0.06720661371946335, 0.031484462320804596, -0.09755618870258331, -0.012754281051456928,
      0.03119351901113987, -0.08477310091257095, 0.02366586960852146, -0.02834361046552658],
     [0.06197763979434967, 0.06924981623888016, 0.10001008957624435, 0.07133644819259644,
      -0.061494942754507065, -0.051777761429548264, 0.010626325383782387, -0.07564186304807663,
      -0.001729790004901588, 0.012203525751829147, 0.016947075724601746, 0.000791416852734983,
      0.07711382210254669, 0.004203277640044689, -0.06684059649705887, -0.10205348581075668,
      -0.11238028854131699, -0.048630423843860626, 0.05910937115550041, -0.007465109694749117,
      -0.07160366326570511, -0.005150035489350557, -0.04089297354221344, -0.018225427716970444,
      0.040207359939813614, -0.09076125174760818, 0.039459921419620514, -0.00775670912116766,
      0.0497540682554245, -0.041988372802734375, -0.008508170023560524, 0.09802432358264923,
      -0.0648031085729599, -0.05191513150930405, -0.03897402063012123, 0.04537995904684067,
      0.0249952245503664, 0.018020037561655045, 0.054034311324357986, -0.0516902431845665,
      -0.08864273130893707, 0.007487778086215258, 0.07570866495370865, 0.0069223428145051,
      0.0824737474322319, -0.07791706174612045, -0.07078812271356583, -0.06436264514923096,
      -0.05004749447107315, -0.0881209596991539, 0.08028187602758408, 0.02747596986591816,
      0.023301605135202408, 0.0023148199543356895, -0.035895004868507385, 0.09588617086410522,
      0.08236132562160492, -0.06573057919740677, -0.07418878376483917, 0.061401110142469406,
      0.08874096721410751, 0.1061810553073883, -0.010486072860658169, -0.08308158814907074,
      0.025053901597857475, 0.09335928410291672, 0.0007427876116707921, -0.044016338884830475,
      0.05275300145149231, 0.07872544974088669, 0.09879479557275772, -0.00822711456567049,
      -0.01491477806121111, 0.04494481906294823, 0.049595724791288376, -0.08416282385587692,
      -0.05628909170627594, -0.025873148813843727, -0.05275334045290947, -0.043411459773778915,
      -0.0940665751695633, -0.010518312454223633, 0.06286458671092987, -0.06608735769987106,
      -0.08585568517446518, 0.006242360919713974, -0.10330013930797577, 0.027056917548179626,
      0.03360233083367348, -0.052857328206300735, 0.012143231928348541, 0.04053829610347748,
      0.05078929662704468, 0.052492786198854446, 0.08664286136627197, 0.03395823761820793,
      0.04090949147939682, 0.0918533205986023, -0.08730732649564743, 0.015421885065734386,
      -0.0454578772187233, -0.05856642499566078, -0.037355199456214905, 0.07058064639568329,
      -0.06838039308786392, -0.08350292593240738, -0.07805785536766052, -0.029861921444535255],
     [-0.0006927521899342537, -0.07002910226583481, -0.07028929144144058, -0.03192448988556862,
      0.008692795410752296, 0.05582861602306366, -0.025297056883573532, 0.07481702417135239,
      0.015046272426843643, -0.05027048662304878, -0.09245849400758743, 0.021288426592946053,
      0.08494466543197632, -0.040169842541217804, 0.06303080171346664, 0.012388575822114944,
      0.009277771227061749, 0.020221173763275146, -0.005347034893929958, 0.07822325080633163,
      0.11010383069515228, 0.025312134996056557, 0.06778684258460999, 0.07795247435569763,
      0.10814289003610611, 0.074191614985466, -0.01469435915350914, -0.04299170896410942,
      0.03856363147497177, 0.08855301141738892, 0.04987845569849014, 0.06675003468990326,
      -0.01311328075826168, -0.05957737937569618, -0.07186802476644516, 0.021916963160037994,
      -0.041381772607564926, 0.06923828274011612, -0.04774914309382439, -0.08829257637262344,
      -0.09969162940979004, -0.08233319967985153, 0.08794640004634857, 0.08028703927993774,
      0.03893386572599411, 0.06828068941831589, 0.005081350449472666, 0.10375653952360153,
      0.07009391486644745, 0.06434816122055054, 0.0806005671620369, 0.010082850232720375,
      -0.038953255861997604, 0.08317121863365173, 0.05774882808327675, -0.07745036482810974,
      -0.05044114217162132, 0.049801114946603775, 0.0060906498692929745, 0.03234443813562393,
      0.02205529436469078, 0.07247775048017502, -0.0011656091082841158, 0.013235197402536869,
      -0.08214464038610458, -0.09294135123491287, -0.043032798916101456, -0.00042849828605540097,
      0.0013963367091491818, -0.08717010170221329, 0.006336269434541464, -0.03229508176445961,
      -0.07902596890926361, 0.03505788743495941, 0.04315797612071037, -0.03890986368060112,
      -0.062466174364089966, 0.0798347219824791, 0.029169008135795593, -0.042997099459171295,
      0.08902936428785324, -0.004275226499885321, -0.02029941789805889, 0.00501626543700695,
      0.011884993873536587, -0.10005392134189606, 0.06674180179834366, -0.06656083464622498,
      0.038238413631916046, 0.07176274061203003, -0.009420866146683693, -0.09472758322954178,
      0.005029349587857723, -0.03785528987646103, 0.042995937168598175, -0.11076967418193817,
      -0.06239059567451477, -0.12463153153657913, -0.010456450283527374, 0.06063048541545868,
      -0.011846259236335754, 0.030837075784802437, -0.013930830173194408, -0.025919834151864052,
      -0.028530137613415718, -0.05280809476971626, 0.03226746618747711, 0.0408012755215168],
     [0.03223690018057823, -0.02474800869822502, -0.03112291358411312, -0.09354795515537262,
      -0.06781353801488876, 0.038055624812841415, 0.04456576332449913, 0.0016779581783339381,
      -0.02650018036365509, -0.002052897587418556, 0.030768077820539474, 0.02954452484846115,
      0.03657246381044388, 0.007765099871903658, 0.046765729784965515, -0.031156856566667557,
      0.018323298543691635, 0.01458111871033907, -0.053640082478523254, 0.04401904717087746,
      -0.04858017712831497, 0.06818795949220657, 0.09249963611364365, -0.011440298520028591,
      0.048736777156591415, -0.026072511449456215, -0.027800263836979866, -0.04810754209756851,
      -0.008292429149150848, -0.07012782245874405, 0.027371883392333984, 0.0360012911260128,
      0.036553092300891876, -0.042851731181144714, 0.05633282661437988, 0.05788293108344078,
      -0.05802752077579498, 0.08143706619739532, -0.06254040449857712, -0.020072506740689278,
      0.08027142286300659, -0.00340714561752975, -0.059577781707048416, -0.08587724715471268,
      -0.10345939546823502, -0.022367335855960846, -9.106849756790325e-05, -0.08134707808494568,
      -0.0477188415825367, 0.08489338308572769, -0.07414058595895767, 0.04810193553566933,
      -0.023761002346873283, -0.03254888951778412, 0.0044859424233436584, -0.0299071054905653,
      -0.03540712967514992, -0.0005444081616587937, 0.015699690207839012, 0.061835501343011856,
      -0.017762653529644012, -0.0694265067577362, 0.0024009039625525475, -0.044501423835754395,
      0.07047318667173386, 0.050494901835918427, 0.013124441727995872, 0.014303373172879219,
      -0.09469891339540482, -0.024376312270760536, -0.08169353753328323, -0.05268024653196335,
      -0.09355723857879639, -0.08166734129190445, 0.10396460443735123, 0.01602335460484028,
      -0.0009345925063826144, 0.0313979908823967, 0.060344148427248, 0.04058795049786568,
      -0.051893897354602814, -0.06989653408527374, -0.0792267918586731, 0.09843843430280685,
      0.09072940051555634, 0.0983537882566452, 0.0925743579864502, 0.08325064182281494,
      0.06634797155857086, 0.05529013276100159, 0.07536602765321732, -0.0013194165658205748,
      0.06534887105226517, 0.07314147055149078, 0.015470919199287891, 0.016105731949210167,
      0.0036609105300158262, -0.07556930929422379, 0.03818110004067421, 0.02316497266292572,
      -0.014966338872909546, -0.06510990858078003, -0.030682340264320374, 0.06279579550027847,
      0.05933535099029541, 0.041212741285562515, 0.017819223925471306, -0.09189467877149582],
     [-0.14300155639648438, -0.13316547870635986, 0.0004701169382315129, -0.0855039432644844,
      -0.06471092998981476, -0.0497589148581028, -0.13105274736881256, -0.08049537241458893,
      -0.028655923902988434, 0.05295943841338158, -0.011518016457557678, -0.07769181579351425,
      0.020062392577528954, 0.014339013956487179, 0.022735603153705597, -0.05955575779080391,
      0.04114281386137009, 0.03599056974053383, 0.10786157846450806, 0.15800637006759644,
      0.0700911357998848, 0.04664989560842514, -0.003692497266456485, 0.13859617710113525,
      0.028666311874985695, 0.05025552213191986, 0.09316577762365341, -0.03517085313796997,
      -0.0424155555665493, 0.04071787744760513, 0.059540268033742905, -0.04621371999382973,
      -0.008984017185866833, -0.0705319344997406, -0.03578905388712883, 0.07997148483991623,
      -0.044082313776016235, 0.011743620038032532, 0.08620055764913559, 0.0276613999158144,
      -0.09043809026479721, 0.05497189983725548, 0.010622994042932987, -0.021301181986927986,
      0.048147786408662796, 0.02677321620285511, 0.013297868892550468, 0.035352177917957306,
      0.00631742412224412, -0.08062314242124557, 0.06178295984864235, -0.07964615523815155,
      0.042630430310964584, -0.0419139489531517, 0.027039319276809692, -0.019266406074166298,
      -0.0033308197744190693, -0.0016553933965042233, -0.040361665189266205, 0.0021576895378530025,
      0.005679331254214048, 0.08614777028560638, 0.00555637339130044, 0.014928029850125313,
      0.06827901303768158, -0.02289441041648388, 0.07515852153301239, 0.08666031807661057,
      -0.020163314417004585, -0.0470927432179451, -0.04049120470881462, 0.02767891250550747,
      -0.010852821171283722, -0.06450832635164261, 0.05455167964100838, -0.025871314108371735,
      0.018130963668227196, -0.07254032045602798, -0.0786990150809288, -0.05460327863693237,
      -0.045906953513622284, 0.10183093696832657, -0.05566660687327385, -0.039166200906038284,
      -0.04597620293498039, 0.050217412412166595, -0.008187987841665745, -0.021754741668701172,
      0.005193138029426336, -0.012472778558731079, -0.026574917137622833, -0.043318554759025574,
      -0.012593305669724941, -0.03016524948179722, -0.06657224148511887, -0.0869523361325264,
      -0.1282086968421936, -0.002000437816604972, -0.13680963218212128, -0.002569084521383047,
      -0.17635942995548248, -0.09149392694234848, 0.027589913457632065, 0.021637389436364174,
      -0.057078104466199875, -0.046359214931726456, 0.001635290216654539, 0.0769069492816925],
     [-0.10523050278425217, 0.010401910170912743, -0.04030556604266167, -0.0654778704047203,
      -0.008853250183165073, -0.1127457544207573, -0.06869369000196457, 0.05027776584029198,
      0.04725142568349838, -0.044120609760284424, -4.2090294300578535e-05, -0.020551616325974464,
      0.10571999847888947, 0.07705596089363098, -0.028265785425901413, 0.06332912296056747,
      -0.05261353403329849, -0.059465985745191574, 0.009592815302312374, 0.010193904861807823,
      0.12938988208770752, 0.03777427598834038, 0.10507067292928696, 0.10450512170791626,
      -0.078667551279068, 0.06497236341238022, 0.02579355426132679, 0.0723201259970665,
      0.08786890655755997, 0.07981901615858078, -0.011199143715202808, 0.06230894476175308,
      0.04102976620197296, -0.016799835488200188, -0.057652588933706284, -0.05879289284348488,
      -0.06104493886232376, 0.07356802374124527, -0.03770376369357109, -0.00566458236426115,
      -0.08669393509626389, -0.04840312525629997, -0.0353122241795063, 0.10944937914609909,
      0.08241631835699081, 0.06568527966737747, -0.036598894745111465, 0.0732453316450119,
      0.08082315325737, -0.0751054584980011, 0.09057872742414474, 0.007709193974733353,
      -0.03560810908675194, 0.00596402445808053, -0.03342555835843086, 0.08047298341989517,
      0.08775492757558823, 0.03515790402889252, 0.05820723995566368, 0.08533982932567596,
      0.02599968947470188, -0.0917695015668869, -0.06526420265436172, 0.012789073400199413,
      -0.04733997955918312, -0.04124986380338669, 0.022901449352502823, -0.06411995738744736,
      0.01863451488316059, -0.08472815901041031, 0.0772157609462738, -0.019570866599678993,
      -0.006955188233405352, 0.021735524758696556, 0.04942818731069565, -0.023620223626494408,
      0.0056631918996572495, 0.08360088616609573, 0.0020995503291487694, -0.008250873535871506,
      0.05678751692175865, -0.0026905392296612263, 0.050026558339595795, -0.028095023706555367,
      -0.09741520881652832, 0.000551518052816391, -0.0323198065161705, -0.015431507490575314,
      0.06445389986038208, -0.037438955157995224, -0.035477444529533386, 0.07183553278446198,
      -0.12531667947769165, 0.06121179834008217, -0.11842189729213715, -0.10219115018844604,
      0.044442515820264816, -0.03143208846449852, 0.023439642041921616, -0.002219861838966608,
      -0.1268642246723175, 0.003101637354120612, -0.0641610324382782, 0.06048620119690895,
      0.06853644549846649, -0.03295860439538956, -0.05897516384720802, -0.05800938606262207],
     [-0.036442894488573074, 0.04741812124848366, 0.05643788352608681, -0.018771158531308174,
      0.05012956261634827, -0.06550739705562592, -0.009016948752105236, -0.06897258013486862,
      -0.06174599006772041, 0.06722327321767807, -0.11020868271589279, 0.05658587068319321,
      6.973308336455375e-05, -0.08022947609424591, 0.018228787928819656, 0.04234394058585167,
      0.047392312437295914, 0.049865249544382095, -0.11098892986774445, 0.042267102748155594,
      -0.009316835552453995, -0.029194200411438942, -0.019680170342326164, 0.011814289726316929,
      -0.04605668783187866, 0.1008322685956955, 0.10452565550804138, -0.07105077058076859,
      -0.05801588296890259, 0.018555374816060066, 0.07853412628173828, 0.04414939135313034,
      0.040057867765426636, -0.04808288440108299, 0.06507992744445801, 0.00983788538724184,
      -0.037693653255701065, 0.006271704565733671, 0.07874355465173721, 0.018944527953863144,
      0.0054881758987903595, 0.0444357767701149, 0.006708740256726742, 0.019581645727157593,
      0.03761369735002518, -0.0528811477124691, -0.03953985869884491, -0.08816657960414886,
      -0.0203864686191082, -0.07907058298587799, -0.03162364289164543, 0.028703592717647552,
      0.01781763695180416, 0.04233809933066368, -0.07851211726665497, 0.001102484529837966,
      0.0361543633043766, 0.03192788362503052, 0.04259542375802994, 0.04559312015771866,
      0.09828780591487885, 0.08599098771810532, -0.037957411259412766, -0.01979987323284149,
      0.006664267275482416, 0.03530532866716385, 0.01800750009715557, -0.04516763985157013,
      0.03312560170888901, -0.025351224467158318, -0.0949324443936348, -0.02007661573588848,
      -0.07964722812175751, -0.0010110223665833473, 0.10534870624542236, -0.05921023339033127,
      -0.015938762575387955, -0.05226127430796623, 0.08237321674823761, 0.03488226234912872,
      0.031972773373126984, 0.042186934500932693, 0.07883567363023758, -0.011025051586329937,
      -0.010158064775168896, -0.004167796578258276, -0.06248430907726288, -0.07195010781288147,
      0.12188573181629181, -0.0780821219086647, -0.028594505041837692, -0.06817235052585602,
      -0.06404536217451096, 0.05479397252202034, 0.01896659843623638, -0.07034263759851456,
      0.0354597233235836, -0.08529286086559296, 0.06162619963288307, -0.005686103831976652,
      -0.0027247765101492405, -0.0412253700196743, 0.05713842809200287, -0.022991392761468887,
      0.04373903200030327, -0.001170797972008586, -0.06720542162656784, -0.03994449973106384],
     [0.020186912268400192, -0.04094061255455017, 0.033246491104364395, 0.039846498519182205,
      -0.09426259249448776, -0.12415221333503723, -0.06297939270734787, 0.0015948066720739007,
      0.0010562869720160961, 0.028698040172457695, -0.04556215927004814, 0.01266360655426979,
      0.061403512954711914, -0.017434440553188324, 0.0614098496735096, -0.08624773472547531,
      0.03932645544409752, -0.004139143042266369, 0.03778669610619545, 0.09844750165939331,
      -0.031167635694146156, 0.0886063277721405, 0.11980723589658737, 0.1416078507900238,
      0.03442496061325073, -0.07265878468751907, -0.002657876815646887, 0.03664786368608475,
      0.046620775014162064, -0.11245471239089966, 0.06353060901165009, 0.015374324284493923,
      -0.04738996550440788, -0.03298916295170784, 0.00340695190243423, -0.010433992370963097,
      0.08815298229455948, 0.029909992590546608, 0.0848555713891983, -0.06824420392513275,
      0.021046822890639305, 0.041831206530332565, -0.04280722513794899, 0.04262252151966095,
      -0.05168944597244263, 0.10218594968318939, -0.01783353090286255, 0.024704821407794952,
      -0.04424700140953064, 0.06923108547925949, -0.06745988875627518, 0.04040490463376045,
      -0.028321970254182816, 0.03384029492735863, 0.05435740202665329, -0.04602287337183952,
      -0.0533403716981411, 0.027011813595891, -0.06750919669866562, 0.027642859145998955,
      -0.0067099640145897865, 0.00986478291451931, -0.03079981356859207, -0.03131240978837013,
      -0.019677359610795975, 0.07900361716747284, 0.06887112557888031, 0.07357869297266006,
      0.06755555421113968, -0.07740584760904312, 0.04957059770822525, -0.04441703110933304,
      -0.003511914052069187, -0.029624586924910545, -0.027661575004458427, 0.07076061517000198,
      0.038565970957279205, -0.008407561108469963, 0.029216166585683823, -0.03710532933473587,
      0.045332688838243484, 0.05262123420834541, 0.038525257259607315, 0.08868823200464249,
      0.09390754252672195, -0.028950508683919907, -0.038442060351371765, -0.04320476949214935,
      0.0651780217885971, -0.08202710747718811, -0.07396609336137772, -0.04728385806083679,
      -0.13187651336193085, -0.08926757425069809, -0.08521096408367157, -0.13010649383068085,
      -0.05541123077273369, 0.033892739564180374, -0.11504985392093658, -0.002848620293661952,
      -0.11241220682859421, -0.059564173221588135, 0.11122973263263702, 0.008883549831807613,
      0.07840662449598312, -0.07734882831573486, 0.03845519945025444, -0.007696275133639574],
     [-0.090034119784832, 0.003372984239831567, 0.041802313178777695, -0.07863222807645798,
      -0.08050033450126648, -0.021631106734275818, -0.01188794057816267, -0.013279948383569717,
      -0.059900738298892975, -0.07252322137355804, 0.0715060755610466, 0.036283448338508606,
      0.02036021091043949, 0.026480508968234062, 0.025471340864896774, -0.08728156238794327,
      -0.027887418866157532, 0.02687302976846695, -0.054568711668252945, 0.019457269459962845,
      0.03963803872466087, 0.07679479569196701, 0.030668707564473152, -0.08154554665088654,
      -0.01614522747695446, 0.0758589506149292, -0.04711058735847473, -0.024090999737381935,
      -0.022343454882502556, 0.05805766582489014, 0.06759416311979294, 0.09529922902584076,
      0.015003782697021961, 0.04256967082619667, -0.0863395482301712, -0.012014584615826607,
      0.0245202723890543, 0.014104471541941166, 0.059117238968610764, 0.012558291666209698,
      -0.047590598464012146, 0.05311378091573715, 0.08656521141529083, -0.03751735761761665,
      0.012211651541292667, 0.09573592990636826, 0.02247520536184311, -0.06314358115196228,
      -0.06900767982006073, -0.025960799306631088, 0.04653758928179741, 0.08115307241678238,
      -0.08098170161247253, 0.02010924182832241, 0.03689955174922943, 0.09299451112747192,
      0.008194644004106522, 0.08643338829278946, -0.062271229922771454, -0.031017674133181572,
      -0.051128216087818146, 0.04053876921534538, -0.07475772500038147, -0.08901629596948624,
      -0.011595243588089943, 0.06005275622010231, -0.020878339186310768, -0.08035474270582199,
      0.059397488832473755, 0.0016055282903835177, -0.07789821177721024, 0.0551023967564106,
      -0.02573176845908165, 0.020766060799360275, -0.05182421952486038, 0.07741785049438477,
      0.04300548508763313, 0.0007270256755873561, 0.022924907505512238, 0.006788493599742651,
      0.04833361506462097, -0.018100140616297722, -0.07154271751642227, -0.004476465284824371,
      0.00819740816950798, 0.015347836539149284, 0.08112191408872604, 0.001853178022429347,
      -0.04710236191749573, -0.003964046016335487, -0.0621158592402935, -0.02892080321907997,
      0.05247099697589874, -0.09244102239608765, 0.01640068180859089, -0.09300598502159119,
      -0.06586259603500366, -0.014392953366041183, 0.007171735167503357, 0.008352317847311497,
      -0.04698529839515686, -0.11867843568325043, 0.03762868419289589, 0.010065785609185696,
      -0.011028856039047241, -0.08035805821418762, 0.034617673605680466, 0.015142695978283882],
     [0.008925294503569603, -0.10163924098014832, -0.11399127542972565, 0.007360444404184818,
      0.07652908563613892, -0.059400785714387894, -0.030305085703730583, -0.08852608501911163,
      0.060297153890132904, 0.00592860346660018, -0.030136482790112495, 0.048777155578136444,
      0.002736713271588087, 0.022113896906375885, 0.016951194033026695, 0.07582508027553558,
      0.03062816895544529, -0.03898383304476738, -0.09377110749483109, -0.026353253051638603,
      0.05514932796359062, -0.07947137951850891, 0.0021454649977385998, -0.03835457190871239,
      0.005572246853262186, 0.01601308211684227, 0.036276619881391525, -0.02157687209546566,
      0.006803690455853939, 0.05025050416588783, -0.062333159148693085, 0.06405933946371078,
      -0.03592532128095627, 0.04813589155673981, 0.056280627846717834, 0.08313269168138504,
      0.042565010488033295, 0.05237999185919762, -0.10293178260326385, -0.069565050303936,
      0.030478069558739662, -0.08165594935417175, 0.05872636288404465, 0.028495268896222115,
      0.07993151247501373, -0.06416811794042587, -0.021474821493029594, 0.004473069682717323,
      0.03533722832798958, 0.040156289935112, 0.08654427528381348, -0.04808034747838974,
      0.018286237493157387, -0.09149572253227234, -0.017874911427497864, -0.05730171874165535,
      0.019155824556946754, 0.042153116315603256, 0.08728573471307755, -0.042571064084768295,
      -0.037389084696769714, -0.0742407888174057, -0.035805005580186844, 0.006744480226188898,
      -0.04420777037739754, 0.10083191096782684, -0.00782217737287283, -0.0773639902472496,
      0.04316779971122742, 0.05165141820907593, 0.08497510850429535, 0.085298553109169,
      0.05079381540417671, 0.0709586963057518, -0.0805189236998558, -0.02901618555188179,
      -0.06941399723291397, 0.06929853558540344, -0.07405393570661545, -0.03531574457883835,
      -0.05112938955426216, 0.020123137161135674, -0.019875245168805122, 0.021280068904161453,
      -0.09320595860481262, -0.02356531098484993, 0.07957825809717178, 0.016791319474577904,
      -0.08200052380561829, 0.036633431911468506, 0.080134816467762, -0.0453011691570282,
      -0.08623243123292923, -0.11057959496974945, -0.0274850782006979, 0.026080917567014694,
      0.018420223146677017, 0.014614645391702652, -0.07467925548553467, 0.04475897550582886,
      -0.07368423044681549, 0.05386167764663696, -0.02930257096886635, 0.08154835551977158,
      -0.023217303678393364, 0.018663998693227768, 0.1152828261256218, -0.06690806150436401],
     [-0.12306506186723709, -0.021361324936151505, -0.15264329314231873, -0.04938972368836403,
      -0.009946094825863838, -0.1598639041185379, -0.09148930013179779, 0.008779019117355347,
      -0.06680677831172943, 0.001688699354417622, -0.058658767491579056, -0.10982906073331833,
      0.01830485835671425, 0.10412812978029251, 0.016650818288326263, -0.029701633378863335,
      0.007476487662643194, 0.007767354603856802, 0.06149667501449585, 0.07403318583965302,
      0.06606490910053253, 0.06600011140108109, -0.005618507973849773, 0.08235733211040497,
      0.005783163942396641, 0.011942149139940739, -0.028719237074255943, 0.0022829335648566484,
      0.05495217442512512, 0.004195254761725664, -0.051557835191488266, -0.03498139604926109,
      -0.026457905769348145, -0.042634688317775726, -0.08431074768304825, 0.004816314205527306,
      0.05786895006895065, 0.011000515893101692, 0.07469504326581955, -0.08378845453262329,
      -0.0011528943432494998, -0.0181808490306139, 0.040476247668266296, -0.0017600766150280833,
      0.08936053514480591, -0.10429682582616806, -0.026367073878645897, 0.07482137531042099,
      0.034227289259433746, -0.009180666878819466, 0.07122670114040375, -0.021478530019521713,
      0.07087720185518265, 0.015079168602824211, 0.10686960816383362, 0.09130484610795975,
      -0.03607872128486633, 0.009884483180940151, -0.02980896830558777, -0.04102589190006256,
      -0.01187752652913332, -0.06795869022607803, 0.03849244490265846, 0.02950444631278515,
      0.056871093809604645, 0.007244880776852369, 0.011617016047239304, 0.03966239094734192,
      0.029016314074397087, 0.0632525160908699, 0.033752042800188065, -0.04705873131752014,
      -0.0646815225481987, -0.030454374849796295, -0.08397633582353592, -0.043763987720012665,
      0.03510404750704765, -0.009701356291770935, -0.019512606784701347, -0.006801148410886526,
      -0.021373499184846878, -0.014688173308968544, 0.014286848716437817, -0.0396592803299427,
      0.034892357885837555, 0.047140318900346756, -0.06822559982538223, 0.08555001020431519,
      0.07304077595472336, -0.07099289447069168, -0.1539866179227829, -0.019152387976646423,
      -0.060423873364925385, -0.13975834846496582, 0.024163102731108665, -0.015868423506617546,
      0.007388690486550331, -0.04639241471886635, -0.11834944039583206, -0.06383585184812546,
      -0.04666033014655113, -0.10785898566246033, -0.009311458095908165, -0.025313839316368103,
      -0.019910678267478943, -0.04358312860131264, -0.02170809730887413, 0.048611681908369064],
     [-0.046712473034858704, 0.04586409404873848, -0.02095973677933216, -0.038026127964258194,
      -0.053538478910923004, -0.04347001761198044, -0.05877896398305893, -0.0020552503410726786,
      0.03580235317349434, -0.05995335057377815, 0.02822473645210266, 0.039029043167829514,
      0.05142349377274513, 0.06467428803443909, -0.06035745143890381, 0.10996881127357483,
      0.03649270534515381, -0.004249229561537504, -0.09288187325000763, 0.058342985808849335,
      -0.07499969750642776, -0.054796405136585236, -0.05412636324763298, -0.021888883784413338,
      0.10039709508419037, 0.08871904015541077, -0.04962436482310295, -0.011540479958057404,
      -0.019527321681380272, 0.015735842287540436, -0.08961913734674454, -0.08799660950899124,
      0.05932111665606499, -0.015912998467683792, -0.029064778238534927, -0.0435539074242115,
      -0.0007371838437393308, -0.008111342787742615, -0.028615958988666534, 0.015671199187636375,
      -0.06606511026620865, -0.007548135239630938, 0.0048268879763782024, -0.09789880365133286,
      0.009095283225178719, -0.026335636153817177, -0.0771336704492569, -0.021075936034321785,
      -0.04661640524864197, -0.08612436801195145, 0.09743566811084747, 0.04209296777844429,
      0.07868029922246933, -0.04167577996850014, 0.07985210418701172, 0.020865125581622124,
      0.05340692773461342, -0.09351252019405365, -0.05786977708339691, 0.017484737560153008,
      -0.018277432769536972, -0.01086720172315836, -0.006695637479424477, -0.05014194920659065,
      0.016778668388724327, -0.04984767362475395, 0.07376774400472641, -0.0631239041686058,
      -0.02931104600429535, -0.005041136872023344, 0.018251003697514534, -0.07835308462381363,
      -0.015011291019618511, -0.10304554551839828, -0.10323596000671387, 0.008485304191708565,
      0.05784141644835472, -0.029259514063596725, 0.08457852900028229, -0.038305945694446564,
      -0.05039995536208153, 0.041485533118247986, -0.10994867980480194, 0.05533864349126816,
      0.07484395056962967, 0.07001234591007233, 0.027877897024154663, 0.0733397826552391,
      0.057895250618457794, -0.06299420446157455, -0.05966172367334366, -0.06887633353471756,
      0.11716848611831665, 0.06534455716609955, -0.01840296760201454, -0.009068057872354984,
      0.09962907433509827, 0.038159530609846115, 0.10318812727928162, 0.08403541892766953,
      0.051170799881219864, 0.023562995716929436, -0.08548944443464279, -0.07556271553039551,
      -0.04168247431516647, -0.08540289849042892, -0.026939310133457184, -0.07031700015068054],
     [-0.04619338735938072, -0.03969603031873703, -0.07614455372095108, 0.041436731815338135,
      -0.12604597210884094, -0.1161772683262825, -0.017322173342108727, 0.01880345679819584,
      -0.0680980309844017, -0.08707021921873093, -0.08761246502399445, 0.05858710780739784,
      0.05473441630601883, 0.04116455093026161, 0.006663904991000891, 0.0821441262960434,
      0.1057378277182579, -0.07783110439777374, 0.146391823887825, 0.024593062698841095,
      0.036157846450805664, -0.015253647230565548, 0.07203027606010437, -0.014685350470244884,
      0.09402637183666229, -0.037936124950647354, -0.03758322447538376, -0.05170978233218193,
      -0.009368564933538437, -0.005435100290924311, 0.0074888258241117, -0.058848924934864044,
      -0.07928220927715302, 0.050739239901304245, -0.0550316646695137, 0.0034081176854670048,
      -0.05973345786333084, -0.056714095175266266, 0.00502540310844779, -0.005815504118800163,
      -0.02253892831504345, -0.0017851815791800618, -0.06761142611503601, 0.04635564610362053,
      -0.04606574773788452, -0.09563397616147995, 0.022962016984820366, -0.029562681913375854,
      0.08811873197555542, 0.060505032539367676, 0.0761093720793724, 0.028760967776179314,
      -0.02552971802651882, -0.04908284917473793, 0.03595349192619324, -0.02480502612888813,
      0.014073826372623444, -0.06942065060138702, -0.03180312365293503, -0.0336761400103569,
      -0.004032985307276249, 0.00797177292406559, -0.04647941514849663, -0.02650608867406845,
      -0.0372222401201725, 0.07465945929288864, -0.07520098984241486, -0.06159227341413498,
      -0.04784611240029335, 0.07066138833761215, 0.06951173394918442, 0.045290157198905945,
      -0.05400563403964043, -0.027230141684412956, 0.0035687698982656, -0.06646516919136047,
      0.04957406222820282, -0.028907770290970802, -0.08184255659580231, 0.04071309417486191,
      0.05867210775613785, 0.02658323384821415, -0.018925059586763382, -0.04431924596428871,
      0.06102289631962776, -0.08940823376178741, -0.0638512596487999, -0.07252325862646103,
      0.05016368255019188, -0.015775863081216812, -0.12869517505168915, -0.015748998150229454,
      0.04682128503918648, -0.02911224588751793, -0.07992330193519592, -0.02510858327150345,
      0.021436739712953568, -0.12074872106313705, -0.11846787482500076, -0.07506388425827026,
      0.03374920040369034, -0.13727109134197235, 0.08497916907072067, 0.09489913284778595,
      0.012478343211114407, 0.018479766324162483, 0.05677300691604614, -0.08017665147781372],
     [0.012677250429987907, 0.04326438531279564, -0.0699528232216835, -0.003918755333870649,
      0.09844420850276947, 0.04287240281701088, -0.03368857875466347, -0.02546764351427555,
      -0.06881234049797058, 0.00745766144245863, 0.06381085515022278, -0.011061428114771843,
      0.08480715751647949, -0.0740591511130333, -0.06696023792028427, 0.039544135332107544,
      0.00953313335776329, 0.0017017261125147343, -0.051397696137428284, 0.0016462149797007442,
      0.0960526242852211, 0.03334012255072594, -0.03590242192149162, -0.04079414904117584,
      -0.05991633981466293, -0.02807229571044445, 0.033578090369701385, -0.06849579513072968,
      -0.09714700281620026, 0.03594743087887764, 0.06712847948074341, 0.06922153383493423,
      -0.05902924761176109, 0.011010861955583096, 0.06928491592407227, -0.01578456535935402,
      0.00212867371737957, -0.032625533640384674, -0.06633662432432175, 0.05145501345396042,
      0.05013132095336914, -0.06776899099349976, 0.04752001911401749, -0.013016927056014538,
      0.08287113904953003, -0.02314283698797226, 0.07203047722578049, -0.021502681076526642,
      0.047416314482688904, -0.08098349720239639, -0.021881822496652603, 0.07528341561555862,
      -0.02986430749297142, -0.04327903687953949, -0.02329183556139469, 0.0045339264906942844,
      -0.0565611831843853, -0.02713949792087078, -0.031857557594776154, 0.048896390944719315,
      -0.07927582412958145, -0.049248114228248596, -0.0696839839220047, -0.03142905235290527,
      0.031082462519407272, 0.07016485929489136, -0.08431053906679153, 0.017418021336197853,
      -0.043373242020606995, 0.013141406700015068, 0.058011166751384735, 0.08615943044424057,
      0.015694089233875275, -0.03893432021141052, 0.03234431520104408, 0.05070678889751434,
      -0.07445502281188965, 0.005989940371364355, 0.022110963240265846, 0.08052404224872589,
      -0.04139476269483566, 0.08727045357227325, 0.04124196991324425, -0.06910864263772964,
      -0.03003728948533535, -0.03735647723078728, -0.04399457573890686, 0.05673318728804588,
      -0.07461445778608322, -0.019590018317103386, 0.027182593941688538, 0.04073936864733696,
      0.06019189581274986, 0.018192535266280174, 0.02901582419872284, -0.045298121869564056,
      0.01242460124194622, 0.03133663535118103, -0.059984076768159866, 0.06538832932710648,
      -0.06313049048185349, -0.0722738578915596, -0.05197391286492348, 0.09376958757638931,
      -0.07953616976737976, -0.07658494263887405, 0.0039678229950368404, -0.0028259805403649807],
     [0.039594803005456924, 0.052240654826164246, -0.08004093170166016, 0.07680986821651459,
      -0.07909160852432251, 0.07227486371994019, 0.06670120358467102, -0.01953502558171749,
      0.025904396548867226, 0.00686006061732769, -0.07153376936912537, 0.0027634541038423777,
      -0.10219014436006546, 0.018024874851107597, 0.05697057023644447, 0.07132512331008911,
      0.03094622679054737, 0.0009761337423697114, 0.032987821847200394, 0.086140476167202,
      -0.07835744321346283, 0.01470556203275919, 0.06248234957456589, -0.011626643128693104,
      0.008479258976876736, 0.08639593422412872, 0.0855695977807045, 0.029265768826007843,
      -0.1256455034017563, -0.11770030111074448, 0.05016641318798065, 0.007993476465344429,
      0.07564935088157654, 0.08766932040452957, -0.02861451357603073, 0.060484379529953,
      0.07045529782772064, -0.05989309400320053, -0.05054694041609764, 0.08795971423387527,
      0.018742822110652924, 0.07545559853315353, -0.07102297991514206, -0.06342559307813644,
      0.04766746610403061, -0.07283812016248703, 0.06364581733942032, -0.03647613152861595,
      0.08180635422468185, 0.08439435064792633, -0.030134551227092743, -0.07397256791591644,
      0.0789891928434372, 0.08132939785718918, -0.05894233658909798, 0.023001553490757942,
      0.07336386293172836, 0.06714184582233429, 0.032035425305366516, 0.02830655314028263,
      0.07682787626981735, 0.027248023077845573, 0.0007710321224294603, -0.07832570374011993,
      -0.0011307535460218787, 0.06954246759414673, 0.015223463997244835, -0.06105269119143486,
      0.06576632708311081, 0.05026084929704666, -0.029707446694374084, 0.06952181458473206,
      -0.0516754649579525, -0.07643060386180878, 0.028491497039794922, 0.024328116327524185,
      -0.021120304241776466, 0.01554050762206316, 0.06629050523042679, 0.06465945392847061,
      0.07976500689983368, 0.0604747049510479, 0.0327930673956871, -0.005179733969271183,
      0.06372486799955368, 0.023469936102628708, -0.04305436834692955, 0.08770004659891129,
      0.06839444488286972, -0.018058478832244873, 0.01040312647819519, -0.07278619706630707,
      0.09812562167644501, 0.10022463649511337, 0.0892309695482254, 0.06303007900714874,
      0.02936757355928421, 0.05056637153029442, 0.0893324688076973, -0.053543440997600555,
      0.08051209896802902, 0.0012554905842989683, 0.04667405039072037, 0.037079378962516785,
      -0.05669067054986954, -0.07383280247449875, -0.07217266410589218, -0.03684302791953087],
     [0.0012517698341980577, 0.11919514834880829, 0.07643865048885345, 0.10906822979450226,
      0.07944542914628983, 0.08181754499673843, -0.00274991849437356, 0.06654369086027145,
      0.03033931739628315, -0.02925475686788559, 0.08382448554039001, 0.025628484785556793,
      -0.02831888571381569, 0.03633540868759155, -0.10255494713783264, -0.005478673614561558,
      -0.1128525361418724, 0.0631842166185379, -0.03809506446123123, -0.08632845431566238,
      0.02709565870463848, 0.08173656463623047, 0.11581455171108246, 0.10745331645011902,
      0.04825595021247864, -0.00040825881296768785, -0.04490652680397034, 0.03945019096136093,
      -0.01799021288752556, -0.016404349356889725, -0.11227594316005707, -0.058497458696365356,
      -0.09159359335899353, 0.0017756040906533599, -0.08345093578100204, 0.05007604882121086,
      -0.05062034726142883, -0.029733816161751747, 0.026098990812897682, -0.06979027390480042,
      -0.0750625729560852, -0.027299262583255768, -0.0023849059361964464, -0.08759415149688721,
      0.09741947799921036, -0.05137207731604576, 0.043462123721838, -0.062366362661123276,
      0.022213079035282135, -0.05146532878279686, 0.07037457823753357, -0.023960264399647713,
      0.020964231342077255, -0.08193174749612808, -0.06581716239452362, -0.0908697247505188,
      -0.08337346464395523, 0.08000543713569641, 0.06897301226854324, 0.0914159044623375,
      0.024213479831814766, -0.08595090359449387, -0.04405825957655907, 0.01629807986319065,
      0.07336314022541046, -0.0714109018445015, 0.04107898846268654, -0.06752235442399979,
      0.034143246710300446, -0.09838502109050751, -0.028913527727127075, -0.007614555303007364,
      -0.06623480468988419, 0.07103799283504486, -0.055051736533641815, 0.029420141130685806,
      -0.058311667293310165, -0.08137296140193939, -0.07625690847635269, -0.06249747425317764,
      0.024255283176898956, -0.030262218788266182, 0.08224403858184814, 0.06903235614299774,
      -0.03975055739283562, -0.04163452610373497, 0.06584188342094421, 0.04037325829267502,
      0.006751085165888071, 0.06096014007925987, -0.07584700733423233, -0.04766518250107765,
      -0.08769627660512924, -0.06465603411197662, 0.044422414153814316, 0.056275565177202225,
      -0.055516134947538376, 0.027841724455356598, 0.04165994003415108, 0.09458063542842865,
      0.05079338327050209, -0.05226082354784012, 0.04681830853223801, 0.002094496274366975,
      -0.0095664719119668, -0.0461609847843647, 0.05276195704936981, -0.035132553428411484],
     [-0.05229886248707771, 0.05301795154809952, -0.07104748487472534, -0.049073364585638046,
      -0.08314210921525955, 0.008494719862937927, -0.07263312488794327, 0.04795241728425026,
      0.07252390682697296, -0.011888612993061543, -0.05165412276983261, -0.052149057388305664,
      -0.06337036937475204, -0.03908652812242508, 0.0701168105006218, -0.10970280319452286,
      -0.030398298054933548, 0.042772404849529266, -0.05916699767112732, -0.010262995958328247,
      -0.056860774755477905, 0.0030517505947500467, -0.08142056316137314, 0.005542652681469917,
      0.01948697865009308, 0.062178339809179306, -0.006886053364723921, -0.01974598504602909,
      -0.02173476107418537, -0.034495897591114044, 0.02702517807483673, -0.07674279808998108,
      0.03154030442237854, 0.002725717844441533, 0.05936671420931816, 0.029590047895908356,
      0.07953641563653946, -0.0038843443617224693, 0.07727482914924622, -0.05223358795046806,
      0.07152851670980453, -0.026434781029820442, -0.05933461710810661, -0.09111420065164566,
      0.08409950882196426, -0.04482204467058182, -0.076967753469944, -0.04439914599061012,
      -0.0026164879091084003, 0.05559588596224785, 0.04252414032816887, -0.07899460196495056,
      -0.012738917022943497, -0.05601010099053383, 0.04193965718150139, -0.008553972467780113,
      -0.08033322542905807, 0.0035660944413393736, -0.07438898086547852, -0.009695461019873619,
      -0.09375657141208649, -0.05426495894789696, -0.04954918101429939, -0.06024378165602684,
      -0.07339295744895935, -0.007633278612047434, 0.013786526396870613, 0.06642837077379227,
      0.07274940609931946, -0.01806996576488018, 0.0764104425907135, 0.08808588236570358,
      0.013952458277344704, -0.02668164111673832, -0.07812264561653137, 0.09642644971609116,
      -0.048836600035429, -0.09596209228038788, 0.058402832597494125, 0.0795862227678299,
      0.08949492871761322, -0.022255048155784607, 0.09554223716259003, -0.05983210727572441,
      -0.03150058165192604, 0.04329310357570648, -0.01994839310646057, 0.06705526262521744,
      0.0391593798995018, 0.0755874291062355, -0.04929601028561592, 0.08746993541717529,
      0.03843945637345314, 0.032649099826812744, 0.03572915494441986, -0.020360589027404785,
      0.09712853282690048, -0.07597596198320389, -0.04329381883144379, 0.07532758265733719,
      0.0897151306271553, 0.011381442658603191, -0.04745786637067795, -0.022821126505732536,
      -0.047172583639621735, -0.015732839703559875, 0.02852182649075985, -0.08951927721500397],
     [-0.06185277923941612, 0.08694278448820114, 0.09864103049039841, -0.03421446308493614,
      -0.06731972843408585, 0.0538514107465744, -0.07388637959957123, -0.04357488825917244,
      0.08988557010889053, 0.10715954750776291, 0.025626877322793007, -0.001516798511147499,
      -0.0947020947933197, -0.006801209412515163, -0.005263626109808683, -0.120825856924057,
      0.015696756541728973, -0.07915299385786057, 0.034824687987565994, 0.007978454232215881,
      0.041798774152994156, -0.021797943860292435, 0.04514690861105919, -0.008367148227989674,
      -0.04295036569237709, -0.07787933945655823, -0.09495639055967331, 0.0745377391576767,
      -0.06455233693122864, -0.06279014050960541, -0.08661538362503052, 0.04800331965088844,
      -0.012923812493681908, -0.032490190118551254, -0.0821164920926094, 0.0715719610452652,
      -0.0688619464635849, 0.01850910112261772, -0.03421393036842346, -0.01232718862593174,
      -0.034090686589479446, -0.010903979651629925, 0.06363218277692795, -0.02457420714199543,
      0.036650508642196655, -0.07439861446619034, 0.03348414972424507, 0.0816897377371788,
      -0.018617356196045876, -0.08191478997468948, 0.05829603224992752, 0.05167300999164581,
      0.060784440487623215, 0.01845044083893299, -0.08444865792989731, -0.039893344044685364,
      -0.06844527274370193, -0.06821057200431824, -0.09989459067583084, 0.02971387840807438,
      -0.07308381795883179, -0.02018469199538231, -0.09147705882787704, -0.03673820197582245,
      -0.06371007859706879, 0.0911700427532196, -0.029679633677005768, -0.033042144030332565,
      0.06348729133605957, -0.0734066441655159, -0.02299676649272442, -0.03444347158074379,
      0.018610404804348946, -0.0028327179607003927, -0.06988682597875595, 0.030085885897278786,
      0.04172196239233017, -0.06342123448848724, -0.08286190032958984, -0.07835835963487625,
      0.04645204544067383, -0.0007776168058626354, -0.002258120570331812, 0.05429040640592575,
      -0.07729164510965347, -0.011167090386152267, 0.015313293784856796, 0.08663886785507202,
      0.0845295786857605, -0.08388105034828186, 0.05793727561831474, 0.08307137340307236,
      -0.001842521596699953, 0.04515043646097183, -0.06485457718372345, -0.05630122497677803,
      -0.08252712339162827, 0.013012251816689968, -0.1052914559841156, -0.03336162492632866,
      -0.03370906412601471, 0.0019194897031411529, 0.08378393948078156, -0.08416277170181274,
      0.029565133154392242, -0.03146862983703613, 0.025396911427378654, -0.08506685495376587],
     [0.0006589130498468876, -0.10674040764570236, -0.012064016424119473, -0.06136104092001915,
      -0.09832628071308136, -0.03820927068591118, -0.004435442388057709, -0.07748090475797653,
      -0.09613875299692154, 0.003756485879421234, 0.00829335954040289, 0.04890485852956772,
      -0.014669657684862614, 0.06240880489349365, 0.06779824197292328, 0.03269054740667343,
      -0.026300417259335518, -0.042111653834581375, 0.13505031168460846, -1.5156660083448514e-05,
      0.07362660020589828, 0.10149086266756058, 0.022012514993548393, -0.04538911581039429,
      0.006182130426168442, -0.07098235189914703, -0.08024301379919052, 0.07390560209751129,
      0.05174015089869499, -0.11148446053266525, 0.028895460069179535, 0.08613680303096771,
      0.02504080720245838, -0.09173865616321564, 0.03812076523900032, -0.09932469576597214,
      0.04948467016220093, 0.01187571045011282, -0.04037070646882057, -0.06085634231567383,
      0.08202898502349854, 0.069669209420681, 0.061805129051208496, -0.011601190082728863,
      0.09594345092773438, -0.0026275431737303734, 0.06252916902303696, -0.010418384335935116,
      0.07673507183790207, -0.06241718307137489, -0.001559442956931889, -0.01952934078872204,
      -0.0989503413438797, 0.04400645196437836, -0.0032777932938188314, -0.01773197203874588,
      0.1098824068903923, -0.015797263011336327, -0.09679284691810608, -0.014909228309988976,
      0.07401657849550247, -0.015407540835440159, 0.09780498594045639, 0.0768444761633873,
      -0.06482066214084625, -0.07647310942411423, -0.0705932155251503, -0.08460555970668793,
      0.0785813182592392, -0.006222852505743504, 0.04281245172023773, 0.09623175114393234,
      -0.018185842782258987, -0.08477408438920975, -0.05488342046737671, 0.02515881508588791,
      0.07852958887815475, -0.06639821827411652, 0.023773662745952606, -0.04339759796857834,
      -0.006288849748671055, -0.0005734747392125428, -0.07993040233850479, -0.08172357082366943,
      -0.05816091224551201, 0.054503243416547775, -0.07449423521757126, 0.039424192160367966,
      0.03718506172299385, -0.07784324139356613, 0.020145196467638016, -0.0456123948097229,
      -0.1255878210067749, 0.00792717281728983, 0.05190158262848854, 0.07255883514881134,
      -0.019855136051774025, -0.006183508317917585, -0.09625569730997086, -0.03498844802379608,
      0.02521073818206787, -0.10057838261127472, 0.009328791871666908, 0.1114528477191925,
      -0.08215388655662537, -0.05938563123345375, 0.007502933032810688, 0.03616701439023018],
     [0.023784596472978592, 0.07336272299289703, -0.006943076848983765, 0.03666088730096817,
      -0.10678021609783173, 0.058163248002529144, -0.08720644563436508, 0.05781444534659386,
      -0.044518716633319855, 0.010364989750087261, 0.01934913359582424, 0.03408421203494072,
      0.0544319674372673, 0.04023616760969162, 0.025990813970565796, -0.016634508967399597,
      0.09482774138450623, -0.01917937584221363, 0.030418436974287033, -0.029658596962690353,
      -0.08990347385406494, -0.10191284865140915, -0.010436109267175198, 0.05684977024793625,
      -0.011419803835451603, -0.017663167789578438, -0.013177650049328804, -0.04883567988872528,
      -0.06077612563967705, 0.02681105211377144, -0.06454888731241226, 0.008551178500056267,
      0.08307257294654846, -0.017699381336569786, 0.03628949075937271, -0.03755928948521614,
      0.05259568244218826, -0.034907419234514236, 0.013487662188708782, 0.08014034479856491,
      0.07749030739068985, 0.047267697751522064, -0.07421296834945679, 0.0696764662861824,
      -0.03420581668615341, 0.004190790466964245, 0.027643581852316856, -0.021540483459830284,
      0.0113461809232831, -0.06952781975269318, 0.04717974737286568, 0.02283616177737713,
      0.016568854451179504, 0.07503970712423325, -0.034012820571660995, -0.05686859041452408,
      0.023847347125411034, 0.09198377281427383, 0.08723848313093185, -0.01267174445092678,
      0.06532403826713562, -0.0029842397198081017, 0.08425191044807434, 0.05346181243658066,
      7.267460023285821e-05, 0.07905721664428711, 0.0724843293428421, 0.1041237860918045,
      -0.04652855172753334, -0.0898762196302414, 0.029845770448446274, 0.0646611824631691,
      0.013885222375392914, -0.01320609450340271, -0.08001494407653809, 0.09234462678432465,
      0.045271385461091995, 0.03904812037944794, 0.04881371930241585, 0.06936518102884293,
      -0.07960204780101776, 0.04290346801280975, -0.02336721494793892, -0.010384450666606426,
      -0.09917329251766205, 0.06690751016139984, -0.08780284225940704, 0.06440922617912292,
      0.04914461076259613, 0.054997313767671585, -0.004523130599409342, -0.016632411628961563,
      0.03809863701462746, 0.04268651083111763, 0.07809513807296753, 0.013250836171209812,
      -0.08693575859069824, -0.022158151492476463, -0.08651650696992874, 0.002720892196521163,
      -0.01817283406853676, -0.05630829185247421, 0.05175317823886871, 0.07342975586652756,
      -0.009665889665484428, 0.09796100109815598, -0.027397869154810905, -0.02636929787695408],
     [0.0399191714823246, -0.02128136344254017, -0.016907988116145134, -0.08308501541614532,
      -0.10826912522315979, -0.04711814597249031, -0.09966325014829636, -0.02939588390290737,
      0.017764154821634293, -0.0790880024433136, -0.003602744545787573, 0.0441754125058651,
      0.08401325345039368, 0.09132187068462372, 0.06939976662397385, 0.06762032210826874,
      0.028292030096054077, -0.03906339406967163, 0.028881046921014786, 0.07702390104532242,
      0.11095522344112396, 0.03393139690160751, 0.07004675269126892, 0.0633719339966774,
      0.06302908807992935, 0.03018302470445633, 0.0009767834562808275, -0.006224514450877905,
      0.10350368171930313, 0.07064926624298096, -0.08732445538043976, -0.04201686382293701,
      0.04767262935638428, 0.08649713546037674, 0.03657788038253784, 0.07251349091529846,
      -0.07903315126895905, 0.08745937049388885, 0.07407832890748978, 0.029897062107920647,
      -0.011691351421177387, 0.05697248876094818, 0.0883108526468277, -0.05160331353545189,
      0.08498258143663406, -0.06018628552556038, -0.010988438501954079, -0.0669819563627243,
      -0.03081422857940197, -0.025566386058926582, 0.08042191714048386, -0.03779737278819084,
      0.08183636516332626, 0.09384499490261078, 0.0949041023850441, 0.05102439969778061,
      -0.04464510828256607, 0.031872935593128204, -0.08500809222459793, -0.0730474665760994,
      -0.028472641482949257, -0.06983283907175064, 0.07138636708259583, 0.011289233341813087,
      -0.0025094011798501015, -0.07843034714460373, 0.001521295984275639, 0.01700257509946823,
      0.005125793628394604, 0.001093971193768084, -0.029620392248034477, -0.0006235710461623967,
      -0.015526586212217808, -0.03239196538925171, 0.013168360106647015, -0.05214430391788483,
      -0.054172057658433914, -0.06883986294269562, -0.00036928756162524223, -0.053756751120090485,
      -0.012041001580655575, -0.04526568949222565, -0.07270874828100204, 0.0868770182132721,
      0.032780420035123825, 0.07690627127885818, 0.04992539808154106, 0.05254071578383446,
      0.049574799835681915, -0.050061244517564774, -0.05170704051852226, -0.07917908579111099,
      -0.036246586591005325, -0.008467505685985088, -0.1348666101694107, 0.013060108758509159,
      -0.06476399302482605, 0.012682215310633183, -0.13754165172576904, -0.05137772485613823,
      0.019920164719223976, 0.02583281695842743, -0.016544746235013008, 0.004562611225992441,
      0.0336182564496994, -0.08986059576272964, -0.052169252187013626, 0.05281354486942291],
     [-0.0005002118414267898, -0.024850880727171898, -0.09288841485977173, -0.005754933692514896,
      0.012736687436699867, -0.020610766485333443, -0.01436204370111227, 0.037436071783304214,
      -0.006179818417876959, -0.12425217032432556, 0.062174588441848755, -0.10138218849897385,
      -0.058338019996881485, -0.009754301980137825, 0.009968419559299946, 0.09973045438528061,
      0.029741894453763962, 0.05370252579450607, -0.031107336282730103, -0.036534473299980164,
      -0.039447128772735596, 0.10331623256206512, 0.060744743794202805, 0.02259611152112484,
      0.021810028702020645, 0.03698742017149925, -0.034222863614559174, -0.0534081906080246,
      -0.042937420308589935, -0.009406311437487602, -0.09741559624671936, -0.010046561248600483,
      -0.11799906194210052, -0.016949355602264404, 0.015406178310513496, 0.05760449916124344,
      -0.05704757571220398, 0.047501757740974426, -0.06193231791257858, 0.04641564562916756,
      -0.0030551492236554623, -0.012054064311087132, 0.05265827104449272, -0.0010658572427928448,
      -0.015073675662279129, -0.05309387668967247, 0.05108414217829704, 0.01015415322035551,
      0.06198727339506149, 0.019977591931819916, -0.023657426238059998, -0.038266122341156006,
      0.07718995958566666, 0.04995846003293991, -0.0415201373398304, 0.05059514939785004,
      0.02108610048890114, 0.0844847783446312, -0.031584661453962326, -0.011966544203460217,
      -0.0036409369204193354, 0.09271321445703506, 0.01805693656206131, -0.07399644702672958,
      -0.03825836256146431, 0.023289497941732407, -0.05814064294099808, 0.04354655742645264,
      0.07927526533603668, 0.04693952202796936, -0.045454077422618866, 0.09065388143062592,
      -0.04168303310871124, 0.05428805202245712, -0.05056256055831909, -0.028142480179667473,
      0.037308283150196075, 0.04891471937298775, -0.011445914395153522, -0.015416931360960007,
      -0.05865364894270897, -0.0679280087351799, -0.005131660029292107, -0.037873607128858566,
      0.09874529391527176, 0.009024270810186863, 0.020318681374192238, 0.028693782165646553,
      0.0946897342801094, -0.04975496605038643, -0.04741301015019417, 0.07913196831941605,
      -0.08962464332580566, -0.06738892942667007, 0.0028770086355507374, -0.04605567455291748,
      -0.05400943383574486, -0.10266910493373871, 0.026939263567328453, 0.015905430540442467,
      -0.038152020424604416, 0.040669217705726624, -0.03813876211643219, 0.02290874347090721,
      0.0683688148856163, 0.011172265745699406, 0.0899297297000885, -0.023248283192515373],
     [-0.0048677739687263966, -0.032347165048122406, -0.004102520179003477, -0.05609540641307831,
      -0.024430720135569572, -0.023319456726312637, 0.016336362808942795, -0.038662660866975784,
      0.010756421834230423, 0.009941263124346733, 0.0900142714381218, -0.030898211523890495,
      0.09066413342952728, 0.015742041170597076, -0.0710199773311615, -0.08694031089544296,
      -0.0464143231511116, -0.006209036335349083, 0.06287935376167297, 0.03369133174419403,
      -0.05670325458049774, -0.04966910928487778, -0.03662210330367088, -0.05734611675143242,
      0.039145611226558685, -0.0008458152879029512, 0.049324896186590195, -0.00720883859321475,
      0.0006783372373320162, 0.051591821014881134, 0.06301900744438171, 0.040810272097587585,
      0.06771254539489746, 0.10148928314447403, 0.09062065184116364, 0.03600282967090607,
      0.04240523278713226, -0.08473876863718033, -0.07537160813808441, -0.05981844663619995,
      0.05682886391878128, 0.041573576629161835, -0.02442074380815029, 0.0782783254981041,
      0.08811153471469879, 0.04369010403752327, -0.10105346888303757, 0.05554133653640747,
      0.07190703600645065, 0.011253009550273418, -0.0622112937271595, -0.011905076913535595,
      -0.0008353501907549798, 0.08969731628894806, 0.011003213003277779, -0.07266614586114883,
      0.08341597020626068, 0.03000793233513832, -0.03832231089472771, 0.01749645732343197,
      -0.04630059003829956, -0.05987173691391945, 0.05068286880850792, -0.05640239641070366,
      -0.06512197107076645, -0.04813117906451225, -0.02778046205639839, 0.010196489281952381,
      0.08887872099876404, 0.0034001274034380913, 0.06681139767169952, 0.017818551510572433,
      0.04511142149567604, -0.09949879348278046, -0.031265318393707275, -0.03869090974330902,
      0.08550550043582916, -0.06094150245189667, -0.08566255867481232, 0.004103765822947025,
      0.03186554089188576, 0.06691119074821472, -0.011943989433348179, -0.05814136192202568,
      0.0677662119269371, -0.013763505034148693, -0.08889968693256378, 0.05595358461141586,
      0.08345450460910797, 0.008528895676136017, -0.07374463230371475, 0.08261606842279434,
      -0.02797004021704197, 0.03404126688838005, -0.04827629402279854, 0.09635818004608154,
      -0.023845260962843895, -0.006392079871147871, 0.014732330106198788, -0.08338334411382675,
      -0.05322040617465973, -0.08459031581878662, 0.03598000854253769, 0.0011828546412289143,
      0.033344317227602005, 0.02043307013809681, -0.0560905896127224, -0.015708770602941513],
     [-0.06972742825746536, 0.0033701732754707336, 0.01314778532832861, -0.024423522874712944,
      -0.037051986902952194, 0.05857676640152931, 0.04793239384889603, -0.02301912009716034,
      -0.09574376046657562, -0.041582975536584854, -0.053283751010894775, -0.0794343650341034,
      0.11861012876033783, 0.09860686957836151, 0.0854322612285614, 0.03781645745038986,
      0.05043273791670799, 0.03216850012540817, -0.07996353507041931, -0.029198352247476578,
      0.09772425144910812, 0.02709001675248146, 0.008954105898737907, -0.09945657849311829,
      0.036112092435359955, -0.024272916838526726, -0.05704834312200546, -0.07510238140821457,
      0.0742998942732811, -0.0044819023460149765, -0.05882612615823746, -0.014409175142645836,
      -0.05721836909651756, 0.017558488994836807, 0.059282224625349045, -0.016672955825924873,
      -0.0567198246717453, -0.03194807469844818, -0.040339015424251556, 0.05623121187090874,
      -0.09088914096355438, 0.003911212552338839, 0.07112995535135269, 0.06925655156373978,
      0.06938688457012177, 0.02245338261127472, 0.0946572944521904, -0.0874210000038147,
      -0.011805405840277672, 0.04886883497238159, -0.07028105109930038, 0.0025713478680700064,
      0.0772331953048706, 0.09807479381561279, 0.023656174540519714, -0.00022277707466855645,
      0.06114722788333893, -0.03563139960169792, -0.02707725018262863, 0.019779903814196587,
      -0.03914310783147812, -0.06624437868595123, -0.03195734694600105, 0.019993139430880547,
      0.07103682309389114, -0.07839462161064148, -0.030936501920223236, -0.0027682229410856962,
      -0.03307012841105461, -0.09376339614391327, 0.018589625135064125, 0.00667174905538559,
      -0.09595894068479538, -0.08081928640604019, 0.04827186465263367, 0.020999738946557045,
      -0.017903191968798637, 0.014820566400885582, 0.04257992282509804, 0.07320845127105713,
      -0.08095549046993256, 0.07984167337417603, -0.01216896902769804, 0.09229692816734314,
      -0.027004355564713478, 0.12019533663988113, -0.04297485202550888, -0.055711280554533005,
      -0.03684096410870552, -0.09000251442193985, 0.0826042965054512, 0.06471601873636246,
      0.03738541528582573, 0.09757065773010254, 0.09117525070905685, 0.06315752863883972,
      0.10765428841114044, 0.008811046369373798, 0.04863189533352852, -0.03955754637718201,
      0.02371497079730034, 0.09250132739543915, 0.06657915562391281, 0.007126145996153355,
      0.012020711787045002, -0.04488470405340195, -0.0099462466314435, -0.019746989011764526],
     [0.008394870907068253, -0.02518397569656372, -0.0018579531461000443, 0.04154348745942116,
      0.022640448063611984, 0.0639951303601265, -0.046556539833545685, -0.02590300142765045,
      0.06428176909685135, 0.11378857493400574, 0.025292756035923958, 0.08713556081056595,
      -0.0140488650649786, -0.006410547066479921, 0.03185930475592613, 0.05189778283238411,
      -0.0961141511797905, 0.05955483764410019, -0.014868680387735367, 0.04291364923119545,
      -0.0085348691791296, 0.01467230636626482, -0.05517289415001869, -0.018902160227298737,
      0.016291284933686256, -0.02375185117125511, 0.03708585351705551, -0.010710077360272408,
      0.027050642296671867, 0.036743611097335815, -0.004711137618869543, 0.06370553374290466,
      -0.049651212990283966, 0.03706853464245796, 0.031169209629297256, -0.05360611900687218,
      0.046881742775440216, 0.04889271780848503, 0.0012566655641421676, -0.07568415254354477,
      -0.019290870055556297, 0.022254152223467827, -0.07310304790735245, 0.018210940062999725,
      -0.04384101554751396, -0.014924765564501286, 0.06397039443254471, 0.037997372448444366,
      0.05918752774596214, -0.059972453862428665, 0.07368497550487518, 0.03166522830724716,
      0.08078991621732712, 0.046601083129644394, 0.09016603231430054, 0.0957995057106018,
      0.015852920711040497, 0.06203526258468628, 0.06399641931056976, -0.05006628483533859,
      0.08396302908658981, 0.018003957346081734, -0.10034506022930145, 0.05364672467112541,
      0.018815042451024055, -0.056278470903635025, -0.0672478973865509, -0.041064370423555374,
      0.08539943397045135, -0.08663508296012878, 0.03501124307513237, 0.008763966150581837,
      0.07351379841566086, -0.08335916697978973, 0.030060669407248497, 0.06917118281126022,
      -0.030661659315228462, 0.012833282351493835, -0.009096493944525719, 0.031852807849645615,
      -0.034231096506118774, 0.012858363799750805, -0.024481283500790596, 0.06275561451911926,
      0.030218368396162987, 0.04775097593665123, -0.01583620347082615, 0.021463917568325996,
      0.04897070676088333, -0.07432424277067184, 0.06576284766197205, -0.053875070065259933,
      -0.01388270128518343, 0.0024707845877856016, 0.09112470597028732, 0.034513331949710846,
      0.0570056326687336, 0.026418164372444153, 0.1106797531247139, 0.07347816973924637,
      0.11733750998973846, -0.05422331765294075, -0.007232033647596836, -0.04400612786412239,
      -0.05040905997157097, -0.04141789674758911, -0.01598442904651165, 0.005243532359600067],
     [0.08701225370168686, -0.0035393796861171722, -0.0680704265832901, -0.04921379312872887,
      0.008550511673092842, -0.028731880709528923, -0.01680261455476284, 0.020941294729709625,
      0.052912019193172455, -0.023646553978323936, 0.060644764453172684, 0.02433222346007824,
      -0.06666646152734756, -0.01162694115191698, -0.07024413347244263, -0.10603258013725281,
      -0.054272204637527466, 0.005213134456425905, 0.05004553124308586, -0.014149701222777367,
      -0.02236609347164631, 0.02722400240600109, -0.03297099471092224, -0.06376446783542633,
      -0.025370480492711067, 0.014873056672513485, 0.07731036096811295, 0.08105196058750153,
      -0.0172122735530138, 0.03712410479784012, 0.06278292089700699, -0.03276212140917778,
      -0.013554426841437817, 0.04395630583167076, 0.009153636172413826, 0.014795942232012749,
      -0.00011096714297309518, -0.09611047059297562, 0.08404899388551712, -0.04875181242823601,
      0.03785251826047897, 0.011231301352381706, 0.035740528255701065, -0.002655612537637353,
      0.01423596777021885, 0.09205398708581924, 0.022758184000849724, -0.09694527089595795,
      -0.007316019851714373, -0.08144859224557877, 0.019965676590800285, -0.005462476052343845,
      -0.05819869413971901, 0.0019099796190857887, 0.047163620591163635, 0.08848629146814346,
      -0.07934874296188354, 0.0657133087515831, -0.08305829763412476, 0.005279623903334141,
      -0.09480908513069153, -0.08287888020277023, -0.07465903460979462, -0.004514179192483425,
      -0.06974449008703232, -0.010791139677166939, 0.06342290341854095, -0.052959926426410675,
      0.026224587112665176, -0.09938587248325348, 0.0414331778883934, 0.08863886445760727,
      -0.009859790094196796, -0.07733867317438126, 0.0045511769130826, 0.049739956855773926,
      0.033374086022377014, -0.004658891819417477, -0.012538662180304527, -0.06738171726465225,
      0.07374871522188187, -0.04207172989845276, 0.0395958349108696, 0.06375765800476074,
      0.03238072618842125, 0.04287891089916229, -0.09542316198348999, 0.05845304951071739,
      0.050196725875139236, -0.07719644159078598, 0.02360425889492035, 0.05683101341128349,
      0.04888533055782318, 0.05854252725839615, -0.03816714882850647, 0.018030758947134018,
      -0.07720078527927399, -0.07443803548812866, 0.04277162626385689, 0.030341127887368202,
      -0.010860650800168514, 0.005393035709857941, 0.02331222966313362, 0.015817759558558464,
      0.02113116718828678, 0.03776462748646736, -0.10091009736061096, 0.03581426292657852],
     [0.04221457615494728, -0.05046958476305008, 0.0820087194442749, -0.03767915442585945,
      0.02779018133878708, -0.010422197170555592, 0.054671529680490494, -0.0817529559135437,
      0.05408832058310509, -0.0038598498795181513, 0.05104999244213104, 0.07180727273225784,
      0.08511590957641602, -0.051073718816041946, -0.09684844315052032, 0.05894524231553078,
      0.034187786281108856, -0.009326813742518425, 0.015511305071413517, -0.09012016654014587,
      -0.03392054885625839, -0.10213347524404526, -0.045558493584394455, -0.02807958796620369,
      0.031194569543004036, 0.007672162726521492, 0.06437414139509201, 0.011057628318667412,
      -0.03950228542089462, 0.0670471340417862, 0.03358549252152443, -0.08361782878637314,
      -0.03478024899959564, -0.04143787920475006, -0.007850464433431625, -0.046019237488508224,
      0.04690440371632576, 0.005104565527290106, -0.014790957793593407, 0.07048922032117844,
      -0.07775687426328659, 0.0865308865904808, 0.02646099217236042, 0.08776847273111343,
      0.04540690779685974, 0.06311509013175964, 0.050850022584199905, -0.020074225962162018,
      -0.06400714069604874, 0.057677511125802994, 0.003191874362528324, -0.06433063000440598,
      -0.039958756417036057, 0.07869134098291397, 0.023119648918509483, -0.021717838943004608,
      -0.07873781025409698, 0.01382774580270052, 0.0632556825876236, 0.0298213679343462,
      -0.017094044014811516, 0.05144815519452095, 0.0597713403403759, 0.013092701323330402,
      0.047996051609516144, -0.055239297449588776, 0.04365247115492821, -0.05511702597141266,
      -0.07906735688447952, -0.08273448050022125, -0.062371876090765, 0.016388501971960068,
      0.06961289793252945, -0.09963507950305939, 0.04808208718895912, 0.02952856570482254,
      0.002935352036729455, 0.04601728916168213, -0.009441956877708435, -0.030668264254927635,
      -0.04083414003252983, 0.016094783321022987, 0.048947375267744064, 0.08649802207946777,
      -0.029894864186644554, -0.10011416673660278, -0.10874633491039276, 0.016854384914040565,
      0.09856535494327545, 0.012975392863154411, -0.049410704523324966, 0.006718463730067015,
      -0.03153083100914955, -0.09296625107526779, 0.046964362263679504, -0.03188885748386383,
      -0.06796064227819443, 0.031215131282806396, 0.08458521962165833, -0.0474356971681118,
      0.057015784084796906, -0.01674267090857029, 0.06767182052135468, 0.07329634577035904,
      -0.027532635256648064, -0.038251522928476334, -0.07304324954748154, -0.00208894070237875],
     [-0.0006469842046499252, -0.07801374793052673, 0.023362595587968826, -0.003712613834068179,
      0.012907874770462513, 0.058926958590745926, -0.0022046163212507963, 0.05241404101252556,
      -0.022457581013441086, -0.08071214705705643, -0.07681164890527725, -0.08456507325172424,
      0.015920691192150116, -0.034749146550893784, -0.06394611299037933, 0.046159036457538605,
      0.01470657717436552, 0.01729416847229004, -0.04492148756980896, -0.04908761382102966,
      0.08221697807312012, -0.07565537095069885, -0.03181971237063408, -0.022266598418354988,
      0.04765114560723305, -0.08498639613389969, -0.0390792116522789, -0.030271437019109726,
      -0.009231610223650932, -0.07083137333393097, -0.061906374990940094, 0.09540700912475586,
      -0.028329189866781235, -0.04482395946979523, -0.08362036198377609, 0.08323456346988678,
      0.0005477199447341263, -0.050906479358673096, -0.007252630311995745, 0.07401547580957413,
      0.053846389055252075, -0.019167330116033554, -0.065847247838974, 0.025622088462114334,
      -0.006418882869184017, 0.08003605902194977, 0.020319847390055656, -0.07162649929523468,
      -0.02353256195783615, 0.09755603969097137, 0.017849326133728027, -0.019363177940249443,
      -0.04129745438694954, -0.0726967602968216, 0.0004075793258380145, -0.0004543309332802892,
      -0.038345202803611755, -0.016996029764413834, -0.0786246508359909, -0.07218214869499207,
      -0.07262177020311356, 0.011756464838981628, 0.057365093380212784, 0.06348303705453873,
      -0.06839722394943237, -0.0824701264500618, 0.005917789880186319, -0.0008025587303563952,
      -0.02098286896944046, -0.013083655387163162, -0.07804600894451141, -0.07293105125427246,
      0.07463088631629944, 0.004465074744075537, -0.0703578069806099, -0.08527585864067078,
      -0.004126952961087227, 0.0817938968539238, 0.04201802983880043, 0.06900925934314728,
      0.032266274094581604, 0.03563481196761131, 0.02763080783188343, -0.050378888845443726,
      -0.03640037029981613, -0.0698990672826767, -0.02696903422474861, 0.02367325872182846,
      0.06825290620326996, 0.0890735387802124, -0.048800691962242126, 0.06977850943803787,
      0.07484618574380875, 0.07891860604286194, 0.010110374540090561, -0.029089253395795822,
      0.022285811603069305, 0.05859803408384323, 0.036698855459690094, 0.05335807800292969,
      0.057198379188776016, 0.02220277488231659, 0.031370487064123154, 0.019471460953354836,
      0.07346033304929733, 0.09050324559211731, -0.024841399863362312, 0.017552077770233154],
     [-0.09935683012008667, -0.07664509117603302, -0.008434084244072437, -0.052855465561151505,
      -0.02989387884736061, -0.15919129550457, -0.04350360482931137, -0.13891048729419708,
      -0.1409984678030014, -0.008790543302893639, -0.06289904564619064, -0.014267547987401485,
      0.09836425632238388, 0.1283922791481018, 0.04828820750117302, 0.02695329859852791,
      0.08577297627925873, 0.02751968428492546, 0.1447593867778778, 0.07299569994211197,
      0.10181976854801178, -0.004537490196526051, 0.1776815801858902, 0.17973428964614868,
      0.010686349123716354, 0.055448807775974274, -0.08568534255027771, 0.06062335520982742,
      0.04296943545341492, 0.01469471026211977, -0.06390584260225296, 0.017561685293912888,
      0.05649604648351669, -0.08033712953329086, -0.0644805058836937, -0.040113989263772964,
      0.11167972534894943, 0.03644123300909996, 0.027383768931031227, -0.02225637063384056,
      -0.039377763867378235, 0.06436295807361603, 0.10850135236978531, 0.0898752436041832,
      -0.05428965762257576, 0.03703191131353378, 0.026398032903671265, -0.001814096001908183,
      0.028828214854002, 0.004082978703081608, 0.08102301508188248, 0.008333566598594189,
      -0.021049315109848976, 0.04437601566314697, 0.0884130448102951, 0.08913376182317734,
      0.00028389247017912567, 0.046044252812862396, 0.009285351261496544, -0.0017367604887112975,
      0.07548084110021591, 0.037792500108480453, -0.059073690325021744, 0.058170024305582047,
      -0.04620983079075813, 0.03035225346684456, 0.037286426872015, -0.02868889458477497,
      -0.0024686851538717747, 0.012277023866772652, -0.011183317750692368, -0.05191292613744736,
      0.07376101613044739, -0.04682866856455803, 0.06860434263944626, -0.061648428440093994,
      -0.05584254488348961, -0.012286992743611336, -0.05209634453058243, 0.10932102054357529,
      -0.0434601716697216, -0.021466512233018875, 0.05899236723780632, -0.033665820956230164,
      0.07656250894069672, 0.03507712855935097, 0.09751609712839127, 0.07748890668153763,
      0.06369653344154358, 0.0373750664293766, -0.11119945347309113, -0.025527117773890495,
      -0.13096067309379578, -0.12461834400892258, -0.12110655009746552, -0.0563376322388649,
      -0.055045537650585175, -0.0785536915063858, -0.11667726933956146, -0.13356676697731018,
      -0.02270805649459362, -0.10653786361217499, -0.001980808563530445, 0.08313126862049103,
      -0.024813545867800713, 0.04838483780622482, 0.10041885823011398, 0.0461033470928669],
     [-0.09688547998666763, 0.02394997514784336, -0.00995587557554245, -0.03547017276287079,
      -0.03204190731048584, -0.05507832020521164, -0.004515982698649168, -0.07429178804159164,
      -0.0009380344417877495, 0.05232899636030197, 0.036529429256916046, -0.006447669118642807,
      -0.053850799798965454, 0.02433365024626255, 0.05871214717626572, -0.06285745650529861,
      -0.020965829491615295, 0.0937969833612442, 0.04455770179629326, 0.16189859807491302,
      0.04175416752696037, 0.17326825857162476, 0.15211816132068634, 0.11106062680482864,
      0.06140090525150299, -0.07834452390670776, 0.04069559648633003, 0.07252521067857742,
      -0.04471294954419136, 0.060325853526592255, -0.018156765028834343, 0.05107560381293297,
      -0.030691709369421005, -0.10772709548473358, -0.10028637945652008, -0.018373114988207817,
      -0.032614294439554214, 0.04196067899465561, 0.10413698107004166, 0.0467628538608551,
      0.017273256555199623, 0.06586936116218567, -0.0044366284273564816, -0.019806746393442154,
      -0.023287102580070496, -0.09715999662876129, -0.08621922135353088, 0.0403415784239769,
      -0.03296877443790436, 0.08210792392492294, -0.0723123773932457, -0.003051587613299489,
      0.07522527873516083, -0.026683460921049118, -0.06397120654582977, 0.11145244538784027,
      0.07482612133026123, -0.027198754251003265, 0.08364535123109818, 0.04908210411667824,
      0.029011918231844902, 0.05698554962873459, -0.020650651305913925, 0.005444752983748913,
      0.00788789987564087, -0.02070455439388752, -0.0766984075307846, -0.05731593072414398,
      -0.09031446278095245, -0.015020376071333885, 0.04116121679544449, 0.0648675486445427,
      4.6936296712374315e-05, -0.10458175837993622, -0.0703035518527031, 0.002044852590188384,
      0.018413018435239792, 0.05345631763339043, -0.07728495448827744, 0.003270251676440239,
      0.0669194683432579, 0.05117952823638916, 0.001984350150451064, -0.028868887573480606,
      -0.026879440993070602, 0.03528915345668793, 0.03242095559835434, -0.013353066518902779,
      -0.01857970841228962, -0.04013797640800476, -0.16319811344146729, -0.11410947144031525,
      -0.2145802229642868, -0.10912121832370758, -0.07165104895830154, -0.10851603746414185,
      -0.13355445861816406, -0.016456516459584236, -0.06847555935382843, -0.15581248700618744,
      -0.06196313351392746, -0.09948410093784332, 0.10880651324987411, -0.026570577174425125,
      -0.05582713335752487, 0.02851715125143528, 0.04130418226122856, -0.04197794571518898],
     [0.01344413310289383, 0.0036880667321383953, 0.09488838165998459, 0.10822589695453644,
      0.0832991674542427, -0.05336850881576538, 0.07636074721813202, 0.0373813696205616,
      -0.002539827488362789, 0.026401333510875702, 0.017095113173127174, 0.05411006510257721,
      0.004512848798185587, 0.07816550880670547, 0.025110704824328423, -0.08629160374403,
      -0.0603402704000473, -0.050285108387470245, -0.005620111711323261, -0.024168385192751884,
      0.047281187027692795, 0.10129303485155106, 0.019428327679634094, 0.07301931828260422,
      -0.09304703772068024, -0.006246206350624561, -0.06532364338636398, -0.07358443737030029,
      0.05227138474583626, 0.030414516106247902, -0.1300841122865677, -0.09198269993066788,
      -0.12017647922039032, -0.10887524485588074, -0.13413268327713013, -0.0585651695728302,
      0.03252825513482094, -0.0424305759370327, -0.044867318123579025, -0.02128659561276436,
      -0.030222760513424873, 0.05073346197605133, -0.03441251441836357, -0.06669212877750397,
      -0.07412353157997131, -0.01978178322315216, -0.05689443275332451, 0.04344158247113228,
      -0.02242906205356121, 0.07018465548753738, -0.0207977257668972, 0.06716284900903702,
      -0.003893028711900115, -0.057282984256744385, -0.04990220442414284, -0.057162750512361526,
      0.03365113213658333, -0.06253530085086823, 0.08627834171056747, -0.08563719689846039,
      0.08389037847518921, 0.02731737121939659, 0.0691274031996727, 0.06598052382469177,
      0.1172047033905983, 0.07993854582309723, 0.036159031093120575, 0.007057409733533859,
      -0.07105202227830887, -0.09188644587993622, -0.02130514197051525, 0.023600632324814796,
      -0.009713206440210342, -0.07881686836481094, 0.030742812901735306, 0.03468410298228264,
      -0.01969844289124012, 0.014922589994966984, 0.07591462135314941, 0.05131759122014046,
      -0.048145830631256104, 0.07602311670780182, -0.07588742673397064, 0.04085838049650192,
      0.07001136988401413, 0.08133161067962646, -0.04103745147585869, -0.04650590568780899,
      0.011665602214634418, -0.037374548614025116, -0.09174562990665436, -0.014866439625620842,
      -0.03446321189403534, 0.032442983239889145, -0.1346709281206131, -0.026692397892475128,
      0.03224639222025871, -0.08427642285823822, -0.03471663221716881, 0.029212146997451782,
      -0.05549642816185951, -0.0213024765253067, 0.008690260350704193, -0.05991290137171745,
      0.0646548941731453, 0.00862604845315218, 0.010187139734625816, -0.0032711485400795937],
     [-0.062256261706352234, 0.09466290473937988, 0.05534890666604042, 0.007428435143083334,
      -0.03976549953222275, -0.061952292919158936, -0.05663527920842171, -0.05027623847126961,
      -0.0927506536245346, 0.016390599310398102, -0.010163988918066025, 0.06200489029288292,
      -0.036131326109170914, 0.033281467854976654, -0.09276563674211502, -0.03161979839205742,
      -0.11150389909744263, 0.048923272639513016, 0.05919680744409561, -0.03688574209809303,
      0.010320881381630898, 0.052659038454294205, -0.10315974056720734, 0.04094080999493599,
      0.07177738845348358, -0.06337377429008484, 0.08016131073236465, -0.06754729896783829,
      0.04797261580824852, 0.0769730880856514, 0.015128079801797867, 0.09708068519830704,
      0.03160637244582176, 0.03308703750371933, 0.043923430144786835, -0.0722968578338623,
      -0.011346067301928997, 0.025894131511449814, 0.04393822327256203, 0.1083015650510788,
      0.0887746587395668, -0.038183536380529404, -0.022318115457892418, 0.03153516352176666,
      0.07272067666053772, -0.01976308971643448, 0.0023463843390345573, 0.003213118063285947,
      -0.006274082697927952, -0.08940884470939636, -0.05522361025214195, 0.01322957593947649,
      -0.05094709247350693, 0.04137161746621132, -0.07827161252498627, 0.0652565136551857,
      -0.054922014474868774, 0.09419725835323334, 0.09620621800422668, -0.023920249193906784,
      -0.01372025441378355, 0.08390729129314423, -0.049576494842767715, -0.017469683662056923,
      -0.04842158406972885, 0.04091135039925575, 0.018565146252512932, 0.08977242559194565,
      0.036698684096336365, -0.04381110891699791, 0.0986587181687355, -0.004878010600805283,
      0.06832575052976608, 3.0004179279785603e-05, -0.03891929239034653, -0.0071396310813724995,
      0.093731589615345, -0.008939999155700207, -0.019276762381196022, 0.07009546458721161,
      -0.044672951102256775, 0.008317382074892521, 0.07032433897256851, -0.05470674857497215,
      -0.017493197694420815, -0.005520106293261051, 0.014692871831357479, -0.08851607143878937,
      -0.007000517565757036, 0.08486302196979523, 0.02901451475918293, -0.0927099660038948,
      0.050173863768577576, 0.028055232018232346, 0.05411200225353241, 0.07093027234077454,
      -0.04340835288167, -0.00629168888553977, -0.05240356922149658, 0.06850925087928772,
      0.08313675969839096, -0.008527212776243687, 0.01374439150094986, 0.05289873108267784,
      0.015850786119699478, -0.04594197869300842, -0.0905400887131691, -0.02541501820087433],
     [-0.08254173398017883, -0.08879248797893524, -0.07015819102525711, 0.030964598059654236,
      -0.03411389887332916, 0.06946808844804764, -0.02415008284151554, 0.03598349168896675,
      -0.009323462843894958, -0.00319775496609509, 0.06469480693340302, -0.05017080530524254,
      0.06706366688013077, -0.0258614644408226, 0.0038968203589320183, 0.04017629474401474,
      -0.06666447967290878, -0.06765575706958771, 0.11890960484743118, 0.1357044279575348,
      0.125289186835289, 0.12800414860248566, 0.03381355106830597, 0.033555611968040466,
      0.07491574436426163, -0.07056525349617004, 0.030262472108006477, 0.016361301764845848,
      0.008750511333346367, 0.008810593746602535, -0.040631525218486786, 0.03423108905553818,
      0.07008704543113708, -0.06172168254852295, -0.0022723188158124685, 0.020852699875831604,
      -0.02005390264093876, -0.010329981334507465, 0.03406538814306259, 0.09882890433073044,
      0.034954771399497986, 0.08671557158231735, 0.11058708280324936, 0.09055017679929733,
      0.09576176106929779, 0.04264732077717781, 0.02191183902323246, 0.03566095232963562,
      -0.048520371317863464, -0.018969079479575157, 0.04142087325453758, -0.0694243460893631,
      -0.05906696617603302, -0.06056566908955574, -0.03714619576931, -0.009609400294721127,
      0.06326477229595184, -0.040766093879938126, -0.07454157620668411, 0.024117015302181244,
      -0.05555315315723419, 0.0467909537255764, -0.08092036098241806, -0.05757194012403488,
      0.08308622986078262, 0.0766298919916153, -0.0857812911272049, -0.0649489238858223,
      -0.08275719732046127, 0.03254999965429306, 0.01819217950105667, -0.0023563611321151257,
      0.05156988278031349, 0.030532168224453926, -0.05216560512781143, -0.01786739006638527,
      -0.00999670010060072, 0.01056941319257021, 0.05681498721241951, 0.002683500060811639,
      0.019936557859182358, -0.02557041309773922, 0.09093345701694489, 0.03173397481441498,
      0.02453436702489853, 0.008019549772143364, 0.0185908954590559, 0.05266107991337776,
      -0.05658911168575287, -0.03037097677588463, 0.025211427360773087, -0.08106472343206406,
      0.01315213367342949, -0.1205935850739479, -0.09017616510391235, -0.011817838065326214,
      -0.12123648077249527, -0.02628449909389019, -0.03429301083087921, -0.13497762382030487,
      -0.01729014329612255, -0.13842493295669556, 0.12571212649345398, 0.10892745107412338,
      0.013904851861298084, -0.03966196998953819, -0.054230403155088425, -0.02638297528028488],
     [-0.05380193516612053, 0.031063249334692955, 0.013294017873704433, -0.08595249056816101,
      -0.10379502922296524, -0.14661364257335663, -0.01043708436191082, 0.04415849968791008,
      -0.10894681513309479, -0.028681360185146332, -0.06537356227636337, -0.006611271761357784,
      0.03924224153161049, -0.024910995736718178, 0.01768651232123375, 0.10835577547550201,
      -0.02772236056625843, 0.08743204176425934, 0.09651628881692886, 0.13469739258289337,
      0.10434051603078842, 0.0342206135392189, -0.009085861966013908, 0.05731365084648132,
      0.08664773404598236, 0.015635564923286438, 0.02284533716738224, 0.09125562757253647,
      -0.042570408433675766, 0.056777726858854294, 0.015611746348440647, -0.002214090432971716,
      0.07045924663543701, 0.058495644479990005, 0.016226794570684433, 0.05665558576583862,
      -0.053956352174282074, 0.0793800875544548, -0.026873109862208366, 0.00884496234357357,
      0.04014734923839569, -0.05917877331376076, 0.07771402597427368, 0.10377385467290878,
      -0.08212658762931824, -0.10296560078859329, -0.04813689738512039, -0.053901661187410355,
      0.012076987884938717, 0.09416412562131882, 0.08313117921352386, -0.03851759061217308,
      0.04335852712392807, 0.034450970590114594, -0.07869637757539749, 0.09489836543798447,
      0.04595090448856354, -0.0737609714269638, 0.061290159821510315, 0.09555814415216446,
      0.08653413504362106, 0.06042877212166786, -0.06007905304431915, -0.02337586134672165,
      0.11679817736148834, 0.06898992508649826, -0.00101980019826442, 0.054462090134620667,
      0.030894652009010315, -0.07136959582567215, -0.022345148026943207, -0.09436376392841339,
      0.006508472841233015, -0.01548024918884039, -0.011558929458260536, 0.06974868476390839,
      -0.0075187599286437035, 0.08581968396902084, 0.008486473001539707, 0.02198886312544346,
      -0.05244623124599457, 0.04169042035937309, 0.07322540134191513, -0.02976212464272976,
      0.08591851592063904, 0.05538949370384216, -0.04466554522514343, -0.057986825704574585,
      -0.06499689072370529, 0.043899308890104294, -0.08807974308729172, -0.002710734261199832,
      -0.05427718162536621, 0.03988964855670929, -0.11841195076704025, -0.008105537854135036,
      -0.08574502915143967, -0.01387040875852108, -0.01311594806611538, 0.009223130531609058,
      -0.04829805716872215, -0.059464361518621445, 0.06370726972818375, 0.06077144667506218,
      -0.05874387547373772, -0.00974961370229721, 0.04226544126868248, -0.04334430769085884],
     [-0.05902014300227165, -0.08679608255624771, 0.0472501777112484, 0.02781105600297451,
      0.049706846475601196, 0.05179425701498985, 0.0714043527841568, 0.02573280595242977,
      -0.024572305381298065, 0.012624291703104973, -0.06387430429458618, 0.04265137016773224,
      0.03563164919614792, 0.07879906892776489, 0.004559399094432592, 0.1042889803647995,
      -0.04672928899526596, -0.01118629239499569, 0.07504034787416458, -0.07355685532093048,
      0.040257956832647324, 0.036304786801338196, -0.015599201433360577, 0.009349031373858452,
      0.02321641519665718, 0.04116145148873329, -0.01096623670309782, -0.06491585820913315,
      0.05476438254117966, 0.1183384358882904, -0.08121950924396515, -0.1051265075802803,
      0.011552179232239723, 0.04217281937599182, -0.030500484630465508, 0.06339102238416672,
      -0.023290809243917465, -0.020492834970355034, 0.01843278482556343, -0.046878356486558914,
      0.07349924743175507, -0.09543697535991669, -0.11242443323135376, 0.0441267192363739,
      -0.05049905180931091, -0.005527112167328596, -0.014164588414132595, 0.019659120589494705,
      0.054623402655124664, 0.07845905423164368, -0.026030361652374268, -0.01171967014670372,
      -0.08763903379440308, -0.06868083029985428, -0.08406805247068405, -0.06744526326656342,
      -0.031851816922426224, 0.0839538648724556, -0.07532062381505966, -0.04816710576415062,
      0.06349536031484604, -0.0018564534839242697, 0.07514390349388123, -0.04019668698310852,
      0.043971672654151917, 0.05883374437689781, 0.012171132490038872, -0.04130557179450989,
      -0.022791899740695953, 0.05508619546890259, 0.007444923743605614, -0.046282440423965454,
      -0.07103559374809265, -0.11527614295482635, 0.02886647917330265, -0.09021703153848648,
      -0.019696742296218872, -0.022142676636576653, -0.05312404781579971, 0.07805388420820236,
      -0.044272277504205704, 0.01869293861091137, -0.04228321462869644, -0.003407825017347932,
      0.10098645091056824, 0.03126320615410805, -0.005833946168422699, -0.009853927418589592,
      -0.031979769468307495, 0.05887492746114731, 0.07347556203603745, -0.03960225358605385,
      0.08428394794464111, 0.03575095161795616, 0.06125801056623459, 0.05995064973831177,
      0.006993635557591915, 0.12175142019987106, 0.07108926773071289, -0.08270642161369324,
      0.09848340600728989, -0.012265241704881191, -0.03147733956575394, -0.04507184028625488,
      0.03756585717201233, -5.721551497117616e-05, 0.008968077600002289, 0.09784019738435745],
     [0.12438742816448212, -0.0021781534887850285, -0.03807365149259567, 0.08452696353197098,
      0.1103106364607811, 0.04558738321065903, -0.017931107431650162, -0.07927551120519638,
      0.01900183968245983, 0.05356844142079353, 0.03527257964015007, -0.032018646597862244,
      0.05295190587639809, -0.08682951331138611, 0.054355915635824203, -0.11385253071784973,
      -0.13009469211101532, 0.003878864459693432, 0.006830054335296154, 0.028047814965248108,
      -0.00946127437055111, 0.07425902783870697, 0.0811719074845314, -0.028810525313019753,
      -0.044875893741846085, -0.043752528727054596, 0.0003077065630350262, 0.04891187325119972,
      0.008300350047647953, 0.0628247931599617, -0.03544826805591583, -0.0934123694896698,
      -0.10927890986204147, 0.03175753355026245, -0.08843309432268143, 0.0258256234228611,
      0.01828870363533497, -0.01839565485715866, 0.0930173248052597, 0.0821739211678505,
      0.025687165558338165, -0.017225421965122223, -0.05000339448451996, 0.048260752111673355,
      -0.0586966909468174, 0.04015543311834335, -0.10250675678253174, 0.023049527779221535,
      0.09226840734481812, -0.044325217604637146, -0.04007072374224663, 0.058476418256759644,
      -0.031112145632505417, -0.04022378847002983, 0.06254284083843231, -0.06295987218618393,
      -0.002771806437522173, 0.09120549261569977, 0.03234393894672394, 0.08678731322288513,
      -0.014206808060407639, 0.044589534401893616, -0.06337051093578339, 0.06812368333339691,
      0.03151542693376541, 0.07991582900285721, -0.020690161734819412, -0.02957037463784218,
      -0.07066094130277634, 0.07318618893623352, -0.062164146453142166, 0.07759138941764832,
      -0.060404352843761444, -0.06255316734313965, 0.07129697501659393, -0.058967649936676025,
      -0.00708753289654851, -0.030642082914710045, -0.010622024536132812, 0.02638903446495533,
      0.06480959802865982, 0.015552634373307228, 0.08887708187103271, -0.006546607706695795,
      0.038541167974472046, -0.09458181262016296, -0.0560784675180912, -0.029291197657585144,
      -0.09732390940189362, 0.09702218323945999, 0.0057760016061365604, -0.012427405454218388,
      0.03027600422501564, 0.018489494919776917, 0.01945248246192932, -0.0809250995516777,
      0.06587756425142288, 0.016711706295609474, 0.012972681783139706, -0.021646056324243546,
      -0.10327225923538208, 0.01935664936900139, -0.036442484706640244, 0.03357554227113724,
      -0.10814444720745087, -0.046494100242853165, -0.10557447373867035, -0.030223043635487556],
     [0.06798280775547028, -0.03828347474336624, 0.004563275724649429, -0.058304451406002045,
      0.10057719051837921, -0.0749855488538742, 0.04329287260770798, -0.02108103781938553,
      0.007244049105793238, -0.09608971327543259, 0.06336694955825806, 0.06715631484985352,
      -0.0035676483530551195, 0.06619942933320999, 0.023829583078622818, 0.030105847865343094,
      -0.014577459543943405, -0.0852043628692627, 0.023809274658560753, -0.06230610981583595,
      -0.06645913422107697, 0.01771477982401848, -0.019430428743362427, 0.0065382374450564384,
      -0.08303868770599365, -0.039715368300676346, 0.0486740842461586, -0.07646278291940689,
      -0.06466484814882278, 0.010650848969817162, 0.04472155496478081, -0.07821834087371826,
      0.07544360309839249, 0.06839536875486374, -0.04883941635489464, 0.022978421300649643,
      0.02098562754690647, 0.06889276951551437, 0.05184665322303772, 0.01800672523677349,
      0.05307003855705261, -0.05504295602440834, 0.09064735472202301, -0.033846765756607056,
      0.025292430073022842, 0.03762705996632576, 0.016282137483358383, 0.02528873085975647,
      -0.012359156273305416, 0.08711844682693481, -0.02503623254597187, 0.06492232531309128,
      0.09159883111715317, 0.059693772345781326, 0.03336033225059509, -0.07841794937849045,
      -0.06173460930585861, 0.003810867667198181, 0.08807764202356339, 0.07043802738189697,
      -0.043676991015672684, 0.04900362342596054, 0.004240244161337614, -0.07800073176622391,
      0.027227018028497696, -0.054209575057029724, 0.05173800140619278, -0.07915578782558441,
      0.08229772746562958, -0.08220894634723663, 0.07563918828964233, -0.0947704091668129,
      0.06157076358795166, -0.1088506206870079, 0.018466390669345856, -0.002230506855994463,
      -0.03872482851147652, -0.061988022178411484, 0.05375281721353531, 0.08091218769550323,
      -0.016986824572086334, 0.003199154743924737, 0.04052788391709328, -0.06925090402364731,
      -0.019697129726409912, -0.0745055079460144, -0.0747966468334198, 0.06147044897079468,
      0.053879689425230026, -0.0673808678984642, 0.03313508629798889, 0.08043438196182251,
      -0.08174263685941696, 0.05352352187037468, -0.07492829114198685, 0.03169085085391998,
      -0.010465984232723713, -0.05058610439300537, -0.030235614627599716, -0.09047941118478775,
      -0.04829678684473038, 0.05667794868350029, -0.09353228658437729, -0.02597111091017723,
      0.023125533014535904, -0.08231231570243835, -0.07263870537281036, -0.009901233948767185],
     [-0.0441029891371727, 0.04595581069588661, 0.08031206578016281, 0.07620805501937866,
      0.09668227285146713, -0.07910150289535522, -0.05374511331319809, 0.07103115320205688,
      -0.006843151990324259, -0.07038894295692444, 0.014455972239375114, 0.06263872236013412,
      -0.054488569498062134, -0.07312265038490295, 0.04037593677639961, 0.03804655745625496,
      -0.014064141549170017, -0.0057821329683065414, 0.019062750041484833, 0.10071631520986557,
      0.15953265130519867, 0.03478509932756424, 0.0648648738861084, 0.13748851418495178,
      -0.049893222749233246, 0.07764343917369843, -0.04517069086432457, -0.048824064433574677,
      0.05883257836103439, -0.02275681681931019, -0.050625164061784744, -0.11471918225288391,
      -0.014003396965563297, -0.116304911673069, 0.023306645452976227, 0.06367959082126617,
      0.032285869121551514, -0.00783633440732956, 0.09738900512456894, 0.10735024511814117,
      -0.07736089825630188, -0.014883922412991524, 0.03262878209352493, -0.02482425421476364,
      -0.04431947320699692, 0.06255797296762466, -0.08449526876211166, -0.08108729124069214,
      -0.07347362488508224, 0.07152412086725235, 0.009925834834575653, -0.014763780869543552,
      -0.008281955495476723, 0.10594147443771362, 0.0660369023680687, -0.05266253650188446,
      -0.08181309700012207, 0.009956330992281437, 0.018372738733887672, 0.06343694776296616,
      -0.07037998735904694, 0.026685310527682304, -0.03167659789323807, 0.020242024213075638,
      0.04525496065616608, 0.0037724929861724377, -0.011232040822505951, 0.0014335857704281807,
      0.08336303383111954, 0.054568685591220856, 0.05980280041694641, 0.02529318444430828,
      -0.03292352333664894, -0.014556936919689178, -0.011800253763794899, 0.007523476146161556,
      0.031999312341213226, 0.05638335272669792, -0.05318764969706535, 0.09355388581752777,
      0.03236931562423706, 0.06547922641038895, -0.048604220151901245, -0.044672589749097824,
      0.022830715402960777, 0.04094519838690758, 0.07088259607553482, 0.07105667144060135,
      -0.057963695377111435, -0.015190882608294487, 0.040623992681503296, 0.04074260964989662,
      -0.02628612332046032, 0.06100558117032051, -0.0279341209679842, -0.1139928475022316,
      -0.04223797097802162, -0.034686360508203506, -0.1564621478319168, 0.048014409840106964,
      -0.025026697665452957, -0.10476014018058777, -0.05027059093117714, -0.00855183880776167,
      -0.06918419897556305, 0.06352634727954865, -0.06396108865737915, 0.01183562446385622],
     [0.06809556484222412, 0.0717962309718132, -0.07774083316326141, 0.08535880595445633,
      0.07031367719173431, 0.06270546466112137, 0.0010774724651128054, 0.004780930001288652,
      0.06715132296085358, -0.0030181757174432278, 0.01862669736146927, 0.06141206622123718,
      -0.03872140869498253, -0.002694018417969346, 0.11146754026412964, 0.0223370511084795,
      0.03998393565416336, 0.019234726205468178, -0.0927281528711319, -0.01858687400817871,
      -0.0526394359767437, 0.061469677835702896, 0.05047496035695076, 0.009734753519296646,
      0.01638820581138134, -0.08305657655000687, -0.052836306393146515, 0.06430306285619736,
      0.061096902936697006, 0.0120052769780159, -0.04866022616624832, 0.0167261753231287,
      -0.08938539773225784, 0.02194931171834469, 0.03602948784828186, -0.07204711437225342,
      0.028208322823047638, -0.09349870681762695, 0.05651379004120827, -0.01116250455379486,
      -0.02404535748064518, -0.08649090677499771, -0.028098436072468758, -0.044982559978961945,
      -0.05682883411645889, 0.010539230890572071, 0.06768736988306046, 0.06368760764598846,
      -0.08762392401695251, 0.020472679287195206, -0.04074596241116524, -0.024140747264027596,
      0.03185388818383217, -0.03205490857362747, -0.0774734765291214, -0.1129847913980484,
      0.04989292845129967, 0.03831632435321808, -0.05886485427618027, -0.07462483644485474,
      0.004048798233270645, -0.08318652957677841, 0.029927508905529976, -0.016514351591467857,
      0.015474114567041397, 0.09577970206737518, 0.07555089145898819, -0.00019984343089163303,
      -0.08879722654819489, 0.03813552483916283, -0.09497898072004318, 0.09222424030303955,
      -0.06416958570480347, -0.0872100219130516, 0.0765041932463646, -0.0005907167796976864,
      -0.03195873275399208, 0.020843299105763435, -0.10102944821119308, 0.044332101941108704,
      0.08466740697622299, 0.006511315703392029, 0.0302025955170393, -0.08118637651205063,
      -0.04950074851512909, 0.04044349119067192, -0.038393016904592514, -0.06402039527893066,
      -0.018457962200045586, 0.04629989340901375, -0.012046676129102707, -0.04723604395985603,
      -0.036398615688085556, -0.09154385328292847, 0.09656304121017456, 0.0562552884221077,
      0.010624849237501621, -0.02772010676562786, 0.053699761629104614, 0.013341963291168213,
      -0.05046588554978371, 0.04937969520688057, -0.05021543428301811, -0.004434457514435053,
      0.006412172224372625, -0.0199330635368824, 0.022136669605970383, -0.0028224310372024775],
     [0.11550980061292648, -0.033325716853141785, -0.05617591366171837, 0.03834376111626625,
      0.0860745757818222, -0.043982721865177155, 0.04041575267910957, 0.059895679354667664,
      0.11950483918190002, -0.015549201518297195, -0.03376397117972374, -0.005915555637329817,
      -0.06087829917669296, -0.02614171802997589, 0.07033923268318176, 0.044858649373054504,
      -0.01489707175642252, -0.08452184498310089, 0.024843359366059303, -0.03587223216891289,
      -0.014695502817630768, -0.12495292723178864, -0.04569030925631523, -0.023176243528723717,
      0.047612156718969345, -0.032387569546699524, 0.011886333115398884, -0.05735073238611221,
      -0.07054340839385986, -0.028812602162361145, -0.07238751649856567, -0.014763914979994297,
      0.036872487515211105, 0.07152018696069717, -0.018349340185523033, 0.02495955303311348,
      0.04896035045385361, 0.0599425807595253, 0.006890346761792898, -0.03509238362312317,
      0.0437748022377491, -0.0033823938574641943, 0.06409363448619843, 0.08040144294500351,
      0.06766460835933685, 0.0823824554681778, 0.0848497524857521, -0.008781432174146175,
      -0.07928825169801712, 0.005140388384461403, 0.07341854274272919, 0.010290294885635376,
      0.05182134732604027, 0.026235438883304596, -0.060541845858097076, -0.024074792861938477,
      -0.02913149818778038, 0.008967806585133076, 0.009916834533214569, 0.03807676583528519,
      -0.023507513105869293, 0.0327618271112442, 0.02904200553894043, 0.020775670185685158,
      0.08104612678289413, -0.032280728220939636, -0.011039938777685165, 0.061757784336805344,
      -0.03134357929229736, -0.02537851221859455, -0.028429431840777397, 0.030532680451869965,
      0.0871846079826355, 0.05722411721944809, 0.039353273808956146, 0.058123499155044556,
      0.030527736991643906, 0.030665775761008263, -0.033679183572530746, 0.004354088567197323,
      -0.013768407516181469, 0.0067752148024737835, -0.06794308125972748, -0.06818357110023499,
      -0.041274912655353546, 0.009113963693380356, -0.02748391032218933, 0.050837546586990356,
      0.011160863563418388, 0.08180064707994461, 0.033190153539180756, 0.008574094623327255,
      0.0041844905354082584, 0.03593835234642029, 0.0033621718175709248, -0.057333290576934814,
      -0.0014317892491817474, -0.09581729024648666, -0.057226475328207016, -0.027495048940181732,
      0.03926694020628929, -0.06567531824111938, -0.06911899894475937, -0.04296618700027466,
      -0.033974844962358475, 0.042776256799697876, 0.06033729016780853, -0.035805631428956985],
     [0.041542086750268936, 0.027304571121931076, -0.015354684554040432, -0.02478189952671528,
      0.01692609302699566, 0.06777767837047577, -0.04986707121133804, -0.07202363014221191,
      -0.10778584331274033, -0.03424849733710289, -0.07241235673427582, -0.005143890157341957,
      0.024216290563344955, -0.10751359909772873, -0.10753320902585983, 0.021383488550782204,
      -0.024968352168798447, -0.035352084785699844, -0.054699093103408813, -0.028957825154066086,
      0.08302483707666397, -0.007871197536587715, -0.035666704177856445, -0.012996627017855644,
      -0.11035110056400299, -0.046862415969371796, 0.04401964694261551, -0.06806940585374832,
      -0.09024570137262344, 0.039946600794792175, 0.019816633313894272, 0.009889048524200916,
      -0.08522733300924301, -0.06197739765048027, -0.09824264049530029, 0.025694536045193672,
      -0.11057665199041367, -0.1042868047952652, -0.023788072168827057, 0.02109675295650959,
      -0.06446532905101776, -0.1319330334663391, -0.004723900463432074, -0.034285373985767365,
      0.0538182370364666, 0.048805318772792816, -0.01216062344610691, -0.040454618632793427,
      0.03789842873811722, 0.07066146284341812, -0.07794515788555145, -0.10609094798564911,
      0.04654594138264656, 0.05822010338306427, -0.035312067717313766, -0.003830032190307975,
      -0.0387532003223896, -0.067506805062294, 0.03121250867843628, 0.0032410845160484314,
      0.08077243715524673, 0.08701049536466599, -0.13301652669906616, 0.07256364077329636,
      0.09073855727910995, 0.08570782840251923, -0.037382565438747406, 0.0006298574735410511,
      -0.09100306779146194, 0.046908263117074966, 0.07565892487764359, 0.011000621132552624,
      -0.004312898963689804, -0.015643635764718056, 0.02960159257054329, 0.07213453203439713,
      0.01540867704898119, 0.07712016999721527, 0.03682477027177811, -0.12738245725631714,
      -0.0800544023513794, -0.02200547605752945, -0.04960907623171806, -0.10040655732154846,
      0.07051881402730942, 0.05845306068658829, 0.00497357826679945, 0.037160441279411316,
      0.06366628408432007, -0.028949102386832237, 0.1266954243183136, 0.1368878334760666,
      0.08247774094343185, -0.038669370114803314, -0.02498645894229412, 0.05378735065460205,
      -0.022862425073981285, 0.006978918332606554, -0.0019466446246951818, 0.015583686530590057,
      0.006601529661566019, 0.009135977365076542, -0.004977117292582989, 0.034423016011714935,
      0.10280916839838028, -0.04630047827959061, 0.07764922082424164, 0.023784123361110687]], dtype=np.float32).T + np.asarray([-0.07165274769067764, -0.07560425996780396, -0.009687228128314018, 0.00662515452131629,
     -0.0016397658037021756, -0.03097822703421116, -0.02178289368748665, -0.0051564620807766914,
     -0.0767449215054512, -0.11084186285734177, -0.004215196240693331, 0.022778021171689034,
     0.02336885780096054, -0.07921639829874039, -0.05059126392006874, 0.0028490314725786448,
     -0.12051782757043839, -0.05177443474531174, 0.09004011005163193, -0.08269207924604416,
     -0.06300263851881027, 0.1286609172821045, -0.08207336813211441, -0.0727984830737114,
     -0.007571977097541094, 0.02275403030216694, -0.05658309534192085, 0.003177694510668516,
     -0.12410629540681839, -0.06956925988197327, -0.05104604363441467, 0.06510622799396515,
     -0.03726320341229439, -0.13344284892082214, -0.14402742683887482, -0.05229152739048004,
     -0.08763959258794785, -0.05780104547739029, -0.004806792829185724, -0.1141989678144455,
     -0.06514966487884521, -0.11113785952329636, -0.1321917325258255, -0.028253667056560516,
     0.07359348982572556, 0.029566532000899315, -0.0049754539504647255, -0.05405614897608757,
     0.03006337210536003, -0.06140867993235588, -0.02038695104420185, -0.06562113761901855,
     -0.04750829562544823, 0.029026906937360764, 0.04095446690917015, 0.0891430675983429,
     0.09121011197566986, -0.13019035756587982, -0.10057861357927322, 0.01154259778559208,
     0.08764519542455673, -0.07218444347381592, 0.020518850535154343, 0.05145468935370445,
     -0.11361388117074966, 0.009322820231318474, 0.005427637603133917, -0.04311476647853851,
     0.06889347732067108, -0.12074383348226547, -0.028939319774508476, -0.11033423990011215,
     0.0023562870919704437, 0.08952382206916809, -0.1408817023038864, -0.06536675244569778,
     -0.05980305001139641, -0.022504480555653572, 0.11865020543336868, -0.006324522662907839,
     0.016185766085982323, -0.03174837678670883, 0.060983188450336456, -0.04362465813755989,
     -0.09854190051555634, 0.03535920009016991, -0.08765165507793427, -0.048186399042606354,
     -0.11305364221334457, 0.053427983075380325, -0.06509952247142792, -0.0708719789981842,
     -0.07850941270589828, 0.08995600789785385, 0.02846762165427208, -0.08320445567369461,
     -0.1265796720981598, 0.006527699530124664, 0.05755690485239029, -0.07095123827457428,
     0.04673374816775322, 0.12675152719020844, -0.0444563589990139, 0.000248522701440379,
     -0.04454546421766281, -0.04447836056351662, -0.010780638083815575, 0.03491062670946121], dtype=np.float32)
    trajectory_h1 = trajectory_h1 / (1.0 + np.exp(-np.clip(trajectory_h1, -30.0, 30.0)))
    trajectory_h2 = trajectory_h1 @ np.asarray([[0.017601413652300835, -0.09501273185014725, -0.07340333610773087, -0.07862967252731323,
      -0.03782941773533821, 0.04697483032941818, -0.06440909951925278, -0.059859778732061386,
      0.019794035702943802, 0.02333441749215126, 0.07569600641727448, 0.04673440381884575,
      0.06398215889930725, -0.02465319074690342, 0.07518451660871506, 0.05068574100732803,
      0.11197687685489655, -0.04135071486234665, 0.05722757801413536, -0.09503455460071564,
      0.03907226771116257, 0.05999051779508591, -0.00787469930946827, 0.04038126394152641,
      0.007295120973140001, 0.058276016265153885, 0.020053617656230927, 0.07396716624498367,
      -0.09029930084943771, 0.012310697697103024, 0.0693662241101265, -0.026803279295563698,
      -0.07877111434936523, -0.030458996072411537, -0.06730369478464127, 0.028990721330046654,
      -0.0526130385696888, 0.027557620778679848, 0.0027254929300397635, 0.010128164663910866,
      -0.0713450014591217, 0.04248971492052078, 0.017173008993268013, -0.03491685166954994,
      0.07821550965309143, -0.031300000846385956, -0.026989774778485298, 0.08191096037626266,
      0.01976335234940052, -0.07919249683618546, -0.06445062905550003, -0.048596929758787155,
      -0.05374918505549431, 0.0691106840968132, -0.06127699837088585, 0.005703532136976719,
      0.04670366272330284, 0.07601986080408096, 0.001925928983837366, -0.06236325949430466,
      -0.06292466074228287, 0.019224591553211212, -0.08528513461351395, 0.03905359283089638,
      0.09766151010990143, -0.09354089945554733, -0.07088977098464966, 0.03515753149986267,
      -0.004907859489321709, 0.0694829598069191, -0.03207937628030777, 0.08596537262201309,
      0.06228563189506531, 0.06921322643756866, -0.014967349357903004, 0.0715818777680397,
      0.035439372062683105, 0.030814234167337418, -0.07546019554138184, 0.0789073035120964,
      0.015525457449257374, 0.026687102392315865, -0.0726100280880928, -0.005732707679271698,
      0.009276323020458221, 0.046536508947610855, -0.048997845500707626, 0.018340110778808594,
      0.030618075281381607, -0.011064527556300163, 0.013693557120859623, 0.07104063779115677,
      0.02507861517369747, -0.04585551097989082, 0.06404133141040802, 0.031569577753543854,
      -0.09540136903524399, -0.027593541890382767, 0.043175533413887024, -0.010045826435089111,
      0.08905502408742905, -0.05251030623912811, -0.0009426174801774323, 0.025853781029582024,
      -0.05245409905910492, -0.08198577165603638, 0.003064497373998165, -0.044596053659915924],
     [0.0298371110111475, -0.03565111383795738, -0.056706931442022324, -0.05483756586909294,
      -0.07834326475858688, 0.06926984339952469, -0.021585870534181595, -0.07333845645189285,
      0.02944204770028591, 0.07225997000932693, -0.0766017884016037, -0.019993111491203308,
      -0.03732278570532799, 0.07029430568218231, -0.010047534480690956, 0.010093354620039463,
      0.0607917383313179, -0.07685945928096771, 0.012712987139821053, -0.02145818993449211,
      -0.019469451159238815, 0.023945093154907227, 0.07509267330169678, 0.05287644639611244,
      -0.07341600209474564, -0.0019351239316165447, -0.029991038143634796, 0.02070748060941696,
      0.05624443665146828, 0.006970650050789118, 0.07363355159759521, -0.03591051325201988,
      -0.06030840054154396, 0.09588514268398285, -0.015943022444844246, 0.06899762153625488,
      -0.08148615807294846, -0.039976682513952255, 0.08186168968677521, 0.06527560204267502,
      0.025733759626746178, 0.09525696933269501, 0.06934750825166702, -0.021059945225715637,
      -0.0015957456780597568, 0.057731129229068756, 0.068763867020607, 0.038389239460229874,
      0.02657100185751915, 0.06606265902519226, 0.03489355742931366, 0.014701283536851406,
      -0.047863930463790894, 0.06581234931945801, 0.06693979352712631, -0.10426618903875351,
      -0.09841941297054291, -0.05008663609623909, 0.101955845952034, -0.011329594068229198,
      0.05299149453639984, -0.04133566841483116, 0.06589872390031815, -0.06841519474983215,
      -0.04715161770582199, 0.019886860623955727, 0.015894781798124313, 0.011143835261464119,
      -0.0575457438826561, 0.03874681144952774, 0.006616827566176653, -0.026994824409484863,
      0.10240696370601654, -0.046724122017621994, 0.04426354914903641, 0.04497483745217323,
      0.05860460177063942, 0.004466786049306393, -0.03281925618648529, 0.02242193929851055,
      -0.08827127516269684, 0.023906398564577103, -0.1070558950304985, -0.0024116002023220062,
      -0.004942622967064381, 0.07394906878471375, -0.055181004106998444, 0.06464769691228867,
      0.0602128729224205, -0.08571596443653107, -0.03889577463269234, -0.04523284360766411,
      -0.027664149180054665, -0.06824658811092377, 0.01643183082342148, -0.05650407820940018,
      0.016094719991087914, -0.007757664192467928, 0.006001520901918411, -0.021949322894215584,
      -0.034638840705156326, -0.043590422719717026, 0.01736929826438427, -0.030195359140634537,
      0.011488684453070164, 0.01914776861667633, 0.027252355590462685, 0.012882973067462444],
     [-0.08290208131074905, -0.10010751336812973, 0.04381776228547096, -0.07120916247367859,
      -0.012920824810862541, 0.10209330916404724, -0.0647663101553917, 0.06514012813568115,
      0.001644651754759252, -0.07401644438505173, 0.14547239243984222, 0.07362347096204758,
      -0.020792856812477112, 0.00518482131883502, 0.03427981585264206, 0.08542975783348083,
      -0.060932666063308716, -0.041222769767045975, 0.07173038274049759, 0.03677219897508621,
      0.03363770246505737, 0.035598065704107285, -0.04156838729977608, -0.018262913450598717,
      0.06873243302106857, 0.036233071237802505, 0.046988122165203094, -0.0775601714849472,
      0.08349355310201645, -0.03960426151752472, 0.07360158115625381, -0.06484859436750412,
      0.07860402017831802, 0.0790068581700325, 0.023961378261446953, -0.09093557298183441,
      -0.03742615878582001, 0.10704346746206284, 0.0393441766500473, 0.06494861841201782,
      0.08362656086683273, -0.019058063626289368, 0.010635671205818653, 0.010377107188105583,
      0.04609719291329384, 0.0874919667840004, 0.04995536431670189, 0.03204919770359993,
      0.04238634556531906, -0.07119587063789368, 0.02590666338801384, -0.003094900632277131,
      -0.08727800846099854, 0.13886432349681854, 0.07493090629577637, 0.01963597536087036,
      -0.02363346330821514, -0.06656182557344437, 0.02953963726758957, 0.13985195755958557,
      -0.07246597856283188, 0.009397828951478004, -0.0035763464402407408, 0.008046618662774563,
      0.07677964121103287, -0.031544122844934464, -0.03466218337416649, 0.029185814782977104,
      0.05524555593729019, -0.013971026055514812, 0.033616822212934494, 0.010815823450684547,
      -0.06813123822212219, 0.043528731912374496, -0.03756807744503021, -0.013880405575037003,
      0.07416903972625732, 0.12609174847602844, 0.0066028134897351265, -0.003936351742595434,
      -0.0017809157725423574, -0.0675506740808487, -0.06155069172382355, 0.007439373526722193,
      0.06806101649999619, -0.060303766280412674, 0.045301925390958786, 0.034131716936826706,
      0.07416468113660812, 0.026538044214248657, 0.05270020291209221, -0.04131902754306793,
      0.03648209944367409, 0.039414580911397934, -0.00819055549800396, 0.09033620357513428,
      0.06385476142168045, 0.01611201837658882, 0.004893668927252293, 0.10679958015680313,
      0.10720045119524002, -0.05782739073038101, -0.011650796048343182, -0.09473875910043716,
      -0.06588868796825409, 0.0053977929055690765, -0.04531317949295044, -0.07270236313343048],
     [-0.0018379554385319352, 0.06649596244096756, 0.08862960338592529, -0.11992308497428894,
      0.028250452131032944, 0.04355932027101517, -0.0020141464192420244, -0.06280908733606339,
      -0.03463449701666832, 0.04303256422281265, 0.15837423503398895, -0.12726573646068573,
      -0.03999663516879082, 0.0283969659358263, -0.025658687576651573, -0.007508900482207537,
      0.11164671927690506, 0.12299708276987076, -0.025767143815755844, -0.05714116245508194,
      0.06310543417930603, -0.1427309513092041, -0.08604524284601212, 0.06317917257547379,
      -0.06434577703475952, 0.023842910304665565, -0.0896260216832161, 0.041481003165245056,
      0.03388453647494316, 0.01178919430822134, -0.043000344187021255, -0.07894103229045868,
      -0.05146003141999245, 0.13379324972629547, 0.11098188161849976, -0.10672596842050552,
      -0.00849723070859909, 0.04226021096110344, 0.0725659430027008, -0.0023018037900328636,
      -0.05190384015440941, -0.0560942105948925, 0.013885618187487125, 0.05825816094875336,
      -0.013277767226099968, 0.11875101178884506, -0.01167221274226904, 0.06384667009115219,
      -0.052449289709329605, -0.015267934650182724, 0.06759323179721832, 0.014127364382147789,
      -0.037860333919525146, 0.13899968564510345, -0.00304974801838398, -0.07881290465593338,
      -0.06913964450359344, 0.01438195165246725, 0.0919477567076683, 0.07416722923517227,
      -0.10236914455890656, 0.018294939771294594, 0.025584282353520393, -0.05544735863804817,
      -0.06817813962697983, -0.04485723376274109, 0.07481515407562256, -0.07893147319555283,
      0.03267179802060127, 0.12010610848665237, -0.07707583904266357, 0.015808017924427986,
      0.08556339889764786, 0.007554987445473671, 0.1390104591846466, 0.1041117012500763,
      -0.09649732708930969, 0.032285865396261215, -0.04305583983659744, 0.08899126201868057,
      0.0077605643309652805, 0.013318688608705997, -0.03571633622050285, -0.01572587341070175,
      -0.05304762348532677, 0.0721777081489563, -0.03720157966017723, 0.061536695808172226,
      -0.007628635503351688, 0.015242647379636765, -0.00076385831926018, 0.038222648203372955,
      0.07627465575933456, -0.06968113780021667, -0.05902237817645073, 0.17953351140022278,
      0.10040020197629929, 0.05141044408082962, 0.08033628761768341, -0.0026032684836536646,
      0.1091722697019577, -0.09885459393262863, -0.05016963556408882, -0.1025276854634285,
      0.04229140654206276, -0.07444089651107788, 0.07491874694824219, -0.027927003800868988],
     [0.056167785078287125, 0.06727180629968643, 0.025541596114635468, -0.007656244095414877,
      -0.013297496363520622, -0.010317294858396053, -0.03545252978801727, -0.023318078368902206,
      0.026441384106874466, 0.00825798325240612, -0.005624547600746155, 0.03875124827027321,
      0.02462739683687687, 0.0025200992822647095, -0.08525400608778, 0.026653612032532692,
      0.0004085924010723829, -0.03197006508708, -0.09886395931243896, -0.05800090357661247,
      0.09479603916406631, -0.11879493296146393, 0.07341772317886353, 0.13844677805900574,
      -0.044748205691576004, 0.12974268198013306, -0.033049702644348145, 0.025715474039316177,
      -0.0423375703394413, -0.038827091455459595, 0.04143746942281723, 0.04677758738398552,
      0.0786280557513237, 0.013204971328377724, 0.0970076248049736, 0.020763445645570755,
      0.0003016535774804652, 0.1400979608297348, 0.07763451337814331, -0.06257381290197372,
      0.08303947746753693, 0.06524347513914108, 0.13225235044956207, -0.003798512974753976,
      0.06253271549940109, 0.03552234172821045, -0.08546309173107147, -0.0106107909232378,
      -0.004764824640005827, 0.04938211664557457, -0.0471443235874176, -0.04369195178151131,
      -0.00975277554243803, 0.04467713087797165, -0.04059722647070885, 0.05455261468887329,
      0.06130014732480049, 0.10138126462697983, -0.04798523709177971, 0.09085916727781296,
      -0.054559264332056046, -0.012413498014211655, 0.02915525995194912, -0.05370072275400162,
      0.06278982758522034, -0.05627070739865303, -0.022027533501386642, 0.01915157400071621,
      0.04292968660593033, 0.03212842717766762, 0.06362061202526093, 0.12209154665470123,
      0.08423799276351929, -0.042900994420051575, 0.11492587625980377, -0.02994832955300808,
      -0.05241489037871361, 0.07890808582305908, -0.08203806728124619, 0.06747492402791977,
      -0.04898922145366669, 0.02341878041625023, -0.029757481068372726, -0.0740644782781601,
      0.04542801156640053, 0.07802101224660873, -0.025236336514353752, 0.1362052708864212,
      0.06839241832494736, -0.022580383345484734, 0.025701558217406273, -0.06692786514759064,
      -0.047434523701667786, -0.027178999036550522, -0.06319355964660645, 0.16442054510116577,
      -0.037267349660396576, 0.04788331687450409, -0.010201343335211277, 0.05412210151553154,
      -0.06493876129388809, -0.014117368496954441, -0.09441815316677094, 0.03275427594780922,
      -0.04524563252925873, -0.007714614737778902, -0.013431123457849026, -0.08055298775434494],
     [0.04779799282550812, -0.06419210135936737, 0.02723230980336666, 0.014209778979420662,
      0.01515891496092081, 0.07867580652236938, -0.04555170610547066, 0.015447226352989674,
      0.046353694051504135, 0.07023151963949203, -0.016104865819215775, 0.0066697197034955025,
      0.0948525071144104, 0.03797203674912453, 0.08192593604326248, 0.026016652584075928,
      0.13883593678474426, -0.041170455515384674, 0.03591006249189377, -0.0807807594537735,
      0.01092152763158083, 0.05169029161334038, 0.004995218478143215, 0.04505520686507225,
      -0.05621585249900818, -0.03836064413189888, -0.09356728941202164, -0.022723162546753883,
      0.03764365240931511, 0.026376016438007355, -0.05654881149530411, -0.07763463258743286,
      0.10851884633302689, 0.07992342114448547, 0.08854275196790695, 0.032408181577920914,
      0.10069267451763153, -0.053034793585538864, -0.08889974653720856, -0.03418949991464615,
      0.00803426280617714, -0.09393738955259323, 0.07301121205091476, 0.057447176426649094,
      -0.017583422362804413, 0.04593717306852341, -0.006036299280822277, 0.032477088272571564,
      0.0810195729136467, 0.05201054736971855, 0.07315076887607574, 0.01738951914012432,
      0.07145608216524124, 0.11394508183002472, 0.003392899874597788, -0.08615624904632568,
      0.02737244963645935, -0.0012283192481845617, 0.012909016571938992, 0.0688168853521347,
      -0.07941468060016632, 0.01689235493540764, -0.08530688285827637, 0.006868191994726658,
      0.03133687749505043, 0.06802009791135788, -0.012797373346984386, -0.04942222312092781,
      0.020240357145667076, 0.06358169764280319, -0.0810559019446373, 0.10999377071857452,
      0.0189605001360178, -0.09740240126848221, 0.10931012034416199, -0.010133947245776653,
      0.05866019055247307, 0.10321103036403656, -0.08116686344146729, 0.07950640469789505,
      0.0002633557887747884, 0.06486047059297562, -0.05437268316745758, -0.0021305850241333246,
      0.0026146366726607084, 0.11033704876899719, -0.06074477732181549, -0.052111778408288956,
      0.0029405297245830297, -0.07467858493328094, 0.009655707515776157, -0.03859364986419678,
      0.05644388496875763, 0.03151766583323479, -0.05109812319278717, 0.16735440492630005,
      0.04841282591223717, -0.0851319432258606, 0.08504556119441986, 0.054438866674900055,
      0.05071009323000908, -0.054681845009326935, -0.012667772360146046, 0.0317191518843174,
      -0.05977408587932587, -0.07149066776037216, 0.09825156629085541, 0.06072220206260681],
     [0.018018880859017372, -0.05785827711224556, 0.14976803958415985, 0.061353445053100586,
      -0.10022380203008652, 0.04684535786509514, -0.06416642665863037, 0.02479073405265808,
      0.06849359720945358, 0.027607960626482964, -0.00909856054931879, -0.07134432345628738,
      0.08228260278701782, -0.02854757197201252, -0.022542428225278854, 0.07479690760374069,
      0.04532099887728691, 0.09086298942565918, 0.03568682819604874, -0.07196927070617676,
      -0.013819687999784946, 0.055891700088977814, 0.044244684278964996, 0.00843807589262724,
      -0.05696433410048485, -0.050086770206689835, 0.06483187526464462, -0.06160268932580948,
      -0.058127377182245255, 0.026740700006484985, 0.06310880184173584, -0.059141356498003006,
      -0.05784754455089569, 0.09070765972137451, -0.02259034849703312, -0.01057359017431736,
      -0.01500059012323618, 0.10049138963222504, 0.0498599149286747, -0.055751409381628036,
      -0.07860177010297775, -0.05394803732633591, -0.02671063132584095, -0.02995731309056282,
      0.01621614769101143, -0.024648714810609818, -0.02319537103176117, 0.07487727701663971,
      -0.016715725883841515, 0.02680140919983387, 0.08907679468393326, -0.03528208285570145,
      0.08557146042585373, 0.037214286625385284, 0.06685204058885574, 0.0412643700838089,
      -0.02445238083600998, 0.047853730618953705, -0.01554610300809145, 0.08403784036636353,
      0.036798521876335144, 0.0773504376411438, -0.03148731589317322, -0.02365449257194996,
      0.07537972182035446, 0.09063537418842316, 0.012777292169630527, 0.0747331902384758,
      0.07692799717187881, -0.06570734083652496, 0.08143534511327744, 0.0019968566484749317,
      0.07488703727722168, 0.06258554756641388, 0.08400896936655045, -0.04073021560907364,
      -0.0898279994726181, 0.044677119702100754, -0.021863820031285286, 0.04168509691953659,
      -0.06110403686761856, -0.06876055151224136, 0.06768238544464111, 0.049693141132593155,
      -0.017658956348896027, -0.05142493546009064, -0.06937786191701889, 0.009959942661225796,
      0.029336923733353615, 0.10094428062438965, 0.037536151707172394, -0.03510073572397232,
      0.09110952168703079, -0.01777711696922779, 0.006961691193282604, 0.1560390591621399,
      -0.046796269714832306, -0.09088277816772461, -0.05091426894068718, 0.029337378218770027,
      0.056463200598955154, 0.07397478818893433, -0.0656697005033493, -0.0015754189807921648,
      -0.006654007360339165, 0.09600900113582611, -0.04872998222708702, 0.05679113045334816],
     [-0.06169499456882477, -0.004859825596213341, 0.03531854227185249, 0.05876006931066513,
      -0.07170922309160233, -0.07278038561344147, -0.056417662650346756, -0.07556862384080887,
      0.066450335085392, -0.022074924781918526, 0.10511577129364014, -0.07281634211540222,
      -0.06095730513334274, 0.08012853562831879, 0.015950435772538185, -0.08697459846735,
      -0.0706760585308075, 0.03992403298616409, 0.07973191142082214, 0.017804136499762535,
      -0.008459020406007767, 0.049097709357738495, 0.08821950107812881, -0.043631430715322495,
      0.06007647141814232, -0.06361178308725357, -0.06201363727450371, -0.025143558159470558,
      0.010751698166131973, 0.008509747684001923, -0.01934618502855301, -0.07146095484495163,
      0.08043239265680313, 0.03940374031662941, 0.05339522287249565, -0.06652512401342392,
      0.010167386382818222, 0.09815537929534912, -0.028574135154485703, 0.08083967864513397,
      -0.050063129514455795, -0.07058511674404144, -0.02853429690003395, 0.08326852321624756,
      0.07004493474960327, -0.029306869953870773, 0.013979769311845303, 0.04590974375605583,
      -0.059928737580776215, 0.022015871480107307, 0.0158549752086401, -0.003838408039882779,
      0.044579777866601944, 0.004595462698489428, -0.039992813020944595, -0.026547040790319443,
      0.034784525632858276, -0.019266046583652496, 0.04154423251748085, -0.024204684421420097,
      -0.10295924544334412, -0.01898016966879368, 0.027680864557623863, -0.0007974131731316447,
      0.0754748210310936, -0.04270360991358757, -0.08568500727415085, 0.04415064677596092,
      -0.07260727137327194, -0.043527498841285706, -0.09778262674808502, -0.034944672137498856,
      0.03500170633196831, 0.07849805802106857, 0.01992006227374077, 0.07627952098846436,
      0.0037581571377813816, -0.03745964914560318, -0.08128896355628967, 0.06485266983509064,
      -0.046442583203315735, -0.04084097221493721, -0.052562166005373, -0.006735723000019789,
      0.011614679358899593, 0.007233262062072754, 0.0021826084703207016, -0.045965537428855896,
      0.05489007383584976, -0.025651857256889343, -0.09916616976261139, -0.07590749114751816,
      0.057611823081970215, -0.020769422873854637, 0.08302772790193558, 0.10738945007324219,
      -0.000709526997525245, -0.06674128770828247, -0.03168053179979324, 0.03684907779097557,
      -0.02715255878865719, 0.03844652697443962, 0.022593950852751732, -0.044326819479465485,
      0.0572427362203598, -0.09848187118768692, 0.010581192560493946, 0.02770266681909561],
     [-0.09273669123649597, 0.045247551053762436, -0.07736140489578247, 0.040523167699575424,
      -0.006101122125983238, 0.06990218907594681, -0.07541901618242264, 0.04755874350667,
      0.03764723613858223, 0.07131201773881912, 0.0794040858745575, 0.07359518855810165,
      -0.0921197161078453, 0.02915124036371708, -0.042648933827877045, 0.006625327747315168,
      -0.07468250393867493, -0.0088726244866848, -0.04494394734501839, -0.01589622162282467,
      -0.016847455874085426, -0.01901063695549965, -0.05503355711698532, 0.06814328581094742,
      0.010370890609920025, -0.04331781715154648, -0.016849828884005547, 0.00332469935528934,
      0.016005337238311768, 0.02042459324002266, 0.07339926809072495, -0.08869278430938721,
      0.09803187847137451, 0.10038960725069046, -0.092364102602005, 0.06762954592704773,
      0.07215513288974762, -0.020562337711453438, -0.06475074589252472, 0.07872307300567627,
      -0.03342897444963455, 0.09797577559947968, 0.06784704327583313, -0.09873808175325394,
      -0.10774527490139008, -0.02484452910721302, -0.00036002029082737863, -0.06714706867933273,
      -0.04224657267332077, -0.052368227392435074, 0.08929988741874695, 0.03750470280647278,
      -0.07617884874343872, 0.03213292360305786, 0.027732303366065025, 0.04982985928654671,
      0.04182247817516327, 0.013935747556388378, -0.010867487639188766, 0.05474760755896568,
      0.1018582433462143, -0.040455207228660583, -0.0651605948805809, -0.07305287569761276,
      0.014092492870986462, -0.03713666647672653, 0.07689222693443298, 0.07747124135494232,
      -0.02412569895386696, 0.0006806529127061367, 0.06795230507850647, -0.06968645751476288,
      -0.03214797005057335, -0.06001511216163635, -0.026604026556015015, 0.05467890948057175,
      0.0035539099480956793, -0.06699205189943314, 0.044017497450113297, 0.09670691192150116,
      -0.046966709196567535, 0.03163900598883629, -0.04339100420475006, 0.022971440106630325,
      0.03370162844657898, -0.10037288069725037, -0.01853133924305439, -0.08255018293857574,
      0.03426092490553856, 0.06803654134273529, -0.029897818341851234, -0.017646389082074165,
      -0.028215330094099045, 0.057780876755714417, 0.005996153224259615, -0.0215119831264019,
      -0.02773229219019413, -0.00315539981238544, -0.0437789112329483, 0.01839224062860012,
      0.044233910739421844, -0.09547402709722519, -0.08572380244731903, -0.002059004968032241,
      -0.055473532527685165, 0.0750756710767746, -0.08963669836521149, -0.04003407806158066],
     [0.07870157808065414, -0.08641542494297028, 0.19303089380264282, -0.015418616123497486,
      0.08444259315729141, 0.08338304609060287, 0.06166974455118179, 0.03812016174197197,
      -0.011733047664165497, -0.0684487596154213, 0.1361038088798523, 0.03644043952226639,
      0.013288885354995728, -0.004523353651165962, -0.0418369397521019, -0.08400145918130875,
      0.15076670050621033, -0.050947707146406174, -0.11387571692466736, -0.11079441010951996,
      -0.002381667960435152, -0.12054790556430817, 0.06364387273788452, 0.09954674541950226,
      -0.09039261937141418, -0.006533646956086159, 0.0589841827750206, 0.05215238407254219,
      0.12163905054330826, -0.03265160322189331, -0.07109122723340988, -0.0061267889104783535,
      0.013182524591684341, 0.06229119375348091, 0.1403570920228958, 0.06647948920726776,
      0.14514365792274475, 0.10069780796766281, -0.10688487440347672, 0.11525697261095047,
      -0.027656827121973038, 0.07898412644863129, 0.01079602725803852, 0.02986796759068966,
      -0.10719014704227448, 0.04458795115351677, -0.024899940937757492, 0.0598016232252121,
      -0.004339289851486683, -0.00965261459350586, -0.11414380371570587, 0.0006049498915672302,
      -0.07986697554588318, -0.006705110892653465, 0.13667388260364532, 0.043573107570409775,
      -0.0925406739115715, 0.024063268676400185, 0.06872983276844025, 0.14183083176612854,
      -0.12344484776258469, -0.11836715787649155, 0.023132648319005966, -0.022827669978141785,
      -0.04607494920492172, -0.004569886717945337, -0.02933824434876442, -0.13017047941684723,
      -0.07081955671310425, 0.13024622201919556, 0.03781760483980179, 0.1531226933002472,
      0.022160252556204796, -0.09026264399290085, 0.028831597417593002, 0.06036669388413429,
      0.012868166901171207, -0.00818033516407013, -0.12488359212875366, -0.01734662801027298,
      -0.009183470159769058, -0.1125655323266983, -0.08928791433572769, -0.06250673532485962,
      0.08990275114774704, 0.1014394536614418, -0.040101103484630585, 0.13623951375484467,
      -0.08543336391448975, -0.042731840163469315, -0.007441829424351454, -0.013197878375649452,
      0.050110332667827606, -0.021156011149287224, -0.051050905138254166, 0.1809404045343399,
      0.11406400054693222, 0.046934761106967926, 0.07958018779754639, 0.15748947858810425,
      0.11059742420911789, -0.09165570139884949, -0.10416179150342941, -0.09990760684013367,
      0.03261769562959671, -0.07195693254470825, 0.023016126826405525, 0.03239831328392029],
     [0.03675369545817375, 0.09597531706094742, 0.033940982073545456, 0.032013047486543655,
      -0.017141643911600113, -0.04537301883101463, -0.03687409684062004, -0.09497369825839996,
      0.03205624967813492, 0.02952713705599308, -0.01755383424460888, 0.034916605800390244,
      0.01271883025765419, 0.04953242093324661, 0.020908670499920845, -0.08780805766582489,
      -0.07920847088098526, 0.07415397465229034, 0.06379960477352142, -0.052806559950113297,
      -0.09533217549324036, -0.007413548417389393, 0.02388725057244301, -0.08168728649616241,
      -0.04231041669845581, 0.04681354761123657, -0.07783143222332001, -0.011754040606319904,
      0.059228427708148956, 0.0849769115447998, -0.06920263171195984, -0.0015102917095646262,
      0.012090746313333511, 0.06409762054681778, 0.05047623813152313, -0.08194668591022491,
      0.029896510764956474, 0.015047883614897728, -0.0511198416352272, 0.09473183751106262,
      0.05840509757399559, -0.05571799352765083, 0.02651279978454113, -0.04799867793917656,
      -0.010409004054963589, -0.06863300502300262, 0.11181196570396423, 0.024928517639636993,
      -0.017330227419734, -0.01093883253633976, -0.06317543983459473, 0.0132774468511343,
      0.05771711468696594, 0.1048479676246643, 0.06950973719358444, -0.07905977964401245,
      0.05153800547122955, 0.10258901864290237, -0.027725890278816223, 0.002415633527562022,
      -0.0634225457906723, 0.07279416173696518, 0.07055306434631348, 0.05784369260072708,
      0.03555792570114136, -0.04475541040301323, -0.08774935454130173, -0.09150940924882889,
      -0.04030134901404381, -0.08349744230508804, -0.05957011133432388, -0.07315269112586975,
      -0.08012901246547699, -0.06726814806461334, 0.06367319077253342, 0.07024995982646942,
      -0.06827441602945328, 0.07118457555770874, -0.058236610144376755, 0.04791318625211716,
      0.08486602455377579, -0.015675965696573257, 0.05732564255595207, 0.061469145119190216,
      0.07238009572029114, 0.02875460498034954, -0.009564263746142387, -0.07574301958084106,
      0.030043968930840492, 0.009246702305972576, -0.024074571207165718, -0.03881710767745972,
      -0.10234013199806213, -0.062155742198228836, -0.006499355658888817, -0.014273048378527164,
      0.06787997484207153, 0.045520178973674774, 0.024477370083332062, 0.07332850992679596,
      -0.08015015721321106, -0.01865289732813835, 0.0489124059677124, -0.010649889707565308,
      -0.011506404727697372, -0.012397808022797108, 0.024746062234044075, 0.039001140743494034],
     [-0.04406595602631569, -0.08763860166072845, 0.13493716716766357, -0.08675539493560791,
      0.017875229939818382, 0.006950035225600004, 0.008380942977964878, -0.09414799511432648,
      0.025601942092180252, 0.0867064893245697, -0.005606804043054581, 0.0015898634446784854,
      0.047994766384363174, 0.08707282692193985, -0.0921124592423439, -0.027348406612873077,
      0.11464256793260574, -0.06987541913986206, -0.04645877331495285, 0.08407893031835556,
      0.07791425287723541, -0.08711601048707962, 0.06715591996908188, 0.029670868068933487,
      -0.087494395673275, 0.07061366736888885, 0.03757237270474434, 0.037216298282146454,
      0.06993766874074936, -7.718965207459405e-05, 0.05971873179078102, -0.08315937221050262,
      -0.042825065553188324, 0.0638728141784668, 0.08921222388744354, 0.06368137151002884,
      0.034137140959501266, 0.11261816322803497, -0.05187274515628815, 0.08445464819669724,
      0.005709016229957342, 0.009793754667043686, 0.033914804458618164, 0.06886261701583862,
      -0.04302104562520981, 0.012293831445276737, -0.047254424542188644, -0.06782559305429459,
      0.05337157100439072, 0.06280921399593353, -0.06228640675544739, -0.03401033207774162,
      -0.0970982164144516, 0.12625949084758759, -0.03282740339636803, -0.024583693593740463,
      -0.06831193715333939, 0.04311574622988701, 0.04169951751828194, 0.12319275736808777,
      -0.0006240108632482588, 0.06592722982168198, 0.0638280063867569, 0.025008613243699074,
      -0.04598182067275047, 0.013842330314218998, 0.08883047848939896, -0.05548889935016632,
      0.01770435832440853, -0.02823753096163273, 0.047502100467681885, 0.04020188748836517,
      -0.08033569157123566, -0.004045700188726187, -0.025948651134967804, -0.055440135300159454,
      -0.03291535750031471, 0.06299328058958054, -0.11168351769447327, 0.07584608346223831,
      0.08100295066833496, 0.009533080272376537, -0.054676905274391174, -0.026606308296322823,
      0.06814117729663849, 0.019722064957022667, -0.02776016853749752, -0.008610665798187256,
      -0.07175945490598679, -0.041622091084718704, -0.045685768127441406, -0.0657704621553421,
      0.10285870730876923, 0.015249027870595455, -0.00853293389081955, 0.118305504322052,
      0.0973455086350441, 0.014446098357439041, -0.05998320132493973, 0.00484659755602479,
      0.06346356868743896, 0.007202997803688049, 0.0004565423878375441, 0.016373351216316223,
      0.010531781241297722, 0.011346705257892609, -0.06786071509122849, -0.0846935585141182],
     [-0.00850706733763218, 0.014389961026608944, -0.08998861908912659, -0.04706203192472458,
      0.10074453055858612, 0.040787890553474426, -0.03178824484348297, 0.045259714126586914,
      0.0184337068349123, -0.040435194969177246, 0.059847962111234665, 0.017838861793279648,
      -0.06943392008543015, -0.03315027430653572, 0.06977524608373642, -0.030088622123003006,
      0.04909875616431236, -0.03439280763268471, 0.03450184315443039, 0.047050993889570236,
      0.04917250573635101, -0.09132513403892517, 0.004960785154253244, -0.023632464930415154,
      -0.0678844153881073, -0.054769665002822876, -0.02272554486989975, -0.05087984353303909,
      0.021837346255779266, -0.046746134757995605, -0.011316142044961452, -0.004278120584785938,
      -0.008510950952768326, -0.02091853693127632, -0.040647242218256, -0.011108743958175182,
      0.05437590926885605, -0.028104033321142197, -0.0901765376329422, -0.03341229632496834,
      -0.05462419614195824, 0.0591287836432457, 0.04149381071329117, -0.09754050523042679,
      -0.09742745757102966, -0.0573086217045784, -0.04938383772969246, -0.00777720520272851,
      0.06376942247152328, 0.010769790038466454, -0.04701238125562668, 0.10405626893043518,
      0.06592699885368347, 0.03321179002523422, -0.03582104668021202, -0.05468904972076416,
      -0.055713117122650146, 0.05165153369307518, -0.003314184257760644, 0.09435076266527176,
      -0.08348485082387924, -0.06225190684199333, 0.030234837904572487, 0.07266394793987274,
      -0.045718640089035034, -0.0509820319712162, -0.07467588037252426, -0.09588585048913956,
      0.07054094970226288, -0.08197709172964096, -0.06686332076787949, 0.05762028321623802,
      -0.007279965095221996, -0.00958791933953762, -0.05401409789919853, -0.03941541165113449,
      0.02494836039841175, 0.0675147995352745, -0.09456457197666168, 0.0030857946258038282,
      0.027076056227087975, -0.10400603711605072, 0.027948986738920212, -0.022230876609683037,
      0.08122196048498154, -0.0972151830792427, 0.07046657055616379, -0.02029973268508911,
      -0.02771659754216671, 0.03963284194469452, -0.08688812702894211, -0.003026630962267518,
      -0.024565570056438446, 0.03043050318956375, -0.10325270146131516, -0.024870675057172775,
      0.08072995394468307, 0.11080248653888702, -0.060334041714668274, -0.05184311792254448,
      0.0933946892619133, 0.04979301616549492, 0.058156806975603104, 0.04098713770508766,
      0.0859432965517044, -0.08446255326271057, 0.05649585649371147, -0.08188709616661072],
     [-0.08670695126056671, 0.06837861984968185, 0.07133735716342926, 0.04716777428984642,
      -0.005518622230738401, 0.08170340210199356, -0.09076432883739471, 0.0002693773421924561,
      -0.07944153994321823, -0.018887463957071304, 0.13547417521476746, -0.08015888184309006,
      -0.008554226718842983, 0.0214842576533556, -0.033345822244882584, 0.07914580404758453,
      0.04457777738571167, 0.07819196581840515, 0.05734992399811745, -0.0037701663095504045,
      0.06372012197971344, 0.049238089472055435, 0.06739610433578491, 0.0027710243593901396,
      -0.051322776824235916, 0.10284129530191422, -0.06250172853469849, 0.022531865164637566,
      0.027789397165179253, -0.004353759810328484, -0.03322763368487358, -0.009994910098612309,
      0.004036226775497198, 0.09411720931529999, -0.022356098517775536, -0.0771045908331871,
      0.09299212694168091, -0.0036351829767227173, 0.08491667360067368, 0.09583732485771179,
      0.0881737470626831, 0.06677060574293137, 0.04343914985656738, 0.03306611254811287,
      0.09003031253814697, 0.135379821062088, -0.08664025366306305, -0.06983060389757156,
      0.07464131712913513, -0.0011566400062292814, -0.07500573992729187, 0.019741056486964226,
      -0.05876942723989487, -0.037362828850746155, -0.030980506911873817, 0.0417424812912941,
      0.04737410321831703, -0.04515911266207695, 0.11058296263217926, 0.02032284066081047,
      0.006396959535777569, -0.08386669307947159, -0.040527261793613434, -0.02052098885178566,
      -0.019271479919552803, 0.09999961405992508, 0.03546781465411186, 0.04225291684269905,
      0.019923489540815353, 0.006578266154974699, -0.0481262169778347, 0.1257016509771347,
      0.034609194844961166, 0.016366710886359215, 0.05142562463879585, 0.07792715728282928,
      -0.01422406267374754, 0.07415503263473511, -0.015766164287924767, 0.04681198671460152,
      0.0065328218042850494, 0.018552590161561966, 0.0599878653883934, -0.05785095691680908,
      -0.06609679758548737, 0.024604177102446556, -0.09008904546499252, 0.025759151205420494,
      0.05602765083312988, 0.021480148658156395, 0.028547758236527443, -0.035095736384391785,
      0.020775482058525085, 0.014947487972676754, -0.04207126051187515, 0.12592041492462158,
      -0.02280963771045208, 0.027485502883791924, -0.011053401976823807, 0.015264620073139668,
      0.007365218363702297, -0.024856604635715485, 0.02699066512286663, 0.06906471401453018,
      0.04749331250786781, -0.007541049737483263, 0.07022453844547272, 0.020352572202682495],
     [0.08903472125530243, -0.015948858112096786, -0.033124640583992004, -0.07486128062009811,
      -0.0456039123237133, 0.008441460318863392, -0.03906511887907982, 0.019419066607952118,
      0.022828759625554085, 0.01171691995114088, -0.05045127496123314, -0.06134714186191559,
      0.01950795203447342, -0.02516743540763855, -0.031594421714544296, -0.09693083167076111,
      0.04581514745950699, -0.08060942590236664, -0.07701021432876587, 0.05551624670624733,
      -0.10762003809213638, 0.08000924438238144, 0.022368501871824265, -0.12303058058023453,
      0.029576802626252174, 0.02762422151863575, 0.07488059997558594, -0.09944406896829605,
      -0.0626341924071312, -0.019734589383006096, -0.07700293511152267, -0.09992125630378723,
      0.022573914378881454, -0.06037978082895279, 0.07074979692697525, -0.023341285064816475,
      0.05856173485517502, 0.0156504288315773, 0.04708617180585861, 0.04454023391008377,
      0.02477281168103218, -0.052644189447164536, 0.02219795249402523, -0.08441732823848724,
      0.007900162599980831, 0.030230967327952385, 0.006082945037633181, -0.02083682082593441,
      -0.09752853959798813, -0.08157679438591003, -0.07183538377285004, 0.017085013911128044,
      -0.01875288411974907, 0.006094449199736118, 0.02958841249346733, 0.07833125442266464,
      -0.0035996739752590656, -0.04679503291845322, 0.0275762677192688, -0.005058120004832745,
      0.010093733668327332, 0.033392272889614105, 0.07192077487707138, -0.030980296432971954,
      0.06763292849063873, -0.10359198600053787, -0.07333751767873764, -0.039201878011226654,
      0.05814814195036888, 0.06452842801809311, 0.037013161927461624, -0.061857983469963074,
      0.0733647421002388, 0.02416687272489071, 0.03691694140434265, -0.08272621035575867,
      -0.03969200700521469, -0.09824750572443008, -0.04507977142930031, -0.08179532736539841,
      0.0550009123980999, 0.0794614851474762, 0.019939469173550606, -0.08133857697248459,
      -0.03714510798454285, -0.042503997683525085, 0.0755050927400589, 0.01993536204099655,
      0.05872344970703125, 0.06658150255680084, 0.032864175736904144, -0.11784999817609787,
      -0.01904013380408287, -0.0909152626991272, -0.06617235392332077, 0.006962873972952366,
      -0.09188266098499298, -0.007917641662061214, 0.03763752803206444, 0.03002278320491314,
      0.023295126855373383, 0.08120893687009811, -0.0318061001598835, 0.022447817027568817,
      0.06400295346975327, -0.0336749367415905, 0.027947532013058662, -0.04280597344040871],
     [0.040400318801403046, -0.004524430725723505, -0.06356827169656754, 0.07370014488697052,
      -0.0771612897515297, -0.07108575850725174, 0.03544287383556366, 0.061935555189847946,
      -0.05436144769191742, -0.06477317214012146, 0.05319445952773094, 0.05140281468629837,
      0.07678750157356262, 0.04396750405430794, 0.061716582626104355, 0.0261307992041111,
      -0.04457873851060867, -0.0049407100304961205, -0.03343873843550682, -0.10261193662881851,
      -0.0724080353975296, -0.06196705624461174, -0.0073724957183003426, 0.0015634765150025487,
      -0.05435330420732498, -0.054229866713285446, -0.07657076418399811, 0.004921221174299717,
      0.04764818772673607, 0.0437188521027565, -0.09065830707550049, 0.06332851201295853,
      0.09639280289411545, 0.0825907289981842, -0.07380864024162292, -0.07366712391376495,
      -0.034223735332489014, -0.0439208559691906, 0.010585487820208073, -0.018469002097845078,
      -0.04072723537683487, 0.03750599920749664, 0.03922667354345322, 0.07494665682315826,
      0.06636150181293488, 0.0784309059381485, 0.09893784672021866, 0.008719204925000668,
      0.024815665557980537, 0.006506165023893118, -0.035487495362758636, 0.02954850159585476,
      -0.005305527243763208, -0.03158479183912277, -0.004653482232242823, -0.09693381935358047,
      0.01594039425253868, 0.033015355467796326, -0.05821746587753296, 0.023891834542155266,
      -0.02667084150016308, -0.05144912004470825, -0.030075160786509514, -0.009209267795085907,
      -0.0730026364326477, 0.08084716647863388, 0.08558652549982071, -0.004650213290005922,
      -0.09449207782745361, -0.016933292150497437, 0.01958128809928894, 0.027249490842223167,
      -0.06829949468374252, 0.09389735758304596, 0.051102932542562485, 0.027279701083898544,
      -0.02549801394343376, 0.02016906440258026, 0.08638691157102585, 0.011086209677159786,
      -0.0006438193377107382, -0.08910263329744339, -0.001341113238595426, 0.03834046050906181,
      -0.022505687549710274, 0.08130264282226562, 0.060754209756851196, 0.0019467289093881845,
      -0.0852741226553917, -0.06058180704712868, -0.02096950262784958, -0.033955685794353485,
      0.024516453966498375, 0.009706716984510422, -0.018648605793714523, -0.06003430485725403,
      0.03219151496887207, 0.09663330763578415, -0.06038783863186836, 0.06356730312108994,
      -0.05227741226553917, 0.03362511470913887, -0.027623707428574562, -0.08503538370132446,
      0.07523024827241898, -0.023478807881474495, -0.06232293322682381, 0.05599727854132652],
     [0.04186223819851875, -0.022984027862548828, 0.07784967869520187, 0.05123426020145416,
      0.048491403460502625, 0.0668225884437561, 0.014732925221323967, -0.05392772704362869,
      0.031284086406230927, -0.0028162761591374874, -0.08008083701133728, 0.04205877333879471,
      0.09188723564147949, 0.04489985108375549, 0.022517135366797447, 0.0021913256496191025,
      0.04273688420653343, 0.07870055735111237, 0.06310153752565384, -0.017293401062488556,
      0.06599921733140945, -0.09471216797828674, 0.03789140284061432, -0.0658850446343422,
      -0.029142683371901512, -0.05652458965778351, 0.07103420048952103, -0.036421213299036026,
      0.09625354409217834, 0.06882784515619278, -0.06564102321863174, -0.0637669488787651,
      0.02180452086031437, -0.016027215868234634, 0.029854312539100647, -0.06958562880754471,
      0.09365245699882507, 0.13295963406562805, -0.00482726376503706, 0.040685661137104034,
      0.03918659687042236, 0.0025967599358409643, -0.06798853725194931, 0.10161114484071732,
      0.02386261522769928, 0.09621472656726837, 0.06791096180677414, -0.07035613059997559,
      -0.029485631734132767, -0.08830110728740692, 0.06257014721632004, 0.0010460460325703025,
      -0.02406906522810459, -0.046344906091690063, -0.08028949797153473, -0.03387301415205002,
      0.04960386082530022, 0.03845307230949402, 0.08124689012765884, -0.021696118637919426,
      -0.0623406246304512, 0.0443921759724617, 0.04019200801849365, 0.061109624803066254,
      -0.05025700852274895, -0.030023306608200073, 0.06515814363956451, 0.041267458349466324,
      -0.10067605972290039, -0.05798189714550972, 0.015002057887613773, -0.059348564594984055,
      -0.06315354257822037, -0.050825342535972595, -0.021304232999682426, -0.05323481559753418,
      0.06984654068946838, -0.053373586386442184, -0.08614107966423035, -0.05830691382288933,
      0.03340328484773636, 0.028658486902713776, -0.09375454485416412, -0.004101071506738663,
      0.04514029994606972, 0.05362316220998764, -0.05660165846347809, 0.008798005059361458,
      0.0492594949901104, 0.05755116790533066, 0.09388803690671921, -0.07641046494245529,
      0.0495675690472126, -0.10757126659154892, 0.042541276663541794, -0.027522943913936615,
      0.05805501714348793, 0.09418603777885437, -0.08068722486495972, 0.08539670705795288,
      -0.022176124155521393, -0.005452530458569527, 0.0772382989525795, -0.06719037890434265,
      0.1085154339671135, 0.02170742303133011, -0.022364439442753792, 0.029781952500343323],
     [-0.07715394347906113, 0.054790351539850235, 0.10281217843294144, -0.04364774376153946,
      0.002994481474161148, -0.008460409007966518, 0.07823699712753296, -0.059850938618183136,
      -0.020391441881656647, -0.07923189550638199, -0.08679448068141937, -0.061471983790397644,
      0.07024484872817993, 0.018995212391018867, -0.02680276334285736, 0.04418366029858589,
      0.02424515038728714, 0.028088752180337906, -0.022792302072048187, 0.02252722531557083,
      -0.021095095202326775, -0.09908454120159149, 0.10108020156621933, -0.08060900866985321,
      -0.07088855654001236, -0.040775854140520096, -0.06421902030706406, 0.08564361184835434,
      0.04441582038998604, 0.07814129441976547, 0.004312954377382994, -0.06696423888206482,
      0.08840478211641312, 0.011653688736259937, -0.07647570222616196, -0.011450793594121933,
      -0.02374466136097908, 0.06632942706346512, -0.08484529703855515, 0.026309851557016373,
      0.00962911732494831, -0.0754425972700119, -0.08453540503978729, -0.019503595307469368,
      -0.019391968846321106, -0.04464314505457878, 0.04771813005208969, 0.013140762224793434,
      -0.026172319427132607, 0.08028310537338257, 0.016386060044169426, -0.039088211953639984,
      -0.03455636277794838, 0.004511544480919838, 0.09659480303525925, 0.003048548474907875,
      0.006191144231706858, -0.05543181672692299, -0.03744019567966461, -0.04236675426363945,
      -0.06781527400016785, -0.07133209705352783, -0.03307822719216347, 0.022586695849895477,
      0.07448653131723404, -0.027121655642986298, 0.07403836399316788, -0.036476895213127136,
      0.047281619161367416, 0.05631101876497269, 0.06395580619573593, -0.028158260509371758,
      -0.014994914643466473, 0.010885807685554028, 0.03517262637615204, -0.04020072519779205,
      -0.08269207924604416, 0.07217694818973541, -0.04943346604704857, 0.0569642148911953,
      0.06325669586658478, -0.0313163697719574, 0.025330008938908577, -0.03881801664829254,
      -0.048191893845796585, -0.0666985884308815, -0.018529674038290977, 0.10361164808273315,
      0.083883136510849, -0.0050019691698253155, 0.042246393859386444, -0.03989170864224434,
      -0.0003385324089322239, 0.06735104322433472, 0.014639410190284252, 0.07405991107225418,
      -0.004048545379191637, -0.012998376041650772, 0.051176898181438446, 0.002337648067623377,
      0.06140147149562836, -0.07473933696746826, -0.05445600673556328, -0.04922731593251228,
      0.011041943915188313, -0.03595602512359619, -0.054983772337436676, -0.0827738344669342],
     [-0.09370371699333191, -0.02522849477827549, 0.15541893243789673, -0.007347750943154097,
      0.06341669708490372, 0.008149507455527782, 0.025743739679455757, 0.0363268107175827,
      0.09815365821123123, 0.052250612527132034, 0.11825680732727051, -0.09943357855081558,
      0.045644666999578476, 0.08944135159254074, -0.035619813948869705, -0.06728506088256836,
      0.06094604358077049, 0.0007678380934521556, -0.007366317790001631, 0.002110743662342429,
      0.08198913931846619, -0.06139195337891579, -0.053295720368623734, 0.11905938386917114,
      0.052341051399707794, 0.11147670447826385, -0.10207949578762054, 0.018504323437809944,
      -0.01275154110044241, 0.005549130029976368, -0.05395124852657318, -0.10669325292110443,
      -0.048675742000341415, 0.021034667268395424, 0.06264461576938629, -0.05359724536538124,
      -0.014395867474377155, 0.015873653814196587, -0.06712906807661057, -0.027392800897359848,
      -0.06483674049377441, 0.09237061440944672, -0.026334092020988464, 0.10622713714838028,
      0.07237990200519562, 0.011136239394545555, -0.0733746662735939, 0.0077096023596823215,
      0.08900359272956848, -0.046078089624643326, -0.009408901445567608, 0.13984747231006622,
      -0.08412473648786545, 0.024093501269817352, 0.08456888794898987, -0.000556455459445715,
      0.015554474666714668, 0.029613863676786423, -0.012625450268387794, 0.16391198337078094,
      -0.1451205611228943, 0.023984331637620926, -0.06070444732904434, -0.02240346372127533,
      0.06659093499183655, -0.046070411801338196, 0.0076173157431185246, -0.011893918737769127,
      -0.07325945794582367, -0.05534040555357933, -0.04065084457397461, 0.10002673417329788,
      0.005375286564230919, 0.07226601988077164, 0.06284026056528091, 0.06484542787075043,
      -0.028424790129065514, 0.04149381071329117, 0.06818234920501709, -0.007057730108499527,
      -0.08279333263635635, 0.04046744853258133, 0.0659174844622612, -0.00684414291754365,
      -0.007208252791315317, 0.09431592375040054, 0.04382618889212608, 0.034632302820682526,
      0.07324825972318649, -0.04251546785235405, -0.08137667179107666, 0.09503882378339767,
      -0.04923773556947708, -0.03354353457689285, -0.024684583768248558, 0.08717549592256546,
      0.08931717276573181, -0.05517490580677986, -0.05502932146191597, 0.11622924357652664,
      -0.023045944049954414, -0.0906689316034317, -0.07044176012277603, -0.013688935898244381,
      -0.06041007488965988, -0.024614587426185608, 0.013373156078159809, 0.02297506108880043],
     [-0.05675679072737694, 0.014590539038181305, 0.051305945962667465, -0.0005091516068205237,
      0.003696930594742298, -0.08660253137350082, 0.014860781840980053, 0.06974884122610092,
      0.05539771169424057, 0.016046598553657532, 0.08584088832139969, 0.004738185089081526,
      0.06290090084075928, -0.07218952476978302, -0.06284649670124054, -0.04244520887732506,
      -0.030708935111761093, 0.050624385476112366, -0.03969583660364151, -0.01177886500954628,
      -0.0742809996008873, 0.0499805212020874, -0.0527619943022728, -0.03378748521208763,
      -0.001613275846466422, -0.0736621543765068, -0.021913960576057434, 0.0672927126288414,
      -0.016226716339588165, -0.06638862937688828, -0.07640259712934494, -0.054932694882154465,
      -0.010934349149465561, 0.06472191959619522, 0.09684469550848007, 0.055619217455387115,
      0.009588376618921757, -0.056392766535282135, 0.055815450847148895, 0.0822887122631073,
      0.06937287002801895, 0.0830182358622551, -0.07659249007701874, -0.007499269675463438,
      0.07191459834575653, -0.07881685346364975, 0.07889806479215622, 0.02533140406012535,
      0.04806226119399071, 0.010146968998014927, -0.06869443506002426, 0.0839046835899353,
      -0.008170420303940773, -0.01387493684887886, 0.05869605392217636, 0.02633894421160221,
      -0.08478471636772156, 0.01605071686208248, 0.014715089462697506, 0.0014792067231610417,
      -0.017747392877936363, 0.03050445392727852, -0.03996745124459267, 0.02391239069402218,
      -0.014724460430443287, -0.04123418778181076, -0.03278616443276405, 0.06020326912403107,
      -0.00991964153945446, -0.027817735448479652, 0.00959617830812931, -0.08952087163925171,
      -0.039293549954891205, -0.046378687024116516, 0.09055452793836594, 0.03794916719198227,
      -0.05080030858516693, -0.026190809905529022, -0.0885217934846878, 0.09379054605960846,
      -0.048578258603811264, -0.032465774565935135, -0.03492187336087227, 0.03508429601788521,
      0.08182194083929062, 0.07374230027198792, -0.01383906602859497, -0.08234141021966934,
      -0.0631360337138176, -0.04427352175116539, -0.08259458094835281, -0.015591374598443508,
      -0.052277617156505585, -0.05190996825695038, -0.03683155030012131, -0.06206453964114189,
      0.07047189772129059, -0.04008074104785919, 0.02325303666293621, -0.028260745108127594,
      -0.06655897200107574, -0.04644263908267021, 0.06221005320549011, 0.056653041392564774,
      0.0994533821940422, -0.0537681058049202, -0.0055181789211928844, -0.10194239020347595],
     [-0.009046305902302265, -0.000758447393309325, 0.0248724278062582, -0.1302952766418457,
      -0.07086195051670074, 0.005534977652132511, -0.06743533164262772, 0.03904405236244202,
      0.05846215412020683, 0.029700668528676033, 0.12553749978542328, -0.02511545829474926,
      0.053575556725263596, 0.02606547251343727, 0.019895188510417938, -0.07152640074491501,
      0.025697510689496994, 0.0873904749751091, -0.044985778629779816, -0.0761522725224495,
      -0.024725226685404778, 0.042877957224845886, -0.03472999855875969, 0.020546376705169678,
      0.027908427640795708, 0.07335729897022247, -0.06091446429491043, 0.04039973393082619,
      -0.00032716128043830395, 0.04431924223899841, -0.03569552302360535, -0.10105175524950027,
      0.031531162559986115, -0.003808249020949006, -0.010103325359523296, -0.10836952179670334,
      0.0950627401471138, 0.08286178857088089, -0.054451167583465576, -0.012881402857601643,
      0.029854487627744675, 0.015590553171932697, 0.10790660977363586, 0.011482201516628265,
      -0.061859406530857086, 0.06952715665102005, -0.024253766983747482, -0.05081802234053612,
      -0.05429577827453613, -0.06214926764369011, -0.08195264637470245, 0.1505262404680252,
      0.048769742250442505, 0.09051698446273804, 0.1256457269191742, -0.04391477257013321,
      0.0737699642777443, 0.02387765608727932, 0.09833820909261703, 0.10988594591617584,
      0.013392739929258823, -0.0691847950220108, -0.08179894834756851, 0.07619894295930862,
      0.0918458104133606, -0.08705320209264755, -0.0348048135638237, -0.07657593488693237,
      -0.05390584096312523, -0.011052537709474564, -0.060457516461610794, 0.18551689386367798,
      0.04087653011083603, -0.06665781140327454, 0.11759601533412933, 0.07168292254209518,
      -0.040703728795051575, 0.051625777035951614, -0.0174699854105711, 0.13446080684661865,
      -0.0507645346224308, -0.09442009031772614, -0.06392273306846619, -0.03804122656583786,
      0.003657320514321327, -0.03796948865056038, 0.047999218106269836, 0.01945180632174015,
      0.030194438993930817, 0.04686383903026581, -0.08985019475221634, -0.06616576015949249,
      -0.013656072318553925, -0.05837036669254303, 0.01826426014304161, 0.18478839099407196,
      0.07858075946569443, -0.049397554248571396, 0.015956120565533638, 0.03063482791185379,
      0.01497503463178873, -0.018775299191474915, 0.06679731607437134, -0.07553257793188095,
      0.04588502645492554, -0.11397388577461243, -0.06292243301868439, -0.08877325057983398],
     [0.0023673824034631252, -0.030077146366238594, -0.06349797546863556, 0.0370931476354599,
      -0.0937420129776001, 0.01238088496029377, -0.04846154898405075, 0.04490366950631142,
      0.05639570578932762, 0.03969587758183479, -0.01437542773783207, 0.03019091673195362,
      -0.03837301954627037, 0.055723901838064194, 0.08638649433851242, 0.068520687520504,
      -0.026352304965257645, -0.05405106022953987, -0.07336029410362244, -0.08176463842391968,
      -0.09728366881608963, 0.05473710969090462, -0.08756382018327713, 0.06445764005184174,
      0.014971483498811722, 0.006353449542075396, 0.05292403697967529, -0.019353460520505905,
      0.06143404543399811, 0.030361710116267204, 0.0838775560259819, -0.07747708261013031,
      0.07504074275493622, 0.10163424164056778, -0.07234375178813934, -0.04090899974107742,
      0.08569356799125671, 0.11458945274353027, 0.009157948195934296, -0.0003650939906947315,
      0.0448312945663929, -0.03960966691374779, -0.0807250514626503, 0.09281527251005173,
      -0.06330472975969315, -0.07134005427360535, 0.09109792858362198, -0.026920951902866364,
      0.01905134879052639, -0.016456838697195053, -0.03110331855714321, -0.00971583928912878,
      -0.08960368484258652, 0.029461754485964775, -0.05225251987576485, -0.09297258406877518,
      -0.0032667957711964846, 0.05118294060230255, -0.0049350508488714695, 0.019449150189757347,
      0.041551265865564346, -0.09909417480230331, 0.02460847981274128, -0.0847054049372673,
      -0.08892498165369034, -0.01580994576215744, -0.03956037014722824, 0.05439538136124611,
      0.0651794970035553, -0.06997502595186234, 0.03518964350223541, -0.06755536794662476,
      0.0772794708609581, -0.012719931080937386, -0.0022261859849095345, 0.0024017083924263716,
      0.08796175569295883, -0.047116316854953766, -0.06224736571311951, -0.012498137541115284,
      -0.07896386086940765, 0.04632965475320816, -0.042625028640031815, -0.03750989958643913,
      0.026044512167572975, 0.08939377963542938, -0.04039543867111206, -0.08482102304697037,
      -0.006303055677562952, 0.025476327165961266, -0.04093463346362114, -0.07707937061786652,
      0.01905534416437149, -0.04004018008708954, 0.043561894446611404, 0.03763415291905403,
      0.08762846887111664, 0.0017208035569638014, 0.019325552508234978, -0.013599242083728313,
      -0.0241794865578413, -0.0179080031812191, 0.0011476814979687333, -0.045822158455848694,
      -0.08340629190206528, -0.019210590049624443, 0.05513637140393257, -0.06275355070829391],
     [-0.00628799619153142, -0.07960984855890274, 0.18174336850643158, 0.03174743801355362,
      -0.08815239369869232, 0.08955935388803482, -0.03232107311487198, 0.07705802470445633,
      -0.07075674086809158, 0.03377528488636017, -0.014039845205843449, -0.06126284599304199,
      -0.06645075976848602, 0.05747053772211075, 0.05860581248998642, 0.07395724952220917,
      0.12107677757740021, 0.0792829692363739, -0.10831575095653534, 0.08260315656661987,
      -0.06935016810894012, -0.09958158433437347, 0.04791934788227081, -0.02007497102022171,
      -0.08950956165790558, 0.08770839869976044, 0.048352986574172974, -0.03702057898044586,
      0.02468007616698742, 0.01132240705192089, 0.0528472401201725, -0.04775967821478844,
      -0.07551911473274231, 0.001722850021906197, 0.006059218198060989, -0.043649110943078995,
      -0.0025720687117427588, 0.05655001848936081, 0.060368623584508896, 0.10561995208263397,
      -0.033139899373054504, 0.0738043263554573, 0.08024038374423981, -0.06445962935686111,
      -0.08404844254255295, 0.0264009740203619, 0.0524001270532608, 0.09117487072944641,
      0.008207898586988449, -0.08688246458768845, -0.00041612444329075515, -0.0015310539165511727,
      -0.05332938954234123, 0.06687513738870621, 0.09873978793621063, 0.01071135327219963,
      -0.004151392262428999, -0.06390324980020523, -0.06689681112766266, 0.09763087332248688,
      -0.07300139218568802, -0.044458188116550446, -0.088046595454216, -0.06885594874620438,
      0.0871175155043602, -0.07896777242422104, 0.012984524480998516, -0.005921907722949982,
      -0.04395843297243118, 0.05785951018333435, -0.0625743642449379, 0.11787566542625427,
      -0.0437266044318676, -0.03610902652144432, -0.014851916581392288, -0.007624911144375801,
      0.06609326601028442, 0.04491513967514038, 0.048257067799568176, 0.11951890587806702,
      -0.03723263740539551, 0.024636855348944664, 0.03229700028896332, -0.059089187532663345,
      -0.0741974264383316, -0.059325672686100006, -0.027817703783512115, 0.02926449105143547,
      -0.045507077127695084, 0.036036014556884766, -0.05105087533593178, 0.056103065609931946,
      -0.06406260281801224, 0.09953311085700989, -0.015655484050512314, 0.1752285212278366,
      0.0805649682879448, -0.06461796164512634, -0.025662977248430252, 0.04905756935477257,
      0.09728816896677017, -0.05552726611495018, -0.02830955572426319, -0.042744800448417664,
      0.08720514923334122, 0.007052111905068159, -0.0027320883236825466, -0.018861154094338417],
     [-0.09375764429569244, 0.039947304874658585, 0.17659218609333038, -0.05033664405345917,
      0.07152364403009415, 0.05304082855582237, -0.006426085717976093, -0.021990591660141945,
      0.04258936271071434, 0.008877350948750973, 0.017716599628329277, 0.04007580131292343,
      0.08195251226425171, -0.04017828404903412, 0.004933943506330252, 0.01835208758711815,
      -0.013332972303032875, 0.008057016879320145, -0.009091037325561047, 0.09671948105096817,
      -0.022316131740808487, -0.11327512562274933, 0.012698829174041748, 0.08221273869276047,
      0.024874141439795494, 0.11361608654260635, 0.030217483639717102, -0.047032639384269714,
      -0.030680952593684196, 0.039394378662109375, -0.0034934720024466515, 0.09585894644260406,
      -0.019517820328474045, 0.07834339141845703, -0.015789702534675598, -0.09652041643857956,
      0.04968696087598801, 0.07765617966651917, 0.02099267579615116, -0.014596120454370975,
      0.02628658525645733, 0.06611058861017227, 0.06323286145925522, 0.0857701301574707,
      0.06560622155666351, -0.009714554995298386, -0.0649838000535965, -0.021793682128190994,
      0.07639309763908386, 0.004520330112427473, -0.07343640923500061, -0.04138735309243202,
      0.06921079009771347, 0.08868089318275452, 0.02823234535753727, 0.07839495688676834,
      -0.052692025899887085, -0.04718860238790512, 0.013445289805531502, 0.1398741453886032,
      -0.06787621229887009, 0.026454225182533264, 0.019754741340875626, 0.05164819583296776,
      0.028330035507678986, 0.010947490110993385, 0.08764951676130295, -0.07306042313575745,
      -0.04407639056444168, 0.10856243968009949, 0.04448103904724121, 0.04653509706258774,
      0.03790528327226639, 0.07536330819129944, 0.020014340057969093, 0.09026434272527695,
      -0.008496810682117939, 0.08906888961791992, -0.029254881665110588, 0.11447862535715103,
      -0.07670526951551437, -0.058803871273994446, 0.03528021648526192, 0.0025588537100702524,
      0.010655608959496021, 0.01935366354882717, -0.0353313647210598, 0.07244756072759628,
      0.05377893149852753, -0.06810887157917023, 0.03742091730237007, -0.055730871856212616,
      0.07638078182935715, 0.08608181029558182, 0.06398838013410568, 0.02774515002965927,
      -0.06637602299451828, -0.07195151597261429, -0.022394347935914993, 0.05793033912777901,
      0.01107233390212059, 0.05320686101913452, 0.10399549454450607, -0.08176998049020767,
      0.0211202222853899, 0.053818877786397934, 0.01726013794541359, -0.08498216420412064],
     [-0.09281298518180847, 0.003311987966299057, -0.03901646286249161, -0.00444573862478137,
      0.09257251769304276, 0.02331729792058468, -0.0001290825312025845, 0.08886704593896866,
      0.04304901510477066, -0.01287956815212965, -0.01809057593345642, -0.009784754365682602,
      0.08955913782119751, -0.041825439780950546, 0.03056427650153637, 0.07166489958763123,
      0.04522775113582611, -0.09103067964315414, 0.0625799149274826, 0.06926260143518448,
      -0.04534919559955597, 0.004368857946246862, 0.08958518505096436, 0.022666536271572113,
      0.013089287094771862, 0.044449616223573685, -0.05821092799305916, -0.024981025606393814,
      0.07605016231536865, 0.07674926519393921, 0.01646987535059452, -0.018646610900759697,
      -0.05989888682961464, 0.0073752314783632755, 0.0720536857843399, 0.061292003840208054,
      0.07328852266073227, 0.008684840053319931, 0.049919188022613525, 0.04791026934981346,
      0.09483043849468231, 0.08002378791570663, 0.08335218578577042, 0.028288237750530243,
      0.06461907178163528, 0.07596440613269806, 0.05769234150648117, 0.016358524560928345,
      0.015418360941112041, 0.08325978368520737, -0.03507770970463753, 0.09663014113903046,
      -0.10114151239395142, -0.05538535118103027, -0.03298438340425491, 0.05172738805413246,
      -0.03286112844944, 0.07409801334142685, 0.018646476790308952, -0.034604329615831375,
      0.04185543954372406, 0.04747919738292694, 0.01612011156976223, -0.02820873074233532,
      0.0622650571167469, -0.012326919473707676, -0.09437136352062225, -0.06125896796584129,
      -0.05434081330895424, 0.1006544902920723, 0.030344339087605476, -0.09576895087957382,
      -0.006446735467761755, 0.005553875584155321, 0.012537354603409767, -0.06148681417107582,
      0.03971748426556587, 0.054743941873311996, -0.022267505526542664, 0.07990925014019012,
      0.024470586329698563, 0.04176776111125946, 0.009112401865422726, 0.028551645576953888,
      0.08617284893989563, 0.035313330590724945, 0.06201164424419403, 0.09164627641439438,
      -0.05169942229986191, -0.02594839781522751, 0.10241326689720154, -0.009598950855433941,
      0.006289360579103231, -0.002672684844583273, -0.05436912178993225, -0.11114084720611572,
      -0.060738518834114075, -0.023982545360922813, 0.06766575574874878, -0.045911043882369995,
      0.07479583472013474, 0.007072518114000559, 0.037221815437078476, -0.04356195032596588,
      -0.005194873083382845, 0.02134842611849308, 0.023314455524086952, -0.059386905282735825],
     [-0.03566909581422806, 0.06284113973379135, 0.0008342370856553316, -0.07090768963098526,
      0.01748322881758213, 0.05691063031554222, -0.02865680493414402, 0.06560242176055908,
      0.08091355860233307, 0.019379805773496628, 0.05460796132683754, -0.0241758581250906,
      -0.02389407530426979, -0.004247008357197046, 0.011377571150660515, -0.008488508872687817,
      0.013341455720365047, -0.06890378892421722, -0.00909196026623249, 0.05902734398841858,
      -0.07500817626714706, -0.08053663372993469, 0.07119574397802353, 0.04753890633583069,
      -0.007537491619586945, -0.01678437367081642, 0.051822129637002945, -0.017628928646445274,
      -0.07491198182106018, 0.03837110102176666, -0.022763840854167938, -0.022848527878522873,
      0.08242498338222504, -0.06919743120670319, -0.06579412519931793, -0.0719977617263794,
      -0.027620097622275352, 0.01564902998507023, 0.04256734251976013, 0.0841708779335022,
      0.1032354012131691, 0.00986944604665041, 0.05337868258357048, 0.012749572284519672,
      -0.006242635194212198, 0.0027614557184278965, 0.007854791358113289, 0.06343242526054382,
      0.01232830435037613, -0.09207800775766373, -0.0005103484145365655, -0.053007859736680984,
      0.010835958644747734, -0.0029552625492215157, -0.02993362583220005, -0.09049905091524124,
      -0.0464320033788681, -0.041379548609256744, 0.06102970987558365, 0.04321042820811272,
      0.09743712097406387, -0.0676981508731842, 0.04887530207633972, -0.08550315350294113,
      0.052501168102025986, -0.07445486634969711, -0.025247206911444664, 0.02951960451900959,
      -0.024756429716944695, -0.07544411718845367, 0.02615886740386486, -0.06615694612264633,
      0.013052897527813911, -0.08172222226858139, -0.06979314982891083, 0.03399098291993141,
      0.05897064507007599, 0.1144619956612587, 0.09277975559234619, -0.05557316169142723,
      0.03388208895921707, -0.01611965335905552, -0.0767434611916542, 0.07611389458179474,
      -0.04072430729866028, 0.03000737726688385, -0.07337819784879684, -0.045978158712387085,
      -0.05142884701490402, 0.0242387056350708, 0.036154210567474365, -0.11696391552686691,
      0.058606818318367004, 0.0011225006310269237, 0.02834799885749817, 0.01592816598713398,
      -0.0033775451593101025, 0.005827136337757111, -0.07499448955059052, 0.06620550900697708,
      0.004914003424346447, -0.06015656888484955, -0.08873838931322098, 0.0640837624669075,
      0.08890575915575027, -0.03492916002869606, -0.019224219024181366, -0.06634502112865448],
     [-0.0787256583571434, 0.02483791671693325, -0.047174107283353806, 0.09069222211837769,
      0.029242053627967834, 0.002078287536278367, 0.06462450325489044, 0.02561121992766857,
      -0.05678102374076843, -0.006965796463191509, 0.04227299615740776, 0.0752846971154213,
      0.03273850306868553, -0.025850934907794, -0.013063107617199421, 0.0033793114125728607,
      -0.006604327820241451, 0.06865409761667252, 0.03795357421040535, 0.04262758791446686,
      -0.10316167026758194, 0.009602286852896214, 0.032314714044332504, 0.025856809690594673,
      0.011280267499387264, 0.09582433849573135, -0.02938585728406906, 0.057414013892412186,
      -0.07653487473726273, -0.019777711480855942, 0.05597329139709473, -0.07322555780410767,
      -0.08528485149145126, -0.04135396331548691, 0.08346989750862122, 0.007599970325827599,
      -0.02987975813448429, 0.08349449187517166, -0.040339965373277664, 0.02655782364308834,
      0.08650337904691696, 0.001513147377409041, 0.08392457664012909, 0.09221654385328293,
      0.022887347266077995, -0.058805208653211594, -0.023733988404273987, 0.055048782378435135,
      -0.030926864594221115, 0.06813417375087738, -0.03707858920097351, 0.045054711401462555,
      -0.021893413737416267, 0.02514156885445118, -0.03838276490569115, -0.010021530091762543,
      -0.043166227638721466, -0.07474688440561295, 0.0698462575674057, -0.04533994570374489,
      -0.0003216709301341325, -0.08823203295469284, 6.723601109115407e-05, -0.09086420387029648,
      0.068080373108387, -0.03203941136598587, -0.05625106394290924, 0.015285281464457512,
      -0.08124829083681107, 0.03947923332452774, -0.04010217264294624, -0.04340340569615364,
      0.07209375500679016, -0.004591639619320631, -0.0020321307238191366, -0.004605191759765148,
      -0.030172107741236687, 0.05523543059825897, 0.03133463114500046, -0.007606642786413431,
      -0.06130359321832657, 0.06034895405173302, -0.06525532901287079, -0.06007709726691246,
      0.017567234113812447, 0.06213526427745819, 0.03461536020040512, 0.0047050099819898605,
      -0.0896596685051918, -0.04068866744637489, -0.04369925335049629, -0.0638655498623848,
      -0.03306981921195984, 0.031717900186777115, 0.017769142985343933, 0.003299964591860771,
      -0.060782887041568756, 0.010121829807758331, -0.030733877792954445, -0.04701957851648331,
      0.047594789415597916, 0.06219933182001114, -0.102786123752594, 0.02356143109500408,
      -0.06696145236492157, 0.05426032468676567, -0.08624181896448135, 0.06197701767086983],
     [-0.08440035581588745, -0.04931184649467468, -0.08259983360767365, -0.05149880051612854,
      -0.040449462831020355, 0.06124269217252731, -0.0943092405796051, -0.022831114009022713,
      0.006508044898509979, -0.019457072019577026, 0.029708512127399445, -0.08143187314271927,
      0.01936502382159233, 0.1125776618719101, -0.04544921964406967, -0.06575724482536316,
      0.0957452803850174, 0.01961737871170044, 0.05596178025007248, -0.03159598261117935,
      0.07826748490333557, -0.04994739592075348, -0.08355804532766342, -0.02412550523877144,
      -0.014528931118547916, 0.06340990215539932, -0.032105933874845505, 0.10607261955738068,
      0.08522696793079376, -0.029690101742744446, 0.07387033849954605, 0.07183043658733368,
      -0.010005163960158825, 0.08969258517026901, -0.027304427698254585, 0.06582997739315033,
      0.10551999509334564, 0.02680417150259018, 0.07618580758571625, -0.06200150400400162,
      0.03679908439517021, 0.07448875159025192, -0.030373288318514824, 0.07751017808914185,
      -0.018922703340649605, 0.09768859297037125, 0.020650483667850494, -0.09739241749048233,
      -0.05619753524661064, -0.0107559310272336, 0.01759643480181694, -0.04211392253637314,
      -0.09758834540843964, 0.07248318195343018, 0.07074893265962601, 0.0561053566634655,
      -0.045143790543079376, -0.05622527748346329, 0.028832774609327316, -0.08034421503543854,
      -0.015409649349749088, 0.07334750145673752, -0.07828380167484283, 0.012005306780338287,
      -0.08503728359937668, 0.013643017038702965, 0.009428257122635841, 0.06733371317386627,
      -0.014624453149735928, 0.05713360384106636, -0.021432463079690933, -0.042594119906425476,
      -0.010903366841375828, -0.011170821264386177, -0.07413884997367859, 0.03419453278183937,
      -0.0005279827164486051, 0.013514719903469086, 0.016575608402490616, 0.09849902987480164,
      0.022180117666721344, -0.07327878475189209, 0.012055594474077225, -0.02124779112637043,
      -0.0641244649887085, 0.07884929329156876, 0.04604564234614372, 0.07000192999839783,
      0.06667729467153549, -0.07268690317869186, 0.029702648520469666, 0.026527317240834236,
      0.02549312822520733, 0.06091834604740143, -0.0014015031047165394, -0.08566399663686752,
      0.053744614124298096, 0.08644841611385345, 0.0663202628493309, -0.029460636898875237,
      0.06685405969619751, -0.015990223735570908, -0.042949873954057693, -0.032913725823163986,
      -0.00011379120405763388, -0.05936388298869133, 0.07872777432203293, 0.02657458931207657],
     [0.010964710265398026, -0.054591916501522064, -0.05492806434631348, -0.019828861579298973,
      -0.07122086733579636, 0.06109275296330452, 0.08740115910768509, -0.08718377351760864,
      -0.054475415498018265, 0.10341791063547134, 0.0676332488656044, 0.04111850634217262,
      -0.00847659818828106, 0.06310475617647171, -0.008591026067733765, 0.046838220208883286,
      0.010239268653094769, -0.0710332840681076, 0.0011626635678112507, 0.01195891946554184,
      -0.08695420622825623, -0.007704823277890682, -0.026688190177083015, 0.023995136842131615,
      -0.010286504402756691, 0.09128912538290024, -0.01844758726656437, -0.03092002309858799,
      0.012154761701822281, -0.0881722941994667, 0.052141107618808746, -0.052225664258003235,
      -0.08257150650024414, 0.06557032465934753, 0.04443729296326637, -0.029272550716996193,
      0.0458785817027092, 0.0678459107875824, 0.08165173977613449, 0.03822442516684532,
      -0.01661812514066696, -0.050271175801754, 0.07360547035932541, 0.09089206904172897,
      0.05471891909837723, 0.056525781750679016, 0.023772096261382103, -0.07264525443315506,
      -0.03145742043852806, 0.04247221723198891, -9.252015297533944e-05, 0.05103428289294243,
      -0.07712352275848389, -0.05932803824543953, 0.022850558161735535, -0.04174627363681793,
      -0.04744241014122963, -0.0030665905214846134, -0.07520955801010132, -0.04251708462834358,
      -0.05579911917448044, 0.053483832627534866, -0.04074161499738693, 0.01857283152639866,
      0.06417826563119888, -0.0162836704403162, 0.04680650681257248, -0.07230602949857712,
      0.007748483680188656, -0.04033447429537773, 0.06413000822067261, -0.09812881797552109,
      0.07961885631084442, 0.06600790470838547, 0.007128844037652016, -0.05113972723484039,
      0.07165110111236572, -0.030210159718990326, -0.008607804775238037, 0.1104649007320404,
      -0.0817430168390274, -0.025453872978687286, 0.06503723561763763, -0.07861189544200897,
      0.00970584899187088, -0.03577074036002159, -0.07583312690258026, -0.05883278697729111,
      0.04668133705854416, -0.10350609570741653, -0.07504869252443314, 0.015627335757017136,
      0.06569043546915054, -0.024774428457021713, -0.04215426743030548, -0.055266257375478745,
      0.09205733984708786, 0.03788292780518532, -0.02299650013446808, -0.06614354252815247,
      0.02040286175906658, -0.05998915806412697, -0.05462808534502983, 0.0798419713973999,
      -0.061145439743995667, -0.07465046644210815, 0.0510946623980999, -0.033204592764377594],
     [-0.08659762144088745, -0.056281089782714844, -0.0736764445900917, -0.012003264389932156,
      -0.077497199177742, -0.0001547778520034626, 0.09165459126234055, -0.0963183045387268,
      0.06870121508836746, -0.033122364431619644, 0.05393354967236519, -0.06618928909301758,
      0.03493881970643997, 0.07363435626029968, 0.011440056376159191, -0.05059171840548515,
      0.09577420353889465, 0.07613123953342438, -0.055747292935848236, -0.09330214560031891,
      0.041078582406044006, 0.06431172043085098, 0.014520438387989998, 0.05821991711854935,
      -0.005533438641577959, -0.04562364146113396, 0.06611466407775879, -0.05938464775681496,
      -0.09773258864879608, 0.05864923447370529, -0.04445548728108406, -0.08244860172271729,
      0.021537259221076965, -0.026301762089133263, 0.018142472952604294, 0.05100982263684273,
      -0.04882313311100006, 0.09605519473552704, 0.023821493610739708, 0.008943505585193634,
      -0.06752977520227432, -0.04376566410064697, -0.05784226953983307, 0.08547782152891159,
      0.005407556891441345, -0.059249572455883026, 0.07180396467447281, 0.005815517622977495,
      -0.05933426320552826, -0.033012062311172485, 0.08390901237726212, 0.07370579242706299,
      -0.008112344890832901, -0.013165969401597977, -0.008302892558276653, 0.07949408888816833,
      0.012475398369133472, 0.004167800769209862, 0.015661420300602913, 0.0020683142356574535,
      0.018550235778093338, 0.0230338666588068, -0.08857162296772003, -0.050817638635635376,
      -0.02407381497323513, 0.0009875481482595205, 0.03916797414422035, -0.0100163035094738,
      0.00348579790443182, -0.04942009225487709, 0.057203903794288635, -0.0060744863003492355,
      0.10605636984109879, 0.00332081806845963, -0.07549002766609192, -0.02443518303334713,
      -0.05243071913719177, 0.08168186247348785, 0.08564874529838562, -0.016779182478785515,
      0.003928488586097956, 0.027169540524482727, -0.012510067783296108, 0.07372083514928818,
      -0.07123783230781555, -0.013199862092733383, 0.10294169932603836, 0.018695393577218056,
      0.09193376451730728, 0.046696506440639496, 0.03137800842523575, -0.10262566804885864,
      -0.06898096203804016, -0.029333172366023064, -0.09406185150146484, -0.04867887124419212,
      0.05899551510810852, 0.059833627194166183, -0.00322950491681695, -0.007626087870448828,
      -0.014716403558850288, -0.01919182389974594, 0.04785243049263954, -0.0017284053610637784,
      0.035700131207704544, -0.04005325958132744, -0.07545577734708786, -0.037508271634578705],
     [-0.07473231852054596, -0.06837216019630432, 0.09777948260307312, 0.06707683950662613,
      -0.014954539947211742, -0.0788828507065773, 0.07518617808818817, 0.015004154294729233,
      -0.01682114042341709, 0.0887703225016594, 0.1336594671010971, -0.05583101883530617,
      -0.04031062871217728, -0.05437891185283661, 0.06249062344431877, -0.0025611098390072584,
      -0.06024183705449104, -0.027335714548826218, -0.030553439632058144, 0.02979447692632675,
      0.05880552530288696, 0.015483919531106949, 0.08285301923751831, 0.047338634729385376,
      0.05777199566364288, 0.0029010195285081863, 0.02912256494164467, 0.07283642888069153,
      0.12014996260404587, 0.08476681262254715, -0.018367258831858635, 0.08583742380142212,
      -0.049591727554798126, -0.05752869322896004, 0.09541005641222, 0.08448728173971176,
      -0.0013667496386915445, 0.10988488048315048, 0.04531756415963173, -0.026976443827152252,
      -0.019322291016578674, -0.028888914734125137, -0.011612001806497574, -0.021584929898381233,
      -0.08517462015151978, -0.028193986043334007, 0.04550839960575104, 0.04714403301477432,
      -0.030203143134713173, -0.031903646886348724, -0.009510096162557602, 0.006377479061484337,
      0.012710824608802795, 0.15057207643985748, 0.027761735022068024, 0.034342389553785324,
      -0.10475669056177139, 0.004114546347409487, -0.06561534106731415, -0.03163888305425644,
      -0.11890255659818649, -0.09057211875915527, 0.034244269132614136, -0.08280400186777115,
      -0.014875542372465134, -0.05271805450320244, 0.03520085662603378, -0.09406235814094543,
      -0.08822700381278992, 0.07783947885036469, 0.021617429330945015, 0.16938158869743347,
      0.05925833433866501, 0.026393797248601913, 0.11444146186113358, -0.02274528332054615,
      0.039774902164936066, 0.047098200768232346, 0.013748371973633766, 0.05860227346420288,
      0.05598221346735954, 0.040924832224845886, 0.041423141956329346, -0.07333211600780487,
      -0.08082042634487152, 0.03158646449446678, 0.0804733857512474, 0.0780448243021965,
      -0.036257244646549225, -0.07651853561401367, 0.03385651111602783, -0.03350714594125748,
      0.04025409370660782, -0.0040705096907913685, -0.03228071704506874, 0.12175784260034561,
      0.09769988059997559, 0.008790395222604275, -0.03878564015030861, 0.074696846306324,
      -0.029091371223330498, 0.04560978338122368, -0.027337858453392982, 0.021754048764705658,
      0.08605607599020004, 0.08489711582660675, 0.06660237908363342, -0.057272735983133316],
     [-0.012851503677666187, -0.06169673427939415, -0.022117342799901962, -0.0943746492266655,
      -0.0312587209045887, -0.09951013326644897, -0.008740273304283619, -0.0009066665079444647,
      -0.06014986336231232, 0.09270468354225159, -0.05136512219905853, 0.07454795390367508,
      0.0024620075710117817, -0.010457750409841537, 0.010221673175692558, -0.06097506359219551,
      0.014697377569973469, 0.0807933658361435, 0.04645080864429474, -0.04100874438881874,
      -0.012474733404815197, 0.08240712434053421, -0.02733917348086834, -0.05654289573431015,
      -0.02535836584866047, 0.03933700919151306, 0.07727210223674774, 0.08787788450717926,
      0.004909823182970285, 0.009109394624829292, 0.017651809379458427, 0.03600520268082619,
      0.011803827248513699, 0.05039571598172188, -0.053382374346256256, 0.004252777434885502,
      -0.01323734037578106, 0.0665997713804245, -0.058662280440330505, 0.01088859885931015,
      -0.03078460320830345, 0.005681599024683237, 0.052216723561286926, 0.103446826338768,
      0.047288425266742706, 0.06006425619125366, -0.015028211288154125, -0.05395691096782684,
      -0.10759098082780838, -0.029590478166937828, -0.0406142957508564, -0.05211401358246803,
      -0.05482340604066849, 0.11599428206682205, -0.02776835858821869, -0.057082649320364,
      -0.0774204432964325, -0.06715130060911179, -0.019752101972699165, -0.022065984085202217,
      -0.004447708837687969, 0.04885619506239891, 0.025080909952521324, -0.03899921476840973,
      0.09517787396907806, 0.022766100242733955, 0.016323745250701904, -0.09113063663244247,
      0.03239510580897331, -0.007245445623993874, -0.031207170337438583, -0.01799006201326847,
      -0.04047026485204697, -0.05278504267334938, 0.07877638936042786, 0.04463476315140724,
      -0.06337470561265945, 0.0910499095916748, 0.09735511988401413, -0.029628213495016098,
      0.05231466889381409, 0.06757808476686478, 0.048024892807006836, -0.05652542784810066,
      0.003166917245835066, 0.07262741029262543, -0.06264381855726242, -0.02724211849272251,
      0.07296513766050339, -0.022646110504865646, -0.02417946793138981, 0.0603671558201313,
      -0.016837358474731445, 0.025180907920002937, 0.052554868161678314, -0.06591280549764633,
      -0.08405966311693192, -0.018376823514699936, 0.07718367129564285, 0.0346536748111248,
      0.07810274511575699, 0.056279417127370834, 0.015959152951836586, -0.016256269067525864,
      -0.015542646870017052, -0.03500969335436821, -0.11086460202932358, -0.04644840210676193],
     [0.003654665080830455, 0.06524057686328888, 0.053975798189640045, 0.0452553927898407,
      -0.039889466017484665, 0.13691912591457367, -0.008939092047512531, -0.029809070751070976,
      0.08787567913532257, -0.05557633563876152, 0.07926502823829651, 0.08190874010324478,
      0.04789639264345169, 0.030425619333982468, -0.0169666800647974, 0.006574702449142933,
      0.11598287522792816, 0.12275049090385437, 0.03456326946616173, 0.06869006901979446,
      0.05696582794189453, -0.07826034724712372, -0.06577353179454803, 0.0728563591837883,
      0.055355485528707504, 0.09371349215507507, -0.011397595517337322, 0.07003866136074066,
      0.062019143253564835, -0.0059392196126282215, 0.01928931288421154, 0.016619645059108734,
      -0.03835105150938034, 0.08732029050588608, 0.06977016478776932, -0.06780380010604858,
      0.10771410167217255, 0.1316678524017334, -0.018655765801668167, 0.0886542797088623,
      0.0820806547999382, 0.07948735356330872, 0.0005429672310128808, 0.04517732188105583,
      0.02468540519475937, 0.045816127210855484, -0.07580294460058212, 0.09245406836271286,
      0.07341840118169785, 0.08714866638183594, 0.016404883936047554, 0.13030748069286346,
      -0.06414247304201126, 0.07479075342416763, 0.038082197308540344, -0.029533520340919495,
      0.04325418174266815, 0.06303541362285614, -0.004699598997831345, 0.12148324400186539,
      -0.07371234148740768, -0.09958890080451965, 0.04338288679718971, 0.02646338939666748,
      -0.05165288969874382, 0.06712283939123154, -0.07714271545410156, -0.007239758037030697,
      -0.06306705623865128, 0.11639757454395294, -0.05171007290482521, 0.12666070461273193,
      0.0074445875361561775, -0.011125766672194004, 0.05905668064951897, 0.058351244777441025,
      -0.07076598703861237, -0.005128990393131971, -0.07119831442832947, 0.12357637286186218,
      -0.06083711236715317, 0.07173670083284378, 0.014466453343629837, -0.05617927387356758,
      0.09414057433605194, -0.04949558898806572, -0.012858494184911251, 0.04286366328597069,
      0.0460241362452507, 0.08890551328659058, -0.04035380855202675, 0.09473114460706711,
      -0.006330517120659351, -0.03577657416462898, -0.023662302643060684, 0.08633947372436523,
      0.05752837285399437, -0.1045023649930954, 0.04895348101854324, 0.03423517569899559,
      -0.02625025250017643, -0.11785206198692322, 0.07482996582984924, 0.013542688451707363,
      -0.04384395480155945, 0.055171456187963486, 0.03186217322945595, 0.013403667137026787],
     [-0.07179797440767288, -0.08558347821235657, -0.03923943266272545, 0.02398928627371788,
      0.09258133172988892, -0.04820499196648598, -0.01059336494654417, 0.04125828295946121,
      -0.05753926560282707, -0.06944745779037476, -0.04278137534856796, -0.0656689703464508,
      0.09526856988668442, 0.06921045482158661, -0.07814331352710724, 0.07486489415168762,
      -0.028520872816443443, -0.006824588403105736, -0.0037838134448975325, 0.03075793944299221,
      -0.07095662504434586, 0.013477635569870472, 0.02608175203204155, 0.024236375465989113,
      0.08081476390361786, 0.06636600941419601, -0.041622914373874664, 0.026304040104150772,
      -0.02702517621219158, 0.06524569541215897, -0.05794062092900276, 0.023743078112602234,
      0.044963400810956955, 0.06751935184001923, -0.02438179962337017, 0.011514091864228249,
      0.07982603460550308, -0.013516423292458057, -0.022129027172923088, 0.07217095792293549,
      0.04515897482633591, -0.0672895535826683, 0.07380703836679459, 0.04353656619787216,
      0.01953139901161194, -0.01212410070002079, -0.023164091631770134, -0.05158758535981178,
      -0.03951483964920044, -0.0532585047185421, 0.078608438372612, 0.06584828346967697,
      0.022012637928128242, -0.001250159228220582, 0.02161552384495735, -0.0006254715262912214,
      0.036757923662662506, 0.1113491952419281, 0.05048847198486328, -0.07265053689479828,
      -0.031217632815241814, -0.09580794721841812, -0.029779257252812386, -0.016554126515984535,
      -0.024439053609967232, -0.0013435982400551438, -0.00734546035528183, 0.0726047232747078,
      -0.07999709248542786, 0.08725793659687042, 0.09412119537591934, 0.01782889850437641,
      -0.03244372084736824, -0.00657883333042264, -0.06247175857424736, -0.10456754267215729,
      0.04865043982863426, -0.02439171075820923, -0.05167188122868538, 0.06292875856161118,
      -0.012092079035937786, 0.10135521739721298, -0.042436257004737854, -0.08151120692491531,
      -0.002714416477829218, 0.08317610621452332, -0.07293178886175156, 0.06857220828533173,
      -0.06993108242750168, 0.06338745355606079, -0.025885174050927162, -0.05491471290588379,
      0.059547897428274155, -0.005824350286275148, 0.002366461558267474, -0.019577208906412125,
      -0.04755067452788353, 0.018554551526904106, 0.07911889255046844, 0.03153273090720177,
      0.001845611841417849, -0.09111330658197403, 0.0007615432841703296, -0.042150579392910004,
      0.10267636924982071, 0.07544207572937012, -0.03994263336062431, -0.07937397807836533],
     [-0.06002906709909439, 0.028828367590904236, -0.05613012984395027, -0.04273718595504761,
      -0.025105159729719162, -0.042589083313941956, 0.003577925032004714, 0.06071378290653229,
      0.10405295342206955, 0.042461007833480835, -0.0419057197868824, -0.02612597495317459,
      -0.04549055173993111, -0.004776421934366226, 0.04150733724236488, 0.040011774748563766,
      -0.018403055146336555, 0.025428172200918198, -0.04067566990852356, -0.09533742815256119,
      0.0059898970648646355, -0.04042293131351471, -0.009487748146057129, -0.060413140803575516,
      -0.0006059066508896649, -0.026159003376960754, 0.03153209015727043, -0.08272791653871536,
      0.08789286017417908, -0.061282217502593994, -0.022885596379637718, -0.0583747997879982,
      -0.021678214892745018, 0.09461795538663864, 0.08533619344234467, -0.06251326948404312,
      0.002291663782671094, 0.08854824304580688, -0.015433702617883682, 0.11155529320240021,
      0.020345056429505348, -0.08048976212739944, -0.07999543845653534, -0.07611677050590515,
      0.037233609706163406, -0.0239164549857378, -0.04168860241770744, 0.06347768008708954,
      0.03730703890323639, 0.09522973746061325, 0.07750684767961502, -0.0571068599820137,
      -0.023046938702464104, 0.059164971113204956, 0.04077786207199097, -0.01056477427482605,
      0.05276324227452278, 0.10328646749258041, -0.07251439243555069, 0.03327029198408127,
      -0.084780752658844, -0.09648050367832184, 0.05176950991153717, 0.04555269330739975,
      0.04032253473997116, -0.08513687551021576, 0.06304657459259033, -0.03112812153995037,
      0.07873471081256866, -0.03666548430919647, 0.0625622496008873, -0.06743429601192474,
      0.029202118515968323, 0.04925934970378876, 0.03852719068527222, -0.08510611206293106,
      0.0003165421076118946, 0.030896130949258804, 0.024821752682328224, -0.027115128934383392,
      -0.03354738652706146, 0.04027537256479263, -0.05958271026611328, -0.001563127851113677,
      -0.03338836506009102, -0.05381973832845688, 0.03393840044736862, -0.05479642003774643,
      0.05502142384648323, -0.0030245690140873194, 0.0053663854487240314, -0.02389627881348133,
      0.03342762589454651, -0.06096808984875679, -0.01607700064778328, 0.08151949942111969,
      0.05952770262956619, -0.05093420669436455, -0.002689128741621971, -0.007688451092690229,
      -0.0661953017115593, -0.017178241163492203, -0.05578780546784401, 0.06765760481357574,
      0.08569245040416718, -0.01932615414261818, 0.06836748868227005, -0.06429081410169601],
     [-0.027989720925688744, 0.04956115037202835, -0.08957748860120773, -0.00013349765504244715,
      -0.08331766724586487, 0.07370846718549728, 0.06445695459842682, 0.0788353681564331,
      -0.08450652658939362, -0.04839304834604263, 0.018531445413827896, 0.04649704694747925,
      0.06898288428783417, -0.07533305883407593, -0.026553235948085785, -0.05691419914364815,
      0.040039658546447754, -0.04150305315852165, 0.039326123893260956, -0.04857328161597252,
      0.019103117287158966, 0.012332177720963955, 0.05321255326271057, -0.09029761701822281,
      -0.06055775284767151, 0.02357698231935501, 0.04902264475822449, -0.04855208843946457,
      -0.08571577817201614, 0.06916875392198563, -0.038433466106653214, -0.04682385176420212,
      0.017495062202215195, -0.07600372284650803, -0.07469134032726288, -0.06610435247421265,
      -0.013571224175393581, -0.02054956555366516, -0.03558644652366638, 0.03270600736141205,
      0.005856323055922985, -0.06307020783424377, -0.10413280129432678, 0.04056360945105553,
      0.06657064706087112, -0.09477882832288742, 0.0032876210752874613, -0.060307685285806656,
      -0.0012850849889218807, 0.08809903264045715, 0.05721797049045563, -0.07465657591819763,
      -0.009658747352659702, -0.12193198502063751, 0.042733944952487946, -0.010022622533142567,
      0.10448908060789108, -0.0030284684617072344, -0.06212454289197922, 0.03767012804746628,
      0.04266881197690964, -0.008899319916963577, 0.06961502879858017, 0.028261356055736542,
      -0.005545427091419697, 0.006378136109560728, -0.02801814116537571, -0.029124276712536812,
      -0.0773259699344635, 0.07512174546718597, 0.072858065366745, -0.0465872660279274,
      -0.04322277009487152, 0.023921199142932892, 0.07078669965267181, 0.015032263472676277,
      0.05070753023028374, 0.0398590974509716, -0.03837534040212631, -0.07509241253137589,
      -0.0729733482003212, 0.060961343348026276, -0.06364060938358307, -0.028772829100489616,
      -0.09487487375736237, -0.06947127729654312, -0.05241601541638374, 0.013669613748788834,
      -0.0052345795556902885, -0.07267121225595474, 0.037308644503355026, -0.019055142998695374,
      -0.03696278855204582, 0.07601737231016159, -0.02291225455701351, -0.09447402507066727,
      0.09266479313373566, 0.0060204132460057735, 0.02120829001069069, 0.05634656175971031,
      0.07203980535268784, 0.06989815086126328, -0.014608914963901043, 0.035162314772605896,
      -0.06371943652629852, 0.05238498002290726, 0.06723764538764954, -0.06461741775274277],
     [0.08248695731163025, 0.06783153861761093, 0.1466648131608963, 0.03749697282910347,
      -0.0793478935956955, 0.0474889762699604, 0.01569809578359127, -0.08806530386209488,
      0.09970445185899734, -0.0016834313282743096, 0.012958531267940998, 0.017310941591858864,
      0.04064458608627319, 0.10013795644044876, 0.09540320932865143, -0.007578260265290737,
      0.0894264355301857, -0.015800805762410164, -0.09375816583633423, 0.0416291169822216,
      -0.009670942090451717, -0.009384937584400177, 0.007950183004140854, 0.0780549943447113,
      -0.07422628998756409, 0.04876618832349777, 0.013904412277042866, 0.01610119827091694,
      0.14084210991859436, -0.05121169984340668, -0.01740257441997528, -0.015420842915773392,
      -0.008753577247262001, 0.020037591457366943, 0.10107831656932831, -0.046386148780584335,
      0.06860342621803284, 0.12196135520935059, 0.031109033152461052, 0.08257140964269638,
      0.030456092208623886, 0.03535957634449005, 0.05138015374541283, 0.04044536128640175,
      -0.02756236493587494, 0.12918585538864136, 0.03846326470375061, -0.045110270380973816,
      -0.028784574940800667, 0.050240084528923035, -0.055969804525375366, 0.11546605825424194,
      0.004567785654217005, 0.1238134577870369, 0.10053100436925888, -0.05570415407419205,
      -0.0864921510219574, 0.10255807638168335, -0.0036287764087319374, 0.11792871356010437,
      -0.1129658967256546, -0.025992797687649727, 0.0215091984719038, 0.028479408472776413,
      0.00213237083517015, -0.06756953150033951, -0.014667173847556114, -0.09498301148414612,
      -0.043067388236522675, 0.01435750350356102, -0.03458859771490097, 0.07440083473920822,
      0.08996736258268356, -0.05115653946995735, -0.018141331151127815, -0.04940483719110489,
      0.07336408644914627, 0.13003796339035034, -0.03225647658109665, 0.13470661640167236,
      0.05890895798802376, 0.029260775074362755, -0.0148782292380929, -0.0023148783948272467,
      0.00011061775148846209, 0.11395499110221863, -0.04592342674732208, -0.03972394019365311,
      -0.05551006272435188, -0.05758841335773468, -0.039617374539375305, -0.08472225815057755,
      -0.05826753377914429, 0.08371084928512573, -0.006952064111828804, 0.14011336863040924,
      -0.05272972956299782, 0.024389347061514854, 0.011321688070893288, 0.11409097164869308,
      0.08156509697437286, 0.05764096602797508, -0.06320749223232269, 0.02947986125946045,
      0.00046699997619725764, -0.09206777811050415, 0.05412815511226654, -0.07705843448638916],
     [-0.06783634424209595, 0.0323554091155529, -0.023053396493196487, 0.050492312759160995,
      0.007178422063589096, -0.027277488261461258, -0.0903969258069992, 0.06737323850393295,
      0.0639156699180603, 0.005772753618657589, -0.030414700508117676, 0.05983338505029678,
      -0.08171534538269043, 0.10334800183773041, 0.021413084119558334, -0.04661329835653305,
      0.0601823627948761, 0.01870804652571678, -0.008341403678059578, 0.002858522580936551,
      0.08808121085166931, 0.04249073565006256, -0.0016325587639585137, -0.09491457045078278,
      0.04243709519505501, 0.09407439082860947, -0.025079084560275078, -0.01996234618127346,
      -0.007485899142920971, 0.05858972668647766, -0.05030784755945206, -0.0710119754076004,
      -0.07625672221183777, -0.015308257192373276, 0.028425054624676704, -0.055705055594444275,
      0.09129389375448227, 0.08356767147779465, -0.05352730676531792, 0.006370498798787594,
      0.06932390481233597, -0.09382263571023941, -0.060918428003787994, 0.005702912341803312,
      -0.025879481807351112, 0.058052193373441696, -0.02561943791806698, 0.031071696430444717,
      0.05915314331650734, -0.028481103479862213, 0.03174164518713951, 0.03634721040725708,
      -0.06881669163703918, 0.06221231073141098, 0.09393928945064545, 0.06462181359529495,
      0.008093719370663166, -0.037787023931741714, -0.01286922674626112, -0.07360268384218216,
      0.0050740716978907585, 0.04400996118783951, -0.016525909304618835, -0.031058477237820625,
      -0.07211180776357651, 0.02557026408612728, 0.04115196317434311, -0.06544698774814606,
      0.021092552691698074, -0.03864924609661102, -0.03212674707174301, 0.0014127137837931514,
      0.05954886972904205, 0.09610817581415176, 0.0486755445599556, 0.06625974178314209,
      0.015892427414655685, 0.04295226186513901, -0.08542080968618393, 0.02417118102312088,
      -0.06170384958386421, 0.06716185808181763, -0.029231268912553787, -0.047517117112874985,
      0.04044582322239876, -0.014616615138947964, 0.0043040309101343155, -0.059258606284856796,
      -0.01960759237408638, 0.06252001971006393, 0.10025399923324585, 0.06184164807200432,
      0.02698419615626335, -0.02562016062438488, 0.07300248742103577, -0.04417452961206436,
      0.03953070938587189, -0.059545304626226425, -0.010752620175480843, 0.028050921857357025,
      0.06580910831689835, -0.03808065876364708, 0.0017055850476026535, 0.0819939598441124,
      -0.04358886182308197, -0.08253514766693115, 0.00958180520683527, 0.037119027227163315],
     [-0.05735654756426811, -0.027744483202695847, -0.03468325734138489, 0.04928608983755112,
      -0.054680753499269485, -0.09218452125787735, -0.07990336418151855, 0.011274985037744045,
      -0.007408716715872288, 0.009285506792366505, -0.08476709574460983, -0.05347805842757225,
      -0.020505880936980247, -0.06960126012563705, -0.06809493154287338, 0.08288130164146423,
      0.02526722475886345, 0.047184307128190994, -0.029161352664232254, 0.02422703430056572,
      0.06069382280111313, -0.06730786710977554, 0.02593671716749668, 0.006660907994955778,
      0.1137431189417839, -0.015261190012097359, -0.09585963934659958, 0.004142770543694496,
      -0.07740290462970734, -0.06996139883995056, -0.025207748636603355, 0.05240561068058014,
      0.007317814975976944, -0.017054086551070213, 0.10493885725736618, -0.04446954280138016,
      -0.08103004842996597, 0.03739834204316139, -0.05123751237988472, -0.01488475501537323,
      -0.08406070619821548, 0.050860706716775894, 0.042897600680589676, 0.024553071707487106,
      -0.06352205574512482, 0.011845934204757214, 0.007088705897331238, 0.02287190593779087,
      0.019305387511849403, 0.06700567901134491, 0.014704696834087372, 0.018466079607605934,
      0.010542506352066994, 0.1168552115559578, -0.021113846451044083, -0.00878783781081438,
      -0.020218927413225174, 0.06058163568377495, 0.10902982950210571, 0.046389784663915634,
      0.052102141082286835, -0.07637843489646912, 0.06305276602506638, -0.04952471703290939,
      -0.05408114567399025, 0.10241886228322983, -0.03705017268657684, 0.04045829549431801,
      -0.07529854029417038, 0.09175027906894684, -0.005838319659233093, -0.029430484399199486,
      -0.07323519140481949, 0.06156504154205322, 0.04038110747933388, 0.046190295368433,
      0.027818800881505013, -0.021141286939382553, -0.05109795928001404, -0.07105810940265656,
      0.02053016610443592, -0.08891234546899796, -0.10784309357404709, -0.07260866463184357,
      -0.05225154384970665, -0.02733232080936432, 0.03891200199723244, -0.036465466022491455,
      -0.013259980827569962, 0.06755693256855011, -0.0649542361497879, 0.02709294483065605,
      -0.07258649170398712, 0.04304065182805061, 0.014777976088225842, -0.06326142698526382,
      0.09567190706729889, -0.07618559151887894, -0.033554255962371826, -0.04117091745138168,
      -0.038032133132219315, 0.04526343196630478, -0.04362252354621887, 0.04999334737658501,
      -0.08823799341917038, -0.028921909630298615, 0.011322645470499992, 0.05637728050351143],
     [0.04971786215901375, 0.05663053318858147, -0.010557913221418858, -0.0013810093514621258,
      0.05977005138993263, -0.0010649951873347163, -0.06625162810087204, -0.09648692607879639,
      0.0707634687423706, 0.0660988837480545, -0.029894676059484482, 0.04011555016040802,
      0.07381471246480942, 0.030184252187609673, 0.05448249727487564, -0.043293703347444534,
      0.050975147634744644, -0.04529612883925438, -0.033637698739767075, 0.031308434903621674,
      -0.0038010822609066963, -0.006655198521912098, -0.09001103788614273, -0.001018526148982346,
      -0.050444960594177246, 0.020459741353988647, 0.01529637910425663, 0.01262484397739172,
      0.002314700512215495, 0.016070010140538216, -0.013941447250545025, 0.01987067610025406,
      -0.07087457925081253, -0.04342762008309364, -0.014778698794543743, -0.053275398910045624,
      0.006793803535401821, 0.06443928182125092, -0.0088596660643816, -0.029120339080691338,
      0.014504529535770416, -0.02871670387685299, 0.001537432661280036, -0.05683966353535652,
      0.07160788774490356, 0.0716327354311943, -0.08958763629198074, -0.10748186707496643,
      -0.09431830793619156, -0.0545339435338974, 0.03091701865196228, -0.037535689771175385,
      -0.06833157688379288, 0.010311402380466461, 0.09251469373703003, -0.013270700350403786,
      -0.0440233089029789, 0.04431748017668724, 0.09355096518993378, -0.047185081988573074,
      -0.038117535412311554, -0.08851677179336548, -0.042894210666418076, 0.01962541788816452,
      0.019641226157546043, -0.0027412709314376116, 0.023328816518187523, 0.04297051206231117,
      -0.029886284843087196, 0.0954776406288147, 0.09449110925197601, 0.017039503902196884,
      0.08389793336391449, 0.01251404918730259, 0.08998603373765945, -0.04978097602725029,
      0.023180002346634865, -0.058160729706287384, -0.002145317615941167, -0.048433225601911545,
      0.08018515259027481, 0.03949917480349541, 0.014455805532634258, -0.051308657974004745,
      -0.09635629504919052, -0.05256679281592369, 0.07179749757051468, 0.08828819543123245,
      0.0352623388171196, -0.10220804810523987, -0.08023548871278763, 0.010220257565379143,
      -0.027512023225426674, 0.06924183666706085, 0.053433675318956375, -0.031103171408176422,
      -0.06455503404140472, 0.06543955951929092, -0.056654639542102814, 0.057946741580963135,
      0.0684826448559761, 0.08387091755867004, 0.04716944694519043, 0.008354452438652515,
      0.011743822135031223, 0.061814162880182266, 0.011091813445091248, -0.10218930244445801],
     [0.061085447669029236, -0.10714298486709595, 0.016738612204790115, -0.007497009821236134,
      -0.005380872171372175, 0.08268456906080246, -0.0418538935482502, 0.06401896476745605,
      -0.0770559161901474, -0.06093152239918709, 0.06614788621664047, 0.09170736372470856,
      0.0936490073800087, -0.08086112886667252, -0.07517940551042557, -0.06473058462142944,
      0.08015409111976624, 0.05371971055865288, -0.09090115129947662, 0.010633318684995174,
      0.0715925544500351, -0.09685633331537247, 0.07543092221021652, 0.02587074227631092,
      -0.043089475482702255, 0.036689866334199905, 0.06054497882723808, -0.07246367633342743,
      0.10530640184879303, 0.08393644541501999, 0.05645826831459999, -0.07762736827135086,
      0.00801687128841877, 0.052425842732191086, 0.06272805482149124, -0.019487975165247917,
      0.022190306335687637, 0.12836401164531708, -0.028406940400600433, -0.056363482028245926,
      0.04292241483926773, -0.004219023510813713, 0.12607325613498688, -0.049998313188552856,
      -0.02845233678817749, 0.05074095353484154, -0.03353472426533699, 0.02057364024221897,
      -0.035790979862213135, -0.09599274396896362, 0.03762995824217796, 0.09751159697771072,
      -0.042853500694036484, 0.06707579642534256, 0.05981684848666191, 0.05474686622619629,
      0.011809021234512329, 0.04684119671583176, 0.018233772367239, 0.08732528239488602,
      0.012577764689922333, 0.023891493678092957, -0.060028351843357086, -0.030626928433775902,
      -0.08840867131948471, -0.07862779498100281, -0.08902733772993088, -0.10040020197629929,
      -0.0011592066148295999, 0.052954815328121185, 0.02591395378112793, 0.05411844328045845,
      -0.04224156588315964, -0.038809195160865784, 0.0585579015314579, 0.008723808452486992,
      0.06911701709032059, 0.005144397262483835, 0.027602605521678925, 0.06399592757225037,
      -0.07121726870536804, 0.08003117889165878, -0.08550798892974854, -0.03229210525751114,
      -0.06246218457818031, 0.0389866828918457, -0.09401725977659225, 0.0037701434921473265,
      -0.09142721444368362, 0.003422643756493926, -0.05819547176361084, 0.07842632383108139,
      0.09567251056432724, 0.06295785307884216, -0.042219504714012146, 0.005402280483394861,
      0.06128137931227684, -0.10034362971782684, -0.06601646542549133, 0.08188844472169876,
      0.07971621304750443, 0.051246993243694305, 0.02380499430000782, 0.02359631285071373,
      0.07368525117635727, -0.07608642429113388, -0.01655563898384571, -0.04040795564651489],
     [-0.08923299610614777, -0.08497609198093414, -0.018315991386771202, 0.043380338698625565,
      0.019052986055612564, -0.10486279428005219, -0.10106337815523148, 0.04316171631217003,
      -0.007472421042621136, 0.08505748957395554, 0.05002492666244507, -0.01912466250360012,
      0.027836285531520844, -0.04582742974162102, -0.023486776277422905, 0.0438016913831234,
      0.07814188301563263, -0.08711916953325272, -0.028402583673596382, 0.05792027711868286,
      0.052962761372327805, -0.004295875784009695, 0.027392592281103134, -0.03732550889253616,
      -0.051182135939598083, -0.02158118598163128, 0.0030753938481211662, 0.008558719418942928,
      -0.01163535751402378, -0.035040903836488724, 0.08752267062664032, 0.02013658545911312,
      -0.03998298570513725, -0.026928165927529335, 0.038321882486343384, -0.0005298986798152328,
      -0.039384253323078156, 0.08875289559364319, 0.05708709731698036, 0.083114393055439,
      -0.05360209941864014, 0.06683140993118286, 0.09974584728479385, 0.10809005796909332,
      -0.09055173397064209, 0.09835474193096161, -0.023926515132188797, 0.05986279249191284,
      -0.08556286245584488, 0.08813095837831497, -0.018486911430954933, -0.03786753490567207,
      -0.019560690969228745, 0.04080422967672348, 0.003988130483776331, -0.07334060966968536,
      -0.06760844588279724, 0.03854504972696304, 0.02307509072124958, -0.0878128930926323,
      -0.0513516329228878, -0.05002850294113159, 0.040994588285684586, 0.011069676838815212,
      -0.031594764441251755, 0.0003298360388725996, -0.10143737494945526, -0.015138478949666023,
      -0.011963906697928905, 0.00024657719768583775, -0.02100873552262783, -0.024913936853408813,
      0.10548748821020126, -0.04185099899768829, -0.04078853875398636, 0.06240793317556381,
      0.03223175182938576, 0.023567456752061844, 0.020093979313969612, -0.01966911368072033,
      0.04627065360546112, -0.04422109201550484, -0.04954402521252632, 0.04992822930216789,
      0.043545544147491455, 0.07953937351703644, -0.010175202041864395, -0.05153808742761612,
      -0.07478100806474686, 0.04268703982234001, -0.0182309802621603, -0.10950221866369247,
      -0.05982466787099838, -0.017157604917883873, -0.03862042725086212, -0.032927513122558594,
      0.008136969991028309, -0.021880974993109703, 0.015434508211910725, -0.07711106538772583,
      -0.007916759699583054, 0.006574703846126795, -0.07523348182439804, -0.04072114825248718,
      0.026525219902396202, -0.08982253074645996, 0.03638860210776329, -0.014754458330571651],
     [-0.06791841983795166, 0.022293204441666603, 0.033335935324430466, 0.0725618302822113,
      -0.05434725061058998, 0.04311627149581909, -0.05350517854094505, -0.017415976151823997,
      0.1003570705652237, -0.03234008327126503, 0.05769382044672966, 0.056242894381284714,
      0.042098160833120346, -0.08266263455152512, 0.09849540144205093, -0.06471650302410126,
      0.03283596411347389, -0.0694744810461998, -0.03176723048090935, 0.053421612828969955,
      0.05723154544830322, 0.04111727327108383, -0.041619230061769485, -0.01599019020795822,
      -0.01609540916979313, 0.060633253306150436, -0.06962980329990387, -0.032291773706674576,
      0.0025784983299672604, 0.06595367938280106, 0.02150714211165905, -0.04420123249292374,
      -0.07278182357549667, 0.03087082877755165, 0.005968953482806683, -0.03813133388757706,
      0.019133806228637695, -0.03599727898836136, -0.03302167356014252, -0.054038163274526596,
      -0.040026139467954636, 0.020778601989150047, -0.06930098682641983, -0.08695255219936371,
      -0.01199851930141449, -0.10768654197454453, -0.08761026710271835, 0.02854553423821926,
      -0.046574536710977554, -0.06503664702177048, -0.020501280203461647, 0.06870869547128677,
      0.07418841123580933, 0.09485965222120285, 0.02531682327389717, -0.07702820748090744,
      0.017456015571951866, 0.03751816600561142, -0.041743677109479904, -0.0067822374403476715,
      -0.012834463268518448, 0.009040898643434048, 0.03762838616967201, -0.049699652940034866,
      0.012127636931836605, -0.037126295268535614, 0.07926146686077118, 0.028642185032367706,
      -0.04763100668787956, 0.023013945668935776, 0.004886940121650696, 0.015346960164606571,
      -0.0054265945218503475, 0.05606384575366974, -0.05348178744316101, 0.04753297194838524,
      0.05321960151195526, 0.04523521289229393, 0.078261598944664, 0.0018737688660621643,
      0.06939534097909927, -0.014792426489293575, -0.008013756945729256, -0.014280683360993862,
      -0.03621095046401024, -0.007416852749884129, -0.027110934257507324, 0.0060727703385055065,
      -0.02263370156288147, -0.0633048266172409, 0.04425685852766037, 0.06802569329738617,
      -0.02310067042708397, -0.09599433839321136, -0.024044333025813103, -0.04359869286417961,
      -0.03575281426310539, 0.011965927667915821, -0.08947397023439407, 0.06489364057779312,
      0.09379881620407104, -0.009728210978209972, 0.07746933400630951, 0.04902605339884758,
      -0.06426282227039337, -0.07670138776302338, -0.07443540543317795, -0.021348437294363976],
     [0.01564396545290947, 0.020213671028614044, -0.060852084308862686, 0.00219765049405396,
      -0.013413483276963234, -0.052356358617544174, 0.05526390299201012, -0.06871777027845383,
      -0.011174134910106659, 0.003984570503234863, 0.04526424780488014, -0.02450917661190033,
      0.012954787351191044, -0.046283967792987823, -0.06821934133768082, 0.06593306362628937,
      -0.06559530645608902, 0.020133202895522118, -0.08944885432720184, 0.06611701101064682,
      -0.015528563410043716, -0.0023623688612133265, 0.09226690232753754, -0.09098297357559204,
      -0.05720154196023941, 0.08818117529153824, -0.02420947514474392, -0.05482204630970955,
      0.03130267560482025, 0.008482193574309349, -0.0023217189591377974, -0.0022854777052998543,
      0.07164940983057022, -0.0511108860373497, 0.07253243774175644, 0.03167057782411575,
      0.06299272924661636, -0.04423798620700836, 0.017936885356903076, -0.06523028016090393,
      0.049286194145679474, -0.04815441370010376, -0.03877859190106392, 0.03812295198440552,
      -0.015808433294296265, -0.017313074320554733, 0.09058190137147903, 0.05596982687711716,
      -0.013076728209853172, 0.08868852257728577, 0.09111877530813217, -0.06117892637848854,
      9.379694529343396e-05, 0.038793254643678665, 0.07596810907125473, -0.0106732789427042,
      0.07437965273857117, 0.008694320917129517, -0.05898278206586838, 0.11110279709100723,
      -0.08732996881008148, -0.06736591458320618, -0.05426663160324097, 0.07056807726621628,
      0.08324018865823746, -0.0250425785779953, 0.0012546321377158165, 0.06545176357030869,
      0.032392989844083786, -0.00830301083624363, -0.013125450350344181, 0.10582546889781952,
      0.0057686856016516685, -0.035981275141239166, -0.0012270223814994097, 0.039419546723365784,
      -0.029415728524327278, 0.01622687466442585, 0.09440812468528748, -0.06616304814815521,
      -0.08202166110277176, 0.05204994976520538, 0.03942304104566574, -0.11165304481983185,
      0.05167608708143234, -0.09317171573638916, 0.05862662196159363, 0.025639455765485764,
      -0.056377556174993515, 0.02665145881474018, -0.0033819577656686306, 0.039447344839572906,
      0.07565914839506149, -0.009734116494655609, -0.0659104436635971, -0.006794594693928957,
      -0.05427424982190132, 0.004079175181686878, 0.07515725493431091, 0.05341152474284172,
      0.0032150966580957174, 0.04907255247235298, 0.009563508443534374, 0.0697198435664177,
      0.02624635212123394, 0.06448715180158615, 0.013597315177321434, 0.024625495076179504],
     [0.05600836127996445, -0.039247021079063416, -0.04387671872973442, -0.04172069951891899,
      -0.0848136618733406, 0.010759505443274975, 0.04989471286535263, -0.04537494108080864,
      0.10144655406475067, 0.06846769154071808, 0.053207866847515106, -0.058206818997859955,
      -0.10591147094964981, 0.08035442233085632, 0.0019376820418983698, -0.07152613997459412,
      0.00988289900124073, -0.09143377095460892, -0.03356287628412247, 0.0706935003399849,
      -0.024807000532746315, -0.029615463688969612, 0.07461132109165192, -0.0612226203083992,
      0.07264775037765503, 0.023009618744254112, 0.038259830325841904, -0.06407859176397324,
      -0.0765799880027771, 0.0788300558924675, -0.010452534072101116, -0.02065698243677616,
      0.057549700140953064, 0.0242769755423069, 0.07468950748443604, -0.03632798045873642,
      -0.006266161799430847, -0.060356512665748596, -0.026397952809929848, -0.0014218557626008987,
      0.010703468695282936, 0.08170375972986221, -0.008420795202255249, -0.06652087718248367,
      0.01859893649816513, 0.017432428896427155, 0.037517063319683075, 0.01793144829571247,
      0.017529074102640152, -0.021715905517339706, -0.08757968246936798, 0.015024995431303978,
      0.06617452949285507, -0.10867476463317871, 0.0005704890936613083, -0.03147551417350769,
      -0.09163118153810501, -0.03289199247956276, 0.08308783173561096, 0.030735602602362633,
      -0.05347893759608269, -0.09709444642066956, -0.05543261766433716, 0.045594364404678345,
      -0.015839962288737297, -0.007627641782164574, 0.03027668595314026, -0.10397839546203613,
      -0.04058528691530228, 0.051552411168813705, 0.08796514570713043, -0.0007353233522735536,
      0.02036289870738983, 0.02863115631043911, 0.10401768237352371, -0.0683964192867279,
      -0.030538275837898254, 0.010029858909547329, -0.09441476315259933, 0.009386038407683372,
      -0.04153145104646683, -0.021751314401626587, 0.05626511946320534, -0.015039157122373581,
      -0.060843326151371, 0.07462400197982788, -0.0025663513224571943, 0.042989786714315414,
      -0.014002466574311256, -0.033597324043512344, 0.03861263394355774, -0.06200604513287544,
      -0.0033646789379417896, 0.005630390252918005, -0.04766739159822464, -0.017648622393608093,
      0.04401133581995964, 0.11100402474403381, -0.0433158278465271, 0.0765657126903534,
      -0.08815742284059525, -1.2899239663966e-05, -0.007200988009572029, -0.050773050636053085,
      0.04677001014351845, 0.10402143001556396, -0.08472263067960739, 0.057156968861818314],
     [0.03809555247426033, 0.046542827039957047, -0.06537408381700516, -0.06115823611617088,
      -0.05508233234286308, -0.002713628113269806, 0.014216175302863121, 0.06668758392333984,
      0.03552136570215225, -0.008195840753614902, 0.09494377672672272, 0.010307499207556248,
      0.02258957549929619, -0.06391968578100204, -0.08542840927839279, -0.0048148841597139835,
      -0.005885415710508823, -0.000821281282696873, 0.03665495663881302, 0.05970289185643196,
      0.03348188474774361, 0.04130270704627037, 0.010068681091070175, -0.06919436156749725,
      0.028081156313419342, -0.06028779596090317, 0.05392039939761162, 0.02872096747159958,
      0.021811634302139282, 0.02216869592666626, -0.088307686150074, 0.07633967697620392,
      -0.02809493988752365, -0.06978576630353928, -0.00862581841647625, 0.05164990946650505,
      0.017559828236699104, 0.07398910820484161, 0.08138619363307953, 0.10499732196331024,
      -0.07421012967824936, -0.007959180511534214, -0.04572540894150734, 0.07379686087369919,
      0.011290778405964375, 0.0752398669719696, -0.05245240032672882, -0.06459635496139526,
      0.010913445614278316, -0.07264117896556854, -0.01720523089170456, 0.02176964282989502,
      -0.04288310557603836, -0.025761688128113747, 0.058348074555397034, -0.04883059859275818,
      -0.08407465368509293, 0.024929841980338097, -0.04512351006269455, -0.05976361036300659,
      0.0010795979760587215, -0.06077907234430313, -0.05704033374786377, -0.08476532250642776,
      0.09110583364963531, -0.059801436960697174, -0.08473961055278778, -0.026278069242835045,
      0.03344612196087837, -0.03569689765572548, -0.048465438187122345, -0.0031071817502379417,
      0.046876002103090286, -0.029150551185011864, -0.08118457347154617, 0.0006847845506854355,
      -0.038071583956480026, -0.017559638246893883, 0.08664762228727341, 0.024137644097208977,
      0.049160365015268326, -0.09754980355501175, 0.013347040861845016, -0.019781429320573807,
      -0.01809052564203739, 0.09163295477628708, -0.012179069221019745, 0.07010119408369064,
      -0.027697021141648293, 0.0534859374165535, -0.007496766280382872, -0.010339600034058094,
      0.006209042854607105, -0.10900680720806122, 0.060247574001550674, -0.07249069213867188,
      -0.05686467885971069, -0.05628616735339165, 0.07992520928382874, 0.03265436738729477,
      0.04268789663910866, -0.0367036908864975, 0.02676759660243988, -0.04142836108803749,
      -0.05028245598077774, 0.056129906326532364, -0.02254377119243145, -0.05760284885764122],
     [-0.06017285957932472, 0.06420668214559555, 0.05487706884741783, -0.07530418783426285,
      -0.030721239745616913, 0.043581150472164154, 0.01217330526560545, -0.04138660058379173,
      -0.05307110771536827, 0.08954701572656631, 0.1428237110376358, -0.021414639428257942,
      -0.06835103780031204, 0.023578915745019913, 0.07238733768463135, 0.08506821095943451,
      -0.037957195192575455, -0.0513157844543457, -0.09284688532352448, -0.0365515761077404,
      0.049532804638147354, -0.0193489920347929, 0.032047927379608154, 0.03188607096672058,
      0.009105999022722244, -0.014290938153862953, 0.012792862951755524, 0.09920382499694824,
      -0.03457801789045334, -0.023377208039164543, 0.031068697571754456, 0.014029216952621937,
      0.0693889930844307, -0.06778167933225632, -0.0019326771143823862, -0.02310755103826523,
      0.04221823066473007, 0.04865143075585365, -0.051995888352394104, -0.008184405975043774,
      -0.022553253918886185, -0.011756629683077335, 0.02767876163125038, -0.0412432998418808,
      -0.07785460352897644, 0.04347365349531174, 0.07866497337818146, 0.019655803218483925,
      -0.05453070253133774, -0.10468801856040955, 0.034344132989645004, 0.01952342689037323,
      -0.032461296766996384, 0.15659421682357788, 0.05276097357273102, 0.038248077034950256,
      -0.07413632422685623, -0.0024881819263100624, -0.0332772359251976, 0.1376047134399414,
      -0.09962137043476105, -0.049622952938079834, 0.012395269237458706, -0.01909056305885315,
      0.03086228109896183, 0.0007023221696726978, -0.030986441299319267, -0.06793277710676193,
      0.06642822921276093, 0.030393682420253754, -0.016368672251701355, 0.1679307371377945,
      0.06373541802167892, 0.06183925271034241, -0.010505991987884045, -0.020910507068037987,
      0.0816148966550827, 0.05817527323961258, -0.10226749628782272, -0.04279763251543045,
      0.04501619562506676, -0.0968504399061203, -0.06045907735824585, 0.043986544013023376,
      -0.05553998798131943, 0.021348528563976288, 0.011550066992640495, 0.04631536826491356,
      -0.05651804804801941, 0.002890657866373658, 0.03374877944588661, 0.10049636662006378,
      -0.05592077970504761, 0.06283769756555557, 0.023764699697494507, 0.06223585829138756,
      -0.07691486924886703, 0.03986354172229767, 0.013000797480344772, 0.011878794059157372,
      -0.005260704550892115, -0.03701941668987274, 0.06695745885372162, -0.0641050711274147,
      0.06696692854166031, -0.03938285633921623, 0.04041760042309761, -0.08969546854496002],
     [0.06283900141716003, 0.002396943513303995, 0.015739362686872482, -0.04779663681983948,
      -0.0694224089384079, 0.04885677993297577, 0.00030338289798237383, -0.05689356476068497,
      0.11312268674373627, -0.02601243369281292, 0.014539037831127644, -0.07260412722826004,
      0.008408460766077042, -0.06091322377324104, 0.023997170850634575, -0.11115219444036484,
      0.08000297844409943, 0.12693290412425995, -0.12095864862203598, 0.049655452370643616,
      -0.05774030089378357, -0.056590475142002106, -0.0557430163025856, 0.15584585070610046,
      0.01582060381770134, 0.024952417239546776, 0.027843523770570755, 0.05629992112517357,
      0.10861749947071075, 0.05852867290377617, 0.0001710687211016193, -0.09280548989772797,
      0.09135667234659195, 0.08622284978628159, 0.16674429178237915, 0.06636273115873337,
      0.1315573751926422, 0.05640355125069618, 0.05436934158205986, 0.1063808798789978,
      -0.09488089382648468, 0.012133232317864895, -0.031425610184669495, -0.03891712427139282,
      0.0511983223259449, 0.09557382017374039, -0.01815812848508358, 0.027921665459871292,
      -0.07708282768726349, 0.018550489097833633, 0.006648166570812464, 0.13372905552387238,
      -0.008224183693528175, 0.1521022617816925, 0.0874367505311966, -0.12066618353128433,
      -0.10421890020370483, 0.058653682470321655, -0.029575373977422714, 0.1615588217973709,
      -0.09476402401924133, -0.021648656576871872, -0.05094234645366669, 0.07366938143968582,
      0.03578370437026024, 0.040800392627716064, 0.04118168354034424, -0.07387135177850723,
      -0.05091847479343414, 0.03221883997321129, 0.06867804378271103, 0.0016794446855783463,
      0.05324341729283333, 0.05482235178351402, -0.010286973789334297, 0.016917744651436806,
      -0.06420579552650452, 0.0835300162434578, -0.056769803166389465, -0.03617549315094948,
      0.06209171563386917, -0.019544098526239395, -0.059031981974840164, 0.05432857200503349,
      0.04759438335895538, 0.0014881605748087168, -0.06867257505655289, 0.13054333627223969,
      0.07177197933197021, 0.06388626247644424, -0.032541196793317795, -0.06218591332435608,
      -0.07652181386947632, 0.012028705328702927, -0.026148296892642975, 0.11769258975982666,
      0.0875541940331459, -0.04424351453781128, -0.0793415904045105, 0.052989471703767776,
      0.0437711663544178, -0.10687068104743958, -0.060063742101192474, 0.004438359290361404,
      0.08405659347772598, 0.05311434715986252, 0.03258669003844261, -0.0017605047905817628],
     [0.06636308878660202, 0.07410930097103119, 0.04134361818432808, 0.06671439856290817,
      -0.0018104607006534934, 0.05523862689733505, 0.05440552160143852, 0.015296694822609425,
      -0.04856475442647934, -0.08811184018850327, 0.007074715103954077, 0.05770208314061165,
      0.038862258195877075, 0.07800523936748505, 0.0460926815867424, -0.05056065693497658,
      -0.05003267154097557, -0.10458619892597198, 0.02136281132698059, 0.0746077448129654,
      0.046465106308460236, 0.03488476201891899, -0.031675226986408234, 0.018790777772665024,
      -0.07457304745912552, -0.08062328398227692, -0.09812498837709427, 0.10329634696245193,
      0.033299654722213745, 0.06961937993764877, 0.060081079602241516, 0.058136820793151855,
      -0.006406279746443033, 0.08516939729452133, 0.07604531943798065, -0.02033188007771969,
      0.0596858412027359, 0.03006657212972641, 0.015723509714007378, -0.06556280702352524,
      0.08625812083482742, -0.0036068232730031013, 0.015089770779013634, -0.01975911855697632,
      0.026548461988568306, 0.054117150604724884, 0.08002251386642456, 0.0779200866818428,
      -0.06760884821414948, -0.09188331663608551, 0.08831121027469635, 0.05576467514038086,
      0.018872587010264397, 0.023519860580563545, 0.08840174227952957, 0.03394137695431709,
      -0.012787527404725552, -0.06891278922557831, -0.034148383885622025, -0.002615691628307104,
      0.0542309395968914, -0.04315893352031708, -0.08269283920526505, -0.10667694360017776,
      0.028851764276623726, -0.061096642166376114, -0.00372719787992537, 0.003882855409756303,
      0.04300859570503235, 0.08884360641241074, 0.060576483607292175, -0.04572565481066704,
      0.10333405435085297, 0.012180548161268234, 0.09748547524213791, 0.06438859552145004,
      -0.03815465793013573, -0.04412807524204254, 0.012695614248514175, -0.047515057027339935,
      -0.021733295172452927, -0.006812434643507004, -0.00816053245216608, 0.03514387831091881,
      -0.08486679196357727, -0.010131712071597576, 0.010785236023366451, 0.026369396597146988,
      0.03631586953997612, -0.08065588772296906, -0.02244577556848526, 0.03771545737981796,
      -0.08030639588832855, -0.08930391818284988, -0.03692999109625816, -0.0755205824971199,
      0.10661453008651733, -0.020578831434249878, -0.024847371503710747, -0.05594995617866516,
      -0.050791967660188675, 0.07528604567050934, -0.0078097255900502205, 0.036614976823329926,
      0.0736103504896164, -0.08548687398433685, 0.029893292114138603, -0.07408270239830017],
     [-0.0008393353200517595, 0.021197933703660965, 0.0834411084651947, -0.018015127629041672,
      -0.01963741146028042, 0.0417758971452713, 0.03022170066833496, 0.0626799464225769,
      -0.06687299907207489, -0.045978233218193054, 0.17631176114082336, -0.0693615972995758,
      0.05698074400424957, -0.06175035983324051, -0.0561005137860775, -0.07568230479955673,
      0.08492399007081985, 0.13049592077732086, -0.08925778418779373, 0.004174578934907913,
      0.03935323655605316, -0.14027494192123413, -0.024083750322461128, 0.04584095627069473,
      0.05566808953881264, 0.046764858067035675, -0.056766558438539505, 0.08420324325561523,
      0.06653697043657303, 0.03647534176707268, -0.028790708631277084, -0.08859444409608841,
      -0.009185689501464367, 0.007554973009973764, 0.03993235155940056, 0.04709602892398834,
      0.03940349444746971, 0.07448440790176392, -0.08842873573303223, 0.09134357422590256,
      -0.0769718587398529, 0.03314407914876938, 0.08931053429841995, 0.0344015508890152,
      -0.006554972380399704, 0.04640315845608711, 0.055865027010440826, 0.07316076755523682,
      0.06989826261997223, -0.038500308990478516, -0.057699449360370636, 0.04987705871462822,
      0.08587772399187088, 0.1474369317293167, -0.008099977858364582, -0.07056666165590286,
      -0.09040448814630508, 0.07386311143636703, 0.11271738260984421, 0.14756959676742554,
      -0.10961618274450302, 0.0077585494145751, -0.009792502969503403, 0.02391461469233036,
      -0.022225696593523026, 0.04694333299994469, -0.03461560606956482, 0.005274869501590729,
      -0.04599104821681976, 0.11427460610866547, 0.06889481097459793, 0.13763684034347534,
      -0.03339171037077904, 0.029223933815956116, 0.1114746555685997, 0.12212663888931274,
      0.07323543727397919, 0.08510711044073105, -0.12453001737594604, 0.04862509295344353,
      0.030451539903879166, -0.09372838586568832, 0.06957696378231049, -0.056138183921575546,
      -0.0030086736660450697, -0.049657195806503296, 0.006926343776285648, -0.00513291172683239,
      0.01172737404704094, -0.059117089956998825, -0.0023406012915074825, 0.02433161251246929,
      0.05741070583462715, -0.03633251413702965, 0.019186317920684814, 0.10314512997865677,
      0.11084458976984024, -0.01992320828139782, 0.07466555386781693, 0.08181735873222351,
      0.12076937407255173, -0.017971664667129517, -0.09108518809080124, 0.07975276559591293,
      0.002566334092989564, 0.0445932038128376, 0.022207168862223625, -0.061024684458971024],
     [0.015503817237913609, 0.06146509572863579, 0.04628508538007736, 0.0720156878232956,
      0.01627928391098976, 0.037710681557655334, 0.04614929482340813, 0.0779779702425003,
      -0.009401585906744003, 0.0769769549369812, 0.14003033936023712, 0.011571235954761505,
      0.07560913264751434, -0.015315026044845581, 0.0798109769821167, 0.017320726066827774,
      0.06619539856910706, -0.046615440398454666, 0.060674313455820084, -0.021174894645810127,
      -0.0022848627995699644, -0.138292595744133, -0.09793934226036072, 0.042941540479660034,
      0.03415415808558464, 0.14009042084217072, 0.051001712679862976, -0.04387717321515083,
      0.11360307037830353, -0.00020020226656924933, -0.054491568356752396, -0.08305899798870087,
      0.04464752972126007, 0.0836072489619255, 0.12065725773572922, -0.0564495325088501,
      0.07279830425977707, 0.037986986339092255, 0.017489472404122353, -0.036579251289367676,
      0.019009452313184738, -0.038176435977220535, -0.03987722098827362, -0.0326550155878067,
      -0.04233020171523094, 0.12348295748233795, -0.03269768878817558, -0.01014690287411213,
      -0.016964565962553024, 0.004938500002026558, -0.11473716050386429, 0.09886304289102554,
      -0.09457998722791672, 0.13766629993915558, 0.021816667169332504, -0.05764263495802879,
      -0.11000847816467285, -0.00516689196228981, 0.04721291735768318, 0.10612822324037552,
      -0.01897248439490795, 0.07712668180465698, -0.027344221249222755, -0.05869206413626671,
      -0.015461200848221779, -0.07243578881025314, -0.06707020103931427, -0.009012027643620968,
      -0.07659655064344406, -0.05158237740397453, -0.07624303549528122, 0.029738401994109154,
      -0.05369405820965767, 0.040856052190065384, 0.07829640805721283, -0.012185070663690567,
      0.0210975781083107, 0.11262793093919754, 0.05429371818900108, -0.02975481003522873,
      -0.09075364470481873, 0.05462520569562912, 0.0277975182980299, -0.034774016588926315,
      -0.08169686794281006, 0.07935117185115814, 0.07587304711341858, 0.08450039476156235,
      -0.08619040250778198, -0.0660119354724884, -0.0018635678570717573, -0.024738378822803497,
      0.08034129440784454, -0.048794664442539215, -0.10029713064432144, 0.05198076739907265,
      0.01889391429722309, 0.06471465528011322, 0.09340375661849976, 0.12056467682123184,
      0.07910856604576111, -0.020173344761133194, 0.019140906631946564, -0.01448361948132515,
      0.006688437890261412, -0.05272948369383812, 0.0446711964905262, 0.004431062377989292],
     [0.057742781937122345, 0.009849402122199535, 0.02591134048998356, 0.06460560113191605,
      -0.055943556129932404, 0.06210494041442871, 0.008053415454924107, 0.06437142193317413,
      0.03359036520123482, 0.045079268515110016, -0.09082336723804474, -0.08615115284919739,
      0.07384280115365982, -0.033969782292842865, -0.08048383891582489, -0.042042315006256104,
      -0.06626448780298233, 0.06927637755870819, 0.032436538487672806, -0.03430037572979927,
      0.08321306854486465, -0.08693603426218033, -0.07335890084505081, 0.041148439049720764,
      0.02104366011917591, 0.0594661571085453, 0.015265248715877533, 0.04167988523840904,
      0.001537495874799788, -0.0921899750828743, 0.01226753182709217, 0.02692817710340023,
      0.10149414092302322, 0.02118810825049877, -0.009126602672040462, -0.07281326502561569,
      -0.014654839411377907, 0.03573187068104744, 0.06562461704015732, -0.05372890457510948,
      0.007513327058404684, -0.09380078315734863, -0.08282386511564255, 0.05377977341413498,
      0.07709889858961105, -0.004071284085512161, -0.030315406620502472, -0.05119737610220909,
      -0.052125610411167145, 0.004506875295192003, 0.03355422616004944, 0.016462037339806557,
      0.06962651014328003, 0.05376681312918663, 0.010514010675251484, -0.035599276423454285,
      0.025965696200728416, 0.11137991398572922, -0.022152980789542198, 0.0355461910367012,
      0.004677545744925737, 0.05522740259766579, 0.08795633167028427, 0.06236689165234566,
      -0.0835665762424469, 0.050934381783008575, 0.020767804235219955, -0.05053028464317322,
      0.05050487071275711, -0.03088955022394657, 0.06520170718431473, -0.02875782735645771,
      -0.045900799334049225, 0.011924716643989086, 0.09140338748693466, 0.024844584986567497,
      -0.0004073228337801993, -0.06841810792684555, -0.06904351711273193, 0.056086283177137375,
      -0.09221884608268738, -0.09059575200080872, -0.0346391424536705, -0.05109555274248123,
      -0.02649332582950592, -0.07605454325675964, 0.10059347003698349, 0.05670643597841263,
      0.0014030939200893044, -0.0552511103451252, 0.036867499351501465, -0.04771360382437706,
      0.04841237515211105, 0.054613515734672546, 0.025770606473088264, -0.04583531245589256,
      -0.021514710038900375, -0.0387897863984108, 0.04669879376888275, 0.04586029052734375,
      0.005967507604509592, -0.09051815420389175, 0.03511396795511246, 0.02906334400177002,
      0.006636453792452812, -0.008575552143156528, -0.0852060467004776, -0.0008837701752781868],
     [0.039647176861763, -0.061730097979307175, -0.026361949741840363, -0.05983354151248932,
      -0.0019151289016008377, 0.05755610018968582, 0.04512910172343254, -0.06496430933475494,
      -0.012762973085045815, 0.0071665141731500626, 0.042353563010692596, -0.07427681237459183,
      0.04941514879465103, 0.07723919302225113, 0.033845141530036926, -0.05019327998161316,
      0.07749935239553452, -0.05953561142086983, 0.08254317194223404, -0.06714446097612381,
      0.015846557915210724, 0.054016102105379105, 0.038211822509765625, 0.04537765681743622,
      -0.04162228852510452, 0.039574794471263885, -0.03052673116326332, -0.04577303305268288,
      0.04641282558441162, 0.029174014925956726, 0.011772500351071358, -0.0190355833619833,
      0.0040998863987624645, 0.06346295028924942, 0.020550761371850967, 0.07832994312047958,
      -0.023833971470594406, -0.034550271928310394, -0.08661089092493057, -0.08797287940979004,
      0.007651014719158411, 0.04897924140095711, -0.0499238483607769, -0.08651040494441986,
      0.04328041896224022, -0.06882999837398529, -0.04359172657132149, -0.05529371649026871,
      -0.0009978327434509993, -0.021584775298833847, 0.01605614647269249, -0.0055659376084804535,
      -0.022918887436389923, 0.10785432159900665, 0.09107395261526108, -0.08753389120101929,
      -0.06546228379011154, 0.09534238278865814, -0.04761404171586037, -0.021129736676812172,
      0.01445339247584343, -0.09533226490020752, 0.029643921181559563, 0.04083261266350746,
      -0.01808091811835766, 0.04989523068070412, 0.0026107789017260075, 0.0907304435968399,
      0.014354095794260502, 0.0856739729642868, 0.07898495346307755, 0.12092843651771545,
      -0.041826050728559494, -0.08280282467603683, 0.006920190528035164, 0.072354257106781,
      -0.06113250553607941, -0.07120197266340256, -0.044994719326496124, -0.058171194046735764,
      -0.09343592077493668, -0.04001016914844513, -0.061103563755750656, 0.07060680538415909,
      -0.02475067228078842, 0.06370223313570023, 0.06167023256421089, 0.026698045432567596,
      -0.023585906252264977, -0.01996941864490509, -0.034716639667749405, -0.0812641903758049,
      -0.035768792033195496, -0.026135846972465515, -0.030553800985217094, 0.158990278840065,
      0.005764927249401808, 0.0062573254108428955, 0.031353749334812164, -0.01830233260989189,
      -0.007688221987336874, 0.06964753568172455, -0.061180684715509415, 0.059055622667074203,
      -0.01294727623462677, 0.004391324240714312, 0.06818457692861557, -0.09269782155752182],
     [-0.09274076670408249, 0.06932277232408524, 0.007221256848424673, 0.03740929812192917,
      -0.04047343134880066, 0.09511137008666992, -0.05100252479314804, 0.05848880112171173,
      -0.03405710309743881, 0.0533798411488533, 0.025655437260866165, -0.043984849005937576,
      0.05765123292803764, 0.02981034852564335, -0.05530625209212303, 0.049267325550317764,
      0.058764148503541946, 0.026054002344608307, -0.054495107382535934, -0.05286141112446785,
      -0.06012057885527611, -0.051013749092817307, 0.0770999938249588, -0.04074574261903763,
      -0.03179648146033287, -0.04035886004567146, 0.09033554047346115, 0.019106579944491386,
      0.028653161600232124, 0.04154486209154129, 0.051135145127773285, -0.057868268340826035,
      -0.00023563532158732414, -0.06889139115810394, 0.04025835543870926, -0.0023314529098570347,
      0.10392396152019501, -0.0670991912484169, -0.04779909551143646, 0.019995134323835373,
      -0.012768224813044071, 0.0573054775595665, 0.08705666661262512, 0.034558892250061035,
      -0.04054014012217522, 0.10530130565166473, 0.041102372109889984, -0.012218165211379528,
      -0.003217224031686783, 0.07207217812538147, 0.02286348305642605, -0.03606031462550163,
      -0.040879614651203156, -0.04358198493719101, 0.03484869375824928, -0.08130728453397751,
      -0.01408669538795948, 0.007943197153508663, -0.06173946335911751, -0.03711968660354614,
      -0.05686426907777786, -0.08131992816925049, 0.038876112550497055, 0.009707387536764145,
      -0.027640128508210182, -0.01903740130364895, 0.019586017355322838, 0.010018710047006607,
      -0.09097560495138168, -0.03413241729140282, -0.04426434636116028, -0.03592554107308388,
      0.012779198586940765, 0.054197512567043304, -0.05295179411768913, 0.0945952832698822,
      -0.017682690173387527, 0.1181221604347229, -0.10948437452316284, 0.0864410474896431,
      0.017108824104070663, 0.10397753864526749, -0.032469555735588074, -0.07698684930801392,
      -0.056271687150001526, -0.006702004000544548, 0.06535132974386215, 0.0018694953760132194,
      0.06668364256620407, 0.01568835787475109, -0.0431697815656662, 0.012800098396837711,
      -0.04730697721242905, -0.007644555997103453, -0.019912803545594215, -0.03592318668961525,
      0.007898906245827675, 0.06476053595542908, 0.06678222119808197, 0.08294165134429932,
      -0.06647907197475433, -0.09329976886510849, 0.040559928864240646, 0.0004900929634459317,
      -0.030629387125372887, 0.003515101969242096, 0.01246695127338171, 0.05532190576195717]], dtype=np.float32).T + np.asarray([0.03106207773089409, -0.10773561149835587, 0.003512574592605233, -0.060747452080249786,
     -0.09014245122671127, -0.12880922853946686, 0.059526171535253525, 0.051800910383462906,
     0.02755708619952202, 0.010270015336573124, 0.01460679154843092, -0.06994640082120895,
     0.006396062672138214, -0.10983945429325104, -0.060507453978061676, 0.009675314649939537,
     -0.09552887827157974, -0.07888409495353699, -0.07160723954439163, 0.03473304212093353,
     -0.044123291969299316, 0.08596508949995041, -0.12551665306091309, -0.01510691363364458,
     0.03430968150496483, 0.07102526724338531, -0.06986916810274124, -0.08222290128469467,
     -0.044397905468940735, -0.09019295126199722, -0.07551302015781403, -0.0022359774447977543,
     -0.08420757949352264, 0.06239280477166176, -0.03753453120589256, -0.045721571892499924,
     -0.12436573952436447, 0.045483626425266266, -0.01569228805601597, -0.01984037645161152,
     -0.018265267834067345, 0.050753962248563766, -0.03736983612179756, -0.11116695404052734,
     -0.10365956276655197, 0.052583612501621246, -0.03423898667097092, -0.1349712461233139,
     0.045029785484075546, -0.07765969634056091, -0.15660521388053894, -0.03405357152223587,
     0.05313638597726822, -0.07721206545829773], dtype=np.float32)
    trajectory_h2 = trajectory_h2 / (1.0 + np.exp(-np.clip(trajectory_h2, -30.0, 30.0)))
    trajectory_raw = (trajectory_h2 @ np.asarray([[-0.102227583527565, -0.07016874849796295, 0.10086701065301895, 0.08054063469171524,
      0.04289204999804497, 0.04341170936822891, 0.010659908875823021, 0.02281496487557888,
      -0.008262686431407928, 0.09072291105985641, -0.10804365575313568, 0.09879661351442337,
      -0.10942588746547699, 0.051066190004348755, 0.017392465844750404, -0.02707725204527378,
      -0.09970723092556, -0.08654651045799255, 0.12472347915172577, -0.06049933284521103,
      0.1947396844625473, -0.03234817460179329, 0.04963081330060959, 0.0330941379070282,
      -0.049116794019937515, -0.06071808561682701, -0.030373351648449898, -0.020254578441381454,
      -0.03692667558789253, -0.05522351711988449, 0.07576441019773483, -0.0864502340555191,
      0.050454434007406235, -0.016309376806020737, -0.047953106462955475, 0.013573072850704193,
      0.15505294501781464, -0.07919325679540634, 0.001592106418684125, -0.010753682814538479,
      0.01534669566899538, -0.06252016872167587, -0.014155656099319458, -0.044963538646698,
      -0.06636017560958862, -0.024318262934684753, 0.022272393107414246, 0.17348559200763702,
      -0.04780032858252525, 0.06792863458395004, 0.04493611678481102, -0.07232119143009186,
      0.0292547307908535, -0.015466864220798016]], dtype=np.float32).T + np.asarray([-0.011176231317222118], dtype=np.float32)).ravel()
    parent_early = model_x[:, [36, 38, 40, 42, 44, 46, 48, 50]]
    parent_late = model_x[:, [37, 39, 41, 43, 45, 47, 49, 51]]
    parent_delta = parent_late - parent_early
    parent_x = np.concatenate([0.5 * (parent_early + parent_late), parent_delta, np.abs(parent_delta), parent_early * parent_late], axis=1)
    parent_mean = parent_x.mean(axis=1, keepdims=True)
    parent_var = ((parent_x - parent_mean) ** 2).mean(axis=1, keepdims=True)
    parent_z = (parent_x - parent_mean) / np.sqrt(parent_var + 1e-5)
    parent_z = parent_z * np.asarray([1.0201048851013184, 1.0095634460449219, 0.9458165764808655, 1.0469108819961548, 0.9637984037399292,
     0.9717417359352112, 0.9348713755607605, 1.045913815498352, 0.8667550086975098, 0.8770984411239624,
     1.1135931015014648, 0.9527961015701294, 0.8916541337966919, 0.9756714105606079, 0.95546555519104,
     0.8765855431556702, 1.064543604850769, 1.1118485927581787, 1.0914266109466553, 1.1269220113754272,
     1.0363377332687378, 1.0138728618621826, 0.9672949314117432, 1.006907343864441, 1.0491176843643188,
     1.047345757484436, 1.018730878829956, 1.044831395149231, 1.0337028503417969, 1.015040636062622,
     0.9534163475036621, 1.0890967845916748], dtype=np.float32) + np.asarray([-0.017205476760864258, -0.0022365122567862272, 0.025036849081516266, -0.0698048323392868,
     -0.05954087898135185, 0.03031015582382679, 0.0037604146637022495, 0.0003087818331550807,
     -0.0032874816097319126, 0.012828869745135307, 0.014517650008201599, 0.0446145199239254,
     -0.05166880413889885, -0.051768708974123, 0.00253941654227674, 0.041463546454906464,
     0.1187310665845871, 0.160554900765419, -0.08864586055278778, 0.14017662405967712,
     0.10752440989017487, 0.07646114379167557, 0.09424398094415665, 0.058017563074827194,
     -0.05248459428548813, -0.07051603496074677, 0.03810543194413185, -0.044385503977537155,
     0.06301525980234146, -0.08913515508174896, -0.042554501444101334, -0.09364452213048935], dtype=np.float32)
    parent_h1 = parent_z @ np.asarray([[0.018191928043961525, 0.012115074321627617, -0.08515895903110504, -0.139485165476799,
      0.07676122337579727, -0.1411188393831253, 0.05125875025987625, 0.09048058837652206,
      0.021779028698801994, -0.1313682645559311, 0.1927756518125534, -0.10147476941347122,
      0.07468163222074509, -0.1373319774866104, 0.0833301916718483, -0.02894117496907711,
      -0.17789821326732635, -0.1237538531422615, -0.21047775447368622, -0.06297972798347473,
      -0.15758074820041656, 0.09603799879550934, -0.1601097136735916, -0.1692381650209427,
      0.004552839789539576, -0.04378538206219673, 0.07777994871139526, -0.007502339314669371,
      -0.139203280210495, 0.06420159339904785, 0.01735002174973488, 0.18221169710159302],
     [0.10621563345193863, -0.04813280701637268, 0.10876543074846268, 0.03277011588215828,
      -0.1344815194606781, -0.10769685357809067, -0.07875509560108185, 0.11250238120555878,
      0.09364712983369827, 0.14277441799640656, 0.26007023453712463, 0.09781953692436218,
      -0.06251060962677002, 0.028054367750883102, -0.02612430974841118, -0.09276703745126724,
      -0.05862460285425186, -0.038483839482069016, 0.11274779587984085, -0.11799193173646927,
      0.10953889787197113, -0.05033322796225548, -0.05435428023338318, -0.045836854726076126,
      -0.08809878677129745, 0.12156996876001358, 0.06162063777446747, -0.09360390901565552,
      -0.0870460495352745, 0.143168643116951, -0.10542554408311844, 0.04462166130542755],
     [-0.08348757773637772, -0.033763494342565536, -0.0028385950718075037, 0.06351098418235779,
      0.11995093524456024, -0.07063586264848709, 0.09389299899339676, 0.17345890402793884,
      -0.02144015021622181, -0.12802013754844666, 0.06657057255506516, -0.1743120551109314,
      -0.17596766352653503, -0.02769855596125126, -0.15181520581245422, 0.0391295850276947,
      0.26991114020347595, 0.06587430089712143, -0.04619031399488449, 0.14276285469532013,
      0.021404583007097244, 0.18570318818092346, -0.1081545278429985, 0.030063388869166374,
      -0.09030504524707794, 0.03482311591506004, 0.19314351677894592, 0.029340723529458046,
      0.2354777455329895, 0.0020234910771250725, 0.15766815841197968, 0.052006397396326065],
     [-0.05474962294101715, 0.06408044695854187, -0.03642755374312401, -0.07854489237070084,
      -0.11905333399772644, 0.13844013214111328, 0.10996881872415543, -0.1430356651544571,
      0.06437501311302185, -0.10300956666469574, -0.15134333074092865, -0.00742173520848155,
      0.08367866277694702, 0.16356220841407776, -0.06634362787008286, -0.11980614811182022,
      -0.19391660392284393, -0.09857409447431564, -0.08112362027168274, 0.10977556556463242,
      -0.0807645171880722, 0.08024346828460693, 0.06382600963115692, -0.07379511743783951,
      0.09806493669748306, 0.09903768450021744, 0.2060292810201645, -0.13861988484859467,
      -0.1198124811053276, -0.023419396951794624, 0.023893585428595543, 0.11863631755113602],
     [-0.1594228297472, 0.1091880351305008, -0.08771438151597977, -0.15450941026210785,
      0.1485365927219391, 0.06810873746871948, -0.13981691002845764, 0.11703260242938995,
      -0.11163826286792755, 0.07381926476955414, -0.1150389313697815, 0.05285242572426796,
      -0.0014812893932685256, 0.020441733300685883, -0.0410652793943882, 0.04529590532183647,
      0.08877292275428772, -0.050511185079813004, -0.09846614301204681, -0.09588057547807693,
      -0.10204882174730301, -0.17865386605262756, -0.149464949965477, 0.014192942529916763,
      -0.009894153103232384, -0.05123867094516754, -0.03617722913622856, -0.02951991744339466,
      -0.009067302569746971, 0.04650285094976425, -0.09206700325012207, -0.07542160153388977],
     [0.09767960757017136, -0.16193965077400208, 0.007517222780734301, 0.19936639070510864,
      0.09404689818620682, -0.1619899421930313, 0.05929676815867424, 0.01995447278022766,
      -0.10099074244499207, 0.05961742624640465, -0.04948101565241814, -0.08041919022798538,
      -0.006175505928695202, -0.04095644876360893, 0.08618590980768204, -0.0699833333492279,
      -0.21275728940963745, -0.0687071830034256, 0.09933469444513321, -0.09555279463529587,
      -0.1046975776553154, -0.15982921421527863, -0.07187837362289429, 0.12815769016742706,
      -0.029100826010107994, 0.22431597113609314, -0.04803483933210373, 0.08261539041996002,
      0.12923628091812134, 0.1283099353313446, -0.02726394310593605, 0.07440946996212006],
     [-0.06457855552434921, -0.11371982842683792, -0.1059955582022667, 0.05361390858888626,
      -0.01858118548989296, 0.06448691338300705, 0.034009333699941635, 0.0446992963552475,
      -0.08810943365097046, -0.09410411864519119, -0.005199076607823372, 0.01474662497639656,
      -0.2125299572944641, 0.15461818873882294, 0.05591932311654091, 0.11592453718185425,
      0.1625676453113556, 0.056843552738428116, -0.04771121218800545, 0.24538305401802063,
      0.18035997450351715, 0.10125551372766495, 0.12305489182472229, 0.0058753592893481255,
      0.13520678877830505, -0.04798578470945358, 0.12780643999576569, 0.04624220356345177,
      0.20915819704532623, 0.14808635413646698, -0.04889293015003204, 0.06767404079437256],
     [0.07325190305709839, 0.09336552023887634, 0.07561340928077698, -0.06777743250131607,
      0.0008584090392105281, 0.10243599116802216, 0.1683284193277359, 0.2526125907897949,
      -0.06198413297533989, 0.11056118458509445, 0.032606031745672226, -0.10860545933246613,
      0.06522874534130096, 0.12820632755756378, -0.16879048943519592, 0.01912173442542553,
      -0.25277194380760193, -0.5038004517555237, -0.17651812732219696, -0.38085392117500305,
      -0.24289050698280334, -0.2917729616165161, -0.12199408560991287, -0.10580544918775558,
      0.21469777822494507, 0.26390400528907776, 0.09165012836456299, 0.24389514327049255,
      -0.38449352979660034, -0.04072070121765137, 0.006953536067157984, 0.19173644483089447],
     [0.04635115712881088, -0.14991949498653412, 0.12082404643297195, -0.13730911910533905,
      -0.09022105485200882, 0.0025401879101991653, 0.10256350040435791, 0.02471836656332016,
      -0.18960994482040405, 0.17831960320472717, 0.045228082686662674, 0.12753179669380188,
      -0.12842614948749542, 0.16769646108150482, -0.12151516228914261, 0.07848217338323593,
      0.15207120776176453, 0.03533487021923065, -0.09900642931461334, -0.014192401431500912,
      -0.16947519779205322, 0.1298459768295288, -0.04051165282726288, -0.06372770667076111,
      -0.04153309389948845, -0.05236507207155228, 0.09196314215660095, 0.10342294722795486,
      -0.03953944146633148, -0.04692039638757706, 0.029488351196050644, -0.07795759290456772],
     [0.10424680262804031, 0.17550505697727203, 0.025887003168463707, -0.02186954766511917,
      0.04470612481236458, -0.0037279201205819845, -0.05573398619890213, -0.15263578295707703,
      -0.17652471363544464, 0.14992254972457886, 0.0836213082075119, -0.05121241882443428,
      -0.086504265666008, 0.019032999873161316, -0.08887510001659393, -0.1346326768398285,
      0.12149744480848312, -0.06770770251750946, -0.021398605778813362, -0.10194811969995499,
      0.04824350029230118, 0.05474879592657089, 0.05058160796761513, 0.09756021201610565,
      -0.13413645327091217, -0.11883290857076645, 0.07818590849637985, -0.16251777112483978,
      0.12257663905620575, -0.004250787664204836, 0.1444459706544876, -0.02510857582092285],
     [0.17776478826999664, -0.05572698637843132, -0.17528624832630157, -0.14214017987251282,
      -0.10898680984973907, 0.07198399305343628, 0.11666227132081985, -0.0520726777613163,
      0.07326225936412811, -0.02389185130596161, 0.08798221498727798, -0.11769066751003265,
      0.04375450313091278, 0.01292592752724886, -0.14967934787273407, -0.046895384788513184,
      0.07676003128290176, -0.1183643788099289, 0.007898199371993542, 0.03490986302495003,
      0.13041408360004425, 0.09986915439367294, -0.034404024481773376, -0.12692809104919434,
      0.12709715962409973, 0.10895762592554092, 0.03404799476265907, 0.19529235363006592,
      -0.15486019849777222, 0.09056702256202698, -0.05563144385814667, 0.04989122226834297],
     [-0.19138796627521515, -0.07214940339326859, -0.049056001007556915, -0.015189643017947674,
      -0.16119679808616638, 0.08265918493270874, 0.06361860036849976, 0.060990698635578156,
      -0.15127445757389069, -0.060795895755290985, -0.08534717559814453, -0.08965818583965302,
      0.13553541898727417, 0.007148893550038338, -0.16015401482582092, -0.09810974448919296,
      -0.11648956686258316, 0.004857740830630064, -0.20407025516033173, 0.15808415412902832,
      0.1468946933746338, 0.11885640025138855, 0.18139778077602386, 0.11214189976453781,
      -0.20794686675071716, 0.09810817241668701, 0.12195328623056412, 0.12027370184659958,
      0.011930765584111214, 0.037481557577848434, -0.11805275827646255, -0.13782425224781036],
     [0.09214109927415848, -0.052348989993333817, 0.09307164698839188, 0.02753813937306404,
      -0.046722736209630966, 0.07099074125289917, -0.19584792852401733, -0.009894994087517262,
      -0.014976158738136292, -0.10749460011720657, 0.13689401745796204, -0.13182562589645386,
      -0.10646450519561768, 0.1680452972650528, -0.10574866831302643, 0.03905586153268814,
      -0.05472163110971451, 0.00840518157929182, 0.17548076808452606, 0.14278973639011383,
      0.07951530814170837, -0.19698238372802734, -0.12596715986728668, 0.0038276552222669125,
      -0.10877916216850281, -0.052777450531721115, 0.06464367359876633, 0.05146471783518791,
      -0.026592187583446503, -0.11335397511720657, -0.09824445843696594, -0.1310979425907135],
     [0.007412806618958712, 0.19397544860839844, -0.12487348914146423, 0.1780247688293457,
      0.14861595630645752, 0.17353305220603943, -0.04252248629927635, 0.19736072421073914,
      -0.06018698588013649, -0.031085072085261345, 0.18648110330104828, 0.05460455268621445,
      0.09472095221281052, 0.13896292448043823, -0.0654778927564621, 0.03453414514660835,
      -0.11142373085021973, -0.2985132932662964, 0.2984241843223572, -0.19586065411567688,
      -0.2177913784980774, -0.10708673298358917, -0.25947314500808716, -0.15078556537628174,
      0.06991977244615555, -0.11966831237077713, 0.0014067342272028327, 0.20314879715442657,
      -0.2367294728755951, 0.04028725624084473, 0.17352214455604553, -0.077117919921875],
     [-0.21145658195018768, -0.16375170648097992, -0.03351309150457382, -0.029981598258018494,
      -0.08232280611991882, 0.0027244912926107645, 0.003345736302435398, 0.15577107667922974,
      0.04304823279380798, 0.04341168701648712, -0.15524478256702423, 0.04586811736226082,
      -0.10960035771131516, -0.17610269784927368, 0.05046809837222099, 0.06418217718601227,
      -0.0180645864456892, 0.04427989944815636, -0.027223940938711166, -0.19434960186481476,
      -0.011868100613355637, 0.08735067397356033, -0.15548181533813477, 0.14613085985183716,
      0.07328284531831741, 0.04083778336644173, 0.08736200630664825, -0.07945939898490906,
      0.023065468296408653, -0.01745021715760231, 0.12128943204879761, -0.14823269844055176],
     [-0.041683729737997055, 0.02676217071712017, 0.14902758598327637, -0.026547549292445183,
      -0.15423083305358887, 0.023982718586921692, -0.006480899639427662, 0.14222422242164612,
      0.10724902898073196, 0.08609882742166519, 0.23084548115730286, 0.07825874537229538,
      -0.0943475067615509, 0.018991492688655853, 0.09817241877317429, 0.0779278352856636,
      -0.16867710649967194, -0.04156796634197235, 0.05312708392739296, 0.06078805774450302,
      0.06121479719877243, -0.07460962980985641, -0.09481534361839294, -0.09559108316898346,
      -0.07889948040246964, -0.10559437423944473, 0.13536621630191803, -0.08375672250986099,
      -0.03401723876595497, 0.13399635255336761, 0.1426745057106018, 0.11564403772354126],
     [-0.05206744745373726, 0.12449822574853897, 0.10332755744457245, -0.0614607147872448,
      -0.0069833374582231045, -0.1515006273984909, -0.10701577365398407, -0.10127086192369461,
      -0.15330585837364197, -0.05921880155801773, -0.21354883909225464, -0.06952393054962158,
      0.1298552006483078, -0.11610398441553116, -0.11522898823022842, 0.16145211458206177,
      0.1140027642250061, -0.1611122190952301, 0.2111845761537552, 0.04286373406648636,
      -0.14047609269618988, -0.0040135569870471954, 0.004661055281758308, 0.02397078461945057,
      0.0040237815119326115, -0.03692350536584854, 0.13024471700191498, -0.04268448054790497,
      0.00998743250966072, -0.08649203181266785, -0.021796640008687973, -0.04534236714243889],
     [-0.05073615908622742, 0.17381057143211365, 0.04856979846954346, -0.1512664258480072,
      0.04476533085107803, -0.10881306231021881, -0.1394999772310257, 0.009405351243913174,
      0.02949388511478901, -0.1210789903998375, 0.06057571992278099, 0.0258781760931015,
      0.0796603262424469, -0.17160478234291077, -0.12521760165691376, 0.04761175066232681,
      0.023072626441717148, 0.17060664296150208, 0.010226886719465256, 0.0829475075006485,
      0.17804935574531555, 0.07804857939481735, 0.04190017655491829, -0.03443842753767967,
      -0.0684705600142479, -0.11062002927064896, -0.11748870462179184, -0.14044180512428284,
      -0.11858638375997543, -0.04346918687224388, 0.11708616465330124, -0.011479880660772324],
     [-0.06351526826620102, 0.04497446492314339, -0.16158433258533478, -0.022725079208612442,
      0.05293088033795357, -0.10421469062566757, -0.17127881944179535, -0.09841699153184891,
      0.02540482021868229, -0.16400043666362762, 0.07635386288166046, 0.04675256088376045,
      0.12120766192674637, -0.07293098419904709, 0.04486290365457535, -0.07206230610609055,
      0.04535376653075218, -0.07170967012643814, 0.03497839719057083, 0.0006355689256452024,
      0.15879057347774506, 0.10784048587083817, -0.1637885570526123, -0.08867336064577103,
      -0.07956597208976746, -0.02879624255001545, 0.016853222623467445, 0.06876131892204285,
      -0.15776841342449188, 0.06295479089021683, -0.08188272267580032, -0.05899602919816971],
     [0.003549302229657769, 0.16435958445072174, -0.007876592688262463, -0.19762755930423737,
      -0.11677137017250061, -0.04361465573310852, 0.1028946191072464, 0.11393933743238449,
      -0.14505597949028015, 0.009632215835154057, 0.04560520872473717, 0.10982601344585419,
      0.0010690806666389108, 0.13240547478199005, -0.002996302442625165, 0.10571810603141785,
      0.12122385948896408, 0.0985562652349472, 0.011651643551886082, 0.12045411020517349,
      0.0014716715086251497, 0.04157499596476555, 0.09155146777629852, 0.033386144787073135,
      0.11440187692642212, -0.12166900932788849, 0.10106799751520157, 0.029897695407271385,
      0.16017986834049225, 0.10936041921377182, -0.16607336699962616, -0.19630223512649536],
     [-0.19387204945087433, -0.019944367930293083, -0.09823476523160934, 0.02935807965695858,
      0.06267797946929932, -0.05311150476336479, 0.1568147838115692, -0.11732963472604752,
      0.1855328530073166, -0.18631424009799957, -0.05581092834472656, -0.0055753071792423725,
      0.11823707073926926, 0.15513204038143158, 0.12159348279237747, -0.1133059486746788,
      -0.11297082901000977, 0.12078174948692322, 0.0662565529346466, -0.14569900929927826,
      0.13362270593643188, 0.04562835022807121, 0.040208183228969574, 0.060432836413383484,
      -0.034446462988853455, -0.07596096396446228, 0.00033125319168902934, -0.18023420870304108,
      0.10165375471115112, 0.14419880509376526, -0.17064674198627472, -0.15738938748836517],
     [0.09542027115821838, -0.1270083487033844, -0.05848332494497299, 0.04921484366059303,
      -0.0032485222909599543, -0.07526285201311111, 0.031926922500133514, -0.147221177816391,
      0.08830157667398453, 0.09089399129152298, 0.14688827097415924, -0.089912049472332,
      -0.008762913756072521, -0.02801484800875187, -0.07417374849319458, 0.11930655688047409,
      -0.10983990132808685, 0.1545156091451645, -0.008595705032348633, 0.032011479139328,
      0.12464343011379242, 0.14410774409770966, 0.14479763805866241, 0.17117249965667725,
      0.12825210392475128, 0.1982049196958542, 0.0900774896144867, -0.016932586207985878,
      0.010316920466721058, -0.10260787606239319, -0.017926521599292755, 0.1226707473397255],
     [-0.1622234731912613, 0.10572724789381027, 0.1599842756986618, -0.11176149547100067,
      -0.054120566695928574, 0.17011885344982147, 0.03586094081401825, -0.11967197805643082,
      -0.07660767436027527, 0.024380244314670563, 0.011939863674342632, 0.017787914723157883,
      -0.061734456568956375, -0.04429168254137039, 0.08334144949913025, -0.034355178475379944,
      -0.002023879438638687, -0.09059034287929535, 0.19896937906742096, 0.010187892243266106,
      0.15406376123428345, -0.10664591193199158, 0.15901973843574524, 0.12145262956619263,
      0.18770475685596466, 0.10564834624528885, 0.1264341026544571, -0.07886890321969986,
      -0.06558273732662201, -0.11511091887950897, 0.02234024554491043, 0.04553353786468506],
     [0.0009283983381465077, 0.050575073808431625, -0.06274071335792542, 0.04138559475541115,
      0.030861446633934975, 0.08605842292308807, -0.09650452435016632, 0.16205281019210815,
      0.021076316013932228, -0.05617458373308182, 0.11608347296714783, -0.14714442193508148,
      0.023096371442079544, -0.018365168944001198, -0.12399329990148544, 0.02094440348446369,
      0.040732260793447495, -0.2197626382112503, 0.2634679973125458, -0.06409696489572525,
      0.006843932904303074, 0.0007986743003129959, -0.09774059802293777, 0.07789924740791321,
      0.027637457475066185, 0.044021811336278915, -0.07267925143241882, -0.09775175899267197,
      0.11312998831272125, -0.0817626491189003, -0.018519869074225426, 0.0019149422878399491],
     [0.15458613634109497, -0.17662222683429718, 0.10067611932754517, 0.1710888147354126,
      -0.05547720938920975, -0.15122422575950623, 0.14973917603492737, -0.15333901345729828,
      0.05185571685433388, -0.12311191856861115, 0.12350200116634369, 0.10934291779994965,
      0.06506873667240143, 0.10209596902132034, -0.06017228588461876, 0.18489886820316315,
      -0.02452724613249302, -0.1736171692609787, -0.054398130625486374, -0.21640343964099884,
      0.11109498888254166, 0.1214575245976448, -0.16341309249401093, -0.20441344380378723,
      -0.0587739460170269, 0.04469377547502518, -0.11530563980340958, 0.07951821386814117,
      -0.1322782337665558, 0.1299263834953308, 0.005610494874417782, 0.0579051673412323],
     [-0.028627123683691025, 0.152123361825943, -0.12624496221542358, 0.011419392190873623,
      -0.057333700358867645, -0.04242612421512604, -0.13199631869792938, 0.12852871417999268,
      -0.11933130770921707, 0.002844669623300433, -0.05703501030802727, -0.0802459567785263,
      0.1277497410774231, -0.048317767679691315, -0.0237983800470829, -0.1250421404838562,
      -0.04698740318417549, -0.15314333140850067, 0.10148824006319046, 0.08818194270133972,
      -0.1878320276737213, -0.12761621177196503, 0.02279028482735157, -0.09195046871900558,
      0.2073475420475006, -0.14038190245628357, 0.10618122667074203, 0.019478140398859978,
      -0.10559161752462387, 0.1668146848678589, 0.05993684381246567, 0.028405936434864998],
     [0.1742115169763565, -0.17664873600006104, 0.058643560856580734, 0.05661790445446968,
      0.03969200327992439, 0.1313600093126297, 0.047846175730228424, -0.0724218562245369,
      0.06016327068209648, -0.13530270755290985, 0.18795974552631378, 0.12580905854701996,
      -0.041927605867385864, 0.12310585379600525, -0.13295748829841614, -0.07476506382226944,
      0.11416680365800858, -0.01630353182554245, -0.004677154123783112, -0.03756178542971611,
      0.08377913385629654, 0.14102964103221893, -0.05836159735918045, 0.04525645077228546,
      -0.054047513753175735, 0.13345248997211456, -0.08986921608448029, -0.023822566494345665,
      0.12255579233169556, 0.18791653215885162, 0.008209701627492905, 0.06803283840417862],
     [0.1325361281633377, -0.05969153717160225, -0.08245806396007538, 0.18652959167957306,
      -0.06500806659460068, -0.03844957426190376, -0.17147599160671234, -0.14635542035102844,
      0.1836160570383072, 0.16089393198490143, -0.03878450021147728, 0.13780392706394196,
      0.00048252110718749464, 0.13564516603946686, 0.1633693426847458, 0.041121769696474075,
      -0.10512795299291611, -0.19541381299495697, -0.08422090858221054, -0.043911706656217575,
      -0.1382148414850235, 0.1222233846783638, 0.09956281632184982, -0.0707404837012291,
      0.16048863530158997, 0.1484423577785492, -0.04625571519136429, 0.0207743551582098,
      0.15720602869987488, 0.18268947303295135, -0.02056930586695671, 0.13454096019268036],
     [-0.09347732365131378, 0.13906465470790863, 0.061237089335918427, -0.033640652894973755,
      -0.0470367893576622, -0.16510067880153656, 0.03249644488096237, -0.16010357439517975,
      -0.10554374009370804, 0.04871590808033943, -0.11349782347679138, -0.06244758516550064,
      -0.13089045882225037, 0.08753405511379242, 0.03644423186779022, 0.13411930203437805,
      0.06864854693412781, -0.12195801734924316, 0.0009347758605144918, 0.09553460776805878,
      -0.12234145402908325, -0.08911211788654327, 0.014931966550648212, 0.05799981951713562,
      -0.045300163328647614, -0.05426901578903198, -0.12136077135801315, -0.10463137179613113,
      -0.05752842500805855, -0.08874043077230453, -0.07768236100673676, -0.06448499858379364],
     [0.004434392787516117, 0.1425633430480957, 0.1591440886259079, 0.008938537910580635,
      0.0455574169754982, -0.09724214673042297, 0.031517598778009415, 0.1255551427602768,
      -0.15526354312896729, -0.13073013722896576, -0.07266346365213394, -0.08889316022396088,
      -0.02338503859937191, 0.09530582278966904, -0.08649186044931412, -0.049934711307287216,
      -0.014847248792648315, -0.20016266405582428, 0.14697086811065674, -0.1725536584854126,
      -0.1615869104862213, -0.007825799286365509, 0.11910610646009445, -0.09550824016332626,
      -0.138185054063797, 0.036470960825681686, 0.1359114646911621, 0.03222985938191414,
      -0.09927782416343689, 0.19402354955673218, 0.0695185512304306, 0.056224387139081955],
     [-0.13931287825107574, -0.09419187903404236, 0.004550427198410034, -0.0486186146736145,
      -0.1959787756204605, -0.13766688108444214, 0.17476458847522736, 0.0263986773788929,
      0.1013626977801323, 0.09014179557561874, -0.08889894187450409, -0.1893371194601059,
      0.049572914838790894, -0.09695678949356079, 0.15959060192108154, -0.035382166504859924,
      0.08442185074090958, 0.004585189279168844, 0.14489200711250305, -0.09754064679145813,
      0.10921013355255127, 0.1397566944360733, -0.11277265101671219, 0.12269924581050873,
      -0.08330611139535904, -0.014629599638283253, 0.08362750709056854, -0.2026243507862091,
      0.18494980037212372, -0.03695160895586014, -0.018967807292938232, 0.14088213443756104],
     [0.15725208818912506, 0.01964803971350193, 0.1371581256389618, -0.13484570384025574,
      0.0023463743273168802, 0.17642144858837128, 0.14218030869960785, -0.01613379269838333,
      -0.015093782916665077, -0.052813686430454254, 0.09812001883983612, 0.02466631680727005,
      -0.06861131638288498, -0.08629157394170761, 0.046484313905239105, -0.14940258860588074,
      -0.053233493119478226, 0.0891868844628334, -0.1714080423116684, 0.020481931045651436,
      -0.12080523371696472, -0.01377197913825512, -0.08542430400848389, 0.07996095716953278,
      -0.07006493210792542, -0.2225891798734665, 0.08563629537820816, 0.06975478678941727,
      -0.12916891276836395, -0.030496608465909958, -0.09941605478525162, -0.14639736711978912],
     [-0.09348104149103165, -0.10875006020069122, -0.04980402812361717, -0.03576570376753807,
      0.0028033729176968336, 0.101429283618927, 0.11744405329227448, 0.12639079988002777,
      0.07555261254310608, -0.11511068046092987, -0.13945914804935455, -0.11704448610544205,
      0.07557704299688339, 0.0031632527243345976, 0.03134271502494812, 0.08745366334915161,
      0.0041114711202681065, -0.13664044439792633, 0.2572440505027771, -0.08414500951766968,
      0.09806951135396957, 0.004624919034540653, 0.056633803993463516, 0.12095630168914795,
      0.17152045667171478, 0.050980012863874435, 0.006420338060706854, -0.09688474237918854,
      -0.12088816612958908, -0.02367347478866577, 0.0024893521331250668, -0.08983173966407776],
     [-0.00041647307807579637, -0.00846475176513195, -0.009073288179934025, -0.14508387446403503,
      0.0591534860432148, 0.10579084604978561, 0.08264420926570892, 0.08556301891803741,
      0.0375145860016346, -0.10833859443664551, -0.04170597344636917, 0.13952629268169403,
      0.1471523493528366, -0.09017437696456909, 0.06996599584817886, -0.061233341693878174,
      -0.17871050536632538, 0.12288578599691391, -0.2145441770553589, 0.1857517808675766,
      -0.10990231484174728, -0.007016474846750498, -0.07616226375102997, 0.0729006975889206,
      -0.19249014556407928, 0.11325395107269287, -0.09965727478265762, -0.2491016536951065,
      -0.044748734682798386, 0.1628449708223343, -0.10085668414831161, 0.05741926655173302],
     [-0.08428502827882767, -0.17008750140666962, -0.1786607801914215, -0.030242785811424255,
      -0.020026791840791702, -0.005448757205158472, 0.10420147329568863, 0.04820331186056137,
      0.0198582224547863, 0.042987216264009476, 0.1375805139541626, 0.08320469409227371,
      -0.02245032973587513, 0.12173577398061752, -0.012460430152714252, -0.043072786182165146,
      0.16820329427719116, -0.07390454411506653, -0.03648727387189865, -0.05316515266895294,
      -0.08097917586565018, -0.074304960668087, 0.145796999335289, 0.12008624523878098,
      -0.10204056650400162, -0.013835281133651733, -0.07329356670379639, 0.021979397162795067,
      -0.1443197876214981, -0.18804778158664703, 0.1361839473247528, -0.1603366583585739],
     [0.07926253229379654, -0.00463168416172266, 0.059880778193473816, 0.14566287398338318,
      0.10596424341201782, 0.026632241904735565, 0.0639398992061615, 0.11978167295455933,
      0.010642815381288528, 0.055178362876176834, -0.09078080207109451, 0.07059118151664734,
      -0.11659049242734909, 0.10275007039308548, 0.00866683665663004, 0.14087140560150146,
      -0.08443736284971237, -0.17659293115139008, -0.014881382696330547, -0.0512404628098011,
      -0.08344623446464539, -0.1743851602077484, -0.0491010881960392, 0.010186697356402874,
      0.2284913808107376, 0.013519597239792347, -0.02572271041572094, 0.2640983760356903,
      -0.028239818289875984, 0.15314596891403198, 0.1481640785932541, 0.11333279311656952],
     [-0.0077892933040857315, 0.0530448779463768, -0.07373283803462982, -0.0398106649518013,
      -0.1649603694677353, -0.07944343239068985, 0.13022132217884064, -0.1624242663383484,
      -0.09273876249790192, 0.023486554622650146, 0.12568455934524536, -0.14353744685649872,
      -0.10137045383453369, -0.06217455118894577, -0.007176835089921951, -0.12259596586227417,
      -0.0738963931798935, -0.1415012627840042, -0.1319255828857422, -0.08593577891588211,
      -0.10666023194789886, 0.11734447628259659, -0.017711954191327095, -0.13548001646995544,
      0.009069124236702919, 0.08644357323646545, 0.19045360386371613, -0.06399957090616226,
      0.07329049706459045, -0.11461477726697922, 0.12107846885919571, -0.14316870272159576],
     [-0.17825718224048615, -0.12764060497283936, 0.01802229881286621, 0.13510234653949738,
      0.07893478870391846, 0.11275968700647354, 0.009425035677850246, 0.05251550301909447,
      0.028415407985448837, -0.011953245848417282, -0.04652322456240654, 0.025610988959670067,
      0.08384053409099579, -0.100070521235466, 0.023081650957465172, 0.0899917408823967,
      0.10354453325271606, 0.10930882394313812, -0.20543785393238068, -0.002924996195361018,
      0.08664927631616592, 0.008220337331295013, -0.1300526261329651, -0.005035076756030321,
      -0.0641864538192749, 0.04973507672548294, 0.10145096480846405, 0.026964828372001648,
      -0.1589827686548233, -0.06558582931756973, -0.15762628614902496, 0.05818873643875122],
     [-0.17467236518859863, -0.10126100480556488, -0.1423598676919937, -0.10691292583942413,
      0.132463738322258, -0.04939236491918564, -0.20245391130447388, 0.12750782072544098,
      -0.0726441815495491, -0.04619699344038963, 0.2665790021419525, -0.13379591703414917,
      -0.07682102918624878, 0.11899926513433456, -0.02056456357240677, -0.14736728370189667,
      -0.18236571550369263, -0.06910905987024307, -0.01050050463527441, -0.03619268164038658,
      -0.00808253325521946, 0.00122545356862247, 0.026107851415872574, -0.1821015328168869,
      0.18864883482456207, -0.13279181718826294, 0.21269652247428894, -0.21348534524440765,
      0.1304554045200348, -0.16701972484588623, -0.10322155058383942, 0.05582508072257042],
     [-0.16684377193450928, 0.11652515828609467, -0.03270388022065163, 0.01826031506061554,
      -0.06077740713953972, -0.018061023205518723, 0.080024354159832, -0.05038028210401535,
      0.048163291066884995, 0.033553596585989, 0.13439017534255981, -0.02810606360435486,
      0.07103349268436432, -0.07327637076377869, 0.03717241808772087, 0.10007400065660477,
      -0.10611309111118317, -0.07372511178255081, -0.14135633409023285, 0.12873698770999908,
      -0.16372686624526978, 0.1826382875442505, 0.1580560803413391, -0.028798161074519157,
      -0.054521799087524414, 0.18484953045845032, -0.11961250007152557, 0.09565231949090958,
      0.1247781291604042, -0.050055764615535736, -0.053049083799123764, -0.0702119916677475],
     [-0.0013096207985654473, 0.10017356276512146, -0.038820911198854446, 0.03231149539351463,
      0.1196858286857605, -0.03238120675086975, 0.1805676370859146, -0.05930589139461517,
      -0.009770122356712818, 0.12340286374092102, -0.2912590503692627, 0.039644189178943634,
      -0.07545817643404007, 0.05478412285447121, -0.03210479021072388, 0.1464027613401413,
      -0.016666967421770096, -0.21058109402656555, 0.2688930630683899, -0.30566835403442383,
      -0.26224273443222046, -0.0997837632894516, -0.30241575837135315, -0.12081756442785263,
      0.12034865468740463, 0.007115091197192669, -0.05858367308974266, 0.16829290986061096,
      -0.33340322971343994, 0.1386346071958542, 0.015939492732286453, 0.11612752825021744],
     [-0.04731902852654457, 0.07777676731348038, -0.03304629027843475, -0.18621546030044556,
      -0.011982216499745846, 0.02201482281088829, -0.0349184088408947, 0.14212334156036377,
      -0.018014471977949142, -0.015042036771774292, -0.04839060455560684, -0.1770913451910019,
      -0.03865527734160423, 0.15016283094882965, 0.11063375324010849, -0.12308604270219803,
      -0.10889824479818344, -0.06074608117341995, -0.07483159750699997, -0.10884407162666321,
      0.1704632043838501, 0.18077197670936584, -0.04390950873494148, 0.1335514336824417,
      -0.02453112229704857, -0.10739713162183762, -0.16790610551834106, -0.04334009811282158,
      -0.1750514656305313, -0.18040429055690765, -0.02012721449136734, -0.21142813563346863],
     [0.011891941539943218, 0.16735686361789703, 0.12262821942567825, -0.1086486428976059,
      0.10861489921808243, 0.14913371205329895, -0.01155160553753376, 0.15270838141441345,
      0.05342623591423035, 0.12032841891050339, -0.1336546540260315, -0.15959180891513824,
      -0.022537091746926308, 0.09398050606250763, 0.09955385327339172, 0.017397291958332062,
      -0.11045394092798233, -0.07832347601652145, -0.13062196969985962, -0.09626277536153793,
      -0.11171852797269821, 0.08615957945585251, -0.1615258902311325, 0.029249535873532295,
      -0.04059065133333206, -0.027968067675828934, 0.1084512323141098, -0.0491744764149189,
      -0.033269934356212616, -0.008950257673859596, -0.09829818457365036, -0.06297793239355087],
     [0.038105059415102005, -0.06717934459447861, 0.017757270485162735, 0.1548435240983963,
      -0.10050688683986664, 0.06404982507228851, 0.10501468926668167, 0.20010414719581604,
      0.06703086942434311, -0.11118894070386887, 0.08696237206459045, -0.008676202967762947,
      0.15947844088077545, 0.08909270912408829, 0.10604604333639145, -0.0435897558927536,
      -0.20384541153907776, -0.03056618943810463, 0.19534476101398468, -0.2312181144952774,
      -0.23294340074062347, -0.1587039977312088, -0.2958492040634155, -0.22498266398906708,
      0.16458386182785034, 0.13676512241363525, 0.14580954611301422, -0.018022678792476654,
      -0.18728239834308624, 0.17601870000362396, 0.18368513882160187, 0.04007825627923012],
     [-0.17419201135635376, 0.1636141985654831, -0.10629389435052872, -0.14044614136219025,
      0.09478309750556946, 0.0564013235270977, -0.01835756003856659, -0.16821758449077606,
      -0.13762378692626953, 0.0493764765560627, -0.042593564838171005, -0.11908005177974701,
      0.14890508353710175, 0.01941634528338909, -0.12275134772062302, 0.004437759518623352,
      0.1278635412454605, -0.21971769630908966, 0.12029486894607544, -0.1558254063129425,
      0.088196761906147, 0.03022591583430767, 0.1345730423927307, -0.013420602306723595,
      -0.0668858215212822, -0.14256098866462708, 0.1914881318807602, 0.10069956630468369,
      0.09632612764835358, -0.14079542458057404, 0.18733184039592743, 0.09129539877176285],
     [-0.047586631029844284, -0.15276440978050232, -0.02339145541191101, 0.08742650598287582,
      0.16717475652694702, 0.07276735454797745, 0.11019773781299591, 0.07623853534460068,
      -0.014244365505874157, -0.04077153280377388, 0.1410149186849594, 0.08329717069864273,
      0.12207509577274323, 0.098953478038311, -0.09718260914087296, 0.19698482751846313,
      -0.06800325959920883, -0.03756890073418617, -0.10014266520738602, -0.10934082418680191,
      0.09144164621829987, 0.048809152096509933, -0.1344633400440216, -0.10809142142534256,
      0.0518161803483963, -0.06715715676546097, -0.15727555751800537, 0.04536031559109688,
      -0.1293271780014038, -0.1390189528465271, 0.004933771677315235, 0.01094627846032381],
     [-0.14960920810699463, 0.04842979833483696, 0.13696689903736115, 0.1694820374250412,
      0.010060964152216911, 0.05092324689030647, -0.02065904252231121, 0.14756691455841064,
      -0.014499443583190441, -0.05017508938908577, -0.09616318345069885, 0.12044785916805267,
      -0.01576595939695835, -0.024604300037026405, 0.032146718353033066, 0.013175382278859615,
      -0.04332916811108589, 0.015886753797531128, 0.3328434228897095, -0.16302290558815002,
      0.07484762370586395, 0.07809175550937653, 0.08252903819084167, 0.10306060314178467,
      -0.10848595201969147, 0.030504563823342323, -0.18221409618854523, -0.21642887592315674,
      -0.007983739487826824, 0.16418573260307312, -0.03381640464067459, -0.1558988243341446],
     [-0.05109900236129761, -0.1322348713874817, 0.005142009351402521, -0.021124351769685745,
      -0.05988200753927231, 0.024715861305594444, 0.12135253101587296, -0.16595454514026642,
      0.1903533786535263, -0.026462964713573456, -0.25962895154953003, -0.025535013526678085,
      -0.01680861786007881, 0.1876882165670395, 0.028079254552721977, -0.07529201358556747,
      -0.012557592242956161, 0.033320244401693344, 0.22212277352809906, -0.11079177260398865,
      -0.07464015483856201, -0.15211841464042664, 0.02900039032101631, 0.14111024141311646,
      0.05607546120882034, -0.004122861195355654, -0.1853824257850647, -0.055197954177856445,
      -0.06302545219659805, -0.08931265026330948, -0.15966646373271942, -0.008639269508421421],
     [-0.09618043154478073, 0.10411413013935089, -0.14392170310020447, -0.09883934259414673,
      0.16236813366413116, -0.062457725405693054, 0.023075252771377563, -0.07270409911870956,
      -0.17105838656425476, -0.09158042073249817, 0.06035328283905983, 0.02931812033057213,
      -0.12701576948165894, 0.16569919884204865, 0.020142285153269768, 0.15640269219875336,
      0.19948124885559082, -0.0867033302783966, 0.08040895313024521, 0.14945363998413086,
      0.009046092629432678, 0.025365715846419334, 0.02393678016960621, -0.13944128155708313,
      -0.05999694764614105, 0.15824715793132782, 0.06042372062802315, -0.0457051657140255,
      0.034209929406642914, 0.05239081755280495, -0.06546729058027267, 0.14072225987911224],
     [0.1770232766866684, 0.12591971457004547, -0.1296502947807312, -0.09809169918298721,
      -0.10942289233207703, 0.040237557142972946, 0.158657968044281, 0.17032204568386078,
      0.1357361376285553, 0.1130329966545105, -0.19256195425987244, -0.14027705788612366,
      0.14243382215499878, -0.11920485645532608, -0.052212636917829514, -0.10838788747787476,
      0.019414296373724937, -0.10242893546819687, -0.05226815119385719, -0.0686807781457901,
      -0.2503345012664795, -0.283356636762619, 0.011479157954454422, -0.0738697275519371,
      0.12621435523033142, 0.2534765899181366, 0.0617007315158844, -0.05035192891955376,
      0.055034488439559937, 0.017964091151952744, 0.15459391474723816, 0.016840647906064987],
     [0.16163037717342377, -0.05846188962459564, 0.031310852617025375, 0.02793974056839943,
      0.11949325352907181, 0.07455962151288986, -0.029582519084215164, 0.1975829154253006,
      -0.006431258283555508, 0.10656038671731949, 0.008807905949652195, -0.041044916957616806,
      -0.16525891423225403, -0.14820018410682678, -0.007433155085891485, 0.0562228299677372,
      -0.06072436645627022, 0.2079530954360962, -0.1381167620420456, 0.031233910471200943,
      0.11557646840810776, 0.1155954897403717, 0.09424827992916107, -0.017266498878598213,
      -0.10173851996660233, 0.002905628178268671, -0.10456369817256927, -0.01760854385793209,
      0.11727488785982132, 0.09447820484638214, -0.20901960134506226, 0.056040409952402115],
     [-0.13805189728736877, -0.07840751111507416, 0.06391932815313339, -0.1197560653090477,
      0.012679066509008408, 0.0570000596344471, 0.1032431572675705, 0.0027978087309747934,
      -0.14866995811462402, 0.022101234644651413, -0.06090132147073746, -0.055105239152908325,
      0.14171849191188812, -0.048861268907785416, 0.2152305543422699, 0.12201745063066483,
      -0.007498522754758596, -0.2588253915309906, -0.052380532026290894, -0.23001088201999664,
      -0.32220202684402466, 0.0155772864818573, -0.017316294834017754, -0.19329392910003662,
      -0.15788534283638, -0.1698453724384308, -0.0730026587843895, -0.07808913290500641,
      0.1679946780204773, -0.04137074947357178, -0.009575705975294113, -0.12618544697761536],
     [-0.12406738847494125, -0.13783089816570282, -0.019033528864383698, -0.063597172498703,
      -0.1625043749809265, 0.07430609315633774, -0.03383595123887062, 0.09314055740833282,
      0.09669952094554901, -0.004319740459322929, -0.08396867662668228, -0.013254680670797825,
      -0.05896160006523132, -0.07117331773042679, -0.15048405528068542, -0.06314067542552948,
      -0.16981330513954163, -0.041282400488853455, 0.040960364043712616, 0.1003817617893219,
      0.03059953637421131, -0.09916756302118301, -0.16537784039974213, 0.15126651525497437,
      -0.04031236469745636, 0.022681113332509995, 0.043670862913131714, 0.03014540672302246,
      0.16203059256076813, 0.027273692190647125, -0.1757589727640152, -0.14759992063045502],
     [0.11497911065816879, 0.24294985830783844, 0.07858339697122574, 0.1842946857213974,
      0.14401136338710785, 0.061071764677762985, 0.030064327642321587, 0.15164460241794586,
      0.01115769986063242, 0.15341347455978394, 0.17740197479724884, -0.06722956895828247,
      0.077586829662323, 0.02483076974749565, -0.09427785128355026, 0.08078895509243011,
      -0.31859734654426575, -0.07867664843797684, -0.018612127751111984, -0.10241328179836273,
      -0.2144414484500885, -0.1715552806854248, -0.1341283619403839, -0.2266119122505188,
      0.12744677066802979, -0.08804882317781448, -0.00295084691606462, 0.12826435267925262,
      -0.20134133100509644, 0.12480979412794113, 0.16835929453372955, 0.14242754876613617],
     [-0.05690687149763107, -0.09563875198364258, 0.042535893619060516, 0.022578516975045204,
      -0.1819894164800644, -0.007250309456139803, 0.10321741551160812, -0.1641923040151596,
      -0.11317024379968643, -0.15230464935302734, -0.1448201686143875, -0.10536422580480576,
      -0.04017427936196327, 0.032405558973550797, -0.03592152148485184, -0.07830850780010223,
      -0.04558352380990982, -0.12630769610404968, -0.01862291432917118, -0.11206408590078354,
      0.03259801119565964, 0.07776578515768051, 0.0013316823169589043, 0.07560531049966812,
      0.10779764503240585, 0.11733841150999069, 0.14494527876377106, 0.040951624512672424,
      0.004963659681379795, 0.05476876720786095, -0.049243781715631485, -0.04340701922774315],
     [-0.060164954513311386, -0.21859486401081085, -0.13525983691215515, 0.11171663552522659,
      -0.08050408214330673, -0.055052317678928375, 0.12556827068328857, -0.17981910705566406,
      0.11039630323648453, -0.11605893075466156, 0.07325831055641174, -0.025097336620092392,
      0.03424917906522751, 0.09553197026252747, -0.03105163387954235, -0.05802503228187561,
      -0.19024479389190674, -0.0861901119351387, -0.045166388154029846, 0.01563173718750477,
      -0.0916508138179779, 0.02065054513514042, -0.19491714239120483, -0.2245805859565735,
      -0.08958318829536438, 0.11509532481431961, 0.08830317854881287, 0.11707406491041183,
      0.06778311729431152, -0.11228149384260178, 0.038405388593673706, 0.09258142858743668],
     [0.06131141260266304, 0.052477046847343445, 0.06748361885547638, 0.057071052491664886,
      -0.02392534911632538, -0.1216873899102211, 0.08833812922239304, 0.08802367001771927,
      0.02060428075492382, 0.029414987191557884, -0.04328020662069321, 0.09493783116340637,
      -0.012566003948450089, 0.12241999804973602, -0.10940632969141006, -0.038604751229286194,
      -0.034781564027071, -0.012229698710143566, 0.25563377141952515, -0.1897706687450409,
      -0.027090495452284813, 0.11648166924715042, 0.07561838626861572, -0.04339906573295593,
      0.16240501403808594, -0.06203475221991539, -0.17798784375190735, -0.12230436503887177,
      -0.14567767083644867, 0.08341709524393082, 0.12837892770767212, -0.09091833233833313],
     [-0.13698112964630127, -0.10484393686056137, 0.11237847059965134, 0.026200007647275925,
      -0.03801316022872925, 0.13077126443386078, -0.16537922620773315, -0.004809108562767506,
      0.014913642778992653, 0.17066678404808044, -0.027110766619443893, -0.018878858536481857,
      -0.13691134750843048, -0.006813565269112587, -0.1349211484193802, -0.09279950708150864,
      0.043944403529167175, -0.10525184124708176, 0.1241210475564003, -0.010406635701656342,
      0.0010782527970150113, 0.1504313349723816, -0.18065309524536133, -0.001268782652914524,
      0.09052819013595581, -0.042150117456912994, -0.07363833487033844, -0.06192511320114136,
      -0.01890411414206028, -0.19864651560783386, -0.10249333083629608, -0.11869478225708008],
     [-0.006414095405489206, 0.09532670676708221, -0.012314530089497566, -0.07552868872880936,
      0.006789187900722027, 0.01076047308743, 0.09648682922124863, 0.2070746272802353,
      -0.11371362954378128, 0.09064290672540665, -0.002616140292957425, 0.10245397686958313,
      0.0035660522989928722, -0.16881722211837769, 0.014712400734424591, -0.008281798101961613,
      0.12177928537130356, 0.20503592491149902, -0.16343331336975098, 0.13566002249717712,
      0.02807074412703514, 0.16271568834781647, 0.03519086539745331, -0.07236543297767639,
      0.09646396338939667, -0.12981052696704865, -0.10478416085243225, -0.0985308513045311,
      0.0622149296104908, -0.09796427935361862, 0.0016570256557315588, -0.14600549638271332],
     [0.09094835817813873, -0.14670728147029877, 0.009830429218709469, 0.13686616718769073,
      0.1828368753194809, 0.010499268770217896, -0.12494503706693649, -0.11404868960380554,
      -0.044330231845378876, 0.02980623207986355, 0.16733910143375397, 0.02843283861875534,
      -0.10193909704685211, 0.028204193338751793, -0.08519217371940613, 0.12693114578723907,
      0.03474477306008339, 0.0009393999353051186, 0.05088433623313904, -0.10071422904729843,
      -0.019304033368825912, -0.10372202098369598, 0.11483060568571091, 0.03182929754257202,
      0.05793421342968941, 0.18641522526741028, -0.02096770517528057, -0.001799608115106821,
      -0.08054597675800323, 0.06163927540183067, -0.10388725250959396, 0.20496602356433868],
     [-0.041783519089221954, 0.017161890864372253, -0.08652954548597336, 0.18866819143295288,
      0.1944241225719452, -0.1156926080584526, -0.07572381943464279, -0.09322921186685562,
      0.19994746148586273, -0.05416174605488777, 0.21806012094020844, 0.09715613722801208,
      -0.113139308989048, -0.17370425164699554, 0.0052036261186003685, -0.13431496918201447,
      0.01674060896039009, -0.11651276051998138, -0.09075946360826492, -0.03974879905581474,
      -0.15815727412700653, -0.2660246789455414, -0.12293969839811325, -0.18107379972934723,
      0.12859444320201874, 0.2086215615272522, -0.2508077025413513, 0.15338128805160522,
      -0.2810991108417511, 0.2649780511856079, 0.010757816955447197, 0.21666128933429718],
     [0.07272132486104965, -0.1367102712392807, -0.16592322289943695, -0.081092968583107,
      -0.013744138181209564, 0.002121641067788005, 0.10317835956811905, 0.1564307063817978,
      0.035131435841321945, -0.12720564007759094, 0.012033878825604916, -0.11271993815898895,
      -0.0299833994358778, -0.1080373153090477, 0.17212823033332825, -0.06834419071674347,
      0.14510546624660492, 0.1569921374320984, 0.007695835083723068, 0.14385555684566498,
      -0.09125683456659317, 0.19224689900875092, -0.048003289848566055, -0.03640547767281532,
      -0.030047494918107986, -0.12726642191410065, -0.01842721365392208, -0.146231010556221,
      0.043963946402072906, 0.127542182803154, 0.10417193919420242, 0.029751429334282875],
     [-0.10062868148088455, -0.04796484112739563, -0.07047810405492783, -0.13069720566272736,
      0.041510164737701416, 0.12823428213596344, -0.1569952666759491, -0.137687548995018,
      0.1518855094909668, 0.07765313237905502, -0.08964965492486954, 0.1370973438024521,
      -0.11336318403482437, 0.15782327950000763, 0.023332001641392708, -0.04573080688714981,
      -0.18744607269763947, -0.019369499757885933, -0.1040143370628357, -0.030247097834944725,
      0.07603626698255539, -0.0796118751168251, -0.12378708273172379, -0.17671509087085724,
      0.14862236380577087, -0.23702365159988403, 0.036306630820035934, -0.2148270457983017,
      -0.04667928069829941, -0.047961048781871796, -0.03031850978732109, 0.15693560242652893],
     [0.03485574945807457, -0.0021170522086322308, -0.018662981688976288, -0.177606999874115,
      -0.22837992012500763, 0.14557483792304993, 0.07263445854187012, -0.0016651933547109365,
      0.04181617125868797, 0.04942520707845688, 0.0875590518116951, -0.15781866014003754,
      0.09581038355827332, 0.002409856067970395, 0.1007855236530304, 0.07570064067840576,
      0.1491594761610031, 0.1939248889684677, -0.1541956663131714, 0.06386752426624298,
      -0.05533187836408615, 0.04030157998204231, -0.041397880762815475, 0.10707888752222061,
      0.007387646473944187, -0.049610137939453125, 0.10522086918354034, -0.011375360190868378,
      -0.05998316407203674, -0.13229034841060638, 0.06818072497844696, 0.028231365606188774]], dtype=np.float32).T + np.asarray([-0.05500571429729462, 0.045074958354234695, 0.06104279309511185, -0.08730878680944443,
     -0.1714816391468048, 0.01555568166077137, 0.20539453625679016, -0.22983098030090332,
     -0.06145523861050606, 0.10472496598958969, 0.009253280237317085, 0.09398338198661804,
     -0.057467326521873474, 0.0885649248957634, -0.0988585576415062, -0.1053408607840538,
     -0.022354917600750923, 0.11707005649805069, -0.07511166483163834, 0.24442555010318756,
     -0.16827695071697235, 0.13504986464977264, -0.07063721865415573, -0.15720881521701813,
     -0.0710587278008461, -0.1829393059015274, 0.1044117659330368, -0.034571535885334015,
     -0.07124949991703033, 0.11577045172452927, 0.1361846774816513, -0.07992736250162125,
     -0.14006347954273224, 0.0520411916077137, 0.06568939238786697, -0.08951756358146667,
     0.0809834897518158, 0.11332564055919647, 0.18607820570468903, -0.10740354657173157,
     -0.004118406679481268, 0.09221439063549042, -0.1289530247449875, 0.03379516676068306,
     0.046663738787174225, -0.15155039727687836, -0.07394914329051971, -0.16203701496124268,
     -0.14390088617801666, -0.16892005503177643, 0.0434749573469162, 0.07011507451534271,
     0.09256269037723541, 0.0015721272211521864, 0.04646768793463707, -0.11572661250829697,
     0.02541261352598667, 0.15498104691505432, -0.02100636251270771, -0.05771161615848541,
     -0.11593052744865417, 0.004789744969457388, -0.08128653466701508, 0.11585111916065216], dtype=np.float32)
    parent_h1 = parent_h1 / (1.0 + np.exp(-np.clip(parent_h1, -30.0, 30.0)))
    parent_h2 = parent_h1 @ np.asarray([[0.00022938137408345938, 0.08628484606742859, 0.044015634804964066, -0.0041190339252352715,
      0.06225963681936264, -0.07181693613529205, 0.0037003513425588608, 0.06523928791284561,
      0.015375095419585705, 0.08302698284387589, -0.010569561272859573, -0.10125993192195892,
      0.06774681806564331, 0.12914273142814636, -0.027293797582387924, -0.053604189306497574,
      -0.08628494292497635, -0.014539513736963272, 0.07427959889173508, -0.02277030609548092,
      -0.059828080236911774, 0.055584657937288284, 0.12103445082902908, 0.0034301639534533024,
      -0.07162052392959595, 0.04066839441657066, -0.08015953004360199, 0.06264925003051758,
      0.0075194681994616985, 0.08354916423559189, -0.11826799809932709, -0.10713478922843933,
      0.01750316470861435, 0.062164727598428726, -0.08621224761009216, -0.06206221505999565,
      -0.09672357141971588, 0.006902133580297232, -0.0544213205575943, 0.0648801326751709,
      0.12871406972408295, -0.014046166092157364, 0.04031688719987869, 0.05252804979681969,
      0.02956085093319416, 0.11366679519414902, -0.0880291759967804, 0.08810318261384964,
      0.08103898912668228, 0.08799328655004501, 0.029902353882789612, 0.05255384370684624,
      0.06824783235788345, -0.06150676682591438, 0.011476855725049973, -0.028921693563461304,
      0.05980297923088074, -0.004132542293518782, -0.126911461353302, -0.05990435555577278,
      0.06134481728076935, 0.032930221408605576, 0.04815134033560753, -0.049649856984615326],
     [0.15352094173431396, -0.08371209353208542, -0.06104504317045212, -0.02892826683819294,
      0.0911136344075203, 0.08186222612857819, -0.09924816340208054, -0.09286762773990631,
      -0.00010751139052445069, 0.08583102375268936, -0.07023357599973679, -0.08167412877082825,
      -0.01754818856716156, -0.15555085241794586, -0.08272432535886765, -0.05646514147520065,
      0.03476462885737419, -0.10384581983089447, -0.0001485356769990176, 0.07129489630460739,
      0.029487723484635353, 0.005174519028514624, 0.007533077150583267, 0.10491719841957092,
      0.03457985445857048, -0.02119031734764576, 0.02768615633249283, 0.0651642382144928,
      0.09405240416526794, -0.01894320547580719, -0.050341565161943436, 0.11736632138490677,
      -0.09700822830200195, -0.09691761434078217, 0.055711884051561356, -0.023362871259450912,
      -0.007846557535231113, 0.0751972422003746, 0.09919050335884094, 0.08033125102519989,
      -0.029673386365175247, 0.09099726378917694, 0.011443295516073704, -0.12221892923116684,
      -0.06735283881425858, 0.11873529851436615, -0.057496123015880585, -0.10531315952539444,
      -0.04203716665506363, -0.03246793523430824, 0.10640900582075119, -0.0911259725689888,
      -0.03175564482808113, -0.06075876206159592, -0.07042668759822845, 0.0674438625574112,
      0.00014790130080655217, -0.040931764990091324, 0.08011626452207565, 0.07717875391244888,
      -0.001488414709456265, -0.07911466062068939, 0.0345846489071846, 0.0006614966550841928],
     [-0.12226320058107376, 0.08672848343849182, -0.08377989381551743, 0.11045166850090027,
      0.08417753875255585, 0.01890886202454567, 0.0407990925014019, -0.04142700135707855,
      -0.08927244693040848, -0.03257595747709274, -0.0958525687456131, 0.09669717401266098,
      0.06616047769784927, -0.0073028248734772205, 0.053505659103393555, 0.05185282230377197,
      -0.05146890506148338, 0.005309784319251776, -0.04894540086388588, -0.1332593858242035,
      0.017763780429959297, -0.10573193430900574, -0.009836787357926369, 0.030741071328520775,
      0.05756950378417969, 0.11010529100894928, 0.02541099674999714, 0.03453942760825157,
      -0.07824623584747314, -0.05102122947573662, 0.08971913903951645, 0.003291473723948002,
      0.1213589534163475, 0.10826145857572556, -0.04353519529104233, 0.02412521280348301,
      0.015698468312621117, 0.01585347391664982, 0.025389431044459343, 0.007364102639257908,
      0.1209503635764122, 0.10963620245456696, 0.0054251267574727535, -0.09778496623039246,
      -0.10081436485052109, 0.07641710340976715, -0.004353613592684269, 0.016461126506328583,
      0.05273580923676491, 0.07025665044784546, -0.06057414412498474, -0.039744310081005096,
      -0.11439890414476395, 0.15389014780521393, 0.09290634840726852, 0.11153331398963928,
      0.11024995893239975, 0.0352875292301178, 0.10108942538499832, 0.05596292018890381,
      -0.09651021659374237, -0.050433199852705, 0.1404443383216858, -0.10902804881334305],
     [-0.07725128531455994, 0.0437907911837101, -0.06263337284326553, -0.05440637841820717,
      0.031166845932602882, -0.11938703060150146, -0.047615572810173035, 0.1007201224565506,
      0.018353432416915894, -0.059842076152563095, -0.11780855804681778, 0.09874337166547775,
      -0.040577299892902374, 0.009449473582208157, 0.11191529035568237, 0.09514535218477249,
      -0.10956814885139465, 0.09609922766685486, 0.11349710077047348, -0.11636655777692795,
      -0.05422531068325043, 0.05680230259895325, 0.03869815170764923, 0.06768786162137985,
      0.09905491769313812, 0.0049811494536697865, 0.09725528210401535, -0.1056801900267601,
      0.10960860550403595, -0.07782687246799469, -0.03390081226825714, 0.08825572580099106,
      -0.12226767092943192, -0.11954354494810104, 0.011378807947039604, 0.029133502393960953,
      0.08161059021949768, 0.07014185190200806, -0.058496423065662384, -0.014655040577054024,
      0.07277150452136993, 0.036302365362644196, 0.08103335648775101, 0.10541954636573792,
      -0.015110837295651436, 0.03246393799781799, 0.08589878678321838, -0.04281705617904663,
      -0.04429858177900314, 0.059562359005212784, -0.08914180845022202, 0.11077608168125153,
      0.027850700542330742, 0.03903064876794815, -0.07135631144046783, 0.10957565903663635,
      -0.12975458800792694, -0.07925010472536087, -0.08281233161687851, -0.09676559269428253,
      -0.10569380223751068, 0.008381938561797142, 0.011846975423395634, 0.0623600110411644],
     [-0.10028600692749023, -0.10835011303424835, -0.06167911738157272, -0.0145683279260993,
      -0.10160592943429947, 0.0301054734736681, -0.04218275472521782, 0.08240197598934174,
      0.046336907893419266, -0.09897062182426453, -0.01929072104394436, -0.027637561783194542,
      0.03955300524830818, 0.11274714767932892, -0.0655035600066185, 0.011098342016339302,
      -0.0734977275133133, 0.05858053267002106, 0.03219064325094223, 0.020149653777480125,
      -0.08628128468990326, 0.027093779295682907, 0.006119014695286751, 0.0809403508901596,
      0.02008357085287571, 0.0659397691488266, -0.05663907527923584, -0.12971706688404083,
      0.027919640764594078, -0.03918374329805374, 0.10435806959867477, 0.11529215425252914,
      -0.10305484384298325, -0.1009017676115036, 0.05400029569864273, 0.061940595507621765,
      -0.10090195387601852, 0.104765884578228, 0.04108351469039917, 0.1112937480211258,
      0.0952300950884819, -0.06619277596473694, 0.05625297129154205, -0.09756195545196533,
      -0.04577600583434105, 0.06456019729375839, -0.02756788767874241, 0.030382532626390457,
      0.09221572428941727, 0.08966146409511566, -0.11039712280035019, -0.08009403198957443,
      0.06526193022727966, 0.09853321313858032, 0.04714426398277283, -0.09540975838899612,
      -0.008757886476814747, -0.05290265381336212, 0.10305078327655792, 0.1094469204545021,
      0.019892603158950806, -0.07927894592285156, 0.11684005707502365, 0.05555181950330734],
     [0.09657049179077148, -0.08994273096323013, -0.1015956923365593, 0.023481333628296852,
      0.017173342406749725, -0.11304232478141785, -0.17533738911151886, 0.08634226024150848,
      -0.10504170507192612, -0.07651074975728989, 0.0071548460982739925, -0.07866734266281128,
      -0.08633488416671753, 0.1915377974510193, 0.12010600417852402, -0.0652998611330986,
      -0.09954831749200821, -0.061969298869371414, -0.014735988341271877, -0.08166110515594482,
      0.019871119409799576, 0.008008447475731373, 0.0893162414431572, -0.10422106087207794,
      0.06055588275194168, -0.10321418195962906, -0.02087792009115219, -0.017984190955758095,
      0.08838700503110886, 0.06276588141918182, -0.015758266672492027, -0.01664077676832676,
      0.03136434778571129, -0.13656160235404968, -0.08547555655241013, 0.04732608422636986,
      -0.004430442117154598, 0.022442521527409554, 0.013135834597051144, 0.0144354784861207,
      0.1936924159526825, -0.10945884138345718, -0.07395455986261368, 0.04808051511645317,
      -0.06019403040409088, 0.08507496863603592, 0.1673717051744461, 0.13716116547584534,
      -0.1132848933339119, 0.02497926726937294, -0.13162052631378174, 0.05745696648955345,
      0.0069222101010382175, 0.23798470199108124, -0.09236588329076767, -0.02558794803917408,
      0.08949101716279984, 0.07248284667730331, -0.15329498052597046, -0.11802474409341812,
      0.08808762580156326, -0.04523557797074318, -0.08328936249017715, -0.1065506637096405],
     [0.10645601153373718, 0.013037506490945816, 0.06449925899505615, -0.09793632477521896,
      -0.10379970073699951, 0.10582586377859116, -0.12473005801439285, 0.07642007619142532,
      -0.037518590688705444, -0.10306813567876816, 0.1098192110657692, -0.03928261250257492,
      0.028453093022108078, 0.01855389215052128, 0.06824325025081635, 0.012709292583167553,
      -0.028989223763346672, -0.09756916016340256, -0.033953916281461716, 0.060218825936317444,
      0.09762323647737503, 0.03199107572436333, -0.07547460496425629, 0.11605287343263626,
      -0.11018571257591248, -0.10714638978242874, -0.08961793035268784, -0.02711804397404194,
      0.0362984836101532, 0.002837322885170579, -0.06946662068367004, 0.1053260862827301,
      -0.04278988763689995, 0.09420807659626007, 0.015424770303070545, 0.08013775944709778,
      -0.11689453572034836, 0.029985226690769196, -0.044524967670440674, -0.08905360102653503,
      -0.05138665810227394, -0.13939258456230164, 0.01698889210820198, -0.11725745350122452,
      -0.015861598774790764, 0.03051106072962284, 0.15306101739406586, 0.10633186995983124,
      0.017231373116374016, -0.10551594197750092, -0.148306205868721, -0.02892492711544037,
      -0.018100067973136902, 0.16255927085876465, 0.03209943696856499, -0.07953499257564545,
      -0.09406012296676636, -0.11482640355825424, 0.049699634313583374, 0.03452511504292488,
      -0.021518675610423088, -0.06505414098501205, -0.10623384267091751, 0.09966234117746353],
     [-0.1511337012052536, 0.07877518981695175, 0.01620703935623169, 0.08353301137685776,
      0.01173557061702013, -0.003170632990077138, 0.05395462363958359, -0.2276524156332016,
      0.07692514359951019, 0.05637190863490105, -0.11174990981817245, 0.041152480989694595,
      0.14663858711719513, -0.10661330819129944, 0.04253920167684555, 0.07956864684820175,
      -0.1089172512292862, -0.039656396955251694, 0.12293468415737152, -0.029444720596075058,
      0.012183903716504574, 0.11440206319093704, 0.02642706222832203, 0.09228534251451492,
      0.10717251896858215, -0.11266312748193741, -0.0913146585226059, -0.11276530474424362,
      -0.026363076642155647, 0.018161453306674957, -0.01880054362118244, -0.096549391746521,
      -0.05417730286717415, 0.040645573288202286, 0.030633267015218735, -0.11029823869466782,
      0.04840092360973358, 0.11007170379161835, -0.0652911439538002, -0.12095809727907181,
      -0.05399958789348602, -0.07890523225069046, -0.10789016634225845, -0.16578936576843262,
      0.017793752253055573, -0.08181080967187881, 0.09966248273849487, 0.0715169832110405,
      0.049380362033843994, -0.07156535983085632, -0.005894818343222141, 0.022332558408379555,
      0.0028034523129463196, -0.13167829811573029, 0.03390125557780266, -0.06300478428602219,
      0.09707946330308914, 0.11016228049993515, -0.08738966286182404, 0.11293857544660568,
      0.03017997182905674, -0.0380605049431324, 0.005725149065256119, -0.07007735222578049],
     [0.01332046091556549, -0.018677078187465668, -0.11285647749900818, 0.047560665756464005,
      7.836276927264407e-05, -0.00782837439328432, 0.006193898618221283, 0.11761701107025146,
      -0.0013947354163974524, -0.032061867415905, 0.02305254526436329, -0.07642324268817902,
      0.04583695903420448, 0.1730610728263855, -0.05811583250761032, -0.025131141766905785,
      0.03612156957387924, 0.10406642407178879, 0.07816486805677414, -0.10762885212898254,
      0.1377088874578476, 0.09838725626468658, -0.041431158781051636, -0.07574708014726639,
      -0.09331002086400986, 0.03956057131290436, -0.12227534502744675, -0.06609292328357697,
      -0.04809181019663811, 0.05753449350595474, -0.021900996565818787, 0.03486698493361473,
      -0.09781937301158905, 0.013231259770691395, -0.09408333897590637, 0.05056343972682953,
      -0.08688913285732269, -0.11523497849702835, -0.0832083523273468, -0.10621490329504013,
      0.13298198580741882, -0.053018368780612946, -0.013019738718867302, -0.030527915805578232,
      -0.0836905762553215, 0.09338991343975067, 0.03846902400255203, 0.08586259186267853,
      -0.00926965195685625, 0.02973509021103382, -0.1370134949684143, -0.06638749688863754,
      0.060322098433971405, 0.06182904914021492, 0.12531223893165588, 0.019936516880989075,
      0.021217068657279015, -0.023660488426685333, -0.04674943536520004, -0.06252173334360123,
      0.022291740402579308, -0.03645247593522072, 0.07799184322357178, -0.012102430686354637],
     [-0.03277578577399254, -0.07816411554813385, 0.019803961738944054, -0.041438695043325424,
      0.10706023126840591, -0.15761463344097137, -0.11185874044895172, 0.15281836688518524,
      0.09861743450164795, -0.014353347942233086, 0.00452773692086339, -0.12237909436225891,
      -0.047700658440589905, -0.11451510339975357, -0.13236741721630096, 0.05951927974820137,
      0.010037780739367008, 0.09309402853250504, 0.04781932011246681, 0.10750341415405273,
      -0.11596750468015671, 0.07947437465190887, 0.037166062742471695, -0.06626028567552567,
      -0.08851480484008789, -0.09440763294696808, -0.011735005304217339, -0.0662226676940918,
      -0.05423134192824364, -0.08898945152759552, -0.060667555779218674, 0.06198236346244812,
      0.04902595281600952, -0.0960737094283104, 0.058836471289396286, 0.008074352517724037,
      0.09411939233541489, -0.027152791619300842, -0.08841845393180847, -0.02269992046058178,
      0.0344446562230587, 0.10048181563615799, 0.06792595982551575, -0.08058404922485352,
      -0.04713737964630127, -0.127472922205925, -0.06737366318702698, -0.047266580164432526,
      -0.062078554183244705, 0.14840243756771088, 0.05794132500886917, 0.01833520270884037,
      0.06572264432907104, 0.0006896802806295455, -0.10175314545631409, -0.04694098234176636,
      0.08079853653907776, -0.028243036940693855, -0.10620396584272385, -0.14623869955539703,
      -0.09764344990253448, -0.031804151833057404, -0.08755242079496384, -0.061155643314123154],
     [0.08540674299001694, -0.08013847470283508, -0.015083509497344494, -0.05986849218606949,
      -0.03254426643252373, -0.0037532434798777103, -0.03993889316916466, -0.10151829570531845,
      -0.05467091500759125, -0.1019061952829361, -0.02647308073937893, -0.060468241572380066,
      -0.09465814381837845, 0.06083715334534645, 0.042656928300857544, -0.07015135139226913,
      0.023063015192747116, -0.07024435698986053, -0.002364453626796603, -0.08234301209449768,
      0.03921351581811905, 0.060767557471990585, 0.025834375992417336, -0.11655562371015549,
      0.055215056985616684, 0.06452678143978119, -0.10641441494226456, -0.08612526953220367,
      -0.14208751916885376, -0.009315752424299717, 0.0063271778635680676, -0.054056283086538315,
      -0.07232533395290375, 0.08825691044330597, 0.07428429275751114, 0.08891323953866959,
      -0.11306443810462952, -0.010259793139994144, 0.08297599107027054, -0.03370029479265213,
      0.08057094365358353, -0.04427270218729973, -0.02198181301355362, -0.05389321967959404,
      -0.10897938907146454, 0.043382368981838226, 0.06886648386716843, 0.0005891984910704195,
      -0.04991676285862923, -0.12161927670240402, -0.08034706115722656, -0.10346700251102448,
      -0.008153761737048626, -0.011166629381477833, 0.006820911541581154, -0.06986887753009796,
      -0.04606267809867859, 0.07346358150243759, -0.11283714324235916, 0.00031178034259937704,
      -0.06874467432498932, -0.009204817935824394, 0.1161026656627655, 0.00355159561149776],
     [0.07315092533826828, -0.049079131335020065, -0.11110654473304749, -0.010772091336548328,
      -0.12234756350517273, 0.04209291934967041, 0.036564137786626816, 0.0555742010474205,
      0.04238560050725937, 0.02763950265944004, -0.03195733577013016, 0.09510406106710434,
      0.05735711008310318, 0.014926276169717312, 0.029693692922592163, -0.024336714297533035,
      -0.001493019168265164, 0.06664109230041504, 0.06415939331054688, -0.04675934836268425,
      -0.0307297445833683, -0.0831121951341629, 0.1091383770108223, -0.04046978801488876,
      0.07196732610464096, -0.02321714721620083, 0.0920303612947464, -0.01956939324736595,
      -0.007132674567401409, 0.02853051759302616, 0.013941353186964989, -0.05911508947610855,
      0.01694122701883316, 0.06999190151691437, -0.02917930856347084, 0.07568633556365967,
      -0.08299631625413895, -0.10095299780368805, 0.01730453409254551, 0.06075497344136238,
      0.08482164144515991, -0.03968820348381996, 0.08454398810863495, 0.044361233711242676,
      0.07108812034130096, 0.021460309624671936, -0.00771464966237545, 0.05868600681424141,
      -0.10192342102527618, 0.05955473706126213, 0.07835617661476135, -0.13130003213882446,
      0.002638049889355898, 0.11513733118772507, -0.0841730535030365, 0.02123160846531391,
      0.05742915719747543, 0.0170283205807209, -0.12678977847099304, -0.04864097386598587,
      0.02011856809258461, 0.016674788668751717, 0.02532593533396721, -0.07527150213718414],
     [-0.12398424744606018, -0.06921609491109848, -0.04166579991579056, -0.09900559484958649,
      -0.10408401489257812, 0.06298594921827316, -0.11942341923713684, 0.007318870630115271,
      -0.057692550122737885, 0.01304165180772543, -0.08814907819032669, 0.003962163347750902,
      0.018933700397610664, -0.06894777715206146, -0.13027413189411163, 0.05089620500802994,
      0.12264835089445114, -0.10764631628990173, -0.00013400550233200192, -0.16080506145954132,
      0.008642171509563923, -0.09595762193202972, 0.0045207226648926735, 0.038580868393182755,
      -0.04336375743150711, -0.005172327160835266, 0.00668534729629755, 0.06607065349817276,
      0.017669977620244026, -0.00022995052859187126, -0.07609772682189941, -0.12140360474586487,
      0.0848655104637146, -0.07410458475351334, 0.09363109618425369, -0.00835365243256092,
      0.053113095462322235, -0.040762513875961304, -0.041977446526288986, -0.011391207575798035,
      -0.01595708355307579, 0.04246583580970764, -0.043941229581832886, 0.039901819080114365,
      0.07787429541349411, -0.01082582026720047, 0.14342081546783447, 0.007761530112475157,
      -0.08744029700756073, -0.03029770217835903, -0.10280993580818176, 0.03623032569885254,
      0.08170883357524872, -0.059414394199848175, -0.0710960105061531, 0.018405083566904068,
      0.08116590231657028, 0.011589213274419308, -0.14192698895931244, -0.10865848511457443,
      -0.10448846220970154, 0.007257523015141487, 0.0518605075776577, 0.044658105820417404],
     [0.03868316113948822, 0.0684342235326767, -0.12237899750471115, 0.04572385922074318,
      -0.09618224203586578, -0.15930049121379852, 0.1150948703289032, 0.0498051717877388,
      0.0782301053404808, 0.08093009144067764, -0.010363591834902763, 0.08230382204055786,
      0.03248102590441704, 0.046773411333560944, -0.017834050580859184, 0.10686700791120529,
      -0.0725090280175209, -0.050155699253082275, 0.055133428424596786, -0.037894438952207565,
      -0.09754622727632523, 0.059321850538253784, 0.045088738203048706, 0.11178755015134811,
      -0.013238021172583103, 0.0509360134601593, -0.04886611923575401, -0.08446405082941055,
      0.0359683483839035, -0.10913846641778946, -0.12072934955358505, -0.08013024926185608,
      -0.08122184127569199, 0.05595393106341362, 0.10230227559804916, -0.057305674999952316,
      -0.021027108654379845, 0.028025908395648003, -0.07075731456279755, -0.0394400879740715,
      0.08884105086326599, -0.00011962920689256862, 0.054449886083602905, 0.04772607982158661,
      -0.06933341175317764, 0.06215324252843857, 0.08564780652523041, 0.09798379987478256,
      0.03383597731590271, 0.06785812973976135, -0.049605730921030045, -0.04224102571606636,
      -0.060258474200963974, -0.1253562867641449, -0.08633962273597717, -0.04304327815771103,
      -0.026157110929489136, -0.09337552636861801, -0.08348173648118973, -0.06894921511411667,
      -0.07090181857347488, -0.04069036617875099, -0.06653651595115662, 0.020150339230895042],
     [0.006620504893362522, -0.10325231403112411, -0.05059807375073433, -0.04499242454767227,
      -0.14398592710494995, -0.0734562873840332, -0.11066142469644547, -0.1354575902223587,
      0.031745076179504395, 0.022460458800196648, 0.01438914705067873, -0.0796637088060379,
      0.006583899725228548, -0.01857507787644863, 0.11118056625127792, -0.06774865835905075,
      -0.08019278943538666, 0.10108678787946701, 0.1274777203798294, 0.06782524287700653,
      0.062354590743780136, -0.06228448078036308, 0.0883290246129036, 0.1116558164358139,
      0.11620219796895981, 0.040346551686525345, 0.09005124121904373, -0.009128779172897339,
      -0.04668060690164566, 0.06413890421390533, -0.061653364449739456, -0.11640245467424393,
      0.09800021350383759, 0.11272309720516205, -0.09355628490447998, 0.07479929178953171,
      0.009165089577436447, 0.029599854722619057, -0.09511831402778625, 0.11645618826150894,
      0.09951786696910858, 0.06806183606386185, -0.03181096166372299, -0.06198414787650108,
      -0.1067844033241272, -0.0929296538233757, -0.022222599014639854, 0.07684937119483948,
      -0.018485086038708687, 0.0314166322350502, -0.14305061101913452, -0.036088600754737854,
      -0.07247021049261093, -0.10401545464992523, 0.12140984833240509, -0.06864804774522781,
      -0.07462046295404434, -0.060392044484615326, 0.078024722635746, -0.11357712000608444,
      0.03974520415067673, 0.06656942516565323, 0.052045516669750214, -0.10782606154680252],
     [0.04048262536525726, 0.09382984787225723, 0.0010104912798851728, 0.09810791909694672,
      -0.015145091339945793, 0.0411534458398819, 0.0844699963927269, 0.02772696316242218,
      0.04899737611413002, -0.10858429968357086, -0.132225900888443, 0.07652361690998077,
      -0.027331586927175522, 0.13598018884658813, -0.01800038106739521, -0.07364829629659653,
      -0.052788909524679184, -0.0023011118173599243, 0.010501696728169918, 0.009564759209752083,
      -0.052586957812309265, -0.0047017899341881275, -0.025013327598571777, -0.07526206970214844,
      -0.008702079765498638, -0.03065810166299343, -0.12434256821870804, -0.03272715210914612,
      0.03203269839286804, 0.03000451624393463, -0.032319843769073486, 0.07071031630039215,
      0.10864195227622986, 0.07376633584499359, -0.09799767285585403, 0.08205246180295944,
      0.02963564731180668, -0.06546852737665176, -0.056124795228242874, 0.07724694907665253,
      0.02319198101758957, 0.08323951810598373, 0.006057229824364185, 0.09632948040962219,
      0.04563915356993675, 0.09061597287654877, -0.06934609264135361, -0.09226547181606293,
      -0.10672692954540253, 0.033838216215372086, -0.05274606868624687, -0.062126047909259796,
      -0.09228868037462234, 0.11672817170619965, 0.053682222962379456, 0.006148492451757193,
      -0.025747142732143402, 0.09557084739208221, -0.09522128850221634, -0.04331674054265022,
      0.009910334832966328, -0.10443063825368881, -0.06023932248353958, 0.022992238402366638],
     [0.08897639065980911, 0.009665258228778839, 0.008995172567665577, 0.08866763114929199,
      -0.0817294716835022, 0.0883830189704895, -0.016902629286050797, -0.005846539977937937,
      0.1248229444026947, 0.11192986369132996, -0.08652371168136597, -0.06602607667446136,
      -0.08567289263010025, 0.0779557004570961, -0.05164065957069397, 0.015427814796566963,
      0.06807474792003632, -0.10400141030550003, 0.07023975253105164, 0.015185829252004623,
      0.019092245027422905, 0.038079358637332916, -0.138426274061203, -0.06884582340717316,
      -0.05793837830424309, -0.09686768054962158, -0.08375809341669083, 0.0804067999124527,
      0.06957653909921646, 0.05726766213774681, 0.10890565812587738, 0.11164291948080063,
      -0.07106386125087738, 0.06476716697216034, -0.0937344878911972, -0.04449641332030296,
      -0.08121352642774582, 0.01460480596870184, -0.020777447149157524, -0.01826055534183979,
      0.07983344048261642, 0.0372316800057888, 0.02064402401447296, -0.04464200139045715,
      0.02083410508930683, 0.08231809735298157, 0.027674388140439987, 0.099895179271698,
      -0.012114470824599266, 0.12846843898296356, 0.025044035166502, 0.0005732009303756058,
      0.02785196527838707, 0.0123093631118536, 0.05196646973490715, 0.0016652693739160895,
      -0.02106502465903759, -0.04333832859992981, 0.03434295952320099, -0.041235145181417465,
      -0.04927787184715271, -0.026406385004520416, -0.01871948316693306, -0.10646450519561768],
     [-0.03231501951813698, 0.03009112738072872, 0.01856972835958004, 0.08213875442743301,
      -0.09890829026699066, 0.04758258908987045, -0.10100077092647552, -0.08368263393640518,
      0.031115896999835968, -0.020001916214823723, -0.07118958234786987, 0.036098744720220566,
      0.02915206551551819, -0.04014954715967178, 0.09267247468233109, -0.04846488684415817,
      -0.020691851153969765, -0.048262253403663635, -0.06253179162740707, -0.10388217121362686,
      -0.07904161512851715, 0.03308508172631264, -0.06087011098861694, 0.0181569904088974,
      -0.0010870246915146708, 0.06226595863699913, -0.10623642057180405, -0.08448437601327896,
      0.003368915058672428, 0.10126696527004242, 0.0850134789943695, 0.008549577556550503,
      -0.011534790508449078, 0.0292676892131567, 0.05855877324938774, 0.10471533238887787,
      -0.1194310337305069, 0.06791344285011292, 0.04404400289058685, -0.06942040473222733,
      -0.033572014421224594, 0.10964985191822052, 0.06387479603290558, 0.11261340975761414,
      0.06839697808027267, 0.029236163944005966, -0.09211158007383347, -0.034914057701826096,
      -0.02974594756960869, 0.10580755770206451, -0.1232794001698494, -0.10601557046175003,
      0.06280945986509323, 0.06958672404289246, 0.11693930625915527, 0.0264962799847126,
      -0.017222516238689423, 0.08545652776956558, 0.03768681362271309, 0.015354364179074764,
      0.01151592843234539, 0.07225437462329865, -0.10987010598182678, 0.06302986294031143],
     [0.10681029409170151, 0.07333867251873016, -0.04566420987248421, -0.043987397104501724,
      -0.10296828299760818, 0.15752586722373962, -0.02058541588485241, 0.014311989769339561,
      0.06200088560581207, -0.07415313273668289, -0.0003771810152102262, 0.10266125202178955,
      0.09019212424755096, 0.042825303971767426, -0.057822804898023605, 0.04577285423874855,
      -0.04334481060504913, -0.10068988800048828, 0.05405285209417343, -0.12540796399116516,
      0.055388931185007095, -0.0074690249748528, -0.137608602643013, -0.07174024730920792,
      0.2016131728887558, -0.06488215178251266, -0.03685896098613739, -0.031228024512529373,
      -0.025483377277851105, -0.12937584519386292, -0.1248411312699318, -0.0883440226316452,
      0.009610313922166824, -0.05137542262673378, 0.004462511744350195, -0.020069319754838943,
      -0.13609567284584045, -0.0018261413788422942, -0.11258995532989502, -0.011566827073693275,
      0.07526788115501404, 0.10113423317670822, -0.08425074815750122, 0.026394840329885483,
      0.022502463310956955, 0.027506178244948387, -0.14773619174957275, 0.028547704219818115,
      0.0714796856045723, -0.12888573110103607, 0.03673342242836952, -0.08174718916416168,
      -0.10514969378709793, 0.06324440985918045, 0.04403424635529518, 0.06709687411785126,
      -0.0858684778213501, 0.08591017127037048, 0.10996700823307037, 0.0012493738904595375,
      0.1426251381635666, 0.018146153539419174, -0.07142480462789536, -0.051220253109931946],
     [-0.08757565170526505, 0.10924448072910309, 0.04005582258105278, -0.008220821619033813,
      0.11162129044532776, 0.0016603865660727024, 0.04554487392306328, -0.1089606061577797,
      -0.008585815317928791, -0.08008827269077301, -0.031227339059114456, -0.0671885535120964,
      -0.0674295499920845, -0.06792492419481277, 0.0755256861448288, 0.025933874770998955,
      0.10368740558624268, -0.04567456245422363, 0.023077167570590973, -0.02332846075296402,
      0.05310298129916191, 0.10635814070701599, -0.06058090925216675, 0.059898801147937775,
      0.07950735092163086, 0.11267153173685074, 0.0921969935297966, 0.06615792214870453,
      0.04032564535737038, 0.007061675190925598, -0.07507304847240448, -0.0014360557543113828,
      0.022941799834370613, 0.07719183713197708, -0.1007048487663269, -0.023634621873497963,
      -0.036738794296979904, -0.0019643139094114304, 0.08625784516334534, -0.054212894290685654,
      -0.05143636837601662, 0.0749211236834526, 0.12125423550605774, 0.05399148538708687,
      -0.09137671440839767, -0.01377121638506651, 0.03215394541621208, -0.026352589949965477,
      -0.024832161143422127, 0.10782880336046219, 0.04599221050739288, 0.01231856644153595,
      0.013361907564103603, 0.05143604055047035, 0.01836569607257843, 0.1138109341263771,
      -0.035566218197345734, 0.10239619761705399, -0.031101902946829796, -0.0684099793434143,
      -0.0006906561320647597, 0.08048471063375473, 0.05125884711742401, 0.10321720689535141],
     [0.027443163096904755, 0.014845440164208412, -0.09480579942464828, -0.06823518127202988,
      0.0005555005045607686, 0.12792061269283295, -0.041119109839200974, -0.07114710658788681,
      -0.07437403500080109, -0.09710238873958588, -0.12604300677776337, -0.09954757988452911,
      -0.08463979512453079, 0.029216047376394272, 0.044682253152132034, 0.09481848776340485,
      0.0750947892665863, 0.008964442647993565, -0.012654608115553856, -0.06689836829900742,
      0.07569395750761032, 0.03377146273851395, -0.11823480576276779, -0.08997632563114166,
      0.07055246084928513, 0.00937417708337307, 0.07069757580757141, -0.005761997774243355,
      -0.030944176018238068, 0.09387855976819992, 0.031397994607686996, -0.018933724611997604,
      -0.10062163323163986, -0.12176260352134705, -0.03405827283859253, 0.05277806520462036,
      0.06487391889095306, -0.09194473922252655, -0.07972417026758194, -0.09350429475307465,
      0.13996388018131256, 0.06746984273195267, 0.019105231389403343, -0.07820511609315872,
      0.003948769066482782, 0.049414075911045074, -0.12293990701436996, -0.0768914446234703,
      0.030994020402431488, 0.02046332135796547, 0.006277753040194511, 0.05920875445008278,
      -0.07841955125331879, -0.08874855190515518, 0.030842777341604233, 0.08688955008983612,
      0.053156036883592606, -0.02147674933075905, 0.031111104413866997, 0.05110841989517212,
      0.12864014506340027, 0.03108086623251438, -0.007954010739922523, -0.06741341203451157],
     [-0.0634358674287796, 0.10596146434545517, 0.10536670684814453, 0.08933012187480927,
      -0.004739733878523111, 0.003761912463232875, -0.114943727850914, -0.08349095284938812,
      -0.0769282653927803, -0.07640337944030762, -0.04923159256577492, 0.06442181020975113,
      -0.054911497980356216, -0.06451001018285751, -0.08382859826087952, -0.05811702087521553,
      0.08782480657100677, -0.06324468553066254, 0.05600961670279503, 0.07213456928730011,
      -0.10956542938947678, 0.04944193735718727, 0.09980840981006622, -0.10811392217874527,
      -0.03045218251645565, -0.060357458889484406, -0.12125226855278015, -0.06762407720088959,
      -0.05877946317195892, -0.02892228215932846, -0.0761154443025589, 0.027313726022839546,
      -0.08231049031019211, -0.0536317378282547, 0.10163556784391403, 0.08244145661592484,
      -0.03596903383731842, -0.013536488637328148, -0.04913052171468735, 0.018217969685792923,
      -0.032501645386219025, 0.020285189151763916, 0.0426030270755291, 0.08243393898010254,
      -0.07328421622514725, 0.10337530821561813, 0.06919963657855988, 0.054219409823417664,
      0.01020775269716978, 0.006392607931047678, 0.045996181666851044, -0.04204166308045387,
      -0.009817465208470821, 0.10064291208982468, 0.03701700270175934, 0.013033888302743435,
      0.12511762976646423, -0.09301543235778809, -0.04990892484784126, -0.01108738873153925,
      0.06622219830751419, 0.09289852529764175, 0.12007834017276764, 0.09271442145109177],
     [0.0953950434923172, 0.10695138573646545, -0.007261363323777914, 0.06294649839401245,
      0.056011613458395004, -0.0982310026884079, -0.09326872229576111, 0.04298226907849312,
      0.04296927526593208, -0.012341506779193878, 0.04510176554322243, -0.1791115701198578,
      0.12884171307086945, 0.0432538241147995, -0.1003546491265297, 0.11300047487020493,
      0.07754350453615189, -0.04838717356324196, -0.015133361332118511, -0.2008643001317978,
      -0.0723956972360611, -0.10428599268198013, -0.0748744085431099, 0.09246069937944412,
      -0.12156932055950165, 0.06692909449338913, -0.061570413410663605, -0.021545061841607094,
      -0.05658169835805893, -0.00042491930071264505, -0.09427276253700256, -0.0500793531537056,
      0.008270177990198135, -0.06212932616472244, -0.04414984956383705, 0.09933338314294815,
      0.02487117610871792, -0.1583043336868286, -0.014528946951031685, -0.14095927774906158,
      0.2309037446975708, -0.013002143241465092, -0.005298240575939417, 0.16487783193588257,
      0.05947710946202278, -0.07301073521375656, 0.09772515296936035, 0.15269283950328827,
      0.09136400371789932, 0.12597878277301788, -0.09365943819284439, -0.0208191629499197,
      -0.15276861190795898, 0.23155398666858673, 0.07027982175350189, 0.0020814966410398483,
      0.10622181743383408, 0.11257963627576828, -0.10795416682958603, 0.09521707147359848,
      0.15286462008953094, -0.07514193654060364, -0.05507039278745651, -0.13851015269756317],
     [-0.06595698744058609, -0.08304055780172348, -0.004061573185026646, -0.050428152084350586,
      -0.059046801179647446, 0.11015103757381439, -0.0721932053565979, -0.09407710283994675,
      -0.00988082680851221, 0.09334807842969894, 0.04714551195502281, -0.08247731626033783,
      -0.13281649351119995, 0.025798790156841278, -0.08449859917163849, 0.08475229144096375,
      -0.08693265169858932, 0.046382684260606766, -0.06328395009040833, -0.004576498176902533,
      -0.11978539079427719, 0.08595007658004761, 0.09408022463321686, -0.09395264834165573,
      0.10171563178300858, 0.11154772341251373, -0.0589669905602932, 0.056541021913290024,
      -0.12165457755327225, 0.09526114165782928, 0.06479854881763458, 0.021771781146526337,
      0.09134580940008163, 0.0880892276763916, -0.059707753360271454, -0.035633206367492676,
      0.06196827068924904, -0.058400750160217285, -0.062009893357753754, 0.001951022888533771,
      0.09327016025781631, 0.027013348415493965, 0.11477974057197571, 0.12261036038398743,
      -0.052205875515937805, 0.0991944670677185, -0.06260562688112259, -0.09181247651576996,
      -0.12703891098499298, 0.052560463547706604, -0.035760924220085144, 0.07805059850215912,
      -0.0854036808013916, -0.14149782061576843, -0.08239633589982986, 0.07168585062026978,
      0.12148118019104004, -0.03304168954491615, -0.07909273356199265, 0.05995490401983261,
      0.13434597849845886, -0.0016426806105300784, -0.0037602807860821486, -0.11019942909479141],
     [-0.07456371188163757, 0.07895813882350922, -0.052573878318071365, 0.01730423979461193,
      0.09610305726528168, 0.16112308204174042, -0.06640733033418655, -0.11281561851501465,
      0.019735082983970642, -0.06541283428668976, 0.06949418783187866, 0.02935878187417984,
      -0.14354149997234344, -0.1377798169851303, 0.09397455304861069, -0.09908880293369293,
      -0.09036609530448914, -0.1090988889336586, 0.06070776283740997, -0.08529312908649445,
      -0.13184209167957306, 0.007317982614040375, -0.09501350671052933, -0.06496486812829971,
      0.05033574625849724, 0.06477637588977814, 0.1262097954750061, 0.13425759971141815,
      0.03395809233188629, 0.025464992970228195, 0.06093841418623924, 0.05031893774867058,
      -0.07198264449834824, 0.10403557121753693, 0.04051864519715309, 0.08931092917919159,
      -0.11327993869781494, -0.04439677298069, -0.021771062165498734, 0.014158270321786404,
      0.08345892280340195, 0.10466629266738892, -0.07226347923278809, 0.14963999390602112,
      -0.10843296349048615, 0.05269623175263405, 0.09440173208713531, 0.015782346948981285,
      0.029959434643387794, 0.11402309685945511, -0.053989775478839874, -0.0602489598095417,
      0.012299732305109501, 0.08448047190904617, -0.10655096918344498, 0.13284964859485626,
      0.056829046458005905, -0.04899775609374046, 0.040083106607198715, 0.02712612971663475,
      0.15614499151706696, 0.032434213906526566, 0.0020592473447322845, -0.004683941137045622],
     [-0.07691038399934769, 0.05032505467534065, -0.024618975818157196, -0.024274379014968872,
      0.06063304468989372, 0.1839897334575653, -0.08442939817905426, 0.02932613529264927,
      -0.12252452969551086, -0.04554316774010658, 0.006856883876025677, -0.11057788878679276,
      0.06864386796951294, -0.023576298728585243, -0.06362342834472656, -0.11409324407577515,
      -0.07389799505472183, 0.04389430209994316, 0.11335497349500656, 0.08168963342905045,
      -0.027223529294133186, 0.013552764430642128, -0.041310884058475494, 0.006533205974847078,
      0.20118126273155212, 0.03127223625779152, 0.09248315542936325, 0.1738102287054062,
      0.041904378682374954, -0.0672912672162056, -0.12794016301631927, -0.06004289910197258,
      -0.08574400097131729, 0.030437814071774483, -0.11359260976314545, 0.1289062201976776,
      0.07045546919107437, 0.04320380836725235, -0.036729611456394196, 0.034192223101854324,
      0.042469531297683716, 0.03771862015128136, -0.0043580192141234875, -0.04523796588182449,
      0.061059921979904175, -0.0352872870862484, 0.016720058396458626, 0.11476215720176697,
      0.008331716060638428, -0.05351077765226364, -0.078314870595932, 0.13554039597511292,
      0.04858255758881569, 0.04206705465912819, -0.028045080602169037, 0.033278509974479675,
      0.09071483463048935, -0.11799195408821106, 0.0692233294248581, 0.09139417856931686,
      0.13802053034305573, -0.029167866334319115, 0.0313773974776268, -0.11286355555057526],
     [-0.05332297459244728, -0.011300943791866302, 0.04002593085169792, 0.004856523126363754,
      -0.024608870968222618, 0.17051267623901367, -0.008684699423611164, 0.10801779478788376,
      0.06397702544927597, -0.007595822215080261, -0.04009823873639107, 0.02264980599284172,
      -0.016500625759363174, -0.08445437997579575, -0.044398944824934006, -0.011190786957740784,
      -0.030001381412148476, -0.09867914766073227, -0.0019559015054255724, -0.018651099875569344,
      -0.047462910413742065, 0.0835307240486145, -0.08720937371253967, 0.12004735320806503,
      -0.03795424848794937, -0.12001475691795349, -0.0174713134765625, 0.08604361116886139,
      -0.052062418311834335, -0.017532506957650185, -0.015646232292056084, 0.04937538132071495,
      -0.02605537697672844, -0.09862639755010605, 0.08092056959867477, 0.036967795342206955,
      -0.02559489943087101, 0.07462774217128754, -0.007960754446685314, 0.011244913563132286,
      0.07290835678577423, 0.003958356101065874, -0.049617525190114975, -0.02929951436817646,
      0.029426373541355133, 0.04358552396297455, -0.032206546515226364, -0.052051834762096405,
      -0.07565037161111832, -0.10464480519294739, -0.08708105236291885, 0.05114372819662094,
      -0.11549185961484909, 0.017535775899887085, 0.04406319186091423, -0.06835062056779861,
      -0.056217703968286514, -0.019279325380921364, 0.008069944567978382, 0.09152482450008392,
      0.16144756972789764, -0.05036362633109093, -0.07658250629901886, 0.031075935810804367],
     [0.1535295695066452, -0.12027859687805176, -0.09716780483722687, 0.013927221298217773,
      -0.004617747850716114, -0.04708761349320412, 0.011051613837480545, -0.013543358072638512,
      -0.05569965019822121, 0.017514770850539207, 0.1014755591750145, -0.11232997477054596,
      -0.018744168803095818, -0.1037791296839714, 0.0291877631098032, 0.02935918979346752,
      -0.03898073360323906, 0.007133428007364273, 0.042145926505327225, 0.07423768937587738,
      0.06155335530638695, -0.11002326756715775, -0.007788041606545448, 0.06279022246599197,
      0.093744657933712, 0.09649336338043213, 0.05726153030991554, 0.02713109366595745,
      0.03574017807841301, -0.08897459506988525, 0.08505590260028839, -0.06796206533908844,
      0.007218503393232822, 0.1192706972360611, -0.10344485938549042, -0.06723631918430328,
      -0.01108913030475378, 0.043678998947143555, -0.09174058586359024, -0.11476871371269226,
      -0.0799587219953537, -0.05226639285683632, 0.0448637530207634, 0.02989811822772026,
      0.057564303278923035, -0.10891477763652802, 0.07971692830324173, -0.03915225714445114,
      -0.048232316970825195, 0.07993262261152267, -0.03848375380039215, 0.05701947957277298,
      0.060922108590602875, -0.010850176215171814, 0.021792050451040268, -0.10891791433095932,
      -0.08449464291334152, -0.049949292093515396, 0.12205294519662857, 0.1270742118358612,
      0.1383320689201355, -0.07779453694820404, -0.0356467179954052, 0.10883381217718124],
     [-0.10788637399673462, -0.05622217059135437, -0.11131726950407028, 0.0006942795589566231,
      0.12766782939434052, 0.11158301681280136, -0.09452398866415024, 0.01978994347155094,
      0.03040575049817562, 0.08337581902742386, -0.023019032552838326, -0.07014195621013641,
      0.013599817641079426, 0.03892553597688675, -0.0646698996424675, 0.05284957215189934,
      0.11291997879743576, 0.005521594081073999, 0.08322102576494217, -0.008669459261000156,
      -0.02772911824285984, -0.0036866755690425634, 0.0676097571849823, 0.08744021505117416,
      0.10644044727087021, -0.09340320527553558, 0.057192422449588776, 0.15336544811725616,
      0.05124932527542114, 0.02539259009063244, -0.08978475630283356, -0.06834671646356583,
      -0.062229253351688385, 0.07775087654590607, 0.07683441787958145, 0.06908582895994186,
      -0.043091513216495514, -0.07417523860931396, 0.07840445637702942, -0.08625922352075577,
      -0.008975498378276825, 0.00505302706733346, 0.1078561544418335, 0.13668736815452576,
      -0.0004474581219255924, 0.11566741019487381, 0.0992085263133049, -0.08605144917964935,
      0.07949016988277435, 0.005681876093149185, -0.08438466489315033, -0.07923578470945358,
      0.07740085572004318, 0.054633960127830505, 0.06859596818685532, 0.12394026666879654,
      -0.07632137835025787, -0.040022239089012146, 0.004709324799478054, 0.0515238381922245,
      0.05473441258072853, 0.06509402394294739, 0.03415607288479805, -0.07563535124063492],
     [0.03072432056069374, 0.05870456248521805, -0.08763914555311203, -0.030875390395522118,
      0.016100246459245682, -0.07433812320232391, -0.026090774685144424, 0.15748178958892822,
      -0.09185249358415604, 0.07001630961894989, 0.030771130695939064, -0.08270025998353958,
      -0.08406272530555725, 0.11725283414125443, 0.009899118915200233, 0.08686355501413345,
      -0.011053246445953846, -0.026363899931311607, -0.11686796694993973, -0.10257986932992935,
      0.025158178061246872, -0.09567178785800934, -0.06342855095863342, -0.12319439649581909,
      -0.059819869697093964, 0.11445587873458862, -0.044355589896440506, 0.0976727157831192,
      0.04141873121261597, -0.0017267899820581079, 0.02870037406682968, -0.07489011436700821,
      -0.08266448229551315, -0.01111220009624958, -0.12707117199897766, 0.03259448707103729,
      -0.004605707712471485, 0.0323878638446331, 0.0524485819041729, -0.06102687120437622,
      0.17623856663703918, -0.13969367742538452, 0.07580643892288208, 0.09302066266536713,
      0.07123702019453049, 0.07560010999441147, -0.08079060912132263, 0.10057757049798965,
      -0.0138338478282094, -0.056589022278785706, 0.0004836035950575024, -0.004485165234655142,
      0.05441999062895775, 0.18157139420509338, -0.027500228956341743, -0.030042506754398346,
      -0.025591537356376648, -0.12195764482021332, -0.16341274976730347, -0.05687293782830238,
      0.08658148348331451, -0.04949016124010086, 0.1023828387260437, -0.08358040452003479],
     [0.08318411558866501, 0.09329476952552795, -0.09553656727075577, 0.08503895252943039,
      -0.028700435534119606, -0.10648982226848602, 0.003391773672774434, 0.024343043565750122,
      0.015447145327925682, 0.02747132070362568, -0.06529144197702408, -0.01401241309940815,
      -0.07286354154348373, -0.06649086624383926, -0.03578663989901543, 0.042887575924396515,
      0.11292790621519089, -0.039035625755786896, -0.06977420300245285, 0.07255709916353226,
      0.09803568571805954, -0.11585325747728348, 0.03069734200835228, 0.04689909517765045,
      0.02359239012002945, 0.04222723841667175, 0.11035823076963425, 0.10842727869749069,
      0.00559209706261754, -0.12381981313228607, 0.11577816307544708, -0.11987128108739853,
      0.07178837060928345, 0.008823955431580544, -0.020220650359988213, 0.015541547909379005,
      0.11915908008813858, 0.057816699147224426, 0.022366294637322426, 0.01495775580406189,
      0.06465986371040344, 0.12175901234149933, -0.06567305326461792, 0.02333180233836174,
      0.07263261824846268, 0.018333226442337036, 0.07230553776025772, -0.03374108299612999,
      -0.1308450698852539, -0.08414193987846375, 0.07262911647558212, -0.031111033633351326,
      0.08797433227300644, 0.10572128742933273, 0.05380453169345856, 0.012416953220963478,
      -0.07785586267709732, -0.09646342694759369, -0.006900763139128685, 0.06434281170368195,
      0.00020667027274612337, -0.08528564870357513, 0.14385858178138733, -0.12048641592264175],
     [-0.007863853126764297, 0.044825442135334015, -0.05974844843149185, -0.06835444271564484,
      -0.01913289539515972, -0.10114569962024689, 0.011390618048608303, 0.021370720118284225,
      -0.07825244963169098, -0.01146838441491127, 0.04483571648597717, -0.019312364980578423,
      -0.10082372277975082, 0.0449286624789238, 0.056339554488658905, -0.08931649476289749,
      0.03135195001959801, -0.05131012946367264, -0.021514976397156715, -0.10265439003705978,
      -0.05325791612267494, -0.08510605990886688, 0.05321192741394043, -0.0006128853419795632,
      -0.08967864513397217, 0.11965934187173843, -0.11856263875961304, 0.07738227397203445,
      0.09301121532917023, -0.017389796674251556, 0.04740237817168236, -0.11836087703704834,
      -0.05180799961090088, 0.028241874650120735, 0.05195971950888634, 0.002781215589493513,
      0.09318289160728455, -0.07708561420440674, -0.02260383777320385, 0.08727061748504639,
      0.12796151638031006, -0.06872276216745377, -0.08762102574110031, 0.051332879811525345,
      -0.013528170064091682, -0.04381098970770836, 0.051786355674266815, 0.12340938299894333,
      0.004334825556725264, 0.05227527767419815, 0.046589598059654236, -0.09040216356515884,
      -0.0837869867682457, 0.05492366850376129, -0.05714048072695732, 0.12777462601661682,
      0.06976672261953354, -0.11104696989059448, 0.03218882158398628, 0.03297260403633118,
      0.04517284780740738, -0.1325376182794571, 0.06343760341405869, 0.004019512329250574]], dtype=np.float32).T + np.asarray([-0.10430194437503815, -0.028591927140951157, -0.1839289367198944, 0.017374180257320404,
     0.029266580939292908, 0.0023690774105489254, 0.007587087340652943, -0.0031839546281844378,
     -0.008515247143805027, -0.09048516303300858, -0.09202039986848831, -0.12478309869766235,
     -0.1979537308216095, -0.10201489180326462, -0.2062355875968933, -0.12394706904888153,
     0.06985616683959961, -0.06504425406455994, -0.11898782104253769, -0.008965125307440758,
     0.008368853479623795, -0.20017766952514648, -0.11793141067028046, 0.08660466969013214,
     -0.053434424102306366, -0.01487476285547018, 0.004336727783083916, -0.04204514995217323,
     -0.05126643180847168, -0.024568777531385422, -0.17691226303577423, -0.04367058724164963], dtype=np.float32)
    parent_h2 = parent_h2 / (1.0 + np.exp(-np.clip(parent_h2, -30.0, 30.0)))
    parent_raw = (parent_h2 @ np.asarray([[0.00227180658839643, -0.010408476926386356, -0.00211982405744493, -0.0014426751295104623,
      -0.0020759832113981247, -0.21084021031856537, -0.06374601274728775, -0.10145442187786102,
      -0.023607762530446053, 0.10233619809150696, -0.009175264276564121, -0.0011284560896456242,
      -0.12301982939243317, 0.05146866664290428, -0.07442333549261093, 0.0017402159282937646,
      0.0053735654801130295, 0.0038553024642169476, 0.09945625811815262, -0.00233592395670712,
      0.07331900298595428, -0.0009393331129103899, -0.1874334067106247, 0.05912269651889801,
      0.1240490972995758, 0.16727745532989502, 0.07359595596790314, 0.03104892186820507,
      0.09235896915197372, -0.10812197625637054, -0.05298618972301483, -0.10388805717229843]], dtype=np.float32).T + np.asarray([-0.06957858800888062], dtype=np.float32)).ravel()
    raw_factor = np.empty(len(data), dtype=np.float32)
    for positions in data.groupby("date", sort=False).indices.values():
        positions = np.asarray(positions, dtype=np.int64)
        y_local = trajectory_raw[positions].astype(np.float64)
        p_local = parent_raw[positions].astype(np.float64)
        design = np.column_stack([np.ones(len(positions)), p_local])
        gram = design.T @ design
        penalty = np.eye(2, dtype=np.float64) * 0.1
        penalty[0, 0] = 0.0
        beta = np.linalg.solve(gram + penalty, design.T @ y_local)
        residual_local = y_local - design @ beta
        residual_local = (residual_local - residual_local.mean()) / max(residual_local.std(ddof=0), 1e-5)
        parent_local = (p_local - p_local.mean()) / max(p_local.std(ddof=0), 1e-5)
        raw_factor[positions] = residual_local + 0.2 * parent_local
    data["factor"] = pd.Series(raw_factor, index=data.index, dtype="float64")
    data["factor"] = pd.to_numeric(data["factor"], errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    daily_median = data.groupby("date")["factor"].transform("median")
    data["factor"] = data["factor"].fillna(daily_median).fillna(0.0)
    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    data = data[(data["date"] >= start_ts) & (data["date"] <= end_ts)]
    return data[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


# AIStudio trial runner. Keep "quick" for a fast platform check; use "full" for 2019-2024.
AISTUDIO_RUN_MODE = "quick"
if AISTUDIO_RUN_MODE == "quick":
    aistudio_start_date = "2024-01-02 00:00:00"
    aistudio_end_date = "2024-06-30 23:59:59"
else:
    aistudio_start_date = "2019-01-02 00:00:00"
    aistudio_end_date = "2024-12-31 23:59:59"

try:
    from bigmodule import M
except ModuleNotFoundError:
    M = None

if M is not None:
    import dai
    from IPython.display import display

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }

    factor_data = main(datasources, aistudio_start_date, aistudio_end_date)
    if factor_data.empty:
        raise RuntimeError("Factor query returned no rows")

    print("AIStudio factor rows:", len(factor_data))
    print(factor_data.head())
    print("\nFactor value diagnostics:")
    display(
        factor_data.groupby("date")["factor"]
        .agg(["count", "mean", "std", "min", "max"])
        .describe()
    )

    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [aistudio_start_date, aistudio_end_date]},
    ).df()

    eval_namespace = getattr(M, "bigalpha_eval")
    eval_function = getattr(eval_namespace, "v4", None)
    if eval_function is None:
        eval_function = getattr(eval_namespace, "_latest")

    eval_attempts = [
        {
            "factor_data": factor_data,
            "factor_pool": factor_pool,
            "process_pools": False,
            "show": True,
        },
        {"factor_data": factor_data, "factor_pool": factor_pool, "show": True},
        {"factor_data": factor_data, "factor_pool": factor_pool},
        {"factor_data": factor_data, "show": True},
        {"factor_data": factor_data},
    ]
    official_result = None
    eval_errors = []
    for eval_kwargs in eval_attempts:
        try:
            official_result = eval_function(**eval_kwargs)
            print("bigalpha_eval parameters:", sorted(eval_kwargs))
            break
        except (TypeError, ValueError) as exc:
            eval_errors.append(f"{sorted(eval_kwargs)} -> {type(exc).__name__}: {exc}")
    if official_result is None:
        for eval_args in ((factor_data, factor_pool), (factor_data,)):
            try:
                official_result = eval_function(*eval_args)
                print("bigalpha_eval positional arguments:", len(eval_args))
                break
            except (TypeError, ValueError) as exc:
                eval_errors.append(f"{len(eval_args)} positional -> {type(exc).__name__}: {exc}")
    if official_result is None:
        raise RuntimeError("bigalpha_eval call failed:\n" + "\n".join(eval_errors))

    def _section(container, key):
        if container is None:
            return None
        if isinstance(container, dict):
            return container.get(key)
        if hasattr(container, key):
            return getattr(container, key)
        try:
            return container[key]
        except (KeyError, TypeError, IndexError):
            return None

    result_root = getattr(official_result, "_result", None)
    detail_root = getattr(official_result, "_detail", None)
    roots = [official_result, result_root, detail_root]

    factor_analyze = next(
        (value for value in (_section(root, "factor_analyze") for root in roots) if value is not None),
        None,
    )
    factor_regression = next(
        (value for value in (_section(root, "factor_regression") for root in roots) if value is not None),
        None,
    )
    per_factor_scores = _section(factor_regression, "per_factor_scores")
    weights_history = _section(factor_regression, "weights_history")
    direct_metric_names = (
        "ic_mean", "ic_ir", "sharpe_ratio", "stress_ic_ir",
        "model_score", "ModelScore", "score",
    )
    direct_metrics = {}
    for metric_name in direct_metric_names:
        metric_value = next(
            (value for value in (_section(root, metric_name) for root in roots) if value is not None),
            None,
        )
        if metric_value is not None:
            direct_metrics[metric_name] = metric_value

    print("\n=== Official single-factor metrics ===")
    print("Expected fields: ic_mean, ic_ir, sharpe_ratio, stress_ic_ir")
    if factor_analyze is not None:
        display(factor_analyze)
    elif direct_metrics:
        display(pd.DataFrame([direct_metrics]))
    else:
        display(official_result)

    print("\n=== Official regression / ModelScore metrics ===")
    display(per_factor_scores if per_factor_scores is not None else factor_regression)

    print("\n=== Rolling ElasticNet weight history ===")
    if hasattr(weights_history, "tail"):
        display(weights_history.tail(20))
    else:
        display(weights_history if weights_history is not None else "weights_history not exposed by this platform build")

    print("\n=== Raw official result object ===")
    official_result
